# har3d — when is calibration measurable? (revision pipeline, v7)

> **v7 — the IEEE Access revision (Access-2026-53441). You already ran v6, so `runs/final` exists on Drive.**
>
> **What to do:** Runtime → Change runtime type → **GPU, High-RAM** → **Runtime → Run all**.
> Everything already on Drive is skipped; only the new work runs:
>
> | New stage | What it does | Needs | Rough time |
> |---|---|---|---|
> | 5b | retrains the small-dataset joint-stream grid under the alternative preprocessings (keeps the logits) | GPU | ~45 min |
> | 8c | **confirmatory replication**: seeds 3 and 4 of ST-GCN, 2s-AGCN and CTR-GCN at 10,000 / 20,000 / 32,202 clips in one pinned environment, with matched-step and best-accuracy checkpoints; then seed 0 at 10,000 clips re-run in today's environment | GPU, High-RAM | **the long one, ~12–18 h**; resumable run by run |
> | 9d | dependence-aware inference: 8 estimators, coupled and subject-clustered nulls, subjects as units | CPU | ~15 min |
> | 9e | power and size of the null test (minimum detectable calibration error) | CPU | ~15 min |
> | 9f | methods with subjects as units, cross-fitted post-hoc calibration, per-class null | CPU | ~10 min |
> | 9g | preprocessing sensitivity, training-only checks of tree and joint order | CPU | ~5 min |
> | 9h | replication, matched training durations, environment re-runs, seed-level ladder analysis | CPU | ~10 min |
>
> **If Colab disconnects** (it will, during Stage 8c): reconnect to the **same GPU type**, then
> **Run all** again. Finished runs are skipped and training continues with the first missing run.
> Stage 8c refuses to continue on a different GPU type or library version, so that the replication
> stays in one environment (`ALLOW_ENV_MIX = True` in Stage 0 overrides this, and the mix is recorded).
>
> **When Stage 11 prints "EVERYTHING FOR THE REVISION IS READY":** download the folder
> `MyDrive/har3d/runs/final/results` from Drive and upload the zip to the chat.
>
> Optional: `RUN_REPL_EXTRA = True` in Stage 0 trains a third new seed (about +6–9 GPU hours). With six
> seeds in total the exact seed-level test can reach p = 0.03; with five its smallest possible value is 0.06.
>
> Nothing stored is retrained or overwritten: the new runs use new model tags (`*_rep`, `*_env`) and
> separate folders (`logits_prep/`, `logits_ckpt/`). Stages 9b, 9c and 10 reproduce the first-version
> tables and figures unchanged.

## Before you start (Drive layout)

```
MyDrive/har3d/
  MSRAction3DSkeletonReal3D.rar, MSRAction3DSkeleton(20joints).rar,
  MSRDailyActivity3D.zip, joints.zip, actionLabel.txt,
  nturgbd_skeletons_s001_to_s017.zip          <- raw inputs (keep)
  cache/            <- parsed skeletons      (KEEP: skips Stages 2-3b parsing)
  preprocessed/ntu/ <- NTU60_CS.npz + performer.txt (KEEP: skips Stage 7)
  runs/final/       <- EVERYTHING this notebook produces (logits, results)
```
Delete the old `logits/` and `results/` folders (after downloading them).
They were produced by the earlier protocol and are never read by this
notebook, which writes only to `runs/final/`.

## Running everything from scratch (not needed for the revision)

| Pass | Runtime | Cells | Rough time |
|---|---|---|---|
| 1 | GPU (T4 is enough) | 0 → 6b | ~4–8 h (720 small runs, resumable) |
| 2 | CPU High-RAM — *only if* `preprocessed/ntu/` is missing | 0, 7 | 1–2 h |
| 3 | GPU, High-RAM (L4 / A100) | 0, 8, 8b | the long one: roughly 2–4 h per backbone per seed for all 7 rungs on an L4, so ~20–35 h in total; spread over several sessions |
| 4 | any (CPU is fine) | 0, 9 → 11 | ~45–60 min |

In pass 3, ST-GCN runs first, then 2s-AGCN, CTR-GCN, the fixed-budget
ablation, and (8b) the frozen-graph ablation. If compute runs short, set
`LADDER_BACKBONES` / `RUN_FIXED_BUDGET_ABLATION` in Stage 0 to trim, and
Stages 9–10 will analyse whatever is present.

## What changed from the earlier notebook (all review points)

| Issue | Fix |
|---|---|
| Ladder used a fixed 6,000-step budget at every rung | early stopping on validation NLL per rung (best weights restored); the old budget kept as the `stgcn_fixed` ablation |
| "The learned graph causes it" was not tested | Stage 8b freezes 2s-AGCN's graph parameters after 1,000 steps (`agcn_frozen`) and compares paired against `agcn` |
| Only ST-GCN and 2s-AGCN | CTR-GCN-style channel-wise topology refinement added, same trunk |
| **Bug:** 2s-AGCN ladder silently used a random 10% validation slice in a fresh runtime | performer ids read from Drive; any failure raises; the protocol (validation subjects, pool hash) is recorded and enforced for every backbone |
| **Bug:** Stage 5 chose preprocessing by *test* accuracy | chosen by *validation* accuracy; test reported only |
| MSR epochs fixed | early stopping on validation NLL for every small-dataset cell |
| MC dropout untuned (p = 0.3) | inference rate chosen from {0.05, 0.1, 0.2, 0.3} on validation NLL |
| 3-member ensembles | 5 seeds → 5-member deep ensembles |
| No Brier decomposition / classwise ECE | Murphy decomposition (REL, RES, UNC) and classwise ECE for every cell and method |
| 567 vs 557 | explicit, documented quality filter; excluded ids in `data_filtering.json` |
| Fig 1 equal-width bins vs adaptive ECE | equal-mass (variable-width) reliability bins |
| Missing error bars | bootstrap CIs on Figs 3–4; ±1 s.d. over seeds on the ladder |
| RQ1/RQ3 tables mixed in LS/focal cells | restricted to ST-GCN cross-entropy |
| **v5:** method comparisons treated the 240 cells as independent, but the 20 cells of a split share its test clips | sign test and cluster-bootstrap intervals over the 12 test sets (Stage 9c) |
| **v5:** backbone and freezing differences were tested only by a bootstrap over test clips, which holds the trained models fixed | sign tests over seed–rung pairs (separately trained runs), one Holm family; run-to-run noise reported (Stage 9b) |
| **v5:** scaling-law interval from a bootstrap over seeds within fixed rungs | OLS t-interval on the rung means, lack of fit included |
| **v5:** environment.json was overwritten every session | environment_history.json keeps one record per software stack |
| **v5:** the manuscript's numbers were computed outside the notebook | Stage 10 runs make_numbers.py and make_figures.py → results/paper/ |
| **v5:** Fig. 3 showed only the floor of the uncalibrated model | the null floor after temperature scaling is computed (Stage 9b) and drawn |
| **v6:** calibration methods were compared by raw ECE, but every method moves the floor its ECE is measured against | each method's floor is computed on its own probabilities (Stage 9c); the methods are tested on the excess over it and on the Brier score and NLL, one Holm family per score; raw ECE is kept for reference |
| **v6:** fused models were judged by raw ECE only | floor, Brier and NLL of every fused model before and after its temperature (Stage 9) |
| **v6:** the 12 test sets were treated as independent | they share one or two subjects pairwise within a dataset: p-values are reported as approximate, with the agreement within each dataset |
| **v6:** run-to-run noise rested on one pair of identical runs | the across-seed s.d. at the large rungs and the standard error of a three-seed difference (Stage 9b) |
| **v7 (R2-1, R2-16, R1-5):** test sets of a dataset share subjects; cells of a split share clips | coupled null (one draw per clip for every model that predicts it), subject-clustered null, per-dataset results, subjects as units (Stage 9d) |
| **v7 (R2-5):** the null assumes independent labels | subject heterogeneity measured; size of the test under subject effects and a cluster-aware null (Stages 9d, 9e) |
| **v7 (R1-6, R2-6):** no power analysis | controlled miscalibration on the logit scale; minimum detectable calibration error per cell, for the pooled test, for the NTU test set and for synthetic (n, K, confidence) profiles (Stage 9e) |
| **v7 (R2-7):** one estimator, 15 bins | equal-mass ECE with 5–30 bins, equal-width ECE, debiased squared calibration error, binning-free KS error, each with its matched null (Stages 9d, 9h) |
| **v7 (R2-2, R2-3, R2-4):** nine seed–rung pairs are not independent; 10,000-clip threshold chosen post hoc; two CUDA builds | new seeds in one pinned environment with the hypothesis registered before training; seeds as units; same-seed environment re-runs (Stages 8c, 9h) |
| **v7 (R2-11):** early stopping may drive the backbone difference | test logits at matched training steps and at the best validation accuracy for every new run (Stages 8c, 9h) |
| **v7 (R2-12):** post-hoc methods fitted on 1–2 validation subjects | cross-fitted over test subjects (4–6 subjects) and fitted on the test set as an upper bound (Stage 9f) |
| **v7 (R2-14):** preprocessing sensitivity; train-only derivation | Stage 5b logits tested against the null (Stage 9g); tree and joint order re-checked on training subjects alone (Stage 9g) |
| **v7 (R2-15):** per-class ECE without counts or uncertainty | class counts, distinct clips and a clip-coupled matched null per class (Stage 9f) |

## Stage 0 — setup and configuration
Edit the CONFIG block only. Re-run after every runtime restart.

In [ ]:
# =====================================================================
# STAGE 0 -- setup, configuration, Drive mount, sentinels, cache restore
# =====================================================================
# RUN THIS FIRST AFTER EVERY RUNTIME RESTART. /content is wiped on restart;
# the sentinels, parsed caches and results live on Drive.
#
# Everything this run produces goes to MyDrive/har3d/runs/<RUN_TAG>/, so it can
# never mix with logits or sentinels from an earlier protocol. The parsed
# skeleton caches (MyDrive/har3d/cache) and the NTU tensors
# (MyDrive/har3d/preprocessed) are protocol-independent and are reused if
# present -- keep those folders to save 1-2 h of preprocessing.

# ============================== CONFIG ===============================
DRIVE_ROOT      = '/content/drive/MyDrive/har3d'
RUN_TAG         = 'final'          # all outputs -> DRIVE/runs/<RUN_TAG>/
SEED_EVERYTHING = 0                # global determinism seed (Stage 0a)

# ---- small datasets (MSR-Action3D, MSR-DailyActivity3D, UTKinect) ----
SEEDS           = [0, 1, 2, 3, 4]  # 5 seeds -> 5-member deep ensembles
ABLATION_SEEDS  = [0, 1, 2]        # Stage 5 preprocessing ablation
STREAMS         = ['joint', 'bone', 'joint_motion', 'bone_motion']
T_FRAMES        = 64
MAX_EPOCHS_MSR  = 120              # early stopping on validation NLL
PATIENCE_MSR    = 15               # epochs without improvement before stop
MIN_EPOCHS_MSR  = 10
MC_PASSES       = 20
MC_P_GRID       = (0.05, 0.1, 0.2, 0.3)   # MC-dropout rate tuned on val NLL
MAX_ZERO_FRAC   = 0.5              # drop sequences >50% all-zero joint-frames
MIN_FRAMES      = 5                # v2: was 10, which dropped 21 genuine short UTKinect clips

# ---- NTU-60 training-set ladder ----
LADDER_N          = [500, 1000, 2000, 5000, 10000, 20000, 40091]   # last rung = whole pool (32,202 clips once the validation performers are removed)
LADDER_SEEDS      = [0, 1, 2]      # >=3 required for error bars
LADDER_BACKBONES  = ['stgcn', 'agcn', 'ctrgcn']
LADDER_STREAM     = 'joint'
NTU_VAL_SUBJECTS  = 4              # subject-disjoint val, from X-Sub TRAIN
LADDER_MAX_STEPS  = 40000          # early-stopping ceiling (v2: was 15000, which capped 13 runs)
LADDER_EVAL_EVERY = 250            # validate every N steps
LADDER_PATIENCE   = 8              # evaluations without improvement
LADDER_MIN_STEPS  = 500

# ---- ablations answering the review ----
RUN_FIXED_BUDGET_ABLATION = True   # ST-GCN with the old fixed 6,000 steps
FIXED_BUDGET_STEPS        = 6000
RUN_FREEZE_ABLATION       = True   # 2s-AGCN with its graph frozen mid-training
FREEZE_AFTER_STEPS        = 1000
FREEZE_RUNGS              = [2000, 5000, 10000, 20000, 40091]

# ---- revision (v7): what the reviewers asked for ----
# Confirmatory replication of the backbone comparison at the large rungs: new
# seeds, one software environment, hypothesis fixed before training (Stage 8c).
REPL_SEEDS          = [3, 4]       # new seeds, trained first
REPL_EXTRA_SEEDS    = [5]          # a third new seed, trained last ...
RUN_REPL_EXTRA      = False        # ... only if True (about +50% GPU time; six seeds in total
                                   #     make the exact seed-level test able to reach p = 0.03)
REPL_RUNGS          = [10000, 20000, 40091]
REPL_SNAPSHOT_STEPS = [2000, 4000, 6000, 8000, 10000, 12000]   # matched training durations
RUN_ENV_RERUNS      = True         # re-train seed 0 at 10,000 clips in today's environment
ENV_RERUN_SEED, ENV_RERUN_RUNG = 0, 10000
ALLOW_ENV_MIX       = False        # True: continue the replication in a different software stack / GPU
ALLOW_CPU_TRAINING  = False        # True: let the GPU stages run on a CPU runtime (very slow)
RUN_PREP_SENSITIVITY = True        # Stage 5b: retrain under the alternative preprocessings
RERUN_OLD_STATS     = False        # True: recompute Stages 9b and 9c even though their tables exist
RERUN_REVIEW_STATS  = False        # True: recompute Stages 9d-9g even though they are marked done
REQUIRE_EXISTING_RUN = True        # v7 CONTINUES the finished run on Drive. If that run cannot be
                                   # seen, stop -- never start an empty run (and days of retraining)
                                   # by accident. False only to run the whole pipeline from scratch.
# =====================================================================

import importlib, json, os, shutil, sys, time
from pathlib import Path

import numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('not on Colab, or Drive already mounted')

DRIVE   = Path(DRIVE_ROOT)
if not DRIVE.exists():
    raise RuntimeError(f'{DRIVE} not reachable -- check the mount and the folder name.')
CACHE   = Path('/content/har3d_cache')
WORK    = Path('/content/har3d_work')
PREP    = DRIVE / 'preprocessed'


def _is_finished_run(run):
    """The v6 run: its stage markers, metrics and a full logit store."""
    res = run / 'results'
    return all((res / f).exists() for f in ('.done_06_msr_logits', '.done_08_ladder',
                                            'metrics.json', 'prep_config.json'))


def _find_run(wait_s=240):
    """Locate DRIVE/runs/<RUN_TAG> WITHOUT creating anything.
    Right after mounting, the Drive file system can answer 'not found' for a
    folder it has not listed yet; creating the folder at that moment makes a
    second, empty 'runs' next to the real one. So: list the folders (which
    loads their entries), look, and wait and look again before giving up. If
    Drive holds more than one folder called 'runs' (shown as 'runs', 'runs (1)',
    ...), the one that contains the finished run is used."""
    t0, seen = time.time(), []
    while True:
        try:
            seen = sorted(os.listdir(DRIVE))
        except OSError as e:
            seen = [f'<listing failed: {e}>']
        cands = [DRIVE / n for n in seen if n == 'runs' or n.startswith('runs (')]
        good = []
        for c in cands:
            try:
                os.listdir(c)
                if (c / RUN_TAG).is_dir():
                    os.listdir(c / RUN_TAG)
                    if (c / RUN_TAG / 'results').is_dir():
                        os.listdir(c / RUN_TAG / 'results')
                if _is_finished_run(c / RUN_TAG):
                    good.append(c / RUN_TAG)
            except OSError:
                pass
        if len(good) == 1:
            if len(cands) > 1:
                print(f'  !! Drive has {len(cands)} folders named runs: {[c.name for c in cands]}. Using '
                      f'{good[0]} (the one with the finished run). Delete the other, empty one in Drive.')
            return good[0]
        if len(good) > 1:
            raise RuntimeError(f'more than one finished run found: {good}. Keep one "runs" folder in '
                               f'{DRIVE} and re-run this cell.')
        if time.time() - t0 > wait_s:
            raise RuntimeError(
                f'The finished run was NOT found under {DRIVE} (looked for runs/{RUN_TAG}/results with '
                f'its stage markers for {wait_s} s). Nothing was created and nothing was changed.\n'
                f'  Folders visible in {DRIVE}: {seen}\n'
                '  Check: (1) Colab is mounted with the Google account that owns the har3d folder; '
                '(2) DRIVE_ROOT points to that folder; (3) in Drive, har3d contains ONE folder "runs" '
                'holding final/logits and final/results -- delete an empty duplicate "runs" if there is '
                'one; (4) then Runtime > Disconnect and delete runtime, and run this cell again.')
        print(f'  waiting for Drive to show runs/{RUN_TAG} ... ({time.time() - t0:.0f} s)')
        time.sleep(15)


if REQUIRE_EXISTING_RUN:
    RUN = _find_run()
else:
    RUN = DRIVE / 'runs' / RUN_TAG
LOGITS  = RUN / 'logits'
RESULTS = RUN / 'results'
if REQUIRE_EXISTING_RUN:
    n_logit_files = sum(1 for e in os.scandir(LOGITS) if e.name.endswith('.npz'))
    if n_logit_files < 1500:
        raise RuntimeError(f'{LOGITS} holds only {n_logit_files} logit files; the finished run has more '
                           'than 2,000. Drive has not synchronised this folder yet, or this is not the '
                           'folder of the finished run. Nothing was changed. Wait a minute and re-run '
                           'this cell.')
    print(f'  finished run found: {RUN}  ({n_logit_files} logit files)')
    for p in (CACHE, WORK):
        p.mkdir(parents=True, exist_ok=True)
    for p in (DRIVE / 'cache', PREP):
        if not p.is_dir():
            raise RuntimeError(f'{p} is not visible on Drive -- wait a minute and re-run this cell.')
else:
    for p in (CACHE, WORK, LOGITS, RESULTS, DRIVE / 'cache', PREP):
        p.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, '/content')

restored = 0
for f in (DRIVE / 'cache').glob('*.npz'):
    if not (CACHE / f.name).exists():
        shutil.copy2(f, CACHE / f.name)
        restored += 1
print(f'run directory: {RUN}')
print(f'restored {restored} parsed caches from Drive:',
      sorted(p.name for p in CACHE.glob('*.npz')) or 'none yet')


def stage_done(tag):
    return (RESULTS / f'.done_{tag}').exists()


def mark_done(tag, result=None):
    """Refuse to mark a stage complete if any sub-stage reported an error."""
    if isinstance(result, dict):
        bad = [k for k, v in result.items() if isinstance(v, dict) and 'error' in v]
        if 'error' in result:
            bad.append(tag)
        if bad:
            print(f'  [stage {tag} NOT marked complete -- failed: {sorted(set(bad))}]')
            return False
    (RESULTS / f'.done_{tag}').write_text(time.ctime())
    print(f'  [stage {tag} complete]')
    return True


def reset(*tags):
    """Force stages to re-run, e.g. reset('06_msr_logits')."""
    for t in tags:
        p = RESULTS / f'.done_{t}'
        if p.exists():
            p.unlink()
            print('cleared', p.name)


print('stages complete:', sorted(p.name[6:] for p in RESULTS.glob('.done_*')) or 'none')

# ------------------------------ preflight ------------------------------
print('\ninputs in', DRIVE)
NEEDED = {
    'MSRAction3DSkeletonReal3D.rar': 'Action3D metric skeletons',
    'MSRAction3DSkeleton(20joints).rar': 'Action3D screen + confidence',
    'MSRDailyActivity3D.zip': 'DailyActivity3D',
    'joints.zip': 'UTKinect part (c)',
    'actionLabel.txt': 'UTKinect part (d)',
    'nturgbd_skeletons_s001_to_s017.zip': 'NTU-60 raw skeletons',
}
for name, why in NEEDED.items():
    hit = DRIVE / name
    print(f'  {name:<38} {"%6.2f GB" % (hit.stat().st_size/1e9) if hit.exists() else "  MISSING"}   {why}')
for name in ('msr_action3d.npz', 'msr_daily.npz', 'utkinect.npz'):
    print(f'  cache/{name:<32} {"present (ingest will be skipped)" if (CACHE/name).exists() else "absent (will be built)"}')
ntu_npz = PREP / 'ntu' / 'NTU60_CS.npz'
print(f'  preprocessed/ntu/NTU60_CS.npz        {"present (Stage 7 will be skipped)" if ntu_npz.exists() else "absent (Stage 7 builds it)"}')

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'CPU runtime'
!free -g | awk 'NR==2{print "system RAM:", $2, "GB"}'
!df -h /content | awk 'NR==2{print "disk free:", $4}'

### Stage 0a — determinism

In [ ]:
# =====================================================================
# STAGE 0a -- global determinism
# =====================================================================
# Seeds every RNG the pipeline can touch. Per-stage code also re-seeds locally
# (training, subsampling, null draws) so results do not depend on cell order.
import os, random
import numpy as _np

os.environ['PYTHONHASHSEED'] = str(SEED_EVERYTHING)
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
random.seed(SEED_EVERYTHING)
_np.random.seed(SEED_EVERYTHING)
try:
    import torch
    torch.manual_seed(SEED_EVERYTHING)
    torch.cuda.manual_seed_all(SEED_EVERYTHING)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print('seeded python, numpy, torch; cuDNN deterministic; seed', SEED_EVERYTHING)
except Exception as _e:
    print('seeded python, numpy (torch unavailable:', _e, ')')

## Modules
Each cell writes one Python module to `/content`. Run all of them after every runtime restart (seconds).

In [ ]:
%%writefile /content/har3d_io.py
"""
har3d_io.py -- archive extraction and tolerant skeleton-text parsing.

Shared by every ingest stage. Two things live here because both MSR datasets
and NTU tripped over them:

ARCHIVES. MSRDailyActivity3D.zip is a zip OF ZIPS -- 9 stored members, the 8
packs plus sample code -- and the packs themselves use Deflate64 (method 9),
which Python's zipfile cannot read and which Info-ZIP `unzip` 6.00 also cannot
read. 7-Zip can. So extraction tries zipfile, then 7z, then unzip, then unar,
installing each on demand, and nested archives are expanded WHEREVER they land
rather than only where the caller expected them.

TEXT. Skeleton files interleave per-frame marker lines with fixed-width joint
rows, so np.loadtxt raises on inconsistent columns. read_rows() classifies each
line by token count instead and keeps only the modal width.
"""
from __future__ import annotations

import shutil
import subprocess
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np

METHOD_NAMES = {0: "stored", 1: "shrunk", 6: "imploded", 8: "deflate",
                9: "DEFLATE64", 12: "bzip2", 14: "lzma", 98: "ppmd"}
ZIPFILE_SUPPORTS = {0, 8, 12, 14}
ARCHIVE_EXT = {".zip", ".rar", ".7z"}
MEDIA_EXT = {".avi", ".mp4", ".mov", ".bin", ".jpg", ".jpeg", ".png",
             ".mat", ".depth", ".wmv"}


def log(m=""):
    print(m, flush=True)


def inspect_methods(archive: Path) -> dict:
    """Compression histogram, and whether zipfile alone can read it."""
    if Path(archive).suffix.lower() != ".zip":
        return {"counts": {}, "names": {"(not a zip)": 1}, "zipfile_can_read": False}
    try:
        with zipfile.ZipFile(archive) as zf:
            counts = Counter(i.compress_type for i in zf.infolist())
    except zipfile.BadZipFile:
        return {"counts": {}, "names": {"(unreadable)": 1}, "zipfile_can_read": False}
    return {"counts": dict(counts),
            "names": {METHOD_NAMES.get(k, f"method {k}"): v for k, v in counts.items()},
            "zipfile_can_read": set(counts) <= ZIPFILE_SUPPORTS}


def ensure_tool(name: str, apt_pkg: str) -> bool:
    if shutil.which(name):
        return True
    subprocess.run(f"apt-get -qq update && apt-get -qq install -y {apt_pkg}",
                   shell=True, check=False,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return shutil.which(name) is not None


def extract_any(archive: Path, dest: Path, quiet: bool = False) -> str:
    """
    Extract into `dest` with whichever tool can read it. Returns the tool used;
    raises listing every attempt. 7z leads the fallbacks because it is the only
    one of them that reads Deflate64 and RAR.
    """
    archive, dest = Path(archive), Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    errs = []
    info = inspect_methods(archive)
    if info["zipfile_can_read"]:
        try:
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(dest)
            return "zipfile"
        except (NotImplementedError, RuntimeError, zipfile.BadZipFile) as e:
            errs.append(f"zipfile: {e}")
    else:
        errs.append(f"zipfile: cannot read {info['names']}")
        if not quiet:
            log(f"    {archive.name}: {info['names']} -- using a system tool")
    for name, cmd, pkg in (
            ("7z", ["7z", "x", "-y", f"-o{dest}", str(archive)], "p7zip-full"),
            ("unzip", ["unzip", "-q", "-o", str(archive), "-d", str(dest)], "unzip"),
            ("unar", ["unar", "-quiet", "-force-overwrite", "-o", str(dest),
                      str(archive)], "unar")):
        if not ensure_tool(cmd[0], pkg):
            errs.append(f"{name}: unavailable")
            continue
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            return name
        errs.append(f"{name}: rc={r.returncode} {(r.stderr or r.stdout).strip()[:160]}")
    raise RuntimeError(f"could not extract {archive.name}\n  " + "\n  ".join(errs))


def expand_archives(root: Path, skip=("samplecode",), max_depth=4) -> int:
    """Recursively expand nested archives under `root`, deleting each after."""
    root = Path(root)
    total = 0
    for _ in range(max_depth):
        found = [p for p in root.rglob("*")
                 if p.is_file() and p.suffix.lower() in ARCHIVE_EXT
                 and not any(s in p.name.lower() for s in skip)]
        if not found:
            break
        for p in found:
            try:
                tool = extract_any(p, p.parent, quiet=True)
                p.unlink()
                total += 1
                log(f"    expanded {p.name} with {tool}")
            except (zipfile.BadZipFile, RuntimeError) as e:
                log(f"    !! leaving in place: {p.name}\n       {e}")
    return total


def reclaim_media(root: Path) -> int:
    """Delete RGB/depth so the bulk never has to live anywhere permanent."""
    freed = 0
    for f in Path(root).rglob("*"):
        if f.is_file() and f.suffix.lower() in MEDIA_EXT:
            freed += f.stat().st_size
            f.unlink()
    return freed


def read_rows(path: Path, widths=(3, 4)) -> tuple:
    """
    Numeric rows of the modal token width, marker/header lines discarded.
    Returns (rows, width). Empty array rather than an exception, so one bad file
    cannot take down an ingest of several hundred.
    """
    buckets = {w: [] for w in widths}
    with open(path, "r", errors="ignore") as fh:
        for line in fh:
            parts = line.split()
            if len(parts) not in buckets:
                continue
            try:
                buckets[len(parts)].append([float(v) for v in parts])
            except ValueError:
                continue
    w = max(buckets, key=lambda k: len(buckets[k]))
    if not buckets[w]:
        return np.zeros((0, max(widths))), 0
    return np.array(buckets[w], np.float64), w


def find_skeleton_files(root: Path, needle="skeleton") -> list:
    """Case-insensitive, layout-agnostic."""
    return sorted(p for p in Path(root).rglob("*")
                  if p.is_file() and p.suffix.lower() == ".txt"
                  and needle in p.name.lower())


In [ ]:
%%writefile /content/har3d_v2.py
"""
har3d_v2.py -- corrected topology, preprocessing, splits and calibration
helpers for the skeleton calibration benchmark.

Three defects found by MEASURING the cached skeletons rather than reading code:

(1) The legacy parents list is not the MSR tree. 11 of its 19 edges are not
    bones -- the worst is 1.587 m against a 0.275 m upper arm. That list
    defined BOTH the ST-GCN adjacency and bone_pairs. (It is in fact the
    correct Kinect SDK kinematic tree paired with MSR's joint ordering, which
    is why it is internally consistent enough to survive a casual read.)
(2) normalise(root=0) centred on joint 0, which the frame geometry identifies
    as the LEFT SHOULDER. The hip centre is joint 6.
(3) fixed_split() carved validation out of the training subjects by index
    slice, so train and val shared subjects.

The corrected tree was derived from the data: for every joint pair, the
coefficient of variation of the inter-joint distance across all clean frames
was computed. Real bones are rigid, so they have low cv.

verify_topology() re-derives these numbers at run time and raises if any edge
is implausible. Everything here is numpy.
"""
from __future__ import annotations

import numpy as np

# =====================================================================
# 1. Topology
# =====================================================================
JOINT_NAMES = [
    "L_shoulder", "R_shoulder", "shoulder_center", "spine",
    "L_hip", "R_hip", "hip_center", "L_elbow",
    "R_elbow", "L_wrist", "R_wrist", "L_hand",
    "R_hand", "L_knee", "R_knee", "L_ankle",
    "R_ankle", "L_foot", "R_foot", "head",
]

# parent index per joint, -1 = root. Root is hip_center (index 6).
MSR_PARENTS = [2, 2, 3, 6, 6, 6, -1, 0, 1, 7, 8, 9, 10, 4, 5, 13, 14, 15, 16, 2]
ROOT = MSR_PARENTS.index(-1)
SPINE_TOP = JOINT_NAMES.index("shoulder_center")
HIP_L = JOINT_NAMES.index("L_hip")
HIP_R = JOINT_NAMES.index("R_hip")

# The legacy list, kept only so the ablation can reproduce the old numbers.
LEGACY_PARENTS = [-1, 0, 1, 2, 2, 4, 5, 6, 2, 8, 9, 10, 0, 12, 13, 14, 0, 16, 17, 18]

MAX_BONE_M = 0.60      # longest plausible human bone (thigh) with margin
MAX_BONE_CV = 0.20     # rigid connections vary little relative to their length
MAX_BONE_SD_M = 0.045  # ...or little in absolute terms, the fairer test

MAX_BODY_SPAN_M = 3.5
MAX_MEDIAN_PAIR_M = 1.5
MIN_MEDIAN_PAIR_M = 0.05

LR_PAIRS = [(0, 1), (4, 5), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16), (17, 18)]


def bone_pairs(parents=MSR_PARENTS):
    """(child, parent) for every joint. Root maps to itself, a zero bone."""
    return [(j, p if p >= 0 else j) for j, p in enumerate(parents)]


def rigid_spanning_tree(xyz, max_cv=MAX_BONE_CV, max_sd=MAX_BONE_SD_M,
                        allow_fallback=False):
    """
    Shortest rigid connections spanning every joint, as an undirected edge list.
    Judges a candidate parse WITHOUT knowing the joint order.
    """
    F = np.asarray(xyz, np.float64)
    F = F[~(np.abs(F) < 1e-9).all(-1).any(-1)]
    if len(F) < 20:
        return []
    D = np.linalg.norm(F[:, :, None, :] - F[:, None, :, :], axis=-1)
    mu, sd = D.mean(0), D.std(0)
    n = len(mu)
    cand = sorted((mu[i, j], i, j) for i in range(n) for j in range(i + 1, n)
                  if sd[i, j] / max(mu[i, j], 1e-9) <= max_cv or sd[i, j] <= max_sd)
    par = list(range(n))

    def find(a):
        while par[a] != a:
            par[a] = par[par[a]]
            a = par[a]
        return a

    def build(edges):
        nonlocal par
        par = list(range(n))
        out = []
        for w, i, j in edges:
            a, b = find(i), find(j)
            if a != b:
                par[a] = b
                out.append((w, i, j))
        return out

    tree = build(cand)
    if len(tree) < n - 1 and allow_fallback:
        # One joint can be so jittery that no edge touching it passes the
        # rigidity filter. A noisy joint must not masquerade as a wrong tree.
        tree = build(sorted((mu[i, j], i, j)
                            for i in range(n) for j in range(i + 1, n)))
    return tree


def tree_optimality(xyz, parents=MSR_PARENTS, max_cv=MAX_BONE_CV,
                    max_sd=MAX_BONE_SD_M):
    """
    Is the proposed tree the best available explanation of this skeleton?

    A noisier RECORDING inflates every edge's sd together and leaves the tree
    near-optimal; a WRONG tree costs far more than the optimum no matter how
    clean the sensor is. `ratio` is the statistic to trust; `shared` is kept
    only as a catastrophe check.
    """
    tree = rigid_spanning_tree(xyz, max_cv=max_cv, max_sd=max_sd,
                               allow_fallback=True)
    n_needed = np.asarray(xyz).shape[1] - 1
    if len(tree) < n_needed:
        return {"ratio": float("inf"), "shared": 0, "n_edges": n_needed,
                "mst_len_m": float("nan"), "proposed_len_m": float("nan")}
    F = np.asarray(xyz, np.float64)
    F = F[~(np.abs(F) < 1e-9).all(-1).any(-1)]
    mu = np.linalg.norm(F[:, :, None, :] - F[:, None, :, :], axis=-1).mean(0)
    prop = [(j, p) for j, p in enumerate(parents) if p >= 0]
    prop_len = float(sum(mu[j, p] for j, p in prop))
    mst_len = float(sum(w for w, _, _ in tree))
    shared = len({frozenset(e) for e in prop} & {frozenset((i, j)) for _, i, j in tree})
    return {"ratio": prop_len / max(mst_len, 1e-9), "shared": shared,
            "n_edges": len(prop), "mst_len_m": mst_len, "proposed_len_m": prop_len}


def verify_topology(sequences, parents=MSR_PARENTS, names=JOINT_NAMES,
                    max_len=MAX_BONE_M, max_cv=MAX_BONE_CV, raise_on_fail=True,
                    min_frames=100, max_sd=MAX_BONE_SD_M,
                    max_ratio=1.10, min_shared=8):
    """
    Re-derive bone statistics from real data and check the tree is anatomical.
    Frames containing an all-zero joint are dropped -- a dropped joint sits at
    the origin and would fabricate a metre-long bone.

    Raises AssertionError on failure. A hard gate, not a warning.
    """
    F = np.concatenate([np.asarray(s, np.float64) for s in sequences], 0)
    if F.ndim != 3:
        raise ValueError(f"expected (T, J, 3) sequences, got trailing shape {F.shape}")
    keep = ~(np.abs(F) < 1e-9).all(-1).any(-1)
    if keep.sum() < min_frames:
        raise ValueError(f"only {int(keep.sum())} frames with all joints present; "
                         f"need {min_frames} for a stable estimate")
    F = F[keep]
    report, bad = [], []
    for j, p in enumerate(parents):
        if p < 0:
            continue
        d = np.linalg.norm(F[:, j] - F[:, p], axis=-1)
        m, s = float(d.mean()), float(d.std())
        cv = s / max(m, 1e-9)
        ok = (m <= max_len) and (cv <= max_cv or s <= max_sd)
        report.append({"child": j, "parent": p, "name": names[j],
                       "parent_name": names[p], "len_m": m, "cv": cv,
                       "sd_m": s, "ok": ok})
        if not ok:
            bad.append(f"  j{j:02d} {names[j]} -> j{p:02d} {names[p]}: "
                       f"{m:.3f} m, cv {cv:.3f}, sd {s*1000:.0f} mm")

    print(f"  topology check: {len(report) - len(bad)}/{len(report)} edges plausible "
          f"({int(keep.sum()):,} clean frames, {100 * keep.mean():.1f}% of input)")

    if bad:
        opt = tree_optimality(F, parents, max_cv=max_cv, max_sd=max_sd)
        too_long = [r for r in report if r["len_m"] > max_len]
        structural = (not (1.0 / max_ratio <= opt["ratio"] <= max_ratio)
                      or opt["shared"] < min_shared or bool(too_long))
        head = (f"  tree optimality: {opt['ratio']:.3f}x the data-derived "
                f"shortest rigid tree, sharing {opt['shared']}/{opt['n_edges']} "
                f"edges with it")
        if structural:
            msg = ("topology check FAILED -- the tree is not the best "
                   "explanation of this geometry:\n" + "\n".join(bad)
                   + f"\n{head}"
                   + f"\n  (limits: length <= {max_len} m; cv <= {max_cv} or "
                     f"sd <= {max_sd*1000:.0f} mm; ratio <= {max_ratio}; "
                     f"shared >= {min_shared})")
            if raise_on_fail:
                raise AssertionError(msg)
            print("  !! " + msg)
        else:
            print(head + " -- near-optimal, so the tree is right and this "
                         "recording is simply noisier.")
            print("  edges that are anatomical but jittery here:")
            for b in bad:
                print("   " + b.strip())
            print("  Reported, not fatal. Note the affected joints in the paper: "
                  "a noisy joint is a per-class calibration confound, not a "
                  "preprocessing error.")
    return report


def adjacency(parents=MSR_PARENTS, strategy="spatial"):
    """
    ST-GCN spatial partitioning: (3, J, J) = self / centripetal / centrifugal,
    each row-normalised.
    """
    J = len(parents)
    A = np.zeros((J, J), np.float32)
    for j, p in enumerate(parents):
        if p >= 0:
            A[j, p] = A[p, j] = 1.0
    root = list(parents).index(-1)
    hop = np.full(J, np.inf)
    hop[root] = 0.0
    frontier = [root]
    while frontier:
        nxt = []
        for j in frontier:
            for n in np.where(A[j] > 0)[0]:
                if not np.isfinite(hop[n]):
                    hop[n] = hop[j] + 1
                    nxt.append(int(n))
        frontier = nxt
    if not np.isfinite(hop).all():
        raise ValueError(f"parents do not form a connected tree; unreachable: "
                         f"{np.where(~np.isfinite(hop))[0].tolist()}")
    parts = np.zeros((3, J, J), np.float32)
    parts[0] = np.eye(J, dtype=np.float32)
    for i in range(J):
        for j in range(J):
            if A[i, j] > 0:
                parts[1 if hop[j] < hop[i] else 2, i, j] = 1.0
    for k in range(3):
        d = parts[k].sum(1, keepdims=True)
        d[d == 0] = 1.0
        parts[k] = parts[k] / d
    return parts


# =====================================================================
# 2. Preprocessing
# =====================================================================
def impute_missing(seq, max_gap=None):
    """
    Kinect drops a joint by writing (0, 0, 0). Linearly interpolate each joint
    over time; hold the nearest value at the ends. Sequences where a joint is
    missing in EVERY frame keep zeros and are reported by the caller.
    """
    x = np.array(seq, np.float32, copy=True)
    T, J, _ = x.shape
    miss = (np.abs(x) < 1e-9).all(-1)
    t = np.arange(T, dtype=np.float64)
    for j in range(J):
        m = miss[:, j]
        if not m.any() or m.all():
            continue
        good = ~m
        for c in range(3):
            x[m, j, c] = np.interp(t[m], t[good], x[good, j, c])
    return x, miss


def _rotation_to_canonical(frame, spine_top=SPINE_TOP, root=ROOT,
                           hip_l=HIP_L, hip_r=HIP_R):
    """
    Orthonormal basis from one frame: hip->shoulder_center becomes +y,
    L_hip->R_hip becomes +x, z completes a right-handed frame.
    """
    up = frame[spine_top] - frame[root]
    right = frame[hip_r] - frame[hip_l]
    nu = np.linalg.norm(up)
    if nu < 1e-6:
        return None
    up = up / nu
    right = right - np.dot(right, up) * up
    nr = np.linalg.norm(right)
    if nr < 1e-6:
        return None
    right = right / nr
    fwd = np.cross(right, up)
    return np.stack([right, up, fwd], 0)


def normalise_skeleton(seq, centre="perframe", align=True, scale="bone",
                       parents=MSR_PARENTS, root=ROOT):
    """
    Canonicalise one sequence BEFORE any stream is derived.

    centre : 'perframe' removes global translation entirely;
             'frame0' keeps the body's trajectory through the scene as signal;
             'none' leaves absolute coordinates.
    align  : rotate by the frame-0 body basis. Per SEQUENCE, so rotation
             dynamics inside the clip survive.
    scale  : 'bone' divides by the sequence-median bone length.

    Ordering is centre, then align, then scale. Rotating before centring would
    rotate about the camera origin instead of the body.
    """
    x = np.asarray(seq, np.float32)
    if x.ndim != 3 or x.shape[-1] != 3:
        raise ValueError(f"expected (T, J, 3), got {x.shape}")

    if centre == "perframe":
        x = x - x[:, root:root + 1, :]
    elif centre == "frame0":
        x = x - x[0:1, root:root + 1, :]
    elif centre != "none":
        raise ValueError(f"unknown centre={centre!r}")

    if align:
        R = _rotation_to_canonical(x[0])
        if R is not None:
            x = x @ R.T.astype(np.float32)

    if scale == "bone":
        pair = [(j, p) for j, p in enumerate(parents) if p >= 0]
        lens = np.stack([np.linalg.norm(x[:, j] - x[:, p], axis=-1) for j, p in pair], 1)
        s = float(np.median(lens))
        if s > 1e-6:
            x = x / s
    elif scale != "none":
        raise ValueError(f"unknown scale={scale!r}")
    return x.astype(np.float32)


def make_stream(xyz, stream, parents=MSR_PARENTS):
    """
    (T, J, 3) -> stream tensor. joint | bone | joint_motion | bone_motion.
    Call on an ALREADY-normalised sequence so all four share one convention.
    """
    x = np.asarray(xyz, np.float32)
    if stream.startswith("bone"):
        out = np.zeros_like(x)
        for j, p in bone_pairs(parents):
            out[:, j] = x[:, j] - x[:, p]
        x = out
    elif not stream.startswith("joint"):
        raise ValueError(f"unknown stream {stream!r}")
    if stream.endswith("motion"):
        m = np.zeros_like(x)
        m[1:] = x[1:] - x[:-1]
        x = m
    return x


def resample(x, T=64):
    """Uniform temporal resample by nearest index. Deterministic."""
    n = x.shape[0]
    if n == T:
        return x
    idx = np.clip(np.round(np.linspace(0, n - 1, T)).astype(int), 0, n - 1)
    return x[idx]


def build_tensors(cache, stream, T=64, parents=MSR_PARENTS,
                  centre="perframe", align=True, scale="bone", impute=True):
    """
    Ragged cache -> (N, 3, T, J) float32, labels, subjects.

    Order of operations, which is the part that matters:
        impute -> normalise -> derive stream -> resample
    The old code did stream -> normalise -> resample, which normalised bone and
    motion tensors as if they were positions.
    """
    off, meta = cache["offsets"], cache["meta"]
    N = len(off) - 1
    J = len(parents)
    X = np.zeros((N, 3, T, J), np.float32)
    n_all_missing = 0
    for i in range(N):
        seq = cache["xyz"][off[i]:off[i + 1]]
        if impute:
            seq, miss = impute_missing(seq)
            n_all_missing += int(miss.all(0).sum())
        seq = normalise_skeleton(seq, centre=centre, align=align,
                                 scale=scale, parents=parents)
        s = resample(make_stream(seq, stream, parents), T)
        X[i] = s.transpose(2, 0, 1)
    if n_all_missing:
        print(f"    note: {n_all_missing} joint-sequences were missing in every "
              f"frame and remain zero")
    y = meta[:, 0].astype(np.int64) - 1
    subj = meta[:, 1].astype(np.int64)
    return X, y, subj


# =====================================================================
# 3. Subject-disjoint splits
# =====================================================================
def _assert_disjoint(split, subjects, n_classes=None):
    tr, va, te = (set(subjects[split[k]].tolist()) for k in ("train", "val", "test"))
    assert not (tr & va), f"train/val share subjects {sorted(tr & va)}"
    assert not (tr & te), f"train/test share subjects {sorted(tr & te)}"
    assert not (va & te), f"val/test share subjects {sorted(va & te)}"
    n = len(subjects)
    idx = np.concatenate([split["train"], split["val"], split["test"]])
    assert len(np.unique(idx)) == len(idx), "a sample appears in two partitions"
    assert len(idx) == n, f"partitions cover {len(idx)} of {n} samples"
    return True


def subject_splits(subjects, labels=None, n_splits=3, seed=0,
                   n_test_subjects=4, n_val_subjects=2, n_classes=None,
                   verbose=True):
    """
    Subject-disjoint train/val/test splits with an EXPLICIT validation size.
    Warns when a class is absent from a partition, which happens easily on
    small subject-wise splits and quietly breaks per-class calibration.
    """
    subjects = np.asarray(subjects)
    uniq = np.unique(subjects)
    need = n_test_subjects + n_val_subjects + 1
    if len(uniq) < need:
        raise ValueError(f"{len(uniq)} subjects available, need at least {need} "
                         f"({n_test_subjects} test + {n_val_subjects} val + 1 train)")
    rng = np.random.default_rng(seed)
    out = []
    for i in range(n_splits):
        perm = rng.permutation(uniq)
        te_s = set(perm[:n_test_subjects].tolist())
        va_s = set(perm[n_test_subjects:n_test_subjects + n_val_subjects].tolist())
        tr_s = set(perm[n_test_subjects + n_val_subjects:].tolist())
        sp = {
            "name": f"rand{i}",
            "train": np.where(np.isin(subjects, list(tr_s)))[0],
            "val": np.where(np.isin(subjects, list(va_s)))[0],
            "test": np.where(np.isin(subjects, list(te_s)))[0],
            "subjects": {"train": sorted(tr_s), "val": sorted(va_s), "test": sorted(te_s)},
        }
        _assert_disjoint(sp, subjects)
        if labels is not None:
            K = n_classes or int(np.max(labels)) + 1
            for part in ("train", "val", "test"):
                seen = np.unique(np.asarray(labels)[sp[part]])
                if len(seen) < K and verbose:
                    print(f"    !! {sp['name']} {part}: {K - len(seen)} of {K} "
                          f"classes absent")
        if verbose:
            print(f"    {sp['name']}: train {len(sp['train'])} "
                  f"(subj {sp['subjects']['train']}) "
                  f"val {len(sp['val'])} (subj {sp['subjects']['val']}) "
                  f"test {len(sp['test'])} (subj {sp['subjects']['test']})")
        out.append(sp)
    return out


def fixed_split(subjects, train_subjects=(1, 3, 5, 7, 9), val_subjects=(9,),
                labels=None, verbose=True):
    """
    The conventional MSR protocol (odd subjects train, even test), with
    validation held out BY SUBJECT.
    """
    subjects = np.asarray(subjects)
    train_subjects, val_subjects = set(train_subjects), set(val_subjects)
    if not val_subjects <= train_subjects:
        raise ValueError("val_subjects must be drawn from train_subjects")
    tr_s = train_subjects - val_subjects
    if not tr_s:
        raise ValueError("no training subjects left after holding out val")
    sp = {
        "name": "fixed",
        "train": np.where(np.isin(subjects, list(tr_s)))[0],
        "val": np.where(np.isin(subjects, list(val_subjects)))[0],
        "test": np.where(~np.isin(subjects, list(train_subjects)))[0],
        "subjects": {"train": sorted(tr_s), "val": sorted(val_subjects),
                     "test": sorted(set(np.unique(subjects).tolist()) - train_subjects)},
    }
    _assert_disjoint(sp, subjects)
    if verbose:
        print(f"    fixed: train {len(sp['train'])} val {len(sp['val'])} "
              f"test {len(sp['test'])} subjects {sp['subjects']}")
    return sp


# =====================================================================
# 4. Calibration helpers that need more than one split
# =====================================================================
def softmax(z, T=1.0):
    z = np.asarray(z, np.float64) / T
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)


def fit_temperature(logits_val, y_val, lo=0.05, hi=10.0, iters=200):
    """Golden-section search on validation NLL. Deterministic, 1-D, exact."""
    z, y = np.asarray(logits_val, np.float64), np.asarray(y_val)
    idx = np.arange(len(y))

    def loss(T):
        p = softmax(z, T)
        return float(-np.log(np.clip(p[idx, y], 1e-12, 1.0)).mean())

    gr = (np.sqrt(5) - 1) / 2
    a, b = lo, hi
    c, d = b - gr * (b - a), a + gr * (b - a)
    for _ in range(iters):
        if loss(c) < loss(d):
            b, d = d, c
            c = b - gr * (b - a)
        else:
            a, c = c, d
            d = a + gr * (b - a)
        if abs(b - a) < 1e-6:
            break
    return float((a + b) / 2)


def fit_temperature_pooled(val_logits, val_labels):
    """
    One temperature from validation sets pooled across splits or seeds.
    The per-split T is the honest single-run estimate, the pooled T is the
    low-variance one, and the gap between them IS the RQ3 finding.
    """
    z = np.concatenate([np.asarray(v, np.float64) for v in val_logits], 0)
    y = np.concatenate([np.asarray(v) for v in val_labels], 0)
    return fit_temperature(z, y), len(y)


def aggregate_nested(rows, metric, split_key="split", seed_key="seed"):
    """
    Mean over seeds WITHIN a split, then mean and spread ACROSS splits.
    On MSR the split-to-split spread is several times the seed-to-seed spread,
    so a pooled "+/- over seeds" label mostly measures which subjects landed in
    the test set.
    """
    by_split = {}
    for r in rows:
        v = r.get(metric)
        if v is None or not np.isfinite(v):
            continue
        by_split.setdefault(r[split_key], []).append(float(v))
    if not by_split:
        return {"mean": np.nan, "std_across_splits": np.nan,
                "mean_std_within_split": np.nan, "n_splits": 0, "n_seeds": 0}
    per_split_mean = np.array([np.mean(v) for v in by_split.values()])
    per_split_std = np.array([np.std(v, ddof=1) if len(v) > 1 else 0.0
                              for v in by_split.values()])
    return {
        "mean": float(per_split_mean.mean()),
        "std_across_splits": float(per_split_mean.std(ddof=1))
        if len(per_split_mean) > 1 else 0.0,
        "mean_std_within_split": float(per_split_std.mean()),
        "n_splits": len(by_split),
        "n_seeds": int(np.median([len(v) for v in by_split.values()])),
    }


# =====================================================================
# 5. Metrics
# =====================================================================
def _bins_equal_width(conf, M):
    edges = np.linspace(0.0, 1.0, M + 1)
    return np.clip(np.digitize(conf, edges[1:-1], right=True), 0, M - 1), M


def _bins_equal_mass(conf, M):
    """Equal sample count per bin. Far less biased at small n than fixed width."""
    n = len(conf)
    order = np.argsort(conf, kind="stable")
    idx = np.empty(n, np.int64)
    edges = np.linspace(0, n, M + 1).astype(int)
    for b in range(M):
        idx[order[edges[b]:edges[b + 1]]] = b
    return idx, M


def calibration_error(p, y, M=15, mode="equal_width", norm="l1"):
    p = np.asarray(p, np.float64)
    conf = p.max(-1)
    correct = (p.argmax(-1) == np.asarray(y)).astype(np.float64)
    idx, M = (_bins_equal_width if mode == "equal_width" else _bins_equal_mass)(conf, M)
    n = len(conf)
    gaps, weights = [], []
    for b in range(M):
        m = idx == b
        if not m.any():
            continue
        gaps.append(abs(correct[m].mean() - conf[m].mean()))
        weights.append(m.sum() / n)
    if not gaps:
        return float("nan")
    gaps, weights = np.array(gaps), np.array(weights)
    return float(gaps.max() if norm == "max" else (gaps * weights).sum())


def ece(p, y, M=15):
    return calibration_error(p, y, M, "equal_width", "l1")


def adaptive_ece(p, y, M=15):
    return calibration_error(p, y, M, "equal_mass", "l1")


def mce(p, y, M=15):
    return calibration_error(p, y, M, "equal_width", "max")


def brier(p, y):
    """Multiclass Brier. Unbiased and binning-free -- trustworthy at small n."""
    p = np.asarray(p, np.float64)
    oh = np.zeros_like(p)
    oh[np.arange(len(y)), np.asarray(y)] = 1.0
    return float(((p - oh) ** 2).sum(-1).mean())


def nll(p, y):
    p = np.asarray(p, np.float64)
    return float(-np.log(np.clip(p[np.arange(len(y)), np.asarray(y)], 1e-12, 1.0)).mean())


def top1(p, y):
    return float((np.asarray(p).argmax(-1) == np.asarray(y)).mean())


def risk_coverage(p, y):
    """Sorted by confidence descending. Returns (coverage, risk, AURC)."""
    p, y = np.asarray(p, np.float64), np.asarray(y)
    err = (p.argmax(-1) != y).astype(np.float64)[np.argsort(-p.max(-1), kind="stable")]
    k = np.arange(1, len(err) + 1)
    risk = np.cumsum(err) / k
    return k / len(err), risk, float(risk.mean())


def aurc(p, y):
    return risk_coverage(p, y)[2]


def per_class_ece(p, y, M=15):
    """One-vs-rest over samples predicted as each class. RQ4."""
    p, y = np.asarray(p, np.float64), np.asarray(y)
    out = np.full(p.shape[1], np.nan)
    for c in range(p.shape[1]):
        m = p.argmax(-1) == c
        if m.sum() < 2:
            continue
        conf, corr = p[m, c], (y[m] == c).astype(np.float64)
        idx, M_ = _bins_equal_mass(conf, min(M, max(2, int(m.sum() // 5))))
        g, w = [], []
        for b in range(M_):
            bm = idx == b
            if bm.any():
                g.append(abs(corr[bm].mean() - conf[bm].mean()))
                w.append(bm.sum() / m.sum())
        if g:
            out[c] = float((np.array(g) * np.array(w)).sum())
    return out


def classwise_ece(p, y, M=15):
    """
    Classwise ECE (Kull et al., NeurIPS 2019): for every class k, the
    equal-width binned ECE of p[:, k] against the one-vs-rest indicator
    1[y == k], averaged over classes. Unlike top-label ECE it scores the whole
    probability vector, so it sees miscalibration in non-argmax classes that
    top-label metrics ignore (Gupta & Ramdas, ICLR 2022).
    """
    p, y = np.asarray(p, np.float64), np.asarray(y)
    n, K = p.shape
    edges = np.linspace(0.0, 1.0, M + 1)
    tot = 0.0
    for k in range(K):
        pk = p[:, k]
        ok = (y == k).astype(np.float64)
        idx = np.clip(np.digitize(pk, edges[1:-1], right=True), 0, M - 1)
        s = 0.0
        for b in range(M):
            m = idx == b
            if m.any():
                s += m.sum() / n * abs(ok[m].mean() - pk[m].mean())
        tot += s
    return float(tot / K)


def brier_decomposition(p, y, M=15):
    """
    Murphy (1973) decomposition of the multiclass Brier score, computed
    one-vs-rest per class and summed (the multiclass Brier score is exactly the
    sum of the K binary scores). With binned forecasts (Stephenson et al. 2008):

        BS = REL - RES + UNC + WB

    REL  reliability  (calibration; lower is better)
    RES  resolution   (refinement / discrimination; HIGHER is better)
    UNC  uncertainty  (depends on the label distribution only)
    WB   within-bin terms (variance minus covariance inside bins; ~0 for
         fine bins)

    The identity is exact by construction (WB is computed, not assumed), so it
    doubles as a test. It explains why a method can improve Brier without
    improving ECE: a gain in RES (sharper, better-ranked forecasts) lowers the
    score even when REL is unchanged.
    """
    p, y = np.asarray(p, np.float64), np.asarray(y)
    n, K = p.shape
    edges = np.linspace(0.0, 1.0, M + 1)
    REL = RES = UNC = WB = 0.0
    for k in range(K):
        f = p[:, k]
        o = (y == k).astype(np.float64)
        obar = o.mean()
        idx = np.clip(np.digitize(f, edges[1:-1], right=True), 0, M - 1)
        rel = res = wb = 0.0
        for b in range(M):
            m = idx == b
            if not m.any():
                continue
            nb = m.sum()
            fb, ob = f[m].mean(), o[m].mean()
            rel += nb * (fb - ob) ** 2
            res += nb * (ob - obar) ** 2
            wb += ((f[m] - fb) ** 2).sum() - 2 * ((f[m] - fb) * (o[m] - ob)).sum()
        REL += rel / n
        RES += res / n
        UNC += obar * (1 - obar)
        WB += wb / n
    return {"rel": float(REL), "res": float(RES), "unc": float(UNC),
            "wb": float(WB), "brier": float(REL - RES + UNC + WB)}


def quality_mask(flags, max_zero_frac=0.5, min_frames=10):
    """
    Sequence-level data-quality filter, applied at tensor-build time (the cache
    stays raw). flags columns: [n_frames, nonfinite_frac, zero_joint_frac, ...].
    A sequence is dropped if more than `max_zero_frac` of its joint-frames are
    all-zero (the Kinect tracker lost the skeleton, so imputation has nothing to
    interpolate from) or if it has fewer than `min_frames` frames. This is the
    principled version of the conventional 557-of-567 MSR-Action3D subset,
    whose exact exclusion list is not consistently documented.
    """
    flags = np.asarray(flags, np.float64)
    keep = (flags[:, 2] <= max_zero_frac) & (flags[:, 0] >= min_frames)
    keep &= np.isfinite(flags[:, 1]) & (flags[:, 1] < 1.0)
    return keep


def load_cache(npz):
    """Ragged cache file -> dict with xyz, offsets, meta and (if present) flags."""
    z = np.load(npz, allow_pickle=False)
    out = {k: z[k] for k in ("xyz", "offsets", "meta")}
    if "flags" in z.files:
        out["flags"] = z["flags"]
    return out


def masked_tensors(cache, stream, T=64, prep=None, max_zero_frac=0.5,
                   min_frames=10, parents=MSR_PARENTS):
    """
    build_tensors() followed by quality_mask(). The mask is applied BEFORE any
    split is drawn, so every split, seed and method sees the same sample set.
    Returns X, y, subj, keep (boolean over the raw cache order).
    """
    prep = dict(prep or {})
    X, y, subj = build_tensors(cache, stream, T=T, parents=parents,
                               scale="bone", impute=True, **prep)
    if "flags" in cache:
        keep = quality_mask(cache["flags"], max_zero_frac, min_frames)
    else:
        keep = np.ones(len(y), bool)
    return X[keep], y[keep], subj[keep], keep


def bootstrap_ci(fn, p, y, n_boot=1000, alpha=0.05, seed=0):
    """Percentile CI. Non-negotiable on the small datasets."""
    rng = np.random.default_rng(seed)
    p, y = np.asarray(p), np.asarray(y)
    vals = np.empty(n_boot)
    for i in range(n_boot):
        s = rng.integers(0, len(y), len(y))
        vals[i] = fn(p[s], y[s])
    vals = vals[np.isfinite(vals)]
    return (float(fn(p, y)), float(np.quantile(vals, alpha / 2)),
            float(np.quantile(vals, 1 - alpha / 2)))


def evaluate(logits, y, logits_val=None, y_val=None, n_boot=0, seed=0) -> dict:
    """Every metric for one (model, dataset, stream, split, seed) cell."""
    p = softmax(logits)
    out = {"n": int(len(y)), "n_classes": int(p.shape[1]), "top1": top1(p, y),
           "ece_15": ece(p, y), "ece_adaptive": adaptive_ece(p, y),
           "mce_15": mce(p, y), "brier": brier(p, y), "nll": nll(p, y),
           "aurc": aurc(p, y)}
    pc = per_class_ece(p, y)
    out["per_class_ece_mean"] = float(np.nanmean(pc)) if np.isfinite(pc).any() else float("nan")
    out["per_class_ece_max"] = float(np.nanmax(pc)) if np.isfinite(pc).any() else float("nan")
    out["classwise_ece"] = classwise_ece(p, y)
    bd = brier_decomposition(p, y)
    out.update({"brier_rel": bd["rel"], "brier_res": bd["res"],
                "brier_unc": bd["unc"], "brier_wb": bd["wb"]})
    if logits_val is not None and y_val is not None:
        T = fit_temperature(logits_val, y_val)
        pT = softmax(logits, T)
        out.update({"temperature": T, "ece_adaptive_temp": adaptive_ece(pT, y),
                    "brier_temp": brier(pT, y), "nll_temp": nll(pT, y),
                    "top1_temp": top1(pT, y),
                    "classwise_ece_temp": classwise_ece(pT, y)})
    if n_boot:
        for name, fn in (("ece_adaptive", adaptive_ece), ("brier", brier), ("nll", nll)):
            _, lo, hi = bootstrap_ci(fn, p, y, n_boot, seed=seed)
            out[f"{name}_lo"], out[f"{name}_hi"] = lo, hi
    return out


# =====================================================================
# 6. Cross-dataset joint-order matching
# =====================================================================
def distance_signature(xyz, max_frames=20000):
    """Scale-free mean pairwise distance matrix over clean frames."""
    F = np.asarray(xyz, np.float64)
    if F.ndim != 3:
        raise ValueError(f"expected (T, J, 3), got {F.shape}")
    F = F[~(np.abs(F) < 1e-9).all(-1).any(-1)]
    if len(F) < 50:
        raise ValueError(f"only {len(F)} clean frames; need 50+")
    if len(F) > max_frames:
        F = F[np.linspace(0, len(F) - 1, max_frames).astype(int)]
    D = np.linalg.norm(F[:, :, None, :] - F[:, None, :, :], axis=-1).mean(0)
    iu = np.triu_indices(len(D), 1)
    return D / np.median(D[iu])


def _assignment_cost(Dq, Dr, perm):
    return float(np.abs(Dq[np.ix_(perm, perm)] - Dr).sum())


def match_joint_order(Dq, Dr, restarts=30, seed=0):
    """
    perm[c] = q : canonical joint c corresponds to query joint q.
    Quadratic assignment by rank initialisation plus 2-opt, several restarts.
    """
    n = len(Dr)
    rng = np.random.default_rng(seed)
    best = (np.inf, None)
    rank_q, rank_r = np.argsort(Dq.sum(1)), np.argsort(Dr.sum(1))
    for r in range(restarts):
        perm = np.empty(n, int)
        if r == 0:
            perm[rank_r] = rank_q
        else:
            perm = rng.permutation(n)
        c = _assignment_cost(Dq, Dr, perm)
        improved = True
        while improved:
            improved = False
            for i in range(n):
                for j in range(i + 1, n):
                    p2 = perm.copy()
                    p2[i], p2[j] = p2[j], p2[i]
                    c2 = _assignment_cost(Dq, Dr, p2)
                    if c2 < c - 1e-12:
                        perm, c, improved = p2, c2, True
        if c < best[0]:
            best = (c, perm.copy())
    return best[1], best[0] / (n * n)


def _mirror(perm, pairs=LR_PAIRS):
    out = perm.copy()
    for a, b in pairs:
        out[a], out[b] = perm[b], perm[a]
    return out


def derive_canonical_permutation(query_xyz, reference_xyz, parents=MSR_PARENTS,
                                 root=ROOT, verbose=True):
    """
    Recover the permutation putting `query_xyz` into canonical joint order.
    Apply it as xyz_canonical = xyz_query[:, perm].

    A human skeleton is nearly bilaterally symmetric, so the L/R mirror of any
    correct match is almost as cheap. Resolved on geometry, not cost.
    """
    Dr = distance_signature(reference_xyz)
    Dq = distance_signature(query_xyz)
    perm, resid = match_joint_order(Dq, Dr)
    mirror = _mirror(perm)
    resid_m = _assignment_cost(Dq, Dr, mirror) / (len(Dr) ** 2)

    def lr_sign(xyz, p):
        F = np.asarray(xyz, np.float64)
        F = F[~(np.abs(F) < 1e-9).all(-1).any(-1)]
        return float(np.mean(F[:, p[0], 0] - F[:, p[root], 0]))

    want = lr_sign(reference_xyz, np.arange(len(Dr)))
    got = lr_sign(query_xyz, perm)
    flipped = False
    if np.sign(got) != np.sign(want) and abs(resid_m - resid) < 0.25 * max(resid, 1e-9):
        perm, resid, flipped = mirror, resid_m, True

    if verbose:
        print(f"    joint-order match: residual {resid:.4f} "
              f"(mirror {resid_m:.4f})"
              f"{' [mirror chosen on L/R geometry]' if flipped else ''}")
        print(f"    perm (canonical -> query index): {perm.tolist()}")
        if np.array_equal(perm, np.arange(len(Dr))):
            print("    identity -- the query already uses the canonical order")
    return perm, {"residual": resid, "residual_mirror": resid_m,
                  "mirror_applied": flipped, "perm": perm.tolist()}


def body_extent(xyz, max_frames=2000):
    """(median, max) inter-joint distance in metres, over clean frames."""
    F = np.asarray(xyz, np.float64)
    F = F[~(np.abs(F) < 1e-9).all(-1).any(-1)]
    if len(F) < 5:
        return float("nan"), float("nan")
    if len(F) > max_frames:
        F = F[np.linspace(0, len(F) - 1, max_frames).astype(int)]
    D = np.linalg.norm(F[:, :, None, :] - F[:, None, :, :], axis=-1)
    iu = np.triu_indices(D.shape[1], 1)
    return float(np.median(D[:, iu[0], iu[1]])), float(np.nanmax(D))


def score_parse(xyz, max_len=MAX_BONE_M, max_cv=MAX_BONE_CV,
                max_sd=MAX_BONE_SD_M):
    """
    Order-independent plausibility of a candidate parse, in [0, 1].
    PHYSICAL EXTENT first -- a parse mixing metres with raw depth yields a body
    kilometres across, which no heuristic can argue with -- then rigid structure.
    """
    med, mx = body_extent(xyz)
    if not np.isfinite(med) or med > MAX_MEDIAN_PAIR_M or med < MIN_MEDIAN_PAIR_M \
            or mx > MAX_BODY_SPAN_M:
        return 0.0, med
    tree = rigid_spanning_tree(xyz, max_cv, max_sd, allow_fallback=True)
    n_needed = np.asarray(xyz).shape[1] - 1
    if len(tree) < n_needed:
        return 0.0, float("nan")
    lens = [w for w, _, _ in tree]
    return float(np.mean([w <= max_len for w in lens])), float(np.median(lens))


In [ ]:
%%writefile /content/har3d_action3d.py
"""
har3d_action3d.py -- MSR Action3D ingest.

Two archives ship the same 567 sequences in different coordinate systems:
    MSRAction3DSkeletonReal3D.rar      metric (x, y, z) in metres
    MSRAction3DSkeleton(20joints).rar  screen (u, v, d, confidence)

The metric archive is used directly. The screen archive contributes only its
confidence column, which is the per-joint noise proxy RQ4 needs, and the
least-squares intrinsics solve that confirms the two are the same recording.

SETTLED: fitting x/d against u recovers fx = 285.633, cx = 160.002,
fy = -285.574, cy = 120.001, depth scale 0.003998. That is the canonical
Kinect v1 focal length to 0.001%, so the resolution is 320x240 and no
back-projection is needed. The negative fy is the expected image-v-down
against world-y-up flip.

This dataset defines the canonical joint order for the project.
"""
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np

from har3d_io import extract_any, find_skeleton_files, log, read_rows

NAME_RE = re.compile(r"a(\d+)[_-]s(\d+)[_-]e(\d+)", re.IGNORECASE)
N_JOINTS = 20
EXPECTED_FULL = 567
EXPECTED_CONVENTIONAL = 557
KINECT_V1_FX = 285.63


def _resolve(drive_root: Path, patterns):
    for pat in patterns:
        exact = drive_root / pat
        if "*" not in pat and exact.exists():
            return exact
        hits = sorted(drive_root.glob(pat))
        if hits:
            return hits[0]
    return None


def parse_name(path: Path):
    m = NAME_RE.search(path.name)
    return (int(m.group(1)), int(m.group(2)), int(m.group(3))) if m else None


def load_sequence(path: Path, n_joints=N_JOINTS):
    """(T, J, C) or None. Rows must divide evenly by the joint count."""
    rows, w = read_rows(path, widths=(3, 4))
    if rows.shape[0] == 0 or rows.shape[0] % n_joints:
        return None
    return rows.reshape(rows.shape[0] // n_joints, n_joints, w)


def solve_intrinsics(screen: np.ndarray, real: np.ndarray) -> dict:
    """
    Recover fx, fy, cx, cy by least squares rather than guessing.
    x = (u - cx) d / fx  =>  x/d = (1/fx) u + (-cx/fx), so a line fit of x/d
    against u returns fx and cx exactly.
    """
    u, v, d = screen[:, 0], screen[:, 1], screen[:, 2]
    x, y, z = real[:, 0], real[:, 1], real[:, 2]
    ok = np.isfinite([u, v, d, x, y, z]).all(0) & (np.abs(d) > 1e-6)
    if ok.sum() < 100:
        raise ValueError(f"only {int(ok.sum())} usable joints for the fit")
    u, v, d, x, y, z = (a[ok] for a in (u, v, d, x, y, z))

    def fit(px, ratio):
        A = np.stack([px, np.ones_like(px)], 1)
        (slope, icept), *_ = np.linalg.lstsq(A, ratio, rcond=None)
        pred = A @ np.array([slope, icept])
        ss_res = float(((ratio - pred) ** 2).sum())
        ss_tot = float(((ratio - ratio.mean()) ** 2).sum())
        f = 1.0 / slope if abs(slope) > 1e-12 else float("inf")
        return float(f), float(-icept * f), 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    fx, cx, r2x = fit(u, x / d)
    fy, cy, r2y = fit(v, y / d)
    s = float(np.median(z / d))
    return {"fx_raw": fx, "fy_raw": fy, "cx": cx, "cy": cy,
            "depth_scale": s, "fx": fx * s, "fy": fy * s,
            "r2_x": r2x, "r2_y": r2y, "n_points": int(ok.sum())}


def degeneracy(seq: np.ndarray) -> list:
    xyz = seq[..., :3]
    conf = seq[..., 3] if seq.shape[-1] >= 4 else np.ones(seq.shape[:2])
    return [int(seq.shape[0]),
            float(1.0 - np.isfinite(xyz).all(-1).mean()),
            float((np.abs(xyz) < 1e-9).all(-1).mean()),
            float((conf <= 0).mean())]


def ingest(drive_root, cache_dir, work="/content/har3d_work/msr_action3d",
           push_to_drive=True):
    """Both archives -> msr_action3d.npz (ragged, with per-sequence confidence)."""
    drive_root, cache_dir, work = Path(drive_root), Path(cache_dir), Path(work)
    cache_dir.mkdir(parents=True, exist_ok=True)

    real_arc = _resolve(drive_root, ["MSRAction3DSkeletonReal3D.rar", "*Real3D*"])
    scr_arc = _resolve(drive_root, ["MSRAction3DSkeleton(20joints).rar", "*20joints*"])
    if real_arc is None:
        return {"error": f"Real3D archive not found under {drive_root}"}

    d_real, d_scr = work / "real3d", work / "screen"
    if not find_skeleton_files(d_real):
        log(f"  extracting {real_arc.name} with {extract_any(real_arc, d_real)}")
    if scr_arc is not None and not find_skeleton_files(d_scr):
        log(f"  extracting {scr_arc.name} with {extract_any(scr_arc, d_scr)}")

    def index(root):
        out = {}
        for p in Path(root).rglob("*.txt"):
            k = parse_name(p)
            if k:
                out[k] = p
        return out

    real, scr = index(d_real), index(d_scr) if scr_arc is not None else {}
    log(f"  Real3D sequences: {len(real)}   screen sequences: {len(scr)}")
    if len(real) not in (EXPECTED_FULL, EXPECTED_CONVENTIONAL):
        log(f"  !! {len(real)} sequences; expected 567 (full) or 557 (pre-filtered)")

    intr = {}
    shared = sorted(set(real) & set(scr))
    if shared:
        S, R = [], []
        for k in shared[:40]:
            s_, r_ = load_sequence(scr[k]), load_sequence(real[k])
            if s_ is None or r_ is None:
                continue
            T = min(s_.shape[0], r_.shape[0])
            S.append(s_[:T].reshape(-1, s_.shape[-1]))
            R.append(r_[:T].reshape(-1, r_.shape[-1]))
        if S:
            intr = solve_intrinsics(np.concatenate(S), np.concatenate(R))
            off = abs(abs(intr["fx"]) - KINECT_V1_FX) / KINECT_V1_FX * 100
            log(f"  intrinsics: fx {intr['fx']:.3f}  fy {intr['fy']:.3f}  "
                f"cx {intr['cx']:.3f}  cy {intr['cy']:.3f}  "
                f"depth scale {intr['depth_scale']:.6f}")
            log(f"  fit R2 x {intr['r2_x']:.6f} y {intr['r2_y']:.6f}; "
                f"{off:.3f}% off the canonical Kinect v1 285.63 "
                f"-> 320x{int(round(intr['cy'] * 2))}")
            if min(intr["r2_x"], intr["r2_y"]) < 0.999:
                log("  !! poor pinhole fit -- the two archives may not be the "
                    "same recording; do NOT treat them as interchangeable")

    xyz_all, conf_all, lens, metas, flags, has_conf = [], [], [], [], [], []
    for k in sorted(real):
        seq = load_sequence(real[k])
        if seq is None:
            log(f"  !! unreadable, skipped: {real[k].name}")
            continue
        xyz = seq[..., :3].astype(np.float32)
        c = np.full(xyz.shape[:2], np.nan, np.float32)
        ok = False
        if k in scr:
            s_ = load_sequence(scr[k])
            if s_ is not None and s_.shape[-1] >= 4 and s_.shape[0] == xyz.shape[0]:
                c, ok = s_[..., 3].astype(np.float32), True
        xyz_all.append(xyz)
        conf_all.append(c)
        has_conf.append(ok)
        lens.append(xyz.shape[0])
        metas.append(list(k))
        flags.append(degeneracy(seq))

    if not xyz_all:
        return {"error": "no sequence parsed"}

    out = cache_dir / "msr_action3d.npz"
    np.savez_compressed(
        out,
        xyz=np.concatenate(xyz_all, 0),
        conf=np.concatenate(conf_all, 0),
        has_conf=np.array(has_conf, bool),
        offsets=np.concatenate([[0], np.cumsum(lens)]).astype(np.int64),
        meta=np.array(metas, np.int64),
        flags=np.array(flags, np.float64),
        intrinsics=np.array(json.dumps(intr)))
    log(f"  cached -> {out.name} ({out.stat().st_size/1e6:.1f} MB, "
        f"{len(metas)} sequences, {sum(lens):,} frames)")
    log(f"  confidence available for {int(np.sum(has_conf))}/{len(metas)} sequences")

    if push_to_drive:
        (drive_root / "cache").mkdir(parents=True, exist_ok=True)
        import shutil as _sh
        _sh.copy2(out, drive_root / "cache" / out.name)
        log(f"  copied to {drive_root / 'cache' / out.name}")

    fl = np.array(flags)
    return {"n_sequences": len(metas), "n_frames": int(sum(lens)),
            "subjects": sorted({m[1] for m in metas}),
            "n_actions": len({m[0] for m in metas}),
            "intrinsics": intr, "cache": str(out),
            "n_with_confidence": int(np.sum(has_conf)),
            "frac_frames_with_a_dropped_joint": float(fl[:, 2].mean())}


def exclusion_candidates(cache_path, k: int = 10) -> list:
    """
    Rank all 567 by degeneracy to recover the conventional 10 exclusions.
    Reported, never auto-applied: the list goes in the reproducibility
    statement and should be eyeballed before it is fixed.
    """
    z = np.load(cache_path, allow_pickle=False)
    meta, fl = z["meta"], z["flags"]
    n_frames, nonfin, zero_j = fl[:, 0], fl[:, 1], fl[:, 2]
    score = zero_j + nonfin + 0.5 * (n_frames < 20)
    order = np.argsort(-score)
    log(f"  {'sequence':<16}{'frames':>8}{'zero_joints':>14}{'nonfinite':>12}")
    names = []
    for i in order[:k + 5]:
        a, s, e = meta[i]
        nm = f"a{a:02d}_s{s:02d}_e{e:02d}"
        mark = f"  <-- top {k}" if len(names) < k else ""
        log(f"  {nm:<16}{int(n_frames[i]):>8}{zero_j[i]:>14.3f}{nonfin[i]:>12.3f}{mark}")
        if len(names) < k:
            names.append(nm)
    log(f"\n  proposed exclusions ({k}): {', '.join(names)}")
    log("  -> cross-check against a published list before fixing this in the paper")
    return names


In [ ]:
%%writefile /content/har3d_daily.py
"""
har3d_daily.py -- MSR DailyActivity3D ingest, nested-archive aware.

WHY THE ORIGINAL FAILED
-----------------------
MSRDailyActivity3D.zip is a zip OF ZIPS: 9 members, every one of them stored
(compress_type 0) rather than deflated. Stored members copy at raw disk speed,
which is why 3.56 GB "extracted" inside a 101 s run. After extractall there
were nine archives and zero .txt, so find_files() matched nothing.

Fix: expand nested archives wherever they land, recursively.

PARSING
-------
np.loadtxt is not usable here. Skeleton files interleave per-frame header lines
with 4-column joint rows. This parser classifies each line by token count.

Rows per frame is 20 or 40 and divisibility cannot tell them apart. The 40-row
layout interleaves world coordinates (order 1 m) with screen coordinates, so
every candidate layout is tried and scored by bone plausibility.
"""
from __future__ import annotations

import re
import shutil
from pathlib import Path

import numpy as np

from har3d_io import (expand_archives, extract_any, find_skeleton_files,
                      inspect_methods, read_rows, reclaim_media)

NAME_RE = re.compile(r"a(\d+)[_-]s(\d+)[_-]e(\d+)", re.IGNORECASE)
EXPECTED_SEQUENCES = 320


def _log(m=""):
    print(m, flush=True)


def read_joint_rows(path: Path) -> np.ndarray:
    """4-column joint rows; per-frame markers and the file header discarded."""
    rows, _ = read_rows(path, widths=(4,))
    return rows


CANDIDATES = [(20, "all"),      # 20 rows/frame, one coordinate block
              (40, "first"),    # 40 rows/frame, blocked: world first
              (40, "second"),   # 40 rows/frame, blocked: world second
              (40, "even"),     # 40 rows/frame, interleaved: world on even
              (40, "odd")]      # 40 rows/frame, interleaved: world on odd


def _slice(seq, mode, col=slice(0, 3)):
    """seq is (T, rows_per_frame, 4) -> (T, 20, len(col))."""
    if mode == "all":
        return seq[:, :, col]
    if mode == "first":
        return seq[:, :20, col]
    if mode == "second":
        return seq[:, 20:, col]
    if mode == "even":
        return seq[:, 0::2, col]
    if mode == "odd":
        return seq[:, 1::2, col]
    raise ValueError(f"unknown slice mode {mode!r}")


def score_layout(xyz, parents=None, max_len=0.60, max_cv=0.20):
    """
    Plausibility of a candidate parse, INDEPENDENT of joint order. Scoring
    against a fixed tree would assume Action3D's indexing, which this dataset
    does not share.
    """
    from har3d_v2 import score_parse
    return score_parse(xyz, max_len=max_len, max_cv=max_cv)


def choose_layout(paths, parents=None, n_probe=8):
    """Return (rows_per_frame, mode) for the layout that yields a real skeleton."""
    del parents
    probe = [read_joint_rows(p) for p in list(paths)[:n_probe]]
    probe = [r for r in probe if r.shape[0] > 0]
    if not probe:
        return None, None

    results = []
    for rows, mode in CANDIDATES:
        scores, meds = [], []
        for raw in probe:
            n = raw.shape[0]
            if n % rows or n // rows < 2:
                continue
            xyz = _slice(raw.reshape(n // rows, rows, raw.shape[1]), mode)
            if xyz.shape[1] != 20:
                continue
            f, med = score_layout(xyz)
            scores.append(f)
            meds.append(med)
        if scores:
            results.append((float(np.mean(scores)), rows, mode,
                            float(np.nanmedian(meds)), len(scores)))
    if not results:
        _log("  !! no candidate layout divides the row counts evenly")
        return None, None
    results.sort(reverse=True)

    # A pure 20-row file read as 40 rows/frame pairs two consecutive FRAMES into
    # one, and both halves are still real skeletons -- bone plausibility cannot
    # separate them and would silently halve the frame rate.
    top = results[0][0]
    close = [r for r in results if r[0] >= top - 0.03]
    if len(close) > 1:
        simplest = min(close, key=lambda r: (r[1], r[2] != "all"))
        if simplest is not results[0]:
            _log(f"  scores within 0.03 across {len(close)} candidates; "
                 f"preferring the simpler {simplest[1]} rows/frame layout")
            results.remove(simplest)
            results.insert(0, simplest)

    _log(f"  layout scan over {len(probe)} files "
         f"(fraction of edges that look like bones, median bone length):")
    from har3d_v2 import body_extent
    for f, rows, mode, med, n in results:
        mark = "  <-- chosen" if (f, rows, mode, med, n) == results[0] else ""
        ex = body_extent(_slice(probe[0].reshape(-1, rows, probe[0].shape[1]), mode)) \
            if probe[0].shape[0] % rows == 0 else (float("nan"),) * 2
        _log(f"   {rows:>2} rows/frame, world={mode:<6} score {f:5.2f}  "
             f"median edge {med:9.3f} m  body span {ex[1]:9.2f} m{mark}")

    best_f, rows, mode, med, _ = results[0]
    if best_f < 0.8:
        raise RuntimeError(
            f"no layout produces a plausible skeleton (best {best_f:.2f} of 1.00, "
            f"median edge {med:.3f} m). The file format differs from every "
            f"candidate -- inspect a raw file before going further.")
    return rows, mode


def parse_sequence(path: Path, layout: int, mode: str = "first"):
    """-> (xyz (T,20,3) float32, conf (T,20) float32) or None."""
    raw = read_joint_rows(path)
    n = raw.shape[0]
    if n == 0 or n % layout:
        return None
    seq = raw.reshape(n // layout, layout, raw.shape[1])
    xyz = _slice(seq, mode).astype(np.float32)
    conf = _slice(seq, mode, col=slice(3, 4))[..., 0].astype(np.float32)
    if xyz.shape[1] != 20:
        return None
    return xyz, conf


def ingest(drive_root, cache_dir, work="/content/har3d_work/msr_daily",
           reclaim=True, push_to_drive=True):
    """MSRDailyActivity3D.zip -> msr_daily.npz in the same ragged format."""
    drive_root, cache_dir = Path(drive_root), Path(cache_dir)
    work = Path(work)
    work.mkdir(parents=True, exist_ok=True)
    cache_dir.mkdir(parents=True, exist_ok=True)

    zp = None
    for pat in ("MSRDailyActivity3D.zip", "*DailyAct*.zip", "*Daily*.zip"):
        hits = sorted(drive_root.glob(pat))
        if hits:
            zp = hits[0]
            break
    if zp is None:
        return {"error": f"no DailyActivity archive under {drive_root}"}

    if not find_skeleton_files(work):
        info = inspect_methods(zp)
        _log(f"  {zp.name}: {info['names']}")
        _log(f"  extracting with {extract_any(zp, work)} ...")
        n = expand_archives(work)
        _log(f"  expanded {n} nested archives")
        if n == 0:
            _log("  !! no nested archive expanded -- see the messages above")

    skel = find_skeleton_files(work)
    _log(f"  skeleton files: {len(skel)} (expected {EXPECTED_SEQUENCES})")
    if not skel:
        present = sorted({p.suffix.lower() for p in work.rglob("*") if p.is_file()})
        return {"error": "no skeleton files after expansion",
                "suffixes_present": present}

    layout, mode = choose_layout(skel)
    if layout is None:
        return {"error": "could not determine rows per frame"}

    perm, perm_info = None, {}
    ref = cache_dir / "msr_action3d.npz"
    if ref.exists():
        import har3d_v2 as V
        rz = np.load(ref, allow_pickle=False)
        ro = rz["offsets"]
        ref_xyz = rz["xyz"][ro[0]:ro[min(120, len(ro) - 1)]]
        probe_xyz = np.concatenate(
            [parse_sequence(p, layout, mode)[0] for p in skel[:60]
             if parse_sequence(p, layout, mode) is not None], 0)
        perm, perm_info = V.derive_canonical_permutation(probe_xyz, ref_xyz)
    else:
        _log("  !! msr_action3d.npz not in the cache -- cannot check joint order; "
             "storing the dataset's own order")

    xyz_all, conf_all, lens, metas, flags, skipped = [], [], [], [], [], []
    for p in skel:
        m = NAME_RE.search(p.name)
        if m is None:
            skipped.append((p.name, "unparseable name"))
            continue
        parsed = parse_sequence(p, layout, mode)
        if parsed is None:
            skipped.append((p.name, "row count not a multiple of layout"))
            continue
        xyz, conf = parsed
        if perm is not None:
            xyz, conf = xyz[:, perm], conf[:, perm]
        xyz_all.append(xyz)
        conf_all.append(conf)
        lens.append(xyz.shape[0])
        metas.append([int(m.group(1)), int(m.group(2)), int(m.group(3))])
        zero = (np.abs(xyz) < 1e-9).all(-1)
        flags.append([xyz.shape[0], float(1.0 - np.isfinite(xyz).all(-1).mean()),
                      float(zero.mean()), float((conf <= 0).mean())])

    if not xyz_all:
        return {"error": "every skeleton file failed to parse",
                "skipped": skipped[:10]}

    out = cache_dir / "msr_daily.npz"
    np.savez_compressed(
        out,
        xyz=np.concatenate(xyz_all, 0).astype(np.float32),
        conf=np.concatenate(conf_all, 0).astype(np.float32),
        offsets=np.concatenate([[0], np.cumsum(lens)]).astype(np.int64),
        meta=np.array(metas, np.int64),
        flags=np.array(flags, np.float64),
        rows_per_frame=np.array(layout),
        world_rows=np.array(mode),
        joint_perm=np.array(perm if perm is not None else np.arange(20)))
    _log(f"  cached -> {out.name} ({out.stat().st_size/1e6:.1f} MB, "
         f"{len(metas)} sequences)")

    freed = reclaim_media(work) if reclaim else 0
    if reclaim:
        _log(f"  reclaimed {freed/1e9:.2f} GB of RGB/depth")

    if push_to_drive:
        (drive_root / "cache").mkdir(parents=True, exist_ok=True)
        shutil.copy2(out, drive_root / "cache" / out.name)
        _log(f"  copied to {drive_root / 'cache' / out.name}")

    subj = sorted({m[1] for m in metas})
    acts = sorted({m[0] for m in metas})
    _log(f"  subjects {subj}  actions {len(acts)}  frames {sum(lens):,}")

    res = {"n_sequences": len(metas), "rows_per_frame": layout,
           "world_rows": mode, "joint_order": perm_info,
           "subjects": subj, "n_actions": len(acts), "cache": str(out),
           "freed_bytes": freed}
    if len(metas) != EXPECTED_SEQUENCES:
        res["warning"] = f"{len(metas)} sequences, expected {EXPECTED_SEQUENCES}"
        _log(f"  !! {res['warning']}")
    if skipped:
        res["skipped"] = skipped[:10]
        _log(f"  !! {len(skipped)} files skipped, first few: {skipped[:3]}")
    return res


In [ ]:
%%writefile /content/har3d_utkinect.py
"""
har3d_utkinect.py -- UTKinect-Action3D ingest.

WHAT THIS DATASET IS
--------------------
20 continuous recordings (10 subjects x 2 episodes). Each recording holds all
ten actions performed back to back; actionLabel.txt gives the [start, end]
FRAME NUMBER of every segment. There is no pre-segmented release -- the clips
are cut here. 199 valid clips, not 200: s10_e02 has `carry:  NaN NaN`.

FOUR TRAPS, ALL CONFIRMED AGAINST THE PUBLISHED FILES
-----------------------------------------------------
(1) FRAME NUMBERS ARE NOT ROW INDICES. Only frames in which the skeleton was
    tracked were recorded, so the numbering jumps and the effective rate is
    ~15 fps against a nominal 30. Slicing rows [start:end] instead of masking
    on frame number roughly DOUBLES every clip. Every slice here masks on the
    frame number column.
(2) ~2% OF FRAMES ARE DUPLICATED, carrying two skeleton records with slightly
    different joint values. The dataset page says either may be used; the
    first is kept and the drop count is reported.
(3) SEGMENTS OVERLAP AND RUN OUT OF ORDER. s01_e01 lists push before pull;
    s10_e02's push [1944,1994] and pull [1982,2062] genuinely share 13 frames.
    Reported per sequence, never silently merged or reordered away.
(4) JOINT ORDER IS THE KINECT SDK BETA ENUM (NUI_SKELETON_POSITION_INDEX), not
    MSR Action3D's order. Unlike DailyActivity3D this ordering is DOCUMENTED,
    so the permutation is known before the matcher runs. EXPECTED_SDK_PERM is
    that ground truth and derive_canonical_permutation is checked against it
    rather than trusted -- the only dataset in this project where a
    ground-truth check on the matcher is possible at all.

There is NO confidence / tracking column, so UTKinect cannot serve RQ4's
per-joint noise proxy. Action3D remains the only source for that.
"""
from __future__ import annotations

import re
import shutil
from pathlib import Path

import numpy as np

# Canonical modules win when har3d.ipynb has run in this session; the bundled
# fallback is used only when they are absent. Never both.
try:
    from har3d_io import extract_any, log
    import har3d_v2 as _G
    GEOM_SOURCE = 'har3d_v2 + har3d_io (canonical)'
except ImportError:
    import har3d_utk_geom as _G
    from har3d_utk_geom import extract_any, log
    GEOM_SOURCE = 'har3d_utk_geom (bundled fallback)'

# ---------------------------------------------------------------- constants
N_JOINTS = 20
EXPECTED_CLIPS = 199          # 200 nominal, minus s10_e02 carry
EXPECTED_RECORDINGS = 20      # 10 subjects x 2 episodes
MIN_CLIP_FRAMES = 5           # below this a clip cannot be resampled meaningfully
LABEL_URL = "https://cvrc.ece.utexas.edu/KinectDatasets/actionLabel.txt"

# File order in actionLabel.txt. Index+1 becomes the label, matching the other
# caches, because build_tensors() does meta[:, 0] - 1.
ACTIONS = ["walk", "sitDown", "standUp", "pickUp", "carry",
           "throw", "push", "pull", "waveHands", "clapHands"]
ACTION_ID = {a.lower(): i + 1 for i, a in enumerate(ACTIONS)}

# Kinect SDK Beta NUI_SKELETON_POSITION_INDEX -> this project's canonical order.
# Derived from the enum on paper, NOT fitted to data. perm[c] = q, applied as
# xyz_canonical = xyz_query[:, perm]. Identical to the permutation the matcher
# recovered for DailyActivity3D, which is what makes that result checkable.
EXPECTED_SDK_PERM = np.array(
    [4, 8, 2, 1, 12, 16, 0, 5, 9, 6, 10, 7, 11, 13, 17, 14, 18, 15, 19, 3])

SEQ_RE = re.compile(r"^\s*s(\d+)[_-]e(\d+)\s*$", re.IGNORECASE)
SEG_RE = re.compile(r"^\s*([A-Za-z]+)\s*:\s*(\S+)\s+(\S+)\s*$")
FILE_RE = re.compile(r"s(\d+)[_-]e(\d+)", re.IGNORECASE)

# Guard against two live definitions of the skeleton.
assert list(_G.MSR_PARENTS) == [2, 2, 3, 6, 6, 6, -1, 0, 1, 7,
                                8, 9, 10, 4, 5, 13, 14, 15, 16, 2], \
    "the geometry module's MSR_PARENTS is not the canonical corrected tree"


# ------------------------------------------------------------ label parsing
def fetch_labels(dest: Path, url: str = LABEL_URL) -> Path | None:
    """Download actionLabel.txt if it is not already beside the archive."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 100:
        return dest
    try:
        import urllib.request
        log(f"  actionLabel.txt absent -- fetching from {url}")
        urllib.request.urlretrieve(url, dest)
        log(f"  fetched {dest.stat().st_size} bytes")
        return dest
    except Exception as e:                       # offline, 404, blocked proxy
        log(f"  !! could not fetch actionLabel.txt: {type(e).__name__}: {e}")
        log("     Download it by hand and place it beside joints.zip.")
        return None


def parse_action_labels(path) -> tuple[list, list]:
    """
    actionLabel.txt -> (segments, skipped).

    Tokenised, never column-sliced: the shipped file has inconsistent spacing
    ('carry:  1868 2122') and trailing whitespace on many lines.
    """
    segments, skipped, cur = [], [], None
    text = Path(path).read_text(errors="ignore")
    for lineno, line in enumerate(text.splitlines(), 1):
        if not line.strip():
            continue
        m = SEQ_RE.match(line)
        if m:                                     # sNN_eNN header
            cur = (int(m.group(1)), int(m.group(2)))
            continue
        m = SEG_RE.match(line)
        if not m:
            skipped.append((lineno, line.strip(), "unparseable line"))
            continue
        if cur is None:
            skipped.append((lineno, line.strip(), "segment before any header"))
            continue
        name, a, b = m.group(1), m.group(2), m.group(3)
        aid = ACTION_ID.get(name.lower())
        if aid is None:
            skipped.append((lineno, line.strip(), f"unknown action {name!r}"))
            continue
        try:
            start, end = int(float(a)), int(float(b))
        except ValueError:                        # the s10_e02 'NaN NaN' carry
            skipped.append((lineno, line.strip(), "NaN / non-numeric range"))
            continue
        if end < start:
            start, end = end, start
        segments.append({"subject": cur[0], "episode": cur[1], "action": aid,
                         "name": name, "start": start, "end": end})
    return segments, skipped


def find_overlaps(segments) -> list:
    """Segment pairs sharing at least one frame. Reported, not repaired."""
    by_seq: dict = {}
    for s in segments:
        by_seq.setdefault((s["subject"], s["episode"]), []).append(s)
    out = []
    for key, segs in sorted(by_seq.items()):
        segs = sorted(segs, key=lambda s: s["start"])
        for a, b in zip(segs, segs[1:]):
            if b["start"] <= a["end"]:
                out.append({"seq": key, "a": a["name"], "b": b["name"],
                            "shared": a["end"] - b["start"] + 1})
    return out


# ------------------------------------------------------------ joint parsing
def read_joint_file(path, n_joints=N_JOINTS):
    """
    One recording -> (frame_numbers (T,), xyz (T, J, 3), n_bad_rows, n_dupes).

    Rows are `frame_no x1 y1 z1 ... x20 y20 z20` = 61 tokens. Anything else is
    counted and dropped rather than raised on, so one malformed line cannot
    take down the ingest. Duplicate frame numbers keep their first record.
    """
    want = 1 + 3 * n_joints
    frames, rows, bad = [], [], 0
    for line in Path(path).read_text(errors="ignore").splitlines():
        parts = line.split()
        if not parts:
            continue
        if len(parts) != want:
            bad += 1
            continue
        try:
            vals = [float(v) for v in parts]
        except ValueError:
            bad += 1
            continue
        frames.append(int(vals[0]))
        rows.append(vals[1:])

    if not rows:
        return (np.zeros(0, np.int64),
                np.zeros((0, n_joints, 3), np.float32), bad, 0)

    f = np.asarray(frames, np.int64)
    x = np.asarray(rows, np.float32).reshape(-1, n_joints, 3)
    _, first = np.unique(f, return_index=True)    # first record of each frame
    first.sort()
    n_dupes = len(f) - len(first)
    return f[first], x[first], bad, n_dupes


def index_recordings(root: Path) -> dict:
    """(subject, episode) -> path. Layout-agnostic, case-insensitive."""
    out = {}
    for p in Path(root).rglob("*.txt"):
        if "actionlabel" in p.name.lower():
            continue
        m = FILE_RE.search(p.name)
        if m:
            out[(int(m.group(1)), int(m.group(2)))] = p
    return out


# ------------------------------------------------------------------ ingest
def ingest(drive_root, cache_dir, work="/content/har3d_work/utkinect",
           push_to_drive=True, check_perm=True):
    """
    joints.zip + actionLabel.txt -> utkinect.npz, in the same ragged format as
    msr_action3d.npz and msr_daily.npz, so build_tensors() consumes it unchanged.

    Returns a result dict. Any 'error' key makes mark_done() refuse the stage.
    """
    drive_root, cache_dir, work = Path(drive_root), Path(cache_dir), Path(work)
    cache_dir.mkdir(parents=True, exist_ok=True)
    work.mkdir(parents=True, exist_ok=True)
    log(f"  geometry from: {GEOM_SOURCE}")

    # -- 1. locate and extract ------------------------------------------------
    zp = None
    for pat in ("joints.zip", "*joint*.zip", "*UTKinect*.zip", "*utkinect*.zip"):
        hits = sorted(drive_root.glob(pat))
        if hits:
            zp = hits[0]
            break
    if zp is None:
        return {"error": f"no joints archive under {drive_root} "
                         f"(expected joints.zip)"}

    if not index_recordings(work):
        try:
            tool = extract_any(zp, work)
            log(f"  extracted {zp.name} with {tool}")
        except Exception as e:
            return {"error": f"extraction failed for {zp.name}: {e}"}

    rec = index_recordings(work)
    log(f"  recordings found: {len(rec)} (expected {EXPECTED_RECORDINGS})")
    if not rec:
        present = sorted({p.suffix.lower() for p in work.rglob('*') if p.is_file()})
        return {"error": "no sNN_eNN .txt recordings after extraction",
                "suffixes_present": present}

    # -- 2. labels ------------------------------------------------------------
    lbl = fetch_labels(drive_root / "actionLabel.txt")
    if lbl is None:
        return {"error": "actionLabel.txt missing and not fetchable; "
                         "without it there are no labelled clips"}
    segments, skipped = parse_action_labels(lbl)
    log(f"  label segments: {len(segments)} usable, {len(skipped)} skipped")
    for s in skipped:
        log(f"    line {s[0]}: {s[1]!r} -- {s[2]}")

    overlaps = find_overlaps(segments)
    if overlaps:
        log(f"  !! {len(overlaps)} segment pairs share frames "
            f"(boundary frames appear in two clips):")
        for o in overlaps:
            log(f"    s{o['seq'][0]:02d}_e{o['seq'][1]:02d}: "
                f"{o['a']} / {o['b']} share {o['shared']} frame(s)")

    # -- 3. cut the clips -----------------------------------------------------
    xyz_all, lens, metas, flags = [], [], [], []
    dropped, tot_dupes, tot_bad = [], 0, 0
    cache_frames: dict = {}

    for seg in sorted(segments, key=lambda s: (s["subject"], s["episode"],
                                               s["action"])):
        key = (seg["subject"], seg["episode"])
        if key not in rec:
            dropped.append((seg["name"], key, "no recording file"))
            continue
        if key not in cache_frames:
            f, x, bad, dup = read_joint_file(rec[key])
            cache_frames[key] = (f, x)
            tot_dupes += dup
            tot_bad += bad
        f, x = cache_frames[key]
        if len(f) == 0:
            dropped.append((seg["name"], key, "recording parsed to 0 frames"))
            continue

        # THE critical line: mask on frame NUMBER, never on row index.
        m = (f >= seg["start"]) & (f <= seg["end"])
        clip = x[m]
        if len(clip) < MIN_CLIP_FRAMES:
            dropped.append((seg["name"], key,
                            f"only {len(clip)} frames in [{seg['start']},"
                            f"{seg['end']}]"))
            continue

        zero = (np.abs(clip) < 1e-9).all(-1)
        xyz_all.append(clip)
        lens.append(len(clip))
        metas.append([seg["action"], seg["subject"], seg["episode"]])
        flags.append([len(clip),
                      float(1.0 - np.isfinite(clip).all(-1).mean()),
                      float(zero.mean()),
                      float("nan")])           # no confidence column exists

    if not xyz_all:
        return {"error": "every segment failed to cut", "dropped": dropped[:10]}

    log(f"  cut {len(xyz_all)} clips "
        f"({tot_dupes} duplicate frames dropped, {tot_bad} malformed rows)")
    for d in dropped:
        log(f"    dropped {d[0]} s{d[1][0]:02d}_e{d[1][1]:02d}: {d[2]}")

    # -- 4. parse plausibility, BEFORE any permutation ------------------------
    probe = np.concatenate(xyz_all[:60], 0)
    score, med = _G.score_parse(probe)
    ext = _G.body_extent(probe)
    log(f"  parse score {score:.2f} of 1.00, median edge {med:.3f} m, "
        f"body span {ext[1]:.2f} m")
    if score < 0.8:
        return {"error": f"parse does not look like a human skeleton "
                         f"(score {score:.2f}, median edge {med:.3f} m). "
                         f"Inspect a raw file before going further."}

    # -- 5. joint order, checked against the documented SDK enum --------------
    perm = EXPECTED_SDK_PERM.copy()
    perm_info = {"source": "SDK enum (assumed, UNCHECKED)"}
    ref = cache_dir / "msr_action3d.npz"
    if ref.exists():
        rz = np.load(ref, allow_pickle=False)
        ro = rz["offsets"]
        ref_xyz = rz["xyz"][ro[0]:ro[min(120, len(ro) - 1)]]
        recovered, perm_info = _G.derive_canonical_permutation(probe, ref_xyz)
        perm_info["source"] = "distance-signature matching"
        agrees = bool(np.array_equal(recovered, EXPECTED_SDK_PERM))
        perm_info["matches_sdk_enum"] = agrees
        if agrees:
            log("  joint order: matcher output EQUALS the documented Kinect SDK "
                "enum permutation -- ground-truth agreement")
        else:
            log(f"  !! matcher returned {recovered.tolist()}")
            log(f"     SDK enum predicts  {EXPECTED_SDK_PERM.tolist()}")
            if check_perm:
                return {"error": "recovered permutation disagrees with the "
                                 "documented Kinect SDK joint order. One of the "
                                 "two is wrong -- do NOT relax this to proceed."}
            log("     check_perm=False, using the matcher's answer")
        perm = recovered
    else:
        log("  !! msr_action3d.npz absent -- applying the SDK enum permutation "
            "UNCHECKED. Run har3d.ipynb Stage 2 for the cross-check.")

    xyz_all = [c[:, perm] for c in xyz_all]

    # -- 6. topology gate on the permuted result ------------------------------
    try:
        _G.verify_topology(xyz_all)
    except AssertionError as e:
        return {"error": f"topology gate failed after permutation: "
                         f"{str(e).splitlines()[0]}"}

    # -- 7. write the cache ---------------------------------------------------
    out = cache_dir / "utkinect.npz"
    n = len(metas)
    np.savez_compressed(
        out,
        xyz=np.concatenate(xyz_all, 0).astype(np.float32),
        conf=np.full((sum(lens), N_JOINTS), np.nan, np.float32),
        has_conf=np.zeros(n, bool),          # no confidence channel in this set
        offsets=np.concatenate([[0], np.cumsum(lens)]).astype(np.int64),
        meta=np.array(metas, np.int64),
        flags=np.array(flags, np.float64),
        joint_perm=np.asarray(perm))
    log(f"  cached -> {out.name} ({out.stat().st_size / 1e6:.1f} MB, "
        f"{n} clips, {sum(lens):,} frames)")

    if push_to_drive:
        (drive_root / "cache").mkdir(parents=True, exist_ok=True)
        shutil.copy2(out, drive_root / "cache" / out.name)
        log(f"  copied to {drive_root / 'cache' / out.name}")

    res = {"n_sequences": n, "n_frames": int(sum(lens)),
           "subjects": sorted({m[1] for m in metas}),
           "n_actions": len({m[0] for m in metas}),
           "duplicate_frames_dropped": tot_dupes,
           "malformed_rows": tot_bad,
           "overlapping_segment_pairs": len(overlaps),
           "joint_order": perm_info, "cache": str(out)}
    if n != EXPECTED_CLIPS:
        res["warning"] = f"{n} clips, expected {EXPECTED_CLIPS}"
        log(f"  !! {res['warning']}")
    if dropped:
        res["dropped"] = dropped[:10]
    return res


In [ ]:
%%writefile /content/har3d_models_v2.py
"""
har3d_models_v2.py -- ST-GCN + training for the small datasets, on the
corrected topology.

Training protocol (final version, revised after review):

  * EARLY STOPPING on validation NLL, not a fixed epoch count. Each model is
    trained for up to `max_epochs`, the learning rate is halved on a validation
    plateau, and the weights with the lowest validation NLL are restored. Every
    model is therefore evaluated at its own best convergence point, so dataset
    size is not confounded with how over- or under-trained the model is.
    Validation NLL is the selection criterion for every loss (CE, label
    smoothing, focal) because it is a proper scoring rule.
  * MC DROPOUT with the inference dropout rate TUNED on validation. The
    training rate (0.3) is a regulariser; reusing it for MC inference on a few
    hundred samples can starve the network of capacity and flatten
    confidences. The MC rate is chosen from a grid by validation NLL of the
    MC-averaged prediction, and the chosen rate is recorded.
  * loss -- 'ce' | 'ls' (label smoothing) | 'focal'.
  * val+test logits always returned.

Topology is imported from har3d_v2 so there is exactly one definition.
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from har3d_v2 import MSR_PARENTS, adjacency


class STGCNBlock(nn.Module):
    def __init__(self, cin, cout, A, stride=1, residual=True):
        super().__init__()
        self.register_buffer("A", torch.as_tensor(A, dtype=torch.float32))
        self.K = A.shape[0]
        self.gconv = nn.Conv2d(cin, cout * self.K, 1)
        self.tconv = nn.Sequential(
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(cout))
        self.res = (nn.Identity() if (residual and cin == cout and stride == 1)
                    else (nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)),
                                        nn.BatchNorm2d(cout)) if residual else None))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        r = self.res(x) if self.res is not None else 0
        N, _, T, J = x.shape
        y = self.gconv(x).view(N, self.K, -1, T, J)
        y = torch.einsum("nkctv,kvw->nctw", y, self.A)
        return self.relu(self.tconv(y) + r)


class STGCN(nn.Module):
    def __init__(self, num_classes, A, in_ch=3, base=64, dropout=0.3):
        super().__init__()
        self.bn = nn.BatchNorm1d(in_ch * A.shape[1])
        self.layers = nn.ModuleList([
            STGCNBlock(in_ch, base, A, residual=False),
            STGCNBlock(base, base, A),
            STGCNBlock(base, base * 2, A, stride=2),
            STGCNBlock(base * 2, base * 2, A)])
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(base * 2, num_classes)

    def forward(self, x):                       # (N, C, T, J)
        N, C, T, J = x.shape
        x = self.bn(x.permute(0, 1, 3, 2).reshape(N, C * J, T))
        x = x.view(N, C, J, T).permute(0, 1, 3, 2)
        for l in self.layers:
            x = l(x)
        return self.fc(self.drop(F.adaptive_avg_pool2d(x, 1).flatten(1)))


def _loss_fn(kind, smoothing=0.1, gamma=2.0):
    if kind == "ce":
        return lambda o, y: F.cross_entropy(o, y)
    if kind == "ls":
        return lambda o, y: F.cross_entropy(o, y, label_smoothing=smoothing)
    if kind == "focal":
        def focal(o, y):
            logp = F.log_softmax(o, -1)
            lp = logp[torch.arange(len(y)), y]
            return (-(1 - lp.exp()) ** gamma * lp).mean()
        return focal
    raise ValueError(f"unknown loss {kind!r}")


@torch.no_grad()
def _forward_all(model, X, idx, bs=128, device="cpu"):
    return np.concatenate(
        [model(torch.as_tensor(X[idx[b:b + bs]]).to(device)).cpu().numpy()
         for b in range(0, len(idx), bs)], 0).astype(np.float32)


def _state_copy(model):
    return {k: v.detach().clone() for k, v in model.state_dict().items()}


@torch.no_grad()
def _val_nll(model, X, idx, y, device, bs=256):
    model.eval()
    L = _forward_all(model, X, idx, bs=bs, device=device)
    z = torch.as_tensor(L, dtype=torch.float64)
    return float(F.cross_entropy(z, torch.as_tensor(y[idx])).item())


def _set_dropout(model, p=None, train=True):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            if p is not None:
                m.p = float(p)
            m.train(train)


def mc_tuned(model, X, y, split, passes, p_grid, seed, device):
    """
    MC dropout with the inference rate chosen on validation NLL.
    Returns ({'val_mc': (P,n,C), 'test_mc': ...}, chosen_p, {p: val_nll}).
    Batch-norm stays in eval mode, so only dropout is stochastic.
    """
    model.eval()
    vidx = split["val"]
    scores = {}
    for gi, p in enumerate(p_grid):
        _set_dropout(model, p, train=True)
        torch.manual_seed(seed + 10_000 + gi)
        P = np.stack([_forward_all(model, X, vidx, device=device)
                      for _ in range(passes)], 0)
        z = P - P.max(-1, keepdims=True)
        pr = np.exp(z); pr /= pr.sum(-1, keepdims=True)
        pm = pr.mean(0)
        scores[float(p)] = float(-np.log(np.clip(pm[np.arange(len(vidx)), y[vidx]],
                                                 1e-12, 1)).mean())
    best = min(scores, key=scores.get)
    _set_dropout(model, best, train=True)
    out = {}
    for sub in ("val", "test"):
        idx = split[sub]
        torch.manual_seed(seed + 20_000)
        out[sub + "_mc"] = (np.stack([_forward_all(model, X, idx, device=device)
                                      for _ in range(passes)], 0), y[idx])
    _set_dropout(model, None, train=False)
    return out, best, scores


def train_and_logits(X, y, split, num_classes, A=None, max_epochs=120, bs=32,
                     lr=1e-3, seed=0, device=None, verbose=True, loss="ce",
                     smoothing=0.1, gamma=2.0, mc_passes=0, dropout=0.3,
                     early_stop=True, patience=15, min_epochs=10,
                     mc_p_grid=(0.05, 0.1, 0.2, 0.3), epochs=None):
    """
    Train on split['train'] with early stopping on split['val'] NLL; return
    logits for val and test.

    Returns (out, model): out['val'|'test'] = (logits, labels); with
    mc_passes > 0 also out['val_mc'|'test_mc'] = (logits (P,n,C), labels);
    out['info'] records best_epoch, epochs_run, best_val_nll and mc_p.
    `epochs` (legacy) forces a fixed epoch count with no early stopping.
    Deterministic given `seed`.
    """
    if A is None:
        A = adjacency(MSR_PARENTS)
    if epochs is not None:
        max_epochs, early_stop = int(epochs), False
    if early_stop and len(split.get("val", [])) == 0:
        raise ValueError("early stopping needs a non-empty validation split")
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = STGCN(num_classes, A, in_ch=X.shape[1], dropout=dropout).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    if early_stop:
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min",
                                                         factor=0.5, patience=5)
    else:
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max_epochs)
    crit = _loss_fn(loss, smoothing, gamma)

    Xtr = torch.as_tensor(X[split["train"]])
    ytr = torch.as_tensor(y[split["train"]])

    best_nll, best_state, best_ep, since = np.inf, None, -1, 0
    ep_run = 0
    for ep in range(max_epochs):
        model.train()
        perm = torch.randperm(len(Xtr))
        run, tot = 0.0, 0
        for b in range(0, len(perm), bs):
            i = perm[b:b + bs]
            xb, yb = Xtr[i].to(device), ytr[i].to(device)
            opt.zero_grad(set_to_none=True)
            o = model(xb)
            l = crit(o, yb)
            l.backward()
            opt.step()
            run += float(l.detach()) * len(i)
            tot += len(i)
        ep_run = ep + 1
        if early_stop:
            v = _val_nll(model, X, split["val"], y, device)
            sch.step(v)
            if v < best_nll - 1e-6:
                best_nll, best_state, best_ep, since = v, _state_copy(model), ep_run, 0
            else:
                since += 1
            if since >= patience and ep_run >= min_epochs:
                break
        else:
            sch.step()
        if verbose and ep_run % 20 == 0:
            print(f"      ep {ep_run:3d}  train loss {run/tot:.4f}"
                  + (f"  val nll {v:.4f}" if early_stop else ""))

    if early_stop and best_state is not None:
        model.load_state_dict(best_state)
    else:
        best_ep = ep_run
        best_nll = (_val_nll(model, X, split["val"], y, device)
                    if len(split.get("val", [])) else float("nan"))

    model.eval()
    out = {}
    for sub in ("val", "test"):
        idx = split.get(sub, np.array([], int))
        if len(idx) == 0:
            continue
        L = _forward_all(model, X, idx, device=device)
        out[sub] = (L, y[idx])
        if verbose:
            print(f"      {sub:<5} n={len(idx):5d}  top1 {(L.argmax(-1)==y[idx]).mean():.4f}")

    info = {"best_epoch": int(best_ep), "epochs_run": int(ep_run),
            "best_val_nll": float(best_nll), "early_stop": bool(early_stop),
            "mc_p": None, "mc_p_scores": None}
    if mc_passes:
        mc, p_best, scores = mc_tuned(model, X, y, split, mc_passes, mc_p_grid,
                                      seed, device)
        out.update(mc)
        info["mc_p"], info["mc_p_scores"] = p_best, scores
        if verbose:
            print(f"      mc-dropout: {mc_passes} passes, inference p={p_best} "
                  f"(tuned on val NLL)")
    out["info"] = info
    if verbose:
        print(f"      best epoch {best_ep}/{ep_run}  best val NLL {best_nll:.4f}")
    return out, model


In [ ]:
%%writefile /content/har3d_backbones.py
"""
har3d_backbones.py -- the three graph-convolutional backbones used on the
NTU-60 ladder, built on ONE shared trunk so that only the graph layer differs.

    stgcn   ST-GCN (Yan et al., AAAI 2018): a FIXED, hand-defined adjacency.
    agcn    2s-AGCN graph layer (Shi et al., CVPR 2019): fixed adjacency
            + a freely learned graph B + a data-dependent similarity C(x).
    ctrgcn  CTR-GCN graph layer (Chen et al., ICCV 2021): channel-wise
            topology refinement -- a learnable shared topology refined per
            channel from pairwise feature differences.

Shared by all three: four blocks (widths 64, 64, 128, 128; stride 2 in block
3), a 9x1 temporal convolution with batch norm, residual connections, a
batch-norm on the input, mean pooling over time, joints and the two persons,
dropout 0.3, and a linear head. Holding the trunk fixed is deliberate: the
ladder compares GRAPH MECHANISMS at matched depth, width and training recipe,
not published end-to-end systems (CTR-GCN's multi-scale temporal module and
2s-AGCN's two-stream ensembling are intentionally not reproduced).

`graph_parameters(model)` returns the parameters that generate the adaptive
graph, which the adjacency-freezing ablation stops updating mid-training.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

from har3d_v2 import adjacency

ARCHS = ("stgcn", "agcn", "agcn_frozen", "ctrgcn")


# ---------------------------------------------------------------------------
# temporal unit shared by all blocks
# ---------------------------------------------------------------------------
def _tconv(cout, stride):
    return nn.Sequential(nn.Conv2d(cout, cout, (9, 1), (stride, 1), (4, 0)),
                         nn.BatchNorm2d(cout))


def _residual(cin, cout, stride, residual):
    if not residual:
        return None
    if cin == cout and stride == 1:
        return nn.Identity()
    return nn.Sequential(nn.Conv2d(cin, cout, 1, (stride, 1)), nn.BatchNorm2d(cout))


# ---------------------------------------------------------------------------
# ST-GCN block: fixed adjacency
# ---------------------------------------------------------------------------
class STGCNBlock(nn.Module):
    def __init__(self, cin, cout, A, stride=1, residual=True):
        super().__init__()
        self.K = A.shape[0]
        self.register_buffer("A", torch.as_tensor(A, dtype=torch.float32))
        self.gconv = nn.Conv2d(cin, cout * self.K, 1)
        self.bn = nn.BatchNorm2d(cout)
        self.tconv = _tconv(cout, stride)
        self.res = _residual(cin, cout, stride, residual)

    def forward(self, x):                              # (N, C, T, J)
        N, _, T, J = x.shape
        r = self.res(x) if self.res is not None else 0
        y = self.gconv(x).view(N, self.K, -1, T, J)
        y = torch.einsum("nkctv,kvw->nctw", y, self.A)
        y = F.relu(self.bn(y))
        return F.relu(self.tconv(y) + r)


# ---------------------------------------------------------------------------
# 2s-AGCN block: A + B + alpha * C(x)
# ---------------------------------------------------------------------------
class AGCNBlock(nn.Module):
    def __init__(self, cin, cout, A, stride=1, residual=True, embed=None):
        super().__init__()
        self.K = A.shape[0]
        self.register_buffer("A", torch.as_tensor(A, dtype=torch.float32))
        self.B = nn.Parameter(torch.zeros(self.K, A.shape[1], A.shape[2]))
        self.embed = embed or max(cout // 4, 8)
        self.theta = nn.Conv2d(cin, self.embed * self.K, 1)
        self.phi = nn.Conv2d(cin, self.embed * self.K, 1)
        self.alpha = nn.Parameter(torch.zeros(1))
        self.gconv = nn.Conv2d(cin, cout * self.K, 1)
        self.bn = nn.BatchNorm2d(cout)
        self.tconv = _tconv(cout, stride)
        self.res = _residual(cin, cout, stride, residual)

    def forward(self, x):
        N, _, T, J = x.shape
        r = self.res(x) if self.res is not None else 0
        th = self.theta(x).view(N, self.K, self.embed, T, J).mean(3)
        ph = self.phi(x).view(N, self.K, self.embed, T, J).mean(3)
        C = torch.softmax(torch.einsum("nkej,nkel->nkjl", th, ph)
                          / (self.embed ** 0.5), dim=-1)
        Aeff = self.A.unsqueeze(0) + self.B.unsqueeze(0) + self.alpha * C
        y = self.gconv(x).view(N, self.K, -1, T, J)
        y = torch.einsum("nkctv,nkvw->nctw", y, Aeff)
        y = F.relu(self.bn(y))
        return F.relu(self.tconv(y) + r)


# ---------------------------------------------------------------------------
# CTR-GCN block: channel-wise topology refinement
# ---------------------------------------------------------------------------
class CTRGC(nn.Module):
    """One CTR graph convolution for one adjacency partition (Chen et al. 2021)."""

    def __init__(self, cin, cout, rel_reduction=8):
        super().__init__()
        rel = 8 if cin <= 16 else cin // rel_reduction
        self.conv1 = nn.Conv2d(cin, rel, 1)
        self.conv2 = nn.Conv2d(cin, rel, 1)
        self.conv3 = nn.Conv2d(cin, cout, 1)
        self.conv4 = nn.Conv2d(rel, cout, 1)

    def forward(self, x, A, alpha):                    # x (N, C, T, V)
        x1 = self.conv1(x).mean(-2)                    # (N, rel, V)
        x2 = self.conv2(x).mean(-2)
        x3 = self.conv3(x)                             # (N, cout, T, V)
        rel = torch.tanh(x1.unsqueeze(-1) - x2.unsqueeze(-2))   # (N, rel, V, V)
        topo = self.conv4(rel) * alpha + A.unsqueeze(0).unsqueeze(0)
        return torch.einsum("ncuv,nctv->nctu", topo, x3)


class CTRGCNBlock(nn.Module):
    def __init__(self, cin, cout, A, stride=1, residual=True):
        super().__init__()
        self.K = A.shape[0]
        self.PA = nn.Parameter(torch.as_tensor(A, dtype=torch.float32).clone())
        self.convs = nn.ModuleList([CTRGC(cin, cout) for _ in range(self.K)])
        self.alpha = nn.Parameter(torch.zeros(1))
        self.bn = nn.BatchNorm2d(cout)
        nn.init.constant_(self.bn.weight, 1e-6)        # start near the residual
        self.down = (nn.Identity() if cin == cout else
                     nn.Sequential(nn.Conv2d(cin, cout, 1), nn.BatchNorm2d(cout)))
        self.tconv = _tconv(cout, stride)
        self.res = _residual(cin, cout, stride, residual)

    def forward(self, x):
        r = self.res(x) if self.res is not None else 0
        y = sum(self.convs[k](x, self.PA[k], self.alpha) for k in range(self.K))
        y = F.relu(self.bn(y) + self.down(x))
        return F.relu(self.tconv(y) + r)


_BLOCKS = {"stgcn": STGCNBlock, "agcn": AGCNBlock, "agcn_frozen": AGCNBlock,
           "ctrgcn": CTRGCNBlock}

# name fragments of the parameters that GENERATE the graph, per architecture
_GRAPH_PARAM_KEYS = {
    "stgcn": (),
    "agcn": (".B", ".alpha", ".theta.", ".phi."),
    "agcn_frozen": (".B", ".alpha", ".theta.", ".phi."),
    "ctrgcn": (".PA", ".alpha", ".conv1.", ".conv2.", ".conv4."),
}


class GraphNet(nn.Module):
    """Shared trunk. Input (N, C, T, J, M); persons folded into the batch."""

    def __init__(self, arch, num_classes, A, in_ch=3, base=64, dropout=0.3):
        super().__init__()
        if arch not in _BLOCKS:
            raise ValueError(f"unknown arch {arch!r}; choose from {ARCHS}")
        self.arch = arch
        Blk = _BLOCKS[arch]
        self.bn = nn.BatchNorm1d(in_ch * A.shape[1])
        self.layers = nn.ModuleList([
            Blk(in_ch, base, A, residual=False),
            Blk(base, base, A),
            Blk(base, base * 2, A, stride=2),
            Blk(base * 2, base * 2, A)])
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(base * 2, num_classes)

    def forward(self, x):
        N, C, T, J, M = x.shape
        x = x.permute(0, 4, 1, 2, 3).reshape(N * M, C, T, J)
        x = self.bn(x.permute(0, 1, 3, 2).reshape(N * M, C * J, T))
        x = x.view(N * M, C, J, T).permute(0, 1, 3, 2)
        for layer in self.layers:
            x = layer(x)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        x = x.view(N, M, -1).mean(1)
        return self.fc(self.drop(x))


def build(arch, num_classes, parents, in_ch=3, base=64, dropout=0.3):
    return GraphNet(arch, num_classes, adjacency(parents), in_ch, base, dropout)


def graph_parameters(model):
    """Parameters that generate the (adaptive) graph; empty for ST-GCN."""
    keys = _GRAPH_PARAM_KEYS[model.arch]
    out = []
    for name, prm in model.named_parameters():
        dotted = "." + name
        if any(dotted.endswith(k) or k in dotted for k in keys):
            out.append(prm)
    return out


def freeze_graph(model):
    """Stop updating the adaptive graph. Returns the number of tensors frozen."""
    ps = graph_parameters(model)
    for prm in ps:
        prm.requires_grad_(False)
        prm.grad = None
    return len(ps)


def n_params(model):
    return sum(p.numel() for p in model.parameters())


In [ ]:
%%writefile /content/har3d_scale.py
"""
har3d_scale.py -- the NTU-60 TRAINING-SET LADDER. This is the scale axis.

WHY A LADDER AND NOT FOUR DATASETS
----------------------------------
Comparing calibration on small datasets (MSR, n ~ 500) with a large one
(NTU, n ~ 56,000) confounds two things: small TRAIN n (which may genuinely
change calibration) and small TEST n (which inflates the ECE estimator). The
two predict the same curve, so the scale claim is not identifiable. The ladder
holds architecture, recipe, preprocessing and the TEST SET fixed and varies
only the number of training samples, so the estimator floor is constant across
rungs and any movement in the excess ECE is attributable to the model.

CONVERGENCE CONTROL (revised after review)
------------------------------------------
Every rung is trained with EARLY STOPPING on validation NLL: up to `max_steps`
gradient steps, validation every `eval_every` steps, learning rate halved on a
plateau, and the best-validation-NLL weights restored. Each rung is therefore
measured at its own optimal convergence point. The earlier protocol used a
fixed 6,000-step budget for every rung, which gives small rungs hundreds of
epochs (over-fitting) and large rungs a dozen (under-convergence) -- a
confound between scale and convergence state. That protocol is kept as an
explicitly labelled ABLATION (early_stop=False) so the paper can show whether
the scale conclusion survives both.

Validation is a subject-disjoint slice of the X-Sub TRAIN subjects, identical
for every backbone and every rung.
"""
from __future__ import annotations

import time

import numpy as np

# NTU 25-joint kinematic tree, 0-indexed, root at 20 (spine-shoulder).
NTU_PARENTS = [1, 20, 20, 2, 20, 4, 5, 6, 20, 8, 9, 10, 0, 12, 13, 14,
               0, 16, 17, 18, -1, 22, 7, 24, 11]
NTU_JOINTS = 25
NTU_PERSONS = 2

LADDER_DEFAULT = [500, 1000, 2000, 5000, 10000, 20000, 40091]


# =====================================================================
# 1. Tensors
# =====================================================================
def load_ntu_npz(path):
    """
    CTR-GCN's NTU60_CS.npz -> raw arrays. Layout is (N, 300, 150), NOT the
    (N, 3, 300, 25, 2) usually quoted; 150 = 2 persons x 25 joints x 3 coords.
    """
    z = np.load(path, allow_pickle=False)
    return (z["x_train"], z["y_train"].argmax(-1) if z["y_train"].ndim > 1
            else z["y_train"],
            z["x_test"], z["y_test"].argmax(-1) if z["y_test"].ndim > 1
            else z["y_test"])


def reshape_ntu(x):
    """(N, 300, 150) -> (N, 3, 300, 25, 2), matching CTR-GCN's own feeder."""
    N, T, D = x.shape
    assert D == NTU_PERSONS * NTU_JOINTS * 3, f"unexpected trailing dim {D}"
    return x.reshape(N, T, NTU_PERSONS, NTU_JOINTS, 3).transpose(0, 4, 1, 3, 2)


def valid_length(seq):
    """Frames before the zero padding. seq is (3, T, J, M)."""
    nz = np.abs(seq).sum((0, 2, 3)) > 1e-9
    idx = np.where(nz)[0]
    return (int(idx[0]), int(idx[-1]) + 1) if len(idx) else (0, 1)


def valid_resize(seq, T=64):
    """Crop the zero padding, then uniformly resample to T frames."""
    a, b = valid_length(seq)
    s = seq[:, a:b]
    n = s.shape[1]
    if n == T:
        return s
    idx = np.clip(np.round(np.linspace(0, n - 1, T)).astype(int), 0, max(n - 1, 0))
    return s[:, idx]


def to_stream(x, stream, parents=NTU_PARENTS):
    """
    (C, T, J, M) -> stream tensor of the same shape.
    Applied AFTER valid_resize so all four streams share one convention.
    """
    x = np.asarray(x, np.float32)
    if stream.startswith("bone"):
        out = np.zeros_like(x)
        for j, p in enumerate(parents):
            out[:, :, j] = x[:, :, j] - x[:, :, p if p >= 0 else j]
        x = out
    elif not stream.startswith("joint"):
        raise ValueError(f"unknown stream {stream!r}")
    if stream.endswith("motion"):
        m = np.zeros_like(x)
        m[:, 1:] = x[:, 1:] - x[:, :-1]
        x = m
    return x


def prepare_ntu_tensors(x_raw, stream="joint", T=64, parents=NTU_PARENTS,
                        batch=2000, verbose=True):
    """
    (N, 300, 150) -> (N, 3, T, 25, 2) float32 for one stream.

    Processed in batches because the full x_train is 7.2 GB as float32 and
    materialising a second copy OOMs a 13 GB runtime.
    """
    N = len(x_raw)
    out = np.zeros((N, 3, T, NTU_JOINTS, NTU_PERSONS), np.float32)
    for b0 in range(0, N, batch):
        b1 = min(b0 + batch, N)
        blk = reshape_ntu(np.asarray(x_raw[b0:b1], np.float32))
        for i in range(b1 - b0):
            out[b0 + i] = to_stream(valid_resize(blk[i], T), stream, parents)
        if verbose and (b0 // batch) % 5 == 0:
            print(f"      {b1}/{N} sequences", flush=True)
        del blk
    return out


def subsample_indices(y, n, seed=0):
    """
    Stratified subsample of size n. Every class keeps its share, so a small
    rung is not accidentally a class-imbalance experiment as well.
    """
    y = np.asarray(y)
    rng = np.random.default_rng(seed)
    K = int(y.max()) + 1
    per = max(1, n // K)
    picked = []
    for c in range(K):
        idx = np.where(y == c)[0]
        if len(idx) == 0:
            continue
        take = min(per, len(idx))
        picked.append(rng.choice(idx, take, replace=False))
    picked = np.concatenate(picked)
    if len(picked) < n:                        # top up from what is left
        rest = np.setdiff1d(np.arange(len(y)), picked)
        extra = rng.choice(rest, min(n - len(picked), len(rest)), replace=False)
        picked = np.concatenate([picked, extra])
    rng.shuffle(picked)
    return np.sort(picked[:n])


# =====================================================================
# 2. Models -- see har3d_backbones (one shared trunk, three graph layers)
# =====================================================================
def make_model(num_classes, parents=NTU_PARENTS, in_ch=3, base=64,
               dropout=0.3, arch="stgcn"):
    import har3d_backbones as HB
    return HB.build(arch, num_classes, parents, in_ch=in_ch, base=base,
                    dropout=dropout)


# =====================================================================
# 3. One rung
# =====================================================================
def train_rung(Xtr, ytr, Xval, yval, Xte, yte, num_classes, arch="stgcn",
               early_stop=True, max_steps=15000, eval_every=250, patience=8,
               min_steps=500, fixed_steps=6000, bs=64, lr=1e-3, seed=0,
               device=None, parents=NTU_PARENTS, dropout=0.3, loss="ce",
               freeze_graph_after=None, verbose=True, log_every=1000,
               snapshot_steps=None, track_best_acc=False, run_until=0):
    """
    Train one ladder rung and return logits.

    early_stop=True  (main protocol) -- up to `max_steps` steps; validation NLL
                     every `eval_every` steps; LR halved after 3 evaluations
                     without improvement; stop after `patience` evaluations
                     without improvement (never before `min_steps`); restore
                     the best-validation-NLL weights.
    early_stop=False (ablation)      -- exactly `fixed_steps` steps with a
                     cosine schedule, the protocol used before the review.
    freeze_graph_after=S             -- after step S, stop updating the
                     parameters that generate the adaptive graph (the
                     adjacency-freezing ablation for 2s-AGCN).

    snapshot_steps=(s1, s2, ...)     -- also keep the TEST logits of the weights
                     at these steps (multiples of `eval_every`), so backbones
                     can be compared at matched training durations.
    track_best_acc=True              -- also keep the TEST logits of the weights
                     with the best validation ACCURACY (an alternative
                     stopping rule), over every evaluation that was made.
    run_until=S                      -- when early stopping fires before step S,
                     keep training to step S so that the snapshots up to S
                     exist for every backbone. The early-stopping decision
                     and the best-validation-NLL weights are frozen at the
                     moment the rule fires, so nothing in the main output
                     changes; info['steps_run'] stays the step at which the
                     rule fired and info['steps_total'] is the last step.
                     None of the three options touches the optimiser, the
                     scheduler or any random stream before the stopping
                     point: the main output is identical with and without
                     them (tested).

    Returns {'val': (logits, y), 'test': (logits, y), 'info': {...}} and, when
    either option is set, 'extra': {tag: {'logits', 'step', 'val_nll',
    'val_top1'}} with tag 's<step>' or 'acc'.
    Minibatches are drawn with replacement from a seeded generator, so a run
    is deterministic given `seed`.
    """
    import torch
    import torch.nn.functional as F

    import har3d_backbones as HB
    from har3d_models_v2 import _loss_fn

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = make_model(num_classes, parents, in_ch=Xtr.shape[1],
                       dropout=dropout, arch=arch).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    total = max_steps if early_stop else fixed_steps
    if early_stop:
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min",
                                                         factor=0.5, patience=3)
    else:
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, fixed_steps)
    crit = _loss_fn(loss)

    Xt, yt = torch.as_tensor(Xtr), torch.as_tensor(ytr)
    n = len(Xt)
    rng = np.random.default_rng(seed)

    @torch.no_grad()
    def fwd(X, bs_eval=256):
        model.eval()
        return np.concatenate(
            [model(torch.as_tensor(X[b:b + bs_eval]).to(device)).cpu().numpy()
             for b in range(0, len(X), bs_eval)], 0).astype(np.float32)

    def val_stats():
        zv = fwd(Xval)
        z = torch.as_tensor(zv, dtype=torch.float64)
        return (float(F.cross_entropy(z, torch.as_tensor(yval)).item()),
                float((zv.argmax(-1) == np.asarray(yval)).mean()))

    def val_nll():
        return val_stats()[0]

    snapshot_steps = set(int(x) for x in (snapshot_steps or ()))
    extra = {}
    best_acc, best_acc_state, best_acc_step, best_acc_nll = -1.0, None, 0, np.nan
    best, best_state, best_step, since = np.inf, None, 0, 0
    frozen_at, n_frozen, curve = None, 0, []
    stop_step = None                  # step at which the early-stopping rule fired
    t0 = time.time()
    model.train()
    run, seen, corr = 0.0, 0, 0
    step = 0
    for step in range(1, total + 1):
        if freeze_graph_after is not None and step == freeze_graph_after + 1:
            n_frozen = HB.freeze_graph(model)
            frozen_at = freeze_graph_after
            if verbose:
                print(f"      graph frozen after step {frozen_at} "
                      f"({n_frozen} parameter tensors)", flush=True)
        model.train()
        i = torch.as_tensor(rng.integers(0, n, min(bs, n)))
        xb, yb = Xt[i].to(device), yt[i].to(device)
        opt.zero_grad(set_to_none=True)
        out = model(xb)
        l = crit(out, yb)
        l.backward()
        opt.step()
        if not early_stop:
            sch.step()
        run += float(l.detach()) * len(i)
        corr += int((out.argmax(-1) == yb).sum())
        seen += len(i)
        if verbose and step % log_every == 0:
            print(f"      step {step:6d}  loss {run/seen:.4f}  "
                  f"batch acc {corr/seen:.4f}", flush=True)
            run, seen, corr = 0.0, 0, 0
        if early_stop and step % eval_every == 0:
            v, a = val_stats()
            curve.append((step, v))
            sch.step(v)
            if track_best_acc and a > best_acc + 1e-9:
                best_acc, best_acc_step, best_acc_nll = a, step, v
                best_acc_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
            if step in snapshot_steps:
                extra[f"s{step}"] = {"logits": fwd(Xte), "step": int(step),
                                     "val_nll": float(v), "val_top1": float(a)}
            if stop_step is None:
                if v < best - 1e-6:
                    best, best_state, best_step, since = (
                        v, {k: t.detach().clone() for k, t in model.state_dict().items()},
                        step, 0)
                else:
                    since += 1
                if since >= patience and step >= min_steps:
                    stop_step = step
            if stop_step is not None and step >= run_until:
                break

    if early_stop and best_state is not None:
        model.load_state_dict(best_state)
    else:
        best_step, best = step, val_nll()

    out = {"val": (fwd(Xval), np.asarray(yval)),
           "test": (fwd(Xte), np.asarray(yte))}
    out["info"] = {"arch": arch, "early_stop": bool(early_stop),
                   "steps_run": int(stop_step if stop_step is not None else step),
                   "steps_total": int(step), "best_step": int(best_step),
                   "best_val_nll": float(best), "frozen_at": frozen_at,
                   "n_frozen": int(n_frozen), "epochs_at_best":
                   float(best_step * min(bs, n) / n), "seconds": time.time() - t0,
                   "val_curve": curve}
    if track_best_acc and best_acc_state is not None:
        model.load_state_dict(best_acc_state)
        extra["acc"] = {"logits": fwd(Xte), "step": int(best_acc_step),
                        "val_nll": float(best_acc_nll), "val_top1": float(best_acc)}
        out["info"]["best_acc_step"] = int(best_acc_step)
    if extra:
        out["extra"] = extra
    if verbose:
        te = out["test"]
        print(f"      [{arch}] best step {best_step}/{out['info']['steps_run']} "
              f"(~{out['info']['epochs_at_best']:.1f} epochs)  val NLL {best:.4f}  "
              f"test top1 {(te[0].argmax(-1) == te[1]).mean():.4f}", flush=True)
    return out


# =====================================================================
# 4. The ladder protocol (shared by Stage 8 and Stage 8b)
# =====================================================================
def ladder_protocol(x_tr_raw, y_tr, perf_file, n_val_subjects=4, seed=0,
                    record=None):
    """
    Subject-disjoint validation carved from the X-Sub TRAIN performers, and the
    training pool that every rung of every backbone subsamples from.

    There is NO fallback: if the performer ids cannot be recovered, this
    raises. (The earlier pipeline fell back to a random 10% slice in a fresh
    runtime, which gave the 2s-AGCN ladder a different pool and a validation
    set that shared subjects with training.) If `record` (a JSON path) exists,
    the recomputed protocol must match it exactly, so every backbone and every
    session uses the same pool and validation set.
    """
    import hashlib
    import json
    from pathlib import Path
    from har3d_ntu import train_performers

    perf_tr = train_performers(perf_file)
    if len(perf_tr) != len(y_tr):
        raise RuntimeError(f"{len(perf_tr)} recovered performers vs {len(y_tr)} "
                           f"training rows -- the performer file does not "
                           f"match this NTU60_CS.npz")
    uniq = np.unique(perf_tr)
    rng = np.random.default_rng(seed)
    val_subj = sorted(int(s) for s in rng.permutation(uniq)[:n_val_subjects])
    val_mask = np.isin(perf_tr, val_subj)
    val_idx, pool_idx = np.where(val_mask)[0], np.where(~val_mask)[0]
    assert not set(perf_tr[val_idx]) & set(perf_tr[pool_idx])
    proto = {"val_subjects": val_subj, "n_val": int(len(val_idx)),
             "n_pool": int(len(pool_idx)),
             "pool_sha1": hashlib.sha1(pool_idx.tobytes()).hexdigest()[:16],
             "val_sha1": hashlib.sha1(val_idx.tobytes()).hexdigest()[:16]}
    if record is not None:
        record = Path(record)
        if record.exists():
            old = json.loads(record.read_text())
            for k in proto:
                if old.get(k) != proto[k]:
                    raise RuntimeError(f"ladder protocol changed ({k}: {old.get(k)} "
                                       f"-> {proto[k]}); refusing to mix pools")
        else:
            record.write_text(json.dumps(proto, indent=2))
    return val_idx, pool_idx, perf_tr, proto


def run_ladder(store, x_tr_raw, y_tr, pool_idx, Xval, yval, Xte, yte, *,
               model_tag, arch, rungs, seeds, stream="joint", T=64,
               early_stop=True, fixed_steps=6000, max_steps=15000,
               eval_every=250, patience=8, min_steps=500,
               freeze_graph_after=None, protocol=None, device=None,
               verbose=True, snapshot_steps=None, track_best_acc=False,
               run_until=0, extras_dir=None, extra_provenance=None):
    """Train every (rung, seed) for one backbone; resumable via the store.
    Rung subsamples depend only on (n, seed), so all backbones see the same
    training samples at a given rung and seed. With `extras_dir`, the extra
    checkpoints of train_rung (snapshots, best validation accuracy) are written
    there as ck__ntu60__<tag>__<stream>__<split>__s<seed>__<ckpt>.npz BEFORE the
    main cell is stored, so an interrupted run is simply repeated."""
    from store import Key
    K = int(y_tr.max()) + 1
    rows = []
    for n_train in rungs:
        n_train = int(min(n_train, len(pool_idx)))
        split = f"ladder_n{n_train}"
        for seed in seeds:
            if store.has(Key(model_tag, "ntu60", stream, split, seed, "test")):
                if verbose:
                    print(f"  [skip] {model_tag} n={n_train} seed{seed}")
                continue
            sel = pool_idx[subsample_indices(y_tr[pool_idx], n_train, seed=seed)]
            if verbose:
                print(f"\n  === {model_tag} ({arch}) n_train={n_train} seed={seed} ===",
                      flush=True)
            Xtr = prepare_ntu_tensors(x_tr_raw[sel], stream, T=T, verbose=False)
            out = train_rung(Xtr, y_tr[sel], Xval, yval, Xte, yte, K, arch=arch,
                             early_stop=early_stop, fixed_steps=fixed_steps,
                             max_steps=max_steps, eval_every=eval_every,
                             patience=patience, min_steps=min_steps, seed=seed,
                             freeze_graph_after=freeze_graph_after,
                             device=device, verbose=verbose,
                             snapshot_steps=snapshot_steps,
                             track_best_acc=track_best_acc, run_until=run_until)
            prov = dict(arch=arch, method=model_tag, stream=stream,
                        n_train=n_train, T=T, train_info=out["info"],
                        protocol=protocol or {},
                        val="subject-disjoint slice of X-Sub train performers",
                        note="test set fixed across every rung and backbone")
            prov.update(extra_provenance or {})
            if extras_dir is not None:
                from pathlib import Path
                for tag, e in out.get("extra", {}).items():
                    np.savez_compressed(
                        Path(extras_dir) / f"ck__ntu60__{model_tag}__{stream}__{split}__s{seed}__{tag}.npz",
                        logits=e["logits"].astype(np.float32), labels=np.asarray(yte),
                        step=np.int64(e["step"]), val_nll=np.float64(e["val_nll"]),
                        val_top1=np.float64(e["val_top1"]))
            for sub in ("val", "test"):
                L, yy = out[sub]
                store.save(Key(model_tag, "ntu60", stream, split, seed, sub),
                           L, yy, provenance=prov)
            rows.append(dict(model=model_tag, n_train=n_train, seed=seed,
                             best_step=out["info"]["best_step"],
                             steps_run=out["info"]["steps_run"],
                             seconds=float(out["info"]["seconds"]),
                             top1=float((out["test"][0].argmax(-1) == yte).mean())))
            del Xtr
    return rows


In [ ]:
%%writefile /content/har3d_ntu.py
"""
har3d_ntu.py -- NTU RGB+D preprocessing via the CTR-GCN data_gen scripts.

Known traps, all previously hit:
  * git clone refuses a non-empty directory, so cloning AFTER extracting 56,880
    raw files fails silently. Clone first, then move the raw files in.
  * CTR-GCN is 2021 code: np.int (22 sites) and np.string_ (2) were removed in
    NumPy 2.0. Patched with a negative lookahead so np.int64 and np.float32
    survive. Blunt sed would break them.
  * The exclusion-list files open with two prose lines and a blank line. Parse
    by the SsssCcccPpppRrrrAaaa pattern, never by counting lines.
  * The generated tensor layout is (N, 300, 150), not the (N, 3, 300, 25, 2)
    usually quoted.

The released-checkpoint accuracy gate that used to live here is retired: the
CTR-GCN weights were never obtainable. The scale axis now comes from the
training-set ladder in har3d_scale.py, which is methodologically better anyway
-- it holds architecture, recipe and TEST SET fixed so only train n moves.
"""
from __future__ import annotations

import re
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np

CTRGCN_URL = "https://github.com/Uason-Chen/CTR-GCN.git"
NTU60_FILES, NTU120_FILES = 56_880, 114_480
NTU60_MISSING, NTU120_MISSING = 302, 535
XSUB_TRAIN_SUBJECTS = {1, 2, 4, 5, 8, 9, 13, 14, 15, 16, 17, 18, 19, 25, 27, 28,
                       31, 34, 35, 38}


def log(m=""):
    print(m, flush=True)


def clone_repo(repo="/content/CTR-GCN", force=False) -> Path:
    """Clone into a guaranteed-empty path. This must happen BEFORE extraction."""
    repo = Path(repo)
    if (repo / ".git").exists() and not force:
        log(f"  [skip] repo already at {repo}")
        return repo
    if repo.exists():
        log(f"  removing non-empty {repo} so the clone can succeed")
        shutil.rmtree(repo)
    r = subprocess.run(["git", "clone", "--depth", "1", "-q", CTRGCN_URL, str(repo)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"clone failed: {r.stderr.strip()}")
    for s in ("get_raw_skes_data.py", "get_raw_denoised_data.py",
              "seq_transformation.py"):
        assert (repo / "data" / "ntu" / s).exists(), f"missing after clone: {s}"
    log("  clone OK; all three data_gen scripts present")
    return repo


def patch_numpy_aliases(repo="/content/CTR-GCN", subdirs=("ntu", "ntu120")) -> int:
    """
    Fix NumPy 2.0 removals, idempotently. The negative lookahead is the
    important part: a blunt s/np.int/int/ rewrites np.int64 into int64.
    """
    repo = Path(repo)
    pats = [(re.compile(r"\bnp\.int(?![0-9_a-zA-Z])"), "int"),
            (re.compile(r"\bnp\.float(?![0-9_a-zA-Z])"), "float"),
            (re.compile(r"\bnp\.bool(?![0-9_a-zA-Z])"), "bool"),
            (re.compile(r"\bnp\.object(?![0-9_a-zA-Z])"), "object"),
            (re.compile(r"\bnp\.string_"), "np.bytes_")]
    n_files = n_sites = n_scanned = 0
    for sub in subdirs:
        d = repo / "data" / sub
        if not d.exists():
            continue
        for f in d.glob("*.py"):
            n_scanned += 1
            src = f.read_text()
            new = src
            for rx, rep in pats:
                new, k = rx.subn(rep, new)
                n_sites += k
            if new != src:
                f.write_text(new)
                n_files += 1
    log(f"  patched {n_sites} NumPy 2.0 removals across {n_files} of "
        f"{n_scanned} scanned files"
        + ("  (0 is expected on a repo already patched)"
           if n_scanned and not n_sites else ""))
    if not n_scanned:
        raise RuntimeError(
            f"no .py files under {repo}/data -- the clone is incomplete. "
            f"Delete /content/CTR-GCN and re-run.")
    missing = []
    for mod, pkg in (("h5py", "h5py"), ("sklearn", "scikit-learn")):
        try:
            __import__(mod)
        except ImportError:
            missing.append(pkg)
    if missing:
        log(f"  installing {', '.join(missing)} (the data_gen scripts need them)")
        subprocess.run(f"pip install -q {' '.join(missing)}", shell=True,
                       check=False, stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL)
    return n_sites


def _free_gb(path="/content") -> float:
    import os
    p = Path(path)
    while not p.exists() and p != p.parent:
        p = p.parent
    try:
        st = os.statvfs(p)
        return st.f_bavail * st.f_frsize / 1e9
    except OSError:
        return float("nan")


def place_raw(drive_root, repo="/content/CTR-GCN", ntu120=False, force=False) -> Path:
    """
    Extract raw .skeleton files into the layout the scripts expect. RESUMABLE:
    a session dying mid-extract leaves a partial directory, and treating "any
    files present" as "done" once failed the count check on 31,734 of 56,880.
    Extraction runs whenever the count is short, overwriting.
    """
    import zipfile
    from har3d_io import extract_any

    repo, drive_root = Path(repo), Path(drive_root)
    dest = repo / "data" / "nturgbd_raw"
    skel = dest / "nturgb+d_skeletons"
    dest.mkdir(parents=True, exist_ok=True)
    exp = NTU120_FILES if ntu120 else NTU60_FILES

    def count():
        return sum(1 for _ in skel.glob("*.skeleton")) if skel.exists() else 0

    have = count()
    if have >= exp and not force:
        log(f"  {have:,} .skeleton already in place")
        return skel
    if have:
        log(f"  {have:,} of {exp:,} present -- a previous extraction was "
            f"interrupted; resuming (existing files are overwritten)")

    archives = ["nturgbd_skeletons_s001_to_s017.zip"]
    if ntu120:
        archives.append("nturgbd_skeletons_s018_to_s032.zip")

    members = 0
    for name in archives:
        z = drive_root / name
        if not z.exists():
            log(f"  !! {name} not found in {drive_root}")
            continue
        with zipfile.ZipFile(z) as zf:
            m = sum(1 for x in zf.namelist() if x.endswith(".skeleton"))
        members += m
        log(f"  {name}: {m:,} .skeleton members, "
            f"{z.stat().st_size/1e9:.1f} GB compressed")
    if members and members != exp:
        raise RuntimeError(
            f"the archives hold {members:,} .skeleton members, not {exp:,}. "
            f"The download is incomplete -- re-download before extracting.")

    log(f"  disk free before: {_free_gb(dest):.1f} GB "
        f"(raw .skeleton needs ~15 GB for NTU-60)")
    for name in archives:
        z = drive_root / name
        if not z.exists():
            continue
        t0 = time.time()
        log(f"  extracting {name} ...")
        extract_any(z, dest)
        log(f"    done in {time.time()-t0:.0f}s, {count():,} files so far")
    log(f"  disk free after: {_free_gb(dest):.1f} GB")

    n = count()
    log(f"  raw .skeleton files: {n:,} (expected {exp:,})  "
        f"{'OK' if n == exp else '<-- MISMATCH'}")
    if n != exp:
        raise RuntimeError(
            f"expected {exp:,} .skeleton files, found {n:,} after a full pass.\n"
            f"  The archives do hold {members:,} members, so this is not a bad "
            f"download.\n  Most likely the session ran out of disk "
            f"({_free_gb(dest):.1f} GB free now) or extraction was interrupted "
            f"again.\n  Free space, then re-run this cell -- it resumes.")
    return skel


def check_statistics(repo="/content/CTR-GCN", ntu120=False) -> int:
    """skes_available_name.txt is what the scripts consume. Verify the arithmetic."""
    sub = "ntu120" if ntu120 else "ntu"
    stat = Path(repo) / "data" / sub / "statistics"
    avail = stat / "skes_available_name.txt"
    n = sum(1 for ln in avail.read_text().splitlines() if ln.strip())
    total = NTU120_FILES if ntu120 else NTU60_FILES
    miss = NTU120_MISSING if ntu120 else NTU60_MISSING
    ok = n == total - miss
    log(f"  arithmetic: {total:,} - {miss} = {total-miss:,}   "
        f"available list says {n:,}   {'OK' if ok else '<-- CHECK'}")
    if not ok:
        raise RuntimeError("exclusion arithmetic does not match the shipped list")
    return n


def run_datagen(repo="/content/CTR-GCN", ntu120=False) -> None:
    """The three official stages, in order, from the directory they expect."""
    wd = Path(repo) / "data" / ("ntu120" if ntu120 else "ntu")
    for script in ("get_raw_skes_data.py", "get_raw_denoised_data.py",
                   "seq_transformation.py"):
        log(f"\n  === {script} ===")
        t0 = time.time()
        r = subprocess.run(["python", script], cwd=wd, text=True, capture_output=True)
        for ln in (r.stdout or "").strip().splitlines()[-5:]:
            log("    " + ln)
        if r.returncode != 0:
            for ln in (r.stderr or "").strip().splitlines()[-15:]:
                log("    ! " + ln)
            raise RuntimeError(f"{script} failed (exit {r.returncode})")
        log(f"  OK in {time.time()-t0:.0f}s")


def verify_outputs(repo="/content/CTR-GCN", ntu120=False) -> dict:
    wd = Path(repo) / "data" / ("ntu120" if ntu120 else "ntu")
    out = {}
    for f in sorted(wd.glob("*.npz")):
        z = np.load(f)
        shapes = {k: list(z[k].shape) for k in z.files}
        log(f"\n  {f.name} ({f.stat().st_size/1e9:.2f} GB)")
        for k, s in shapes.items():
            log(f"    {k:<10} {tuple(s)}")
        out[f.name] = shapes
    if not out:
        raise RuntimeError("no .npz produced -- data_gen did not complete")
    return out


def push_to_drive(drive_root, repo="/content/CTR-GCN", ntu120=False) -> None:
    sub = "ntu120" if ntu120 else "ntu"
    src = Path(repo) / "data" / sub
    dest = Path(drive_root) / "preprocessed" / sub
    dest.mkdir(parents=True, exist_ok=True)
    tot = 0
    for f in list(src.glob("*.npz")) + [src / "statistics" / "skes_available_name.txt",
                                        src / "statistics" / "performer.txt"]:
        if f.exists():
            shutil.copy2(f, dest / f.name)
            tot += f.stat().st_size
    log(f"  copied {tot/1e9:.2f} GB -> {dest}")


def find_performer_file(drive_root=None, repo="/content/CTR-GCN", ntu120=False):
    """
    Locate statistics/performer.txt. The Drive copy (written by
    push_to_drive in Stage 7) is checked FIRST, because /content -- and with it
    the cloned repo -- is wiped whenever the runtime changes. The previous
    pipeline read only the repo copy, so the 2s-AGCN ladder, run in a fresh
    GPU runtime, silently fell back to a random validation slice.
    """
    sub = "ntu120" if ntu120 else "ntu"
    cands = []
    if drive_root is not None:
        cands.append(Path(drive_root) / "preprocessed" / sub / "performer.txt")
    cands.append(Path(repo) / "data" / sub / "statistics" / "performer.txt")
    for c in cands:
        if c.exists():
            return c
    raise FileNotFoundError(
        "performer.txt not found in any of: " + ", ".join(map(str, cands))
        + ". Re-run Stage 7 (it copies the statistics files to Drive).")


def _read_performers(src):
    src = Path(src)
    if src.is_dir():
        src = src / "data" / "ntu" / "statistics" / "performer.txt"
    return np.array([int(ln) for ln in src.read_text().split() if ln.strip()])


def test_performers(src, train_subjects=XSUB_TRAIN_SUBJECTS) -> np.ndarray:
    """
    Performer id per row of x_test, in the order seq_transformation wrote.
    The npz stores no sample ids, so they are recovered from the shipped
    statistics file by applying the same X-Sub criterion. `src` is the
    performer.txt path (see find_performer_file) or the repo root.
    """
    perf = _read_performers(src)
    return perf[~np.isin(perf, list(train_subjects))]


def train_performers(src, train_subjects=XSUB_TRAIN_SUBJECTS) -> np.ndarray:
    """Performer id per row of x_train, same ordering argument."""
    perf = _read_performers(src)
    return perf[np.isin(perf, list(train_subjects))]


In [ ]:
%%writefile /content/har3d_calib.py
"""
har3d_calib.py -- calibration methods and the NULL MODEL.

THE NULL MODEL IS THE POINT OF THIS FILE. A binned calibration error is a
BIASED estimator: a perfectly calibrated model does not report ECE 0, it
reports a positive floor that grows as n shrinks and K grows. On the MSR
datasets that floor is roughly the size of every effect in the paper, so an
ECE quoted without it is uninterpretable.

ece_null() simulates the floor using THE MODEL'S OWN confidence profile rather
than synthetic logits: draw y_sim ~ Categorical(p_i) for each sample. By
construction that simulated dataset is perfectly calibrated AND has exactly the
observed confidence distribution, so any ECE it reports is pure estimator bias.
The excess over the null is the only part that is evidence of miscalibration.

Every metric is imported from har3d_v2 so there is exactly one definition.
"""
from __future__ import annotations

import numpy as np

from har3d_v2 import (adaptive_ece, brier, ece, fit_temperature, nll, softmax,
                      top1)


# =====================================================================
# 1. The null model
# =====================================================================
def ece_null(p, n_sim=500, metric=adaptive_ece, seed=0, M=15):
    """
    Null distribution of `metric` for a PERFECTLY calibrated model with this
    model's confidence profile.

    p : (n, K) predicted probabilities from the real model.
    Returns (values (n_sim,), mean, p95).
    """
    p = np.asarray(p, np.float64)
    p = p / p.sum(-1, keepdims=True)
    rng = np.random.default_rng(seed)
    n, K = p.shape
    cdf = np.cumsum(p, axis=1)
    vals = np.empty(n_sim)
    for i in range(n_sim):
        u = rng.random((n, 1))
        y_sim = (u > cdf).sum(1)              # inverse-CDF categorical draw
        y_sim = np.clip(y_sim, 0, K - 1)
        vals[i] = metric(p, y_sim, M) if metric is not brier else metric(p, y_sim)
    return vals, float(vals.mean()), float(np.quantile(vals, 0.95))


def ece_significance(p, y, n_sim=500, metric=adaptive_ece, seed=0, M=15):
    """
    Observed calibration error against its own null.

    Returns observed, null mean/p95, EXCESS (observed - null mean) and a
    one-sided p-value = P(null >= observed). A p-value above 0.05 means the
    model is statistically indistinguishable from perfectly calibrated at this
    sample size -- which is the honest reading for most MSR cells.
    """
    p = np.asarray(p, np.float64)
    obs = metric(p, y, M) if metric is not brier else metric(p, y)
    vals, mu, p95 = ece_null(p, n_sim=n_sim, metric=metric, seed=seed, M=M)
    # Monte-Carlo p-value with the +1 correction (Phipson & Smyth 2010): it is
    # never 0, so a multiple-comparison correction stays valid. Its smallest
    # value is 1/(n_sim+1), so n_sim must be large enough for the family size.
    pval = float((1 + (vals >= obs).sum()) / (n_sim + 1))
    return {"observed": float(obs), "null_mean": mu, "null_p95": p95,
            "excess": float(obs - mu), "p_value": pval,
            "separable": bool(pval < 0.05), "n": int(len(y)),
            "n_classes": int(p.shape[1])}


def paired_bootstrap(p_a, p_b, y, metric=adaptive_ece, n_boot=2000, seed=0, M=15):
    """
    Is method A better calibrated than method B ON THE SAME SAMPLES? Resampling
    the pair together removes the sample-to-sample variance that dominates an
    unpaired comparison at n = 200.
    """
    p_a, p_b, y = np.asarray(p_a), np.asarray(p_b), np.asarray(y)
    rng = np.random.default_rng(seed)
    n = len(y)
    call = (lambda p, yy: metric(p, yy, M)) if metric is not brier else metric
    obs = call(p_a, y) - call(p_b, y)
    d = np.empty(n_boot)
    for i in range(n_boot):
        s = rng.integers(0, n, n)
        d[i] = call(p_a[s], y[s]) - call(p_b[s], y[s])
    d = d[np.isfinite(d)]
    return {"delta": float(obs), "lo": float(np.quantile(d, 0.025)),
            "hi": float(np.quantile(d, 0.975)),
            "p_value": float(min(1.0, 2 * (1 + min((d >= 0).sum(), (d <= 0).sum())) / (len(d) + 1))),
            "significant": bool(np.quantile(d, 0.025) > 0 or np.quantile(d, 0.975) < 0)}


# =====================================================================
# 2. Post-hoc calibration
# =====================================================================
def fit_vector_scaling(logits_val, y_val, l2=1e-3, iters=300, lr=0.1, seed=0):
    """
    Per-class scale and bias, z' = w * z + b, fitted on validation NLL by plain
    gradient descent with analytic gradients and an L2 pull toward the identity
    (w = 1, b = 0). The regulariser matters: with K = 20 and 60 validation
    samples, unregularised vector scaling memorises the validation set.
    """
    z = np.asarray(logits_val, np.float64)
    y = np.asarray(y_val)
    n, K = z.shape
    w, b = np.ones(K), np.zeros(K)
    oh = np.zeros((n, K))
    oh[np.arange(n), y] = 1.0
    for _ in range(iters):
        p = softmax(z * w + b)
        g = (p - oh) / n                       # dL/d(logit)
        gw = (g * z).sum(0) + 2 * l2 * (w - 1.0)
        gb = g.sum(0) + 2 * l2 * b
        w -= lr * gw
        b -= lr * gb
    return w, b


def apply_vector_scaling(logits, w, b):
    return softmax(np.asarray(logits, np.float64) * w + b)


def sharpen_probs(p, T=1.0):
    """Temperature applied to PROBABILITIES (for ensembles, where the mean is
    taken in probability space and no logits survive)."""
    p = np.asarray(p, np.float64)
    q = np.power(np.clip(p, 1e-12, 1.0), 1.0 / T)
    return q / q.sum(-1, keepdims=True)


def fit_temperature_probs(p_val, y_val, lo=0.05, hi=10.0, iters=200):
    """Golden-section on NLL, for probability-space inputs."""
    p, y = np.asarray(p_val, np.float64), np.asarray(y_val)
    idx = np.arange(len(y))

    def loss(T):
        q = sharpen_probs(p, T)
        return float(-np.log(np.clip(q[idx, y], 1e-12, 1.0)).mean())

    gr = (np.sqrt(5) - 1) / 2
    a, b = lo, hi
    c, d = b - gr * (b - a), a + gr * (b - a)
    for _ in range(iters):
        if loss(c) < loss(d):
            b, d = d, c
            c = b - gr * (b - a)
        else:
            a, c = c, d
            d = a + gr * (b - a)
        if abs(b - a) < 1e-6:
            break
    return float((a + b) / 2)


# =====================================================================
# 3. Uncertainty from stochastic passes
# =====================================================================
def mc_dropout_probs(mc_logits):
    """(P, n, K) logits -> mean predictive probability (n, K)."""
    return softmax(np.asarray(mc_logits, np.float64)).mean(0)


def mc_uncertainty(mc_logits):
    """
    Split predictive entropy into aleatoric and epistemic.
      total     = H[ E_p[y] ]
      aleatoric = E_p[ H[y] ]
      epistemic = total - aleatoric   (mutual information)
    """
    P = softmax(np.asarray(mc_logits, np.float64))
    mean_p = P.mean(0)
    total = -(mean_p * np.log(np.clip(mean_p, 1e-12, 1))).sum(-1)
    aleatoric = -(P * np.log(np.clip(P, 1e-12, 1))).sum(-1).mean(0)
    return {"total": total, "aleatoric": aleatoric,
            "epistemic": total - aleatoric}


def deep_ensemble_probs(logit_list):
    """Mean probability over independently seeded models."""
    return np.mean([softmax(np.asarray(z, np.float64)) for z in logit_list], 0)


def adaptive_ece_cc(conf, correct, M=15):
    """
    Adaptive (equal-mass) ECE from per-sample confidence and correctness only.
    Identical to har3d_v2.adaptive_ece(p, y) when conf = p.max(-1) and
    correct = (p.argmax(-1) == y), but avoids touching the (n, K) matrix, which
    makes bootstrap resampling of large test sets ~20x faster.
    """
    conf = np.asarray(conf, np.float64)
    correct = np.asarray(correct, np.float64)
    n = len(conf)
    order = np.argsort(conf, kind="stable")
    c, k = conf[order], correct[order]
    edges = np.linspace(0, n, M + 1).astype(int)
    starts = edges[:-1][edges[:-1] < edges[1:]]
    sizes = np.diff(np.append(starts, n))
    gap = np.abs(np.add.reduceat(k, starts) / sizes - np.add.reduceat(c, starts) / sizes)
    return float((gap * sizes / n).sum())


# holm and sign_test have one definition, in har3d_stats (shared with
# make_numbers.py); they are re-exported here for the stages that import C.
from har3d_stats import holm, sign_test  # noqa: E402,F401


def boot_mean_ci(x, n_boot=2000, alpha=0.05, seed=0):
    """Percentile bootstrap CI of a mean over cells."""
    x = np.asarray(x, float)
    if len(x) == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    bs = x[rng.integers(0, len(x), (n_boot, len(x)))].mean(1)
    return float(x.mean()), float(np.quantile(bs, alpha / 2)), float(np.quantile(bs, 1 - alpha / 2))


# =====================================================================
# 4. One row per (method, cell)
# =====================================================================
def evaluate_method(p, y, name="raw", n_sim=0, seed=0):
    """Metrics for one already-calibrated probability matrix."""
    out = {"method": name, "n": int(len(y)), "top1": top1(p, y),
           "ece_15": ece(p, y), "ece_adaptive": adaptive_ece(p, y),
           "brier": brier(p, y), "nll": nll(p, y)}
    if n_sim:
        out.update({f"null_{k}": v for k, v in
                    ece_significance(p, y, n_sim=n_sim, seed=seed).items()})
    return out


In [ ]:
%%writefile /content/har3d_grid.py
"""
har3d_grid.py -- the small-dataset training grid shared by Stage 6 (cross-
entropy + tuned MC dropout) and Stage 6b (label smoothing, focal loss).

One protocol for every method: the same quality-filtered samples, the same
subject-disjoint splits (three random + the conventional fixed split), the same
seeds, early stopping on validation NLL. Only the loss differs, so the six-
method comparison in Stage 9c is like-for-like. Resumable: any cell already in
the logit store is skipped.
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

import har3d_v2 as V
import har3d_models_v2 as M2
from store import Key, LogitStore

DATASETS = (("action3d", "msr_action3d.npz"),
            ("daily", "msr_daily.npz"),
            ("utkinect", "utkinect.npz"))


def splits_for(subj, y, verbose=False):
    sp = V.subject_splits(subj, labels=y, n_splits=3, seed=0,
                          n_test_subjects=4, n_val_subjects=2, verbose=verbose)
    try:
        sp.append(V.fixed_split(subj, labels=y, verbose=verbose))
    except ValueError as e:
        if verbose:
            print(f"    no fixed split: {e}")
    return sp


def run_grid(cache_dir, logits_dir, prep_cfg, prep_tag, model_tag="stgcn",
             loss_kw=None, streams=("joint",), seeds=(0,), mc_passes=0,
             mc_p_grid=(0.05, 0.1, 0.2, 0.3), T=64, max_epochs=120,
             patience=15, min_epochs=10, max_zero_frac=0.5, min_frames=10,
             datasets=DATASETS, device=None, verbose=True):
    """Train every (dataset, stream, split, seed) cell; returns a log dict."""
    loss_kw = dict(loss_kw or {})
    store = LogitStore(logits_dir)
    logits_dir = Path(logits_dir)
    A = V.adjacency(V.MSR_PARENTS)
    log = {"produced": 0, "skipped": 0, "cells": []}
    for ds, npz in datasets:
        f = Path(cache_dir) / npz
        if not f.exists():
            raise FileNotFoundError(f"{f} missing -- run the ingest stages first")
        cache = V.load_cache(f)
        for stream in streams:
            X, y, subj, keep = V.masked_tensors(cache, stream, T, prep_cfg,
                                                max_zero_frac, min_frames)
            nc = int(y.max()) + 1
            for sp in splits_for(subj, y):
                name = f"{sp['name']}_{prep_tag}"
                for seed in seeds:
                    kt = Key(model_tag, ds, stream, name, seed, "test")
                    if store.has(kt):
                        log["skipped"] += 1
                        continue
                    out, _ = M2.train_and_logits(
                        X, y, sp, nc, A, max_epochs=max_epochs, patience=patience,
                        min_epochs=min_epochs, seed=seed, mc_passes=mc_passes,
                        mc_p_grid=mc_p_grid, device=device, verbose=False,
                        **loss_kw)
                    info = out["info"]
                    prov = dict(arch="stgcn", method=model_tag, loss=loss_kw or "ce",
                                stream=stream, T=T, prep=prep_tag, **prep_cfg,
                                parents="corrected", scale="bone", impute=True,
                                n_after_filter=int(len(y)), n_raw=int(len(keep)),
                                subjects=sp["subjects"], train_info=info,
                                mc_passes=mc_passes)
                    for sub in ("val", "test"):
                        L, yy = out[sub]
                        store.save(Key(model_tag, ds, stream, name, seed, sub),
                                   L, yy, provenance=prov)
                        log["produced"] += 1
                        if sub + "_mc" in out:
                            P, _ = out[sub + "_mc"]
                            np.savez_compressed(
                                logits_dir / f"mc__{ds}__{model_tag}__{stream}__"
                                             f"{name}__s{seed}__{sub}.npz",
                                logits=P.astype(np.float32), labels=yy,
                                mc_p=np.float32(info["mc_p"] or np.nan))
                    row = dict(dataset=ds, stream=stream, split=name, seed=seed,
                               best_epoch=info["best_epoch"],
                               epochs_run=info["epochs_run"],
                               best_val_nll=info["best_val_nll"],
                               mc_p=info["mc_p"],
                               test_top1=float((out["test"][0].argmax(-1)
                                                == out["test"][1]).mean()))
                    log["cells"].append(row)
                    if verbose:
                        print(f"  {model_tag:<12}{ds:<9}{stream:<13}{name:<18}s{seed}"
                              f"  ep {info['best_epoch']:>3}/{info['epochs_run']:<3}"
                              f" top1 {row['test_top1']:.3f}"
                              + (f"  mc p={info['mc_p']}" if info["mc_p"] else ""))
    return log


In [ ]:
%%writefile /content/store.py
"""
store.py -- the logit store.

Filename convention:
    {dataset}__{model}__{stream}__{split}__s{seed}__{subset}.npz
e.g. action3d__stgcn__bone_motion__rand0__s2__test.npz

This is the hinge of the project: once (model, dataset, stream, split, seed,
subset) -> logits, labels is on Drive, every table and figure downstream is
CPU-cheap and re-runnable forever. It is the only artifact that costs GPU
hours to rebuild.
"""
from __future__ import annotations

import hashlib
import json
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np


@dataclass(frozen=True)
class Key:
    model: str      # stgcn | ctrgcn | ...
    dataset: str    # daily | action3d | utkinect | ntu60 | ntu120
    stream: str     # joint | bone | joint_motion | bone_motion
    split: str      # xsub | rand0 | fixed | ladder_n5000 ...
    seed: int
    subset: str     # val | test

    def fname(self) -> str:
        return (f"{self.dataset}__{self.model}__{self.stream}__"
                f"{self.split}__s{self.seed}__{self.subset}.npz")


class LogitStore:
    def __init__(self, root):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

    def path(self, k: Key) -> Path:
        return self.root / k.fname()

    def has(self, k: Key) -> bool:
        return self.path(k).exists()

    def save(self, k: Key, logits, labels, sample_ids=None, top1=None,
             provenance: dict = None):
        logits = np.asarray(logits, np.float32)
        labels = np.asarray(labels, np.int64)
        assert logits.ndim == 2 and len(logits) == len(labels), \
            f"{k.fname()}: logits {logits.shape} vs labels {labels.shape}"
        prov = dict(provenance or {})
        prov.update({"saved": time.ctime(), "key": asdict(k),
                     "n": int(len(labels)), "C": int(logits.shape[1])})
        if top1 is None:
            top1 = float((logits.argmax(-1) == labels).mean())
        prov["top1"] = float(top1)
        prov["logit_sha1"] = hashlib.sha1(logits.tobytes()).hexdigest()[:16]
        np.savez_compressed(
            self.path(k), logits=logits, labels=labels,
            sample_ids=np.asarray(sample_ids if sample_ids is not None
                                  else np.arange(len(labels))),
            provenance=np.array(json.dumps(prov)))
        return self.path(k)

    def load(self, k: Key) -> dict:
        z = np.load(self.path(k), allow_pickle=False)
        out = {f: z[f] for f in z.files}
        out["provenance"] = json.loads(str(out["provenance"]))
        return out

    def list_keys(self) -> list:
        keys = []
        for p in sorted(self.root.glob("*.npz")):
            parts = p.stem.split("__")
            if len(parts) != 6:
                continue                      # e.g. the mc__ dumps
            ds, mdl, st, sp, sd, sub = parts
            if not sd.startswith("s") or not sd[1:].isdigit():
                continue
            keys.append(Key(mdl, ds, st, sp, int(sd[1:]), sub))
        return keys

    def inventory(self) -> dict:
        keys = self.list_keys()
        inv = {}
        for k in keys:
            inv.setdefault(k.dataset, {}).setdefault(k.model, set()).add(k.stream)
        print(f"  logit store: {len(keys)} files under {self.root}")
        for ds in sorted(inv):
            for m in sorted(inv[ds]):
                print(f"    {ds:<10} {m:<10} streams: {sorted(inv[ds][m])}")
        return inv

    def missing(self, models, datasets, streams, splits, seeds, subsets):
        """What still has to be produced. Run before booking GPU time."""
        want = [Key(m, d, st, sp, sd, sub)
                for d in datasets for m in models for st in streams
                for sp in splits for sd in seeds for sub in subsets]
        miss = [k for k in want if not self.has(k)]
        print(f"  {len(want) - len(miss)}/{len(want)} cells present, "
              f"{len(miss)} missing")
        return miss

    def ensemble(self, base: Key, streams=("joint", "bone",
                                           "joint_motion", "bone_motion")):
        """
        Multi-stream fusion. Raises if members are not aligned -- silently
        averaging misaligned members would corrupt every ensembling result.
        """
        probs, labels, used = [], None, []
        for st in streams:
            k = Key(base.model, base.dataset, st, base.split, base.seed, base.subset)
            if not self.has(k):
                continue
            d = self.load(k)
            if labels is None:
                labels = d["labels"]
            elif not np.array_equal(labels, d["labels"]):
                raise ValueError(f"label mismatch between streams for {k.fname()}")
            probs.append(_softmax(d["logits"]))
            used.append(st)
        if not probs:
            raise FileNotFoundError(f"no streams found for {base.fname()}")
        return np.mean(np.stack(probs, 0), 0), labels, used

    def ensemble_logits(self, base: Key, streams=("joint", "bone",
                                                  "joint_motion", "bone_motion")):
        """
        Mean LOGITS rather than mean probabilities, so a temperature can be
        fitted to the fusion. RQ5 previously fitted T for every individual
        stream and never for the ensemble, which is why "ensembling hurts
        calibration" may have been an artifact of the omitted step.
        """
        zs, labels, used = [], None, []
        for st in streams:
            k = Key(base.model, base.dataset, st, base.split, base.seed, base.subset)
            if not self.has(k):
                continue
            d = self.load(k)
            if labels is None:
                labels = d["labels"]
            elif not np.array_equal(labels, d["labels"]):
                raise ValueError(f"label mismatch between streams for {k.fname()}")
            zs.append(np.asarray(d["logits"], np.float64))
            used.append(st)
        if not zs:
            raise FileNotFoundError(f"no streams found for {base.fname()}")
        return np.mean(np.stack(zs, 0), 0), labels, used


def _softmax(z, T=1.0):
    z = np.asarray(z, np.float64) / T
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)


In [ ]:
%%writefile /content/har3d_plotstyle.py
"""
har3d_plotstyle.py -- one place that defines publication figure style, so every
figure in the paper is visually consistent and colourblind-safe.

Imported by the figure stages. Calling apply() sets matplotlib rcParams for
vector-friendly, embedded-font, readable-at-column-width output. PALETTE is
Wong's 8-colour colourblind-safe set (Nature Methods 2011), which reads
correctly under deuteranopia, protanopia and tritanopia and in greyscale.
"""
import matplotlib as mpl
import matplotlib.pyplot as plt

# Wong 2011 colourblind-safe palette.
PALETTE = {
    'black':  '#000000', 'orange': '#E69F00', 'skyblue': '#56B4E9',
    'green':  '#009E73', 'yellow': '#F0E442', 'blue':    '#0072B2',
    'vermillion': '#D55E00', 'purple': '#CC79A7',
}
# Fixed, meaningful assignment reused across every figure so a colour means the
# same thing throughout the paper.
ROLE = {
    'observed': PALETTE['blue'], 'null': '#7F7F7F', 'excess': PALETTE['vermillion'],
    'action3d': PALETTE['blue'], 'daily': PALETTE['orange'],
    'utkinect': PALETTE['green'], 'ntu60': PALETTE['purple'],
    'gap_fill': PALETTE['vermillion'], 'bar': PALETTE['skyblue'],
}
CYCLE = [PALETTE[k] for k in ('blue', 'vermillion', 'green', 'orange',
                              'purple', 'skyblue', 'yellow', 'black')]


def apply():
    """Set journal-quality rcParams. Fonts are embedded as editable text in the
    PDF (Type-42 / TrueType) rather than outlines, which most journals require."""
    mpl.rcParams.update({
        'figure.dpi': 150,
        'savefig.dpi': 400,
        'savefig.bbox': 'tight',
        'pdf.fonttype': 42,          # embed TrueType so text stays selectable
        'ps.fonttype': 42,
        'font.family': 'sans-serif',
        'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
        'font.size': 9,
        'axes.titlesize': 10,
        'axes.labelsize': 9,
        'xtick.labelsize': 8,
        'ytick.labelsize': 8,
        'legend.fontsize': 7.5,
        'legend.frameon': False,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.linewidth': 0.8,
        'lines.linewidth': 1.4,
        'lines.markersize': 5,
        'axes.prop_cycle': mpl.cycler(color=CYCLE),
        'figure.constrained_layout.use': True,
    })


def save(fig, path_noext, formats=('pdf', 'png')):
    """Save one figure in every requested format. PDF is the vector master a
    journal typesets from; PNG is the preview."""
    out = []
    for ext in formats:
        p = f'{path_noext}.{ext}'
        fig.savefig(p)
        out.append(p)
    return out


In [ ]:
%%writefile /content/har3d_stats.py
"""
har3d_stats.py -- inference at the level of coarse units, not cells.

Cells that share a test set are not independent evidence. On the small
datasets the 20 cells of a split (4 streams x 5 seeds) are scored on the same
test clips, so a test that treats the 240 cells as 240 observations is
anti-conservative. On the NTU ladder every run is scored on one fixed test set,
and a bootstrap over its clips holds the trained models fixed, so it cannot see
the variation between training runs. Every inferential statement in the paper
is therefore made over coarser units:

  small datasets   test sets (dataset x split), 12 units. They are not fully
                   independent either: the four test sets of a dataset share
                   one or two subjects pairwise, so p-values over them are
                   approximate (optimistic) and are reported together with
                   the per-dataset pattern.
  NTU ladder       seed-rung pairs (separately trained runs on one fixed
                   test set), which carry the run-to-run variance

Pure numpy/scipy, no project imports, so it is shared verbatim by the notebook
(Stages 9b and 9c) and by make_numbers.py: one definition of every test.
"""
from __future__ import annotations

from collections import defaultdict

import numpy as np
from scipy import stats

N_BOOT_UNITS = 20000          # cluster-bootstrap resamples over test sets


# =====================================================================
# 1. Elementary tests
# =====================================================================
def holm(pvals):
    """Holm-Bonferroni adjusted p-values (order-preserving, FWER control)."""
    pvals = np.asarray(pvals, float)
    m = len(pvals)
    adj = np.empty(m)
    run = 0.0
    for rank, idx in enumerate(np.argsort(pvals, kind="stable")):
        run = max(run, (m - rank) * pvals[idx])
        adj[idx] = min(run, 1.0)
    return adj


def sign_test(deltas):
    """Two-sided exact sign test that positive and negative deltas are equally
    likely (exact zeros dropped). Returns (n_positive, n_nonzero, p)."""
    d = np.asarray(deltas, float)
    d = d[d != 0]
    k = int((d > 0).sum())
    return k, len(d), (float(stats.binomtest(k, len(d), 0.5).pvalue) if len(d) else 1.0)


def cluster_boot_ci(unit_values, n_boot=N_BOOT_UNITS, seed=0, alpha=0.05):
    """Percentile bootstrap interval of the mean of per-unit values. Resampling
    whole units (test sets) keeps every cell of a unit together."""
    d = np.asarray(unit_values, float)
    rng = np.random.default_rng(seed)
    bs = d[rng.integers(0, len(d), (n_boot, len(d)))].mean(1)
    return float(np.quantile(bs, alpha / 2)), float(np.quantile(bs, 1 - alpha / 2))


def unit_means(values, units):
    """Mean of `values` within each unit, in order of first appearance."""
    g = defaultdict(list)
    for v, u in zip(values, units):
        g[u].append(v)
    keys = list(g)
    return keys, np.array([np.mean(g[k]) for k in keys])


def ols_loglog(x, y, level=0.95):
    """OLS fit of log y on log x. The t-interval of the slope uses the scatter
    of the points about the line (lack of fit included), unlike a bootstrap
    over seeds within fixed rungs. Returns slope, lo, hi, R^2."""
    lr = stats.linregress(np.log(np.asarray(x, float)), np.log(np.asarray(y, float)))
    t = stats.t.ppf(0.5 + level / 2, len(x) - 2)
    return float(lr.slope), float(lr.slope - t * lr.stderr), float(lr.slope + t * lr.stderr), float(lr.rvalue ** 2)


# =====================================================================
# 2. Small datasets: test sets are the units
# =====================================================================
def pooled_excess(rows):
    """RQ1 in aggregate: excess over the null averaged within each test set
    (dataset x split), then an exact sign test over test sets."""
    keys, u = unit_means([r["null_excess"] for r in rows], [(r["dataset"], r["split"]) for r in rows])
    k, n, p = sign_test(u)
    per_ds = defaultdict(lambda: [0, 0])
    for key, v in zip(keys, u):
        per_ds[key[0]][0] += int(v > 0)
        per_ds[key[0]][1] += 1
    return {"units": keys, "unit_excess": u, "k_pos": k, "n": n, "p": p, "mean": float(u.mean()),
            "per_dataset": {ds: tuple(c) for ds, c in per_ds.items()}}


def method_comparison(per_cell, methods, baseline="raw", n_boot=N_BOOT_UNITS, key="ece"):
    """
    Paired comparison of calibration methods against the baseline at the level
    of test sets. per_cell: dicts with method, dataset, stream, split, seed and
    the score `key` (float; lower is better: 'ece', 'excess', 'brier', 'nll').
    A cell with seed == -1 (the deep ensemble, one per dataset, stream and
    split) is paired with the seed-mean of the baseline.

    Per method: the paired reduction (baseline - method, positive is better) is
    averaged over the cells of each test set; the sign test runs over test
    sets, Holm-corrected over the methods; the interval is a cluster bootstrap
    over test sets. Also returns a cluster-bootstrap interval of each method's
    mean ECE (baseline included).
    """
    base = {(r["dataset"], r["stream"], r["split"], r["seed"]): r[key]
            for r in per_cell if r["method"] == baseline}
    base_mean = defaultdict(list)
    for (ds, st, sp, _), v in base.items():
        base_mean[(ds, st, sp)].append(v)
    res = {}
    for m in methods:
        vals, units = [], []
        for r in per_cell:
            if r["method"] != m:
                continue
            b = (np.mean(base_mean[(r["dataset"], r["stream"], r["split"])]) if int(r["seed"]) == -1
                 else base[(r["dataset"], r["stream"], r["split"], r["seed"])])
            vals.append(b - r[key])
            units.append((r["dataset"], r["split"]))
        keys, d = unit_means(vals, units)
        k, n, p = sign_test(d)
        lo, hi = cluster_boot_ci(d, n_boot=n_boot, seed=0)
        per_ds = defaultdict(lambda: [0, 0])
        for u, v in zip(keys, d):
            per_ds[u[0]][0] += int(v > 0)
            per_ds[u[0]][1] += 1
        res[m] = dict(n=n, k=k, mean=float(d.mean()), lo=lo, hi=hi, p=p,
                      per_dataset={ds: tuple(c) for ds, c in per_ds.items()})
    for m, a in zip(methods, holm([res[m]["p"] for m in methods])):
        res[m]["p_holm"] = float(a)
        res[m]["verdict"] = (("reduces" if res[m]["mean"] > 0 else "increases") if a < 0.05 else "n.s.")
    ece_ci = {}
    for m in [baseline] + list(methods):
        _, d = unit_means([r[key] for r in per_cell if r["method"] == m],
                          [(r["dataset"], r["split"]) for r in per_cell if r["method"] == m])
        ece_ci[m] = cluster_boot_ci(d, n_boot=n_boot, seed=1)
    return res, ece_ci


# =====================================================================
# 3. Ladder: separately trained runs are the units
# =====================================================================
def seed_level_tests(ece, comparisons, regimes, seeds=(0, 1, 2)):
    """
    Exact sign tests over seed-rung pairs. ece: {(model, n_train, seed): aECE}.
    comparisons: [(model_a, model_b)]; regimes: {name: [n_train, ...]}.
    One Holm family over every (comparison, regime) test. Each pair is a
    separate training run of both models on the same subsample, so the
    run-to-run variance is part of the test, unlike a bootstrap over clips.
    """
    out = []
    for a, b in comparisons:
        for name, ns in regimes.items():
            d = [ece[(a, n, s)] - ece[(b, n, s)] for n in ns for s in seeds
                 if (a, n, s) in ece and (b, n, s) in ece]
            k, nz, p = sign_test(d)
            out.append(dict(model_a=a, model_b=b, regime=name, n_pairs=len(d), n_a_higher=k,
                            n_nonzero=nz, mean_delta=float(np.mean(d)), p=p))
    for r, a in zip(out, holm([r["p"] for r in out])):
        r["p_holm"] = float(a)
    return out


def pooled_seed_sd(ece, models, rungs, seeds=(0, 1, 2)):
    """Run-to-run spread: root-mean-square of the across-seed standard
    deviation (ddof=1) over the given models and rungs."""
    sds = [np.std([ece[(m, n, s)] for s in seeds], ddof=1)
           for m in models for n in rungs if all((m, n, s) in ece for s in seeds)]
    return float(np.sqrt(np.mean(np.square(sds)))), len(sds)


def identical_run_gap(rows, a="agcn", b="agcn_frozen", n=2000, seed=0, freeze_step=1000):
    """Run-to-run noise: two runs that restore the same weights from a step at
    or before the freeze are nominally identical models. Returns the gaps in
    aECE and top-1, or None if the condition does not hold."""
    L = {(r["model"], r["n_train"], r["seed"]): r for r in rows if r.get("dataset") == "ntu60"}
    ra, rb = L.get((a, n, seed)), L.get((b, n, seed))
    if ra is None or rb is None or not (ra["best_step"] == rb["best_step"] <= freeze_step):
        return None
    return dict(ece=abs(ra["ece_adaptive"] - rb["ece_adaptive"]), top1=abs(ra["top1"] - rb["top1"]),
                step=int(ra["best_step"]))


In [ ]:
%%writefile /content/har3d_review.py
"""
har3d_review.py -- analyses added in the revision.

Everything here answers one question the first version left open: how far do
the conclusions depend on (a) treating overlapping test sets as independent,
(b) the i.i.d. label draw of the null model, (c) the estimator and its bin
count, and (d) the power of the test at these sample sizes?

Top-label calibration error depends on a prediction only through its
confidence c_i and whether it is correct. Under the null model the label is
drawn from the predicted vector, so the prediction is correct with probability
c_i: for every TOP-LABEL statistic the full-vector draw of har3d_calib.ece_null
and a Bernoulli(c_i) draw of the correctness are the same null. All routines
below therefore work on (confidence, correct) and are vectorised over null
draws, which makes 20,000 draws per cell cheap.

  1. Estimators   equal-mass ECE with 5..30 bins, equal-width ECE, the
                  bias-corrected squared calibration error (Kumar et al. 2019)
                  and the binning-free Kolmogorov-Smirnov calibration error
                  (Gupta et al. 2021), each with its own matched null.
  2. Coupled null a null that preserves the dependence between cells that
                  score the same clips: one uniform variate per clip decides,
                  for every model that predicts it, whether the prediction is
                  correct (comonotone coupling), optionally with a random
                  effect per subject. Gives dependence-aware p-values for
                  statistics pooled over cells and test sets.
  3. Subject-level tests  human subjects are the units for paired comparisons
                  of per-clip proper scores.
  4. Subject heterogeneity  between-subject variance of the calibration
                  residual against its null (over-dispersion, tau).
  5. Power        probability of rejecting the null for a known miscalibration
                  (a shift on the logit scale), minimum detectable calibration
                  error, and the size of the test under within-subject
                  dependence.
  6. Per-class null  matched null for the pooled per-class ECE.
  7. Seed-level summaries for the ladder (seeds, not seed-rung pairs, as units).

Pure numpy/scipy. The estimators are checked against har3d_v2 in test_har3d.py.
"""
from __future__ import annotations

from collections import defaultdict

import numpy as np
from scipy import stats

from har3d_stats import holm, sign_test

ESTIMATORS = ("aece5", "aece10", "aece15", "aece20", "aece30", "ece15", "ce2db15", "ks")
EST_LABEL = {"aece5": "adaptive ECE, 5 bins", "aece10": "adaptive ECE, 10 bins",
             "aece15": "adaptive ECE, 15 bins", "aece20": "adaptive ECE, 20 bins",
             "aece30": "adaptive ECE, 30 bins", "ece15": "equal-width ECE, 15 bins",
             "ce2db15": "debiased squared CE, 15 bins", "ks": "KS calibration error"}


# =====================================================================
# 1. Estimators on (confidence, correct), vectorised over draws
# =====================================================================
class Profile:
    """The confidences of one cell, sorted, with the bin layouts cached."""

    def __init__(self, conf):
        conf = np.asarray(conf, np.float64)
        self.n = len(conf)
        self.order = np.argsort(conf, kind="stable")
        self.c = conf[self.order]
        self._bins = {}

    def mass_bins(self, M):
        """Equal-mass bins exactly as har3d_v2._bins_equal_mass."""
        key = ("m", M)
        if key not in self._bins:
            edges = np.linspace(0, self.n, M + 1).astype(int)
            starts = edges[:-1][edges[:-1] < edges[1:]]
            sizes = np.diff(np.append(starts, self.n))
            self._bins[key] = (starts, sizes, np.add.reduceat(self.c, starts) / sizes)
        return self._bins[key]

    def width_bins(self, M):
        """Equal-width bins exactly as har3d_v2._bins_equal_width (empty bins dropped)."""
        key = ("w", M)
        if key not in self._bins:
            edges = np.linspace(0.0, 1.0, M + 1)
            idx = np.clip(np.digitize(self.c, edges[1:-1], right=True), 0, M - 1)
            starts = np.flatnonzero(np.r_[True, idx[1:] != idx[:-1]])
            sizes = np.diff(np.append(starts, self.n))
            self._bins[key] = (starts, sizes, np.add.reduceat(self.c, starts) / sizes)
        return self._bins[key]

    def sort(self, correct):
        """(n,) or (S, n) correctness in cell order -> float32 (S, n) in sorted order."""
        k = np.asarray(correct)
        k = k[None, :] if k.ndim == 1 else k
        return np.ascontiguousarray(k[:, self.order], dtype=np.float32)


def _binned(K, starts, sizes, cbar, n, kind):
    acc = np.add.reduceat(K, starts, axis=1) / sizes
    if kind == "l1":
        return (np.abs(acc - cbar) * (sizes / n)).sum(1)
    # bias-corrected squared calibration error (Kumar, Liang & Ma 2019, Eq. 5):
    # the plug-in squared gap of a bin over-estimates by acc(1-acc)/(n_b-1)
    corr = np.where(sizes > 1, acc * (1 - acc) / np.maximum(sizes - 1, 1), 0.0)
    return (((acc - cbar) ** 2 - corr) * (sizes / n)).sum(1)


def estimates(prof, K, names=ESTIMATORS):
    """Every estimator in `names` for each row of K (sorted correctness, (S, n))."""
    out = {}
    for nm in names:
        if nm.startswith("aece"):
            out[nm] = _binned(K, *prof.mass_bins(int(nm[4:])), prof.n, "l1")
        elif nm == "ece15":
            out[nm] = _binned(K, *prof.width_bins(15), prof.n, "l1")
        elif nm == "ce2db15":
            out[nm] = _binned(K, *prof.mass_bins(15), prof.n, "l2db")
        elif nm == "ks":
            # Kolmogorov-Smirnov calibration error (Gupta et al. 2021): largest
            # gap between cumulative confidence and cumulative accuracy over
            # predictions sorted by confidence; no bins
            out[nm] = np.abs(np.cumsum(prof.c[None, :] - K, axis=1, dtype=np.float64)).max(1) / prof.n
        else:
            raise KeyError(nm)
    return out


def null_correct(prof, S, rng, q=None, chunk=4_000_000):
    """S null draws of the correctness, sorted order: Bernoulli(c_i), or
    Bernoulli(q_i) when the success probabilities q (sorted order) are given."""
    q = (prof.c if q is None else np.asarray(q, np.float64)).astype(np.float32)
    out = np.empty((S, prof.n), np.float32)
    step = max(1, chunk // max(prof.n, 1))
    for a in range(0, S, step):
        b = min(S, a + step)
        out[a:b] = rng.random((b - a, prof.n), dtype=np.float32) < q
    return out


def mc_p(null_vals, obs):
    """Monte-Carlo p-value with the +1 correction (never zero)."""
    return float((1 + (np.asarray(null_vals) >= obs).sum()) / (len(null_vals) + 1))


def cell_tests(conf, correct, S=20000, seed=0, names=ESTIMATORS):
    """Observed value, matched-null mean and 95th percentile, excess and
    Monte-Carlo p-value of every estimator for one cell."""
    prof = Profile(conf)
    obs = estimates(prof, prof.sort(correct), names)
    null = estimates(prof, null_correct(prof, S, np.random.default_rng(seed)), names)
    return {nm: {"obs": float(obs[nm][0]), "null_mean": float(null[nm].mean()),
                 "null_p95": float(np.quantile(null[nm], 0.95)),
                 "null_sd": float(null[nm].std()),
                 "excess": float(obs[nm][0] - null[nm].mean()),
                 "p": mc_p(null[nm], obs[nm][0])} for nm in names}


# =====================================================================
# 2. Coupled null: cells that score the same clips are not independent
# =====================================================================
def coupled_null(cells, n_clips, S=5000, seed=0, names=("aece15",), tau=0.0,
                 subj_of_clip=None, shift=0.0):
    """
    Null draws for a group of cells that share test clips (one dataset).

    cells : list of dicts with 'conf' (n,), 'correct' (n,), 'clip' (n,) global
            clip index in [0, n_clips).
    One uniform variate per clip and draw decides the correctness of every
    prediction of that clip: u < c_i. Each cell's marginal null is unchanged
    (Bernoulli(c_i)), while a clip that comes out 'easy' in a draw is easy for
    every split, stream and seed that predicts it -- the strongest positive
    dependence compatible with the marginals, so pooled p-values built on it
    are conservative with respect to the overlap between cells.
    tau > 0 adds a subject random effect: the success probability of every
    clip of subject g is shifted by d_g ~ N(0, tau^2), redrawn per draw
    (subj_of_clip maps a clip to its subject index).
    shift != 0 draws from a MISCALIBRATED model instead (success probability
    shift_probs(c, shift)); used for the power of pooled statistics.
    Returns {name: {'obs': (ncell,), 'null': (ncell, S)}}.
    """
    rng = np.random.default_rng(seed)
    U = rng.random((S, n_clips), dtype=np.float32)
    D = None
    if tau > 0:
        if subj_of_clip is None:
            raise ValueError("tau > 0 needs subj_of_clip")
        D = rng.normal(0.0, tau, (S, int(np.max(subj_of_clip)) + 1)).astype(np.float32)
    out = {nm: {"obs": np.empty(len(cells)), "null": np.empty((len(cells), S))} for nm in names}
    for i, c in enumerate(cells):
        prof = Profile(c["conf"])
        clip = np.asarray(c["clip"])[prof.order]
        q = (prof.c if shift == 0 else shift_probs(prof.c, shift)).astype(np.float32)[None, :]
        if D is not None:
            q = np.clip(q + D[:, np.asarray(subj_of_clip)[clip]], 0.0, 1.0)
        K = (U[:, clip] < q).astype(np.float32)
        nul = estimates(prof, K, names)
        obs = estimates(prof, prof.sort(c["correct"]), names)
        for nm in names:
            out[nm]["obs"][i] = obs[nm][0]
            out[nm]["null"][i] = nul[nm]
    return out


def pooled_from_null(obs, null, units, datasets=None):
    """
    Pooled excess with a dependence-aware Monte-Carlo p-value.

    obs (ncell,), null (ncell, S) from coupled_null (cells of several datasets
    may be stacked: their draws are independent across datasets). The floor of
    a cell is its null mean; cells are averaged within their unit (test set),
    units are averaged to the grand mean. Returns the observed grand mean, its
    null s.d., the p-value P(null >= observed), the number of units with a
    positive mean excess and the null probability of at least that many, and
    the variance inflation relative to independent cells.
    """
    obs, null = np.asarray(obs, float), np.asarray(null, float)
    floor = null.mean(1)
    ex_obs, ex_null = obs - floor, null - floor[:, None]
    ukeys = list(dict.fromkeys(units))
    W = np.zeros((len(ukeys), len(obs)))
    for i, u in enumerate(units):
        W[ukeys.index(u), i] = 1.0
    W /= W.sum(1, keepdims=True)
    u_obs, u_null = W @ ex_obs, W @ ex_null
    g_obs, g_null = float(u_obs.mean()), u_null.mean(0)
    k_obs = int((u_obs > 0).sum())
    k_null = (u_null > 0).sum(0)
    # variance of the grand mean if the cells were independent
    w_cell = W.sum(0) / len(ukeys)
    var_indep = float((w_cell ** 2 * null.var(1)).sum())
    res = {"n_units": len(ukeys), "n_cells": len(obs), "mean_excess": g_obs,
           "null_sd": float(g_null.std()), "p": mc_p(g_null, g_obs),
           "k_pos": k_obs, "p_k": mc_p(k_null, k_obs),
           "z": float(g_obs / max(g_null.std(), 1e-12)),
           "var_inflation": float(g_null.var() / max(var_indep, 1e-18)),
           "unit_keys": ukeys, "unit_excess": u_obs}
    if datasets is not None:
        per = {}
        for ds in dict.fromkeys(datasets):
            m = np.array([d == ds for d in datasets])
            sub = pooled_from_null(obs[m], null[m], [u for u, k in zip(units, m) if k])
            per[ds] = {k: sub[k] for k in ("n_units", "n_cells", "mean_excess", "null_sd", "p", "k_pos", "z")}
        res["per_dataset"] = per
    return res


def _unit_weights(units):
    ukeys = list(dict.fromkeys(units))
    W = np.zeros((len(ukeys), len(units)))
    for i, u in enumerate(units):
        W[ukeys.index(u), i] = 1.0
    return W / W.sum(1, keepdims=True)


def pooled_power(groups, shifts, S=2000, alpha=0.05, seed=0, name="aece15"):
    """
    Power of the POOLED test (mean excess over test sets, coupled null) against
    a common miscalibration of every cell.

    groups : list of (cells, n_clips, units) -- one entry per dataset; cells as
             in coupled_null, units the test set of every cell.
    For each shift b the labels of every cell are drawn from the shifted model
    (same coupling between cells); the pooled statistic is compared with the
    1 - alpha quantile of its null distribution. Returns rows
    (b, eps = mean true calibration error over cells, power), b = 0 first.
    """
    def pooled(b, sd):
        tot, nu = 0.0, 0
        for gi, (cells, n_clips, units) in enumerate(groups):
            r = coupled_null(cells, n_clips, S=S, seed=sd + gi, names=(name,), shift=b)[name]
            W = _unit_weights(units)
            tot = tot + (W @ (r["null"] - floors[gi][:, None])).sum(0)
            nu += W.shape[0]
        return tot / nu

    # floors from an independent set of null draws
    floors = [coupled_null(c, n, S=S, seed=seed + 1000 + gi, names=(name,))[name]["null"].mean(1)
              for gi, (c, n, _) in enumerate(groups)]
    crit = np.quantile(pooled(0.0, seed + 2000), 1 - alpha)
    rows = []
    for j, b in enumerate((0.0,) + tuple(shifts)):
        eps = np.mean([np.abs(np.asarray(c["conf"], float) - shift_probs(c["conf"], b)).mean()
                       for cells, _, _ in groups for c in cells])
        rows.append({"b": float(b), "eps": float(eps),
                     "power": float((pooled(b, seed + 3000 + 10 * j) > crit).mean())})
    return rows


# =====================================================================
# 3. Subject-level paired tests on per-clip scores
# =====================================================================
class SubjectAccumulator:
    """Sums per-clip values by (dataset, subject) over every cell."""

    def __init__(self):
        self.s = defaultdict(float)
        self.n = defaultdict(int)

    def add(self, dataset, subjects, values):
        subjects, values = np.asarray(subjects), np.asarray(values, float)
        for g in np.unique(subjects):
            m = subjects == g
            self.s[(dataset, int(g))] += float(values[m].sum())
            self.n[(dataset, int(g))] += int(m.sum())

    def means(self):
        keys = sorted(self.s)
        return keys, np.array([self.s[k] / self.n[k] for k in keys])


def subject_test(keys, values, n_boot=20000, seed=0):
    """
    Units are human subjects. Exact sign test over subjects and a percentile
    bootstrap of the mean over subjects, resampled within each dataset.
    keys: list of (dataset, subject); values: per-subject mean difference
    (positive = the method is better).
    """
    values = np.asarray(values, float)
    k, n, p = sign_test(values)
    ds = [d for d, _ in keys]
    rng = np.random.default_rng(seed)
    boot = np.zeros(n_boot)
    for d in dict.fromkeys(ds):
        v = values[np.array([x == d for x in ds])]
        boot += v[rng.integers(0, len(v), (n_boot, len(v)))].sum(1)
    boot /= len(values)
    per = {d: (int((values[np.array([x == d for x in ds])] > 0).sum()), int(sum(x == d for x in ds)))
           for d in dict.fromkeys(ds)}
    return {"mean": float(values.mean()), "lo": float(np.quantile(boot, 0.025)),
            "hi": float(np.quantile(boot, 0.975)), "k": k, "n": n, "p": p, "per_dataset": per}


def subject_method_tests(acc_by_method, methods, n_boot=20000):
    """subject_test for every method and Holm over the methods (one family)."""
    res = {}
    for i, m in enumerate(methods):
        keys, vals = acc_by_method[m].means()
        res[m] = subject_test(keys, vals, n_boot=n_boot, seed=i)
    adj = holm([res[m]["p"] for m in methods])
    for m, a in zip(methods, adj):
        res[m]["p_holm"] = float(a)
        # The sign test speaks about the number of subjects, the mean about the size. When a
        # significant majority and the sign of the mean disagree, neither label is right.
        maj = 1 if 2 * res[m]["k"] > res[m]["n"] else -1
        res[m]["verdict"] = ("n.s." if a >= 0.05 else "mixed" if maj * res[m]["mean"] <= 0
                             else "reduces" if maj > 0 else "increases")
    return res


# =====================================================================
# 4. Subject heterogeneity of the calibration residual
# =====================================================================
def _msb(R, H, ng):
    """Between-subject mean square of the residuals R (S, n); H one-hot (n, G)."""
    gm = (R @ H) / ng
    grand = R.mean(1, keepdims=True)
    return ((gm - grand) ** 2 * ng).sum(1) / (len(ng) - 1)


def subject_heterogeneity(conf, correct, subjects, S=2000, seed=0):
    """
    Is the calibration residual (correct - confidence) more variable between
    subjects than independent labels allow? Returns the over-dispersion factor
    phi = MSB / E_null[MSB], its Monte-Carlo p-value, and the method-of-moments
    variance tau^2 of a subject effect on the success probability (one-way
    random-effects ANOVA; may be negative for a single cell, pool before use).
    """
    conf, correct = np.asarray(conf, np.float64), np.asarray(correct, np.float64)
    _, g = np.unique(subjects, return_inverse=True)
    G = int(g.max()) + 1
    if G < 2:
        return {"phi": float("nan"), "p": float("nan"), "tau2": float("nan"), "G": G}
    H = np.eye(G)[g]
    ng = H.sum(0)
    rng = np.random.default_rng(seed)
    obs = _msb((correct - conf)[None, :], H, ng)[0]
    Rn = (rng.random((S, len(conf))) < conf).astype(np.float64) - conf
    nul = _msb(Rn, H, ng)
    n0 = (len(conf) - (ng ** 2).sum() / len(conf)) / (G - 1)
    return {"phi": float(obs / nul.mean()), "p": mc_p(nul, obs),
            "tau2": float((obs - nul.mean()) / n0), "G": G}


# =====================================================================
# 5. Power of the null test
# =====================================================================
def shift_probs(conf, b):
    """True success probability when the model is miscalibrated by a shift b on
    the logit scale: b > 0 over-confident, b < 0 under-confident."""
    c = np.clip(np.asarray(conf, np.float64), 1e-6, 1 - 1e-6)
    return 1.0 / (1.0 + np.exp(-(np.log(c / (1 - c)) - b)))


B_GRID = (0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.65, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0)


def power_curve(conf, bs=B_GRID, S_null=20000, R=300, alphas=(0.05,), seed=0, name="aece15"):
    """
    Power of the matched-null test for one confidence profile.

    For every shift b (and -b) the labels are drawn with success probability
    shift_probs(conf, b); the true top-label calibration error of that model
    is eps = mean |c_i - q_i|. Power is the fraction of R replicates whose
    Monte-Carlo p-value is <= alpha. Returns a list of dicts
    (b, eps, signed gap, power per alpha), b = 0 first (the size of the test).
    """
    prof = Profile(conf)
    rng = np.random.default_rng(seed)
    null = np.sort(estimates(prof, null_correct(prof, S_null, rng), (name,))[name])
    rows = []
    for b in (0.0,) + tuple(bs) + tuple(-x for x in bs):
        q = shift_probs(prof.c, b)
        obs = estimates(prof, null_correct(prof, R, rng, q=q), (name,))[name]
        p = (1 + S_null - np.searchsorted(null, obs, side="left")) / (S_null + 1)
        rows.append({"b": float(b), "eps": float(np.abs(prof.c - q).mean()),
                     "gap": float((prof.c - q).mean()),
                     **{f"power_{a:g}": float((p <= a).mean()) for a in alphas}})
    return rows, float(null.mean()), float(null.std())


def mde(rows, alpha=0.05, side=1, target=0.8):
    """Minimum detectable calibration error: the smallest true eps at which the
    power reaches `target`, by linear interpolation along the b grid for
    over-confidence (side=1) or under-confidence (side=-1). inf if never."""
    key = f"power_{alpha:g}"
    pts = sorted(((r["eps"], r[key]) for r in rows if r["b"] == 0 or np.sign(r["b"]) == side))
    for (e0, p0), (e1, p1) in zip(pts[:-1], pts[1:]):
        if p1 >= target > p0:
            return float(e0 + (target - p0) * (e1 - e0) / max(p1 - p0, 1e-12))
        if p0 >= target:
            return float(e0)
    return float("inf")


def synthetic_conf(n, K, mean_conf, seed=0, pool=20000):
    """A deterministic confidence profile of size n for a K-class softmax model
    whose mean confidence is `mean_conf`: logits ~ N(0, s^2 I) with s set by
    bisection, n evenly spaced quantiles of the resulting confidences."""
    z = np.random.default_rng(seed).normal(size=(pool, K))

    def conf(s):
        e = np.exp(s * (z - z.max(1, keepdims=True)))
        return (e / e.sum(1, keepdims=True)).max(1)
    lo, hi = 0.0, 50.0
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        lo, hi = (mid, hi) if conf(mid).mean() < mean_conf else (lo, mid)
    return np.quantile(conf(0.5 * (lo + hi)), (np.arange(n) + 0.5) / n)


def clustered_size(conf, subjects, taus, S_null=2000, R=2000, alpha=0.05, seed=0, name="aece15"):
    """
    Size of the test when labels are NOT independent given the confidences:
    every subject g has a random effect d_g ~ N(0, tau^2) on the success
    probability of all its clips (zero mean, so the model is calibrated on
    average over subjects). Rejection rate at `alpha` of (i) the i.i.d. null
    and (ii) a null that draws the same subject effects (tau known).
    """
    prof = Profile(conf)
    g = np.unique(np.asarray(subjects)[prof.order], return_inverse=True)[1]
    G = int(g.max()) + 1
    rng = np.random.default_rng(seed)

    def draws(tau, n_draw):
        d = rng.normal(0.0, tau, (n_draw, G)) if tau > 0 else np.zeros((n_draw, G))
        q = np.clip(prof.c[None, :] + d[:, g], 0.0, 1.0)
        return (rng.random((n_draw, prof.n)) < q).astype(np.float32)

    null_iid = np.sort(estimates(prof, draws(0.0, S_null), (name,))[name])
    out = []
    for tau in taus:
        obs = estimates(prof, draws(tau, R), (name,))[name]
        null_cl = np.sort(estimates(prof, draws(tau, S_null), (name,))[name])
        p_iid = (1 + S_null - np.searchsorted(null_iid, obs, side="left")) / (S_null + 1)
        p_cl = (1 + S_null - np.searchsorted(null_cl, obs, side="left")) / (S_null + 1)
        out.append({"tau": float(tau), "size_iid": float((p_iid <= alpha).mean()),
                    "size_cluster": float((p_cl <= alpha).mean()),
                    "floor_iid": float(null_iid.mean()), "floor_cluster": float(null_cl.mean())})
    return out


# =====================================================================
# 6. Per-class ECE with a matched null
# =====================================================================
def _pc_layout(conf, pred, K, M=15):
    """Per class: indices of its predictions sorted by confidence and the
    equal-mass bins of har3d_v2.per_class_ece."""
    lay = {}
    for c in range(K):
        idx = np.flatnonzero(pred == c)
        if len(idx) < 2:
            continue
        idx = idx[np.argsort(conf[idx], kind="stable")]
        M_ = min(M, max(2, int(len(idx) // 5)))
        edges = np.linspace(0, len(idx), M_ + 1).astype(int)
        starts = edges[:-1][edges[:-1] < edges[1:]]
        sizes = np.diff(np.append(starts, len(idx)))
        lay[c] = (idx, starts, sizes, np.add.reduceat(conf[idx], starts) / sizes)
    return lay


def per_class_ece_cc(conf, pred, correct, K, M=15, layout=None):
    """har3d_v2.per_class_ece from (confidence, predicted class, correct);
    `correct` may be (n,) or (S, n). Returns (S, K) with NaN where undefined."""
    conf = np.asarray(conf, np.float64)
    k = np.asarray(correct, np.float32)
    k = k[None, :] if k.ndim == 1 else k
    lay = layout or _pc_layout(conf, np.asarray(pred), K, M)
    out = np.full((k.shape[0], K), np.nan)
    for c, (idx, starts, sizes, cbar) in lay.items():
        acc = np.add.reduceat(k[:, idx], starts, axis=1) / sizes
        out[:, c] = (np.abs(acc - cbar) * (sizes / len(idx))).sum(1)
    return out


def per_class_null(conf, pred, correct, clip, n_clips, K, S=5000, seed=0, M=15):
    """
    Matched null of the pooled per-class ECE. Predictions of the same clip by
    different splits and seeds are coupled through one uniform per clip, as in
    coupled_null. Returns per class: n predictions, observed, null mean, null
    95th percentile, p-value.
    """
    conf, pred = np.asarray(conf, np.float64), np.asarray(pred)
    lay = _pc_layout(conf, pred, K, M)
    obs = per_class_ece_cc(conf, pred, correct, K, M, lay)[0]
    U = np.random.default_rng(seed).random((S, n_clips), dtype=np.float32)
    nul = per_class_ece_cc(conf, pred, U[:, np.asarray(clip)] < conf.astype(np.float32)[None, :], K, M, lay)
    rows = []
    for c in range(K):
        if c not in lay:
            continue
        rows.append({"class": c, "n_pred": int(len(lay[c][0])),
                     "n_clips": int(len(np.unique(np.asarray(clip)[lay[c][0]]))),
                     "obs": float(obs[c]), "null_mean": float(nul[:, c].mean()),
                     "null_p95": float(np.quantile(nul[:, c], 0.95)), "p": mc_p(nul[:, c], obs[c])})
    adj = holm([r["p"] for r in rows]) if rows else []
    for r, a in zip(rows, adj):
        r["p_holm"] = float(a)
    return rows


# =====================================================================
# 7. Ladder: seeds as the units
# =====================================================================
def seed_summary(values):
    """
    One value per independently seeded training run (e.g. the difference
    between two backbones averaged over the large rungs of that seed).
    Mean, t-interval and two-sided t-test with n-1 degrees of freedom, the
    exact sign-flip (permutation) p-value and the number of positive seeds.
    """
    v = np.asarray(values, float)
    n = len(v)
    out = {"n": n, "mean": float(v.mean()) if n else float("nan"), "n_pos": int((v > 0).sum())}
    if n >= 2:
        se = v.std(ddof=1) / np.sqrt(n)
        t = stats.t.ppf(0.975, n - 1)
        out.update(sd=float(v.std(ddof=1)), lo=float(v.mean() - t * se), hi=float(v.mean() + t * se),
                   p_t=float(2 * stats.t.sf(abs(v.mean()) / max(se, 1e-300), n - 1)))
        # exact sign-flip test: all 2^n sign assignments
        signs = np.array(np.meshgrid(*[[-1, 1]] * n)).reshape(n, -1).T
        perm = np.abs((signs * np.abs(v)).mean(1))
        out["p_flip"] = float((perm >= abs(v.mean()) - 1e-15).mean())
    else:
        out.update(sd=float("nan"), lo=float("nan"), hi=float("nan"), p_t=float("nan"), p_flip=float("nan"))
    return out


In [ ]:
%%writefile /content/test_har3d.py
"""
test_har3d.py -- correctness tests.

Every assertion here guards something that, if it broke, would SILENTLY corrupt
a calibration number rather than raise. Run before trusting any output.
"""
import numpy as np

import har3d_v2 as V
import har3d_calib as C
import har3d_scale as S

PASS = FAIL = 0


def check(name, cond, extra=""):
    global PASS, FAIL
    if cond:
        PASS += 1
        print(f"  ok   {name}" + (f"  [{extra}]" if extra else ""))
    else:
        FAIL += 1
        print(f"  FAIL {name}" + (f"  [{extra}]" if extra else ""))


def _depth(j, parents):
    d, p = 0, parents[j]
    while p >= 0:
        d += 1
        p = parents[p]
    return d


def synthetic_skeleton(T=200, seed=0):
    rng = np.random.default_rng(seed)
    rest = np.zeros((20, 3), np.float32)
    L = {0: .22, 1: .22, 2: .276, 3: .218, 4: .221, 5: .221,
         7: .275, 8: .275, 9: .293, 10: .293, 11: .07, 12: .07,
         13: .544, 14: .544, 15: .427, 16: .427, 17: .17, 18: .17, 19: .235}
    D = {0: (-1, .2, 0), 1: (1, .2, 0), 2: (0, 1, 0), 3: (0, 1, 0),
         4: (-1, -.2, 0), 5: (1, -.2, 0), 7: (-.4, -1, 0), 8: (.4, -1, 0),
         9: (0, -1, 0), 10: (0, -1, 0), 11: (0, -1, 0), 12: (0, -1, 0),
         13: (0, -1, 0), 14: (0, -1, 0), 15: (0, -1, 0), 16: (0, -1, 0),
         17: (0, 0, 1), 18: (0, 0, 1), 19: (0, 1, 0)}
    for j in sorted(range(20), key=lambda k: _depth(k, V.MSR_PARENTS)):
        p = V.MSR_PARENTS[j]
        if p < 0:
            continue
        d = np.array(D[j], np.float32)
        rest[j] = rest[p] + d / np.linalg.norm(d) * L[j]
    seq = np.repeat(rest[None], T, 0)
    ang = np.linspace(0, 1.2, T)
    for t in range(T):
        for j in (10, 12):
            v = seq[t, j] - seq[t, 8]
            c, s = np.cos(ang[t]), np.sin(ang[t])
            seq[t, j] = seq[t, 8] + np.array([v[0]*c - v[1]*s, v[0]*s + v[1]*c, v[2]])
    seq += rng.normal(0, 1e-4, seq.shape)
    return seq.astype(np.float32)


print("=" * 70)
print("  har3d test suite")
print("=" * 70)

rng = np.random.default_rng(0)
seq = synthetic_skeleton()

print("\n[1] topology")
rep = V.verify_topology([seq], raise_on_fail=False)
check("corrected tree passes on a synthetic anatomical body",
      all(r["ok"] for r in rep))
try:
    V.verify_topology([seq], parents=V.LEGACY_PARENTS)
    check("legacy tree is rejected", False, "no exception raised")
except AssertionError as e:
    check("legacy tree is rejected", str(e).count("->") >= 5,
          f"{str(e).count('->')} bad edges named")
check("tree is a single rooted tree",
      V.MSR_PARENTS.count(-1) == 1 and len(V.MSR_PARENTS) == 20)
check("no self-parent", all(j != p for j, p in enumerate(V.MSR_PARENTS)))
A = V.adjacency()
check("adjacency shape (3,20,20)", A.shape == (3, 20, 20))
rs = A.sum(-1)
check("adjacency rows sum to 1 or 0",
      np.all(np.isclose(rs, 1.0, atol=1e-6) | np.isclose(rs, 0.0, atol=1e-6)))
check("self partition is the identity", np.allclose(A[0], np.eye(20)))
check("centripetal + centrifugal reproduce the bone graph",
      int(((A[1] + A[2]) > 0).sum()) == 2 * 19)
try:
    V.adjacency(parents=[-1, 0] + [5] * 3 + [-1] * 15)
    check("disconnected tree raises", False)
except ValueError:
    check("disconnected tree raises", True)

print("\n[2] NTU topology (the ladder graph)")
check("NTU tree has 25 joints and one root",
      len(S.NTU_PARENTS) == 25 and S.NTU_PARENTS.count(-1) == 1)
ok = True
for j in range(25):
    p, hops = j, 0
    while S.NTU_PARENTS[p] >= 0:
        p = S.NTU_PARENTS[p]
        hops += 1
        if hops > 25:
            ok = False
            break
check("every NTU joint reaches the root, no cycles", ok)
An = V.adjacency(S.NTU_PARENTS)
check("NTU adjacency builds and is connected", An.shape == (3, 25, 25))

print("\n[3] missing-joint imputation")
s2 = seq.copy()
truth = s2[10, 5].copy()
s2[10, 5] = 0.0
fix, miss = V.impute_missing(s2)
check("gap is detected", bool(miss[10, 5]) and miss.sum() == 1)
check("interpolated value within 1 mm of truth",
      np.linalg.norm(fix[10, 5] - truth) < 1e-3)
s3 = seq.copy()
s3[:, 7] = 0.0
fix3, miss3 = V.impute_missing(s3)
check("always-missing joint stays zero and is flagged",
      np.allclose(fix3[:, 7], 0) and miss3[:, 7].all())

print("\n[4] normalisation invariances")
n = V.normalise_skeleton(seq, centre="perframe", align=True, scale="bone")
check("perframe centring puts the root at the origin every frame",
      np.allclose(n[:, V.ROOT], 0, atol=1e-5))
t = np.array([3.0, -2.0, 7.0], np.float32)
check("translation invariance",
      np.allclose(n, V.normalise_skeleton(seq + t, centre="perframe",
                                          align=True, scale="bone"), atol=1e-4))
Q = np.linalg.qr(rng.normal(size=(3, 3)))[0].astype(np.float32)
if np.linalg.det(Q) < 0:
    Q[:, 0] *= -1
check("rotation invariance (align=True)",
      np.allclose(n, V.normalise_skeleton(seq @ Q.T, centre="perframe",
                                          align=True, scale="bone"), atol=1e-3))
check("scale invariance",
      np.allclose(n, V.normalise_skeleton(seq * 1.7, centre="perframe",
                                          align=True, scale="bone"), atol=1e-4))

print("\n[5] streams")
b_s = V.make_stream(n, "bone")
bm = V.make_stream(n, "bone_motion")
check("root bone is exactly zero", np.allclose(b_s[:, V.ROOT], 0))
check("motion first frame is zero", np.allclose(bm[0], 0))
check("bone_motion is the temporal difference of bone",
      np.allclose(bm[1:], b_s[1:] - b_s[:-1]))
check("bone differs from the legacy tree's bone stream",
      not np.allclose(b_s, V.make_stream(n, "bone", parents=V.LEGACY_PARENTS)))

print("\n[6] subject splits")
subjects = np.repeat(np.arange(1, 11), 20)
labels = np.tile(np.arange(20), 10)
sp = V.subject_splits(subjects, labels=labels, n_splits=3, seed=0, verbose=False)
ok = True
for s in sp:
    trs, vas, tes = (set(subjects[s[k]].tolist()) for k in ("train", "val", "test"))
    ok &= not (trs & vas) and not (trs & tes) and not (vas & tes)
    ok &= len(s["train"]) + len(s["val"]) + len(s["test"]) == len(subjects)
check("train/val/test subject-disjoint and exhaustive", ok)
check("validation is never a single subject",
      all(len(s["subjects"]["val"]) >= 2 for s in sp))
fx = V.fixed_split(subjects, verbose=False)
check("fixed split: train and val are subject-disjoint (the old bug)",
      not (set(subjects[fx["train"]].tolist()) & set(subjects[fx["val"]].tolist())))

print("\n[7] temperature")
K, N = 20, 4000
rs2 = np.random.default_rng(7)
z = rs2.normal(size=(N, K)) * 2.0
y = np.array([rs2.choice(K, p=V.softmax(zi)) for zi in z])
T_hat = V.fit_temperature(z / 2.5, y)
check("recovers a known temperature", abs(T_hat - 1 / 2.5) < 0.05,
      f"got {T_hat:.4f}, expected {1/2.5:.4f}")
p0, pT = V.softmax(z), V.softmax(z, T_hat)
check("temperature NEVER changes accuracy",
      float((p0.argmax(-1) == y).mean()) == float((pT.argmax(-1) == y).mean()))
check("temperature does not reorder any prediction",
      np.array_equal(p0.argmax(-1), pT.argmax(-1)))

print("\n[8] metrics")
K2, N2 = 6, 6000
r2 = np.random.default_rng(11)
zc = r2.normal(size=(N2, K2)) * 1.6
yc = np.array([r2.choice(K2, p=V.softmax(zi)) for zi in zc])
pc = V.softmax(zc)
e_big = V.adaptive_ece(pc, yc)
e_small = V.adaptive_ece(pc[:400], yc[:400])
check("a perfectly calibrated model still reports a nonzero ECE floor",
      0.0 < e_big < 0.035, f"n={N2}: {e_big:.4f}")
check("that floor shrinks as n grows (pure estimator bias)",
      e_small > 2 * e_big, f"n=400: {e_small:.4f}")
po = V.softmax(zc, 0.35)
check("overconfidence is detected", V.adaptive_ece(po, yc) > 6 * e_big)
check("MCE >= ECE by construction", V.mce(po, yc) >= V.ece(po, yc))
p2 = np.array([[0.7, 0.3], [0.2, 0.8]])
y2 = np.array([0, 1])
check("Brier matches the closed form",
      abs(V.brier(p2, y2) - ((0.09 + 0.09) + (0.04 + 0.04)) / 2) < 1e-12)
check("NLL matches the closed form",
      abs(V.nll(p2, y2) - (-(np.log(0.7) + np.log(0.8)) / 2)) < 1e-12)
check("better confidence ranking gives lower AURC",
      V.aurc(V.softmax(zc), yc) < V.aurc(V.softmax(r2.permutation(zc)), yc))

print("\n[9] the null model -- the load-bearing new result")
vals, mu, p95 = C.ece_null(pc, n_sim=120, seed=0)
check("null ECE for a calibrated model is positive, not zero", mu > 0,
      f"n={N2} K={K2}: null mean {mu:.4f}")
small = V.softmax(r2.normal(size=(200, 20)) * 1.6)
_, mu_s, _ = C.ece_null(small, n_sim=120, seed=0)
check("the null floor GROWS as n shrinks and K grows", mu_s > 3 * mu,
      f"n=200 K=20: {mu_s:.4f} vs n={N2} K={K2}: {mu:.4f}")
sig_cal = C.ece_significance(pc, yc, n_sim=120)
check("a calibrated model is NOT flagged as miscalibrated",
      not sig_cal["separable"], f"p={sig_cal['p_value']:.3f}")
yo = np.array([r2.choice(K2, p=pi) for pi in V.softmax(zc, 3.0)])
sig_bad = C.ece_significance(V.softmax(zc, 0.3), yo, n_sim=120)
check("a badly miscalibrated model IS flagged",
      sig_bad["separable"] and sig_bad["excess"] > 0,
      f"excess {sig_bad['excess']:.3f}, p={sig_bad['p_value']:.3f}")

print("\n[10] paired bootstrap and vector scaling")
pb = C.paired_bootstrap(V.softmax(zc, 0.3), pc, yc, n_boot=300)
check("paired bootstrap detects a real calibration difference",
      pb["significant"] and pb["delta"] > 0, f"delta {pb['delta']:.4f}")
pb0 = C.paired_bootstrap(pc, pc, yc, n_boot=300)
check("a method compared with itself shows zero difference",
      abs(pb0["delta"]) < 1e-12)
w, b = C.fit_vector_scaling(zc[:2000], yc[:2000], l2=1e-2)
check("vector scaling stays near the identity under L2",
      0.3 < float(np.mean(w)) < 2.0 and abs(float(np.mean(b))) < 1.0,
      f"mean w {np.mean(w):.3f}, mean b {np.mean(b):.3f}")
pv = C.apply_vector_scaling(zc[2000:], w, b)
check("vector-scaled probabilities are a valid distribution",
      np.allclose(pv.sum(-1), 1.0))

print("\n[11] MC-dropout uncertainty decomposition")
mcl = np.stack([zc + r2.normal(0, 0.4, zc.shape) for _ in range(12)], 0)
u = C.mc_uncertainty(mcl)
check("total = aleatoric + epistemic",
      np.allclose(u["total"], u["aleatoric"] + u["epistemic"], atol=1e-9))
check("epistemic uncertainty is non-negative", (u["epistemic"] > -1e-9).all())
check("mc mean probs are a valid distribution",
      np.allclose(C.mc_dropout_probs(mcl).sum(-1), 1.0))

print("\n[12] ladder plumbing (numpy paths)")
fake = np.zeros((6, 300, 150), np.float32)
fake[:, :40] = np.random.default_rng(1).normal(0, 0.3, (6, 40, 150))
r = S.reshape_ntu(fake)
check("NTU reshape gives (N, 3, T, 25, 2)", r.shape == (6, 3, 300, 25, 2))
a, b_ = S.valid_length(r[0])
check("valid_length finds the padding boundary", (a, b_) == (0, 40), f"{(a, b_)}")
vr = S.valid_resize(r[0], T=64)
check("valid_resize returns T frames", vr.shape == (3, 64, 25, 2))
bs = S.to_stream(vr, "bone")
check("NTU bone stream zeroes the root joint",
      np.allclose(bs[:, :, 20], 0))
bm2 = S.to_stream(vr, "bone_motion")
check("NTU bone_motion is the temporal difference of bone",
      np.allclose(bm2[:, 1:], bs[:, 1:] - bs[:, :-1]))
yl = np.repeat(np.arange(10), 50)
idx = S.subsample_indices(yl, 200, seed=0)
check("stratified subsample has the requested size", len(idx) == 200)
cnt = np.bincount(yl[idx], minlength=10)
check("and keeps every class represented", cnt.min() >= 15, f"min {cnt.min()}")
check("subsample is deterministic for a fixed seed",
      np.array_equal(idx, S.subsample_indices(yl, 200, seed=0)))

print("\n[13] Brier decomposition, classwise ECE, data-quality filter")
rb = np.random.default_rng(21)
zb = rb.normal(size=(800, 7)) * 1.4
pb = V.softmax(zb)
yb = np.array([rb.choice(7, p=pi) for pi in pb])
bd = V.brier_decomposition(pb, yb)
check("Murphy identity is exact: REL - RES + UNC + WB == Brier",
      abs(bd["brier"] - V.brier(pb, yb)) < 1e-10, f"{bd['brier']:.6f}")
check("reliability, resolution, uncertainty are non-negative",
      bd["rel"] >= 0 and bd["res"] >= 0 and bd["unc"] >= 0)
yb_rand = rb.integers(0, 7, 800)
check("a forecast unrelated to the labels has ~zero resolution",
      V.brier_decomposition(pb, yb_rand)["res"] < 0.5 * bd["res"] + 0.01)
cw_cal = V.classwise_ece(pb, yb)
cw_bad = V.classwise_ece(V.softmax(zb, 0.3), yb)
check("classwise ECE separates calibrated from overconfident",
      cw_bad > 2 * cw_cal, f"{cw_cal:.4f} vs {cw_bad:.4f}")
fl = np.array([[40, 0.0, 0.00, 0.0], [38, 0.0, 1.00, 0.0],
               [5, 0.0, 0.00, 0.0], [60, 0.0, 0.40, 0.0]])
check("quality mask drops all-zero and too-short sequences only",
      V.quality_mask(fl, 0.5, 10).tolist() == [True, False, False, True])


print("\n[14] shared helpers: masked tensors, Holm, sign test, ladder protocol")
import har3d_calib as C2
rng15 = np.random.default_rng(15)
seqs = [rng15.normal(0, .3, (30, 20, 3)) + np.array([0, 0, 2.5]) for _ in range(6)]
seqs[2][:25] = 0.0
off15 = np.concatenate([[0], np.cumsum([len(q) for q in seqs])])
cache15 = {"xyz": np.concatenate(seqs), "offsets": off15,
           "meta": np.array([[1 + i % 2, 1 + i // 2, 1] for i in range(6)]),
           "flags": np.array([[len(q), 0.0, float((np.abs(q) < 1e-9).all(-1).mean()), 0.0]
                              for q in seqs])}
X15, y15, s15, k15 = V.masked_tensors(cache15, "joint", 16, dict(centre="frame0", align=False))
check("masked_tensors drops exactly the tracker-loss sequence",
      len(X15) == 5 and not k15[2] and k15.sum() == 5)
check("Holm is monotone and never below raw p",
      np.all(C2.holm([0.01, 0.04, 0.03]) >= np.array([0.01, 0.04, 0.03]))
      and np.allclose(C2.holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06]))
check("sign test drops ties", C2.sign_test([1, 1, 0, -1])[:2] == (2, 3))
mu15, lo15, hi15 = C2.boot_mean_ci(rng15.normal(1, 1, 200))
check("bootstrap mean CI brackets the mean", lo15 < mu15 < hi15)
import tempfile, pathlib
from har3d_ntu import XSUB_TRAIN_SUBJECTS
tr_subj = sorted(XSUB_TRAIN_SUBJECTS)
perf15 = np.array([tr_subj[i % len(tr_subj)] for i in range(80)] + [3] * 20)
with tempfile.TemporaryDirectory() as td:
    pf = pathlib.Path(td) / "performer.txt"
    pf.write_text("\n".join(map(str, perf15)))
    y80 = np.arange(80) % 4
    vi, pi, pt, pr = S.ladder_protocol(None, y80, pf, 3, record=pathlib.Path(td) / "p.json")
    check("ladder validation is subject-disjoint from the pool",
          not set(pt[vi]) & set(pt[pi]) and len(vi) + len(pi) == 80)
    vi2, pi2, _, _ = S.ladder_protocol(None, y80, pf, 3, record=pathlib.Path(td) / "p.json")
    check("ladder protocol is reproducible across sessions", np.array_equal(pi, pi2))
    try:
        S.ladder_protocol(None, np.arange(79) % 4, pf, 3)
        check("ladder protocol refuses a performer/row mismatch (no silent fallback)", False)
    except RuntimeError:
        check("ladder protocol refuses a performer/row mismatch (no silent fallback)", True)

print("\n[15] Monte-Carlo p-values are never zero")
_p = np.full((300, 5), 0.2); _p[:, 0] = 0.99; _p /= _p.sum(1, keepdims=True)
_sig = C.ece_significance(_p, np.ones(300, int), n_sim=200, seed=0)
check("null-test p-value has the +1 floor", abs(_sig["p_value"] - 1 / 201) < 1e-12, _sig["p_value"])
_pb = C.paired_bootstrap(_p, np.full_like(_p, 0.2), np.ones(300, int), n_boot=100)
check("paired-bootstrap p-value is never zero", _pb["p_value"] >= 2 / 101 - 1e-12, _pb["p_value"])

_rng = np.random.default_rng(7)
for _n, _K in ((97, 10), (16487, 60)):
    _pp = V.softmax(_rng.normal(0, 2, (_n, _K))); _yy = _rng.integers(0, _K, _n)
    _fast = C.adaptive_ece_cc(_pp.max(-1), (_pp.argmax(-1) == _yy))
    check(f"fast adaptive ECE equals the reference (n={_n})", abs(_fast - V.adaptive_ece(_pp, _yy)) < 1e-12)

print("\n[16] inference over coarse units: test sets and seed-rung pairs (har3d_stats)")
import har3d_stats as HS
check("Holm and the sign test have one definition", C.holm is HS.holm and C.sign_test is HS.sign_test)
_u, _m = HS.unit_means([1.0, 3.0, 10.0, 20.0, 5.0], ["a", "a", "b", "b", "a"])
check("unit means average within units, in order of first appearance",
      _u == ["a", "b"] and np.allclose(_m, [3.0, 15.0]))
_cb = HS.cluster_boot_ci([0.2] * 12, n_boot=500)
check("cluster bootstrap of identical units is degenerate", np.allclose(_cb, (0.2, 0.2)))
_x = np.random.default_rng(4).normal(1, 1, 40)
_lo, _hi = HS.cluster_boot_ci(_x, n_boot=2000)
check("cluster bootstrap interval brackets the mean of the units", _lo < _x.mean() < _hi)
_b, _blo, _bhi, _r2 = HS.ols_loglog([500, 1000, 2000, 5000, 10000], [0.1 * (n / 500) ** -0.4 for n in (500, 1000, 2000, 5000, 10000)])
check("OLS log-log recovers an exact power law", abs(_b + 0.4) < 1e-9 and _r2 > 1 - 1e-12 and _blo <= _b <= _bhi)
_pc = []
for _ds in ("d1", "d2", "d3", "d4"):
    for _sp in ("s0", "s1", "s2"):
        for _st in ("joint", "bone"):
            for _sd in (0, 1):
                _pc.append(dict(method="raw", dataset=_ds, stream=_st, split=_sp, seed=_sd, ece=0.12))
                _pc.append(dict(method="good", dataset=_ds, stream=_st, split=_sp, seed=_sd, ece=0.10))
            _pc.append(dict(method="ens", dataset=_ds, stream=_st, split=_sp, seed=-1, ece=0.13))
_res, _eci = HS.method_comparison(_pc, ["good", "ens"], n_boot=500)
check("method comparison counts test sets, not cells", _res["good"]["n"] == 12 and _res["ens"]["n"] == 12)
check("ensemble is paired with the seed-mean of the baseline", abs(_res["ens"]["mean"] + 0.01) < 1e-12)
check("twelve concordant test sets give the exact two-sided sign-test p",
      abs(_res["good"]["p"] - 2 / 2 ** 12) < 1e-12 and _res["good"]["k"] == 12 and _res["ens"]["k"] == 0)
check("Holm over the methods, verdict from the sign of the mean",
      abs(_res["good"]["p_holm"] - 4 / 2 ** 12) < 1e-12 and _res["good"]["verdict"] == "reduces"
      and _res["ens"]["verdict"] == "increases")
_E = {(m, n, s): (0.05 if m == "a" else 0.02) + 0.001 * s for m in ("a", "b") for n in (1, 2, 3) for s in (0, 1, 2)}
_sl = HS.seed_level_tests(_E, [("a", "b")], {"low": [1], "high": [2, 3]})
check("seed-level tests pool seed-rung pairs per regime",
      [r["n_pairs"] for r in _sl] == [3, 6] and all(r["n_a_higher"] == r["n_pairs"] for r in _sl))
check("seed-level tests are Holm-corrected as one family",
      abs(_sl[1]["p_holm"] - 2 * 2 / 2 ** 6) < 1e-12 and abs(_sl[0]["p_holm"] - 0.25) < 1e-12)
_rows = [dict(dataset="ntu60", model="agcn", n_train=2000, seed=0, best_step=1000, ece_adaptive=0.05, top1=0.50),
         dict(dataset="ntu60", model="agcn_frozen", n_train=2000, seed=0, best_step=1000, ece_adaptive=0.057, top1=0.51)]
_g = HS.identical_run_gap(_rows)
check("identical-run gap is measured when both runs restore the same pre-freeze step",
      _g is not None and abs(_g["ece"] - 0.007) < 1e-12 and abs(_g["top1"] - 0.01) < 1e-12)
_rows[1]["best_step"] = 1250
check("identical-run gap refuses runs that diverged after the freeze", HS.identical_run_gap(_rows) is None)
_pe = HS.pooled_excess([dict(dataset="x", split=s, null_excess=v) for s, v in
                        (("a", 0.02), ("a", -0.01), ("b", 0.03), ("c", 0.01))])
check("pooled excess averages within test sets before the sign test",
      _pe["n"] == 3 and _pe["k_pos"] == 3 and np.allclose(_pe["unit_excess"], [0.005, 0.03, 0.01]))
# methods are tested on the score they are given (Brier, NLL, excess), not on ECE
for _r in _pc:
    _r["brier"] = {"raw": 0.40, "good": 0.42, "ens": 0.38}[_r["method"]]
_resb, _ = HS.method_comparison(_pc, ["good", "ens"], n_boot=500, key="brier")
check("method comparison tests the score it is given",
      _resb["good"]["verdict"] == "increases" and _resb["ens"]["verdict"] == "reduces"
      and _res["good"]["verdict"] == "reduces")
check("method comparison reports the agreement within each dataset",
      _resb["ens"]["per_dataset"] == {d: (3, 3) for d in ("d1", "d2", "d3", "d4")})
_E2 = {("m", 1, s): v for s, v in enumerate((0.10, 0.12, 0.14))}
_E2.update({("m", 2, s): v for s, v in enumerate((0.20, 0.20, 0.20))})
_sd2, _ng2 = HS.pooled_seed_sd(_E2, ["m"], [1, 2])
check("pooled seed s.d. is the root mean square of the per-group s.d.",
      _ng2 == 2 and abs(_sd2 - np.sqrt((0.02 ** 2 + 0.0) / 2)) < 1e-12)
_pf = V.softmax(np.random.default_rng(11).normal(0, 1.5, (300, 10)))
_f_raw = C.ece_null(_pf, n_sim=300, seed=0)[1]
_f_sharp = C.ece_null(C.sharpen_probs(_pf, 0.3), n_sim=300, seed=0)[1]
check("the floor moves with the confidences: sharpening lowers it", _f_sharp < _f_raw, (_f_raw, _f_sharp))

print("\n[18] revision analyses (har3d_review)")
import har3d_review as RV
_rg = np.random.default_rng(21)
_p = V.softmax(_rg.normal(0, 2.0, (400, 12)))
_y = np.array([_rg.choice(12, p=q) for q in _p])
_conf, _corr = _p.max(-1), (_p.argmax(-1) == _y)
_prof = RV.Profile(_conf)
_e = RV.estimates(_prof, _prof.sort(_corr))
check("vectorised equal-mass ECE equals the reference for 5, 15 and 30 bins",
      all(abs(_e[f"aece{M}"][0] - V.adaptive_ece(_p, _y, M)) < 1e-6 for M in (5, 15, 30)))
check("vectorised equal-width ECE equals the reference", abs(_e["ece15"][0] - V.ece(_p, _y)) < 1e-6)
_ks = np.abs(np.cumsum((_conf - _corr)[np.argsort(_conf, kind="stable")])).max() / len(_conf)
check("KS calibration error is the largest cumulative confidence-accuracy gap",
      abs(_e["ks"][0] - _ks) < 1e-6)
_pc_ref = V.per_class_ece(_p, _y)
_pc_new = RV.per_class_ece_cc(_conf, _p.argmax(-1), _corr, 12)[0]
check("per-class ECE from (confidence, prediction, correct) equals the reference",
      np.allclose(np.nan_to_num(_pc_ref, nan=-1), np.nan_to_num(_pc_new, nan=-1), atol=1e-6))
_ct = RV.cell_tests(_conf, _corr, S=2000, seed=0)
_ref_floor = C.ece_null(_p, n_sim=400, seed=0)[1]
check("Bernoulli(confidence) null reproduces the floor of the full-vector label draw",
      abs(_ct["aece15"]["null_mean"] - _ref_floor) < 0.004, (_ct["aece15"]["null_mean"], _ref_floor))
check("Monte-Carlo p-values are never zero and at most one",
      RV.mc_p(np.zeros(99), 1.0) == 0.01 and RV.mc_p(np.ones(99), 0.0) == 1.0)
_nd = RV.estimates(_prof, RV.null_correct(_prof, 4000, np.random.default_rng(1)), ("ce2db15", "aece15"))
check("the debiased squared calibration error is centred on zero under the null",
      abs(_nd["ce2db15"].mean()) < 0.1 * _nd["aece15"].mean() ** 2 + 2e-4, _nd["ce2db15"].mean())
# coupled null: marginals unchanged, cells that share clips move together
_clipA = np.arange(400)
_cells = [dict(conf=_conf, correct=_corr, clip=_clipA),
          dict(conf=_conf, correct=_corr, clip=_clipA),
          dict(conf=_conf, correct=_corr, clip=_clipA + 400)]
_cn = RV.coupled_null(_cells, 800, S=3000, seed=0)["aece15"]
check("coupled null keeps the marginal floor of every cell",
      np.allclose(_cn["null"].mean(1), _ct["aece15"]["null_mean"], atol=0.003))
check("coupled null: cells on the same clips are tied, cells on other clips are not",
      np.corrcoef(_cn["null"][0], _cn["null"][1])[0, 1] > 0.99
      and abs(np.corrcoef(_cn["null"][0], _cn["null"][2])[0, 1]) < 0.08)
_pl = RV.pooled_from_null(_cn["obs"], _cn["null"], ["u1", "u1", "u2"], ["d", "d", "d"])
check("pooled test reports the inflation of the variance by shared clips",
      1.2 < _pl["var_inflation"] < 2.3 and _pl["n_units"] == 2 and 0 < _pl["p"] <= 1,
      _pl["var_inflation"])
# power: size at b = 0, monotone, and the minimum detectable error is finite
_cs = RV.synthetic_conf(300, 20, 0.6)
check("synthetic confidence profile has the requested mean", abs(_cs.mean() - 0.6) < 0.01, _cs.mean())
_rows, _, _ = RV.power_curve(_cs, bs=(0.5, 1.0, 2.0), S_null=4000, R=400, seed=0)
_pw = {r["b"]: r["power_0.05"] for r in _rows}
check("null test has its nominal size", 0.02 < _pw[0.0] < 0.09, _pw[0.0])
check("power grows with the miscalibration", _pw[0.0] < _pw[0.5] < _pw[1.0] <= _pw[2.0] and _pw[2.0] > 0.95, _pw)
_m = RV.mde(_rows)
check("minimum detectable calibration error lies between the tested errors",
      0 < _m < max(r["eps"] for r in _rows), _m)
_sub = np.repeat(np.arange(4), 75)
_cz = RV.clustered_size(_cs, _sub, (0.0, 0.15), S_null=1500, R=600, seed=0)
check("subject effects inflate the size of the i.i.d. null; the cluster-aware null repairs it",
      _cz[0]["size_iid"] < 0.1 and _cz[1]["size_iid"] > 0.3 and _cz[1]["size_cluster"] < 0.1,
      [(r["size_iid"], r["size_cluster"]) for r in _cz])
_het0 = RV.subject_heterogeneity(_cs, np.random.default_rng(3).random(300) < _cs, _sub, S=1000)
_d = np.array([0.25, -0.25, 0.2, -0.2])[_sub]
_het1 = RV.subject_heterogeneity(_cs, np.random.default_rng(3).random(300) < np.clip(_cs + _d, 0, 1), _sub, S=1000)
check("subject heterogeneity: over-dispersion detected only when subjects differ",
      _het0["p"] > 0.01 and _het1["p"] < 0.01 and _het1["tau2"] > 0.01, (_het0["phi"], _het1["phi"]))
_ss = RV.seed_summary([0.03, 0.04, 0.02, 0.05, 0.01])
check("five seeds of one sign give the smallest exact sign-flip p-value, 2/32",
      abs(_ss["p_flip"] - 0.0625) < 1e-12 and _ss["n_pos"] == 5 and _ss["lo"] > 0)
_acc = RV.SubjectAccumulator()
_acc.add("d1", np.array([1, 1, 2, 2]), np.array([1.0, 3.0, -1.0, -3.0]))
_acc.add("d1", np.array([1]), np.array([5.0]))
_k, _v = _acc.means()
check("subject accumulator averages per subject over every cell",
      _k == [("d1", 1), ("d1", 2)] and np.allclose(_v, [3.0, -2.0]))
_st = RV.subject_test([("d", i) for i in range(9)], np.full(9, 0.1) + np.arange(9) * 1e-3, n_boot=500)
check("subject-level test: nine subjects of one sign", _st["k"] == 9 and _st["p"] < 0.01 and _st["lo"] > 0)
_am = RV.SubjectAccumulator()
_am.add("d", np.arange(12), np.r_[np.full(11, 0.01), -1.0])          # better for 11 of 12 subjects, worse on average
_vm = RV.subject_method_tests({"m": _am}, ["m"], n_boot=200)["m"]
check("a significant majority with a mean of the opposite sign is labelled mixed",
      _vm["k"] == 11 and _vm["mean"] < 0 and _vm["verdict"] == "mixed", _vm["verdict"])

print("\n[17] backbones and ladder trainer (torch)")
try:
    import torch
    import har3d_backbones as HB
    torch.manual_seed(0)
    xt = np.random.default_rng(3).normal(0, 0.3, (6, 3, 16, 25, 2)).astype(np.float32)
    yt = np.array([0, 1, 2, 0, 1, 2])
    for arch in ("stgcn", "agcn", "ctrgcn"):
        m = HB.build(arch, 3, S.NTU_PARENTS, base=16)
        with torch.no_grad():
            o = m(torch.as_tensor(xt))
        check(f"{arch} forward gives (N, K)", tuple(o.shape) == (6, 3))
    check("ST-GCN has no adaptive-graph parameters",
          len(HB.graph_parameters(HB.build("stgcn", 3, S.NTU_PARENTS, base=16))) == 0)
    m = HB.build("agcn", 3, S.NTU_PARENTS, base=16)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-2, weight_decay=1e-4)
    HB.freeze_graph(m)
    g0 = [q.clone() for q in HB.graph_parameters(m)]
    for _ in range(2):
        opt.zero_grad(set_to_none=True)
        torch.nn.functional.cross_entropy(m(torch.as_tensor(xt)),
                                          torch.as_tensor(yt)).backward()
        opt.step()
    check("freezing stops every adaptive-graph parameter from updating",
          all(torch.equal(a, b) for a, b in zip(g0, HB.graph_parameters(m))))
    r = S.train_rung(xt, yt, xt, yt, xt, yt, 3, arch="stgcn", max_steps=12,
                     eval_every=4, patience=2, min_steps=4, bs=4, seed=0,
                     verbose=False)
    check("early-stopping rung restores a step no later than it ran",
          1 <= r["info"]["best_step"] <= r["info"]["steps_run"])
    # bit-for-bit comparison of two trainings: on the CPU, where arithmetic is
    # reproducible (two GPU runs of one seed can differ in the last bits)
    r1 = S.train_rung(xt, yt, xt, yt, xt, yt, 3, arch="stgcn", max_steps=12,
                      eval_every=4, patience=2, min_steps=4, bs=4, seed=0,
                      verbose=False, device="cpu")
    r2 = S.train_rung(xt, yt, xt, yt, xt, yt, 3, arch="stgcn", max_steps=12,
                      eval_every=4, patience=2, min_steps=4, bs=4, seed=0,
                      verbose=False, snapshot_steps=(4, 8, 24), track_best_acc=True,
                      run_until=24, device="cpu")
    check("snapshots, the best-accuracy checkpoint and run_until leave the main output unchanged",
          np.array_equal(r1["test"][0], r2["test"][0]) and np.array_equal(r1["val"][0], r2["val"][0])
          and r1["info"]["best_step"] == r2["info"]["best_step"]
          and r1["info"]["steps_run"] == r2["info"]["steps_run"])
    check("matched-step snapshots and the best-accuracy checkpoint are returned",
          {"s4", "s8", "acc"} <= set(r2["extra"]) and r2["extra"]["s4"]["logits"].shape == (6, 3)
          and r2["info"]["steps_total"] >= r2["info"]["steps_run"])
except ImportError:
    print("  (torch not installed -- backbone tests skipped)")

print("\n" + "=" * 70)
print(f"  {PASS} passed, {FAIL} failed")
print("=" * 70)
raise SystemExit(1 if FAIL else 0)


In [ ]:
%%writefile /content/make_numbers.py
"""
make_numbers.py RESULTS_DIR OUT_DIR

Writes every number, table and caption fragment the manuscript quotes, from a
results folder of the notebook (runs/final/results):

  OUT_DIR/numbers.tex              \\newcommand macros used in the prose
  OUT_DIR/tab_*.tex                the tables
  OUT_DIR/methods_split_level.json what make_figures.py needs for Fig. 4

Inference is made over coarse units only (har3d_stats.py): test sets
(dataset x split) on the small datasets, which share subjects within a dataset
so their p-values are approximate, and seed-rung pairs on the ladder. The
manuscript contains no typed result that is not produced here; only constants
of the protocol (bin counts, seeds, grid values) are typed.
"""
import csv
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import har3d_stats as HS  # noqa: E402

R = Path(sys.argv[1])
O = Path(sys.argv[2])
O.mkdir(parents=True, exist_ok=True)
T = R / "tables"
rows = json.loads((R / "metrics.json").read_text())
# replication and environment re-runs (revision) are reported from their own tables
rows = [r for r in rows if not r["model"].endswith(("_rep", "_env"))]
MAC = {}


def mac(name, val):
    assert name.isalpha(), name
    MAC[name] = str(val)


def f3(x):
    return f"{x:.3f}"


def f2(x):
    return f"{x:.2f}"


def sgn(x, d=3):
    if abs(x) < 0.5 * 10 ** (-d):
        return f"{0:.{d}f}"
    return f"{x:+.{d}f}".replace("-", "$-$").replace("+", "$+$")


def neg(s):
    return s.replace("-", "$-$")


def pfmt(p):
    return "$<$0.001" if p < 1e-3 else f"{p:.3f}"


def prel(p):
    """p with its relation, for prose: '\\,=\\,0.012' or '\\,$<$\\,0.001'."""
    return r"\,$<$\,0.001" if p < 1e-3 else r"\,=\," + f"{p:.3f}"


def thou(n):
    return f"{int(n):,}".replace(",", "{,}")


def read_csv(name):
    return list(csv.DictReader(open(T / name)))


DS = ["action3d", "daily", "utkinect"]
DSN = {"action3d": "Action", "daily": "Daily", "utkinect": "UTKinect"}
DSD = {"action3d": "Action3D", "daily": "Daily", "utkinect": "UTKinect"}
ST = ["joint", "bone", "joint_motion", "bone_motion"]
STN = {"joint": "joint", "bone": "bone", "joint_motion": "joint-mot.", "bone_motion": "bone-mot."}
small = [r for r in rows if r["model"] == "stgcn" and r["dataset"] in DS]
assert len(small) == 240, len(small)
ntu = [r for r in rows if r["dataset"] == "ntu60"]
Lr = {(r["model"], r["n_train"], r["seed"]): r for r in ntu}

# ====================================================================== data
filt = json.loads((R / "data_filtering.json").read_text())
mac("nActionKept", filt["action3d"]["n_kept"])
mac("nActionDrop", filt["action3d"]["n_dropped"])
mac("nDailyKept", filt["daily"]["n_kept"])
mac("nUTKept", filt["utkinect"]["n_kept"])
proto = json.loads((R / "ladder_protocol.json").read_text())
mac("nNTUval", thou(proto["n_val"]))
mac("nNTUpool", thou(proto["n_pool"]))
mac("NTUvalSubjects", ", ".join(str(s) for s in proto["val_subjects"]))
_ns_a = sorted({r["n"] for r in small if r["dataset"] == "action3d"})
mac("nActionTestMin", _ns_a[0])
mac("nActionTestMax", _ns_a[-1])

# ====================================================================== RQ1
# per-experiment null tests, Holm over all 240 cells
padj = HS.holm([r["null_p_value"] for r in small])
SEP = {(r["dataset"], r["stream"], r["split"], r["seed"]): (r["null_p_value"], a) for r, a in zip(small, padj)}
nH = sum(a < 0.05 for _, a in SEP.values())
nR = sum(p < 0.05 for p, _ in SEP.values())
mac("nSmallCells", len(SEP))
mac("nSepHolm", nH)
mac("nSepRaw", nR)
mac("pctSepHolm", f"{100 * nH / len(SEP):.1f}")
mac("nExpectedRaw", int(round(0.05 * len(SEP))))
for ds in DS:
    mac(f"nSep{DSN[ds]}", sum(a < 0.05 for k, (_, a) in SEP.items() if k[0] == ds))
mac("nSepActionMotion", sum(a < 0.05 for k, (_, a) in SEP.items() if k[0] == "action3d" and "motion" in k[1]))
mac("nSims", thou(small[0]["null_sims"]))
mac("minHolmP", f"{len(SEP) / (small[0]['null_sims'] + 1):.3f}")

# RQ1 in aggregate: test sets are the units
pe = HS.pooled_excess(small)
mac("nTestSets", pe["n"])
mac("nTestSetsPos", pe["k_pos"])
mac("pTestSetsPos", pfmt(pe["p"]))
mac("meanExcessPooled", f3(pe["mean"]))
# the test sets of a dataset share subjects, so agreement within each dataset is reported too
for ds in DS:
    mac(f"tsPos{DSN[ds]}", "{}/{}".format(*pe["per_dataset"][ds]))


def by_split(sub, key):
    g = defaultdict(list)
    for r in sub:
        g[r["split"]].append(r[key])
    return float(np.mean([np.mean(v) for v in g.values()]))


lines = []
for ds in DS:
    for st in ST:
        sub = [r for r in small if r["dataset"] == ds and r["stream"] == st]
        t, e = by_split(sub, "top1"), by_split(sub, "ece_adaptive")
        fl = np.mean([r["null_null_mean"] for r in sub])
        ex = np.mean([r["null_excess"] for r in sub])
        ns = sum(SEP[(r["dataset"], r["stream"], r["split"], r["seed"])][1] < 0.05 for r in sub)
        lines.append(f"{DSD[ds]} & {STN[st]} & {sub[0]['n_classes']} & {f3(t)} & {f3(e)} & {f3(fl)} & "
                     f"{sgn(ex)} & {ns}/{len(sub)} & {f2(np.mean([r['temperature'] for r in sub]))} & "
                     f"{f3(np.mean([r['brier'] for r in sub]))}\\\\")
        if st == "joint":
            mac(f"floor{DSN[ds]}Joint", f3(fl))
            mac(f"ece{DSN[ds]}Joint", f3(e))
    if ds != DS[-1]:
        lines.append("\\addlinespace[2pt]")
(O / "tab_small.tex").write_text(r"""\begin{table*}[t]
\caption{Small datasets, uncalibrated ST-GCN: mean over the 20 cells of each
dataset and stream (4 subject-disjoint splits $\times$ 5 seeds). Top-1 and
$\aece$ average seeds within a split, then splits. ``Null'' is the matched
estimator floor and ``excess'' the observed $\aece$ minus that floor; ``sep.''
counts cells whose observed $\aece$ is separable from the null after Holm
correction over all """ + str(len(SEP)) + r""" cells. $T$ is the validation-fitted
temperature ($T<1$: under-confident).}
\label{tab:small}
\centering\footnotesize
\begin{tabular}{@{}llcccccccc@{}}
\toprule
Dataset & Stream & $K$ & top-1 & $\aece$ & null & excess & sep. & $T$ & Brier\\
\midrule
""" + "\n".join(lines) + r"""
\bottomrule
\end{tabular}
\end{table*}
""")

# temperatures
mac("nUnderconf", sum(r["temperature"] < 1 for r in small))
mac("meanTSmall", f2(np.mean([r["temperature"] for r in small])))
for m, name in (("stgcn_ls", "LS"), ("stgcn_focal", "Focal")):
    s = [r for r in rows if r["model"] == m]
    mac(f"meanT{name}", f2(np.mean([r["temperature"] for r in s])))
    mac(f"nUnderconf{name}", sum(r["temperature"] < 1 for r in s))
small_all = [r for r in rows if r["dataset"] != "ntu60"]
mac("nTbound", sum(r["temperature"] <= 0.0501 for r in small))
mac("nTboundAll", sum(r["temperature"] <= 0.0501 for r in small_all))
mac("nEpochCap", sum((r["epochs_run"] or 0) >= 120 for r in small_all))
mac("nSmallAll", len(small_all))
_tj = defaultdict(list)
for r in small:
    if r["stream"] == "joint":
        _tj[r["dataset"]].append(r["temperature"])
for ds in DS:
    mac(f"Tj{DSN[ds]}Min", f2(min(_tj[ds])))
    mac(f"Tj{DSN[ds]}Max", f2(max(_tj[ds])))

# worked example: the cells are fixed choices; their values are read from the run
M = {(r["dataset"], r["stream"], r["split"], r["seed"]): r for r in small}
for tag, key in (("WA", ("action3d", "bone_motion", "fixed_frame0", 3)),
                 ("WB", ("daily", "bone", "rand0_frame0", 1))):
    r = M[key]
    mac(f"{tag}obs", f3(r["null_observed"]))
    mac(f"{tag}null", f3(r["null_null_mean"]))
    mac(f"{tag}pninetyfive", f3(r["null_null_p95"]))
    mac(f"{tag}excess", f3(r["null_excess"]))
    mac(f"{tag}p", pfmt(r["null_p_value"]))
    mac(f"{tag}n", r["n"])
    mac(f"{tag}holm", "separable" if SEP[key][1] < 0.05 else "not separable")
_wa = [M[("action3d", "bone_motion", "fixed_frame0", s)]["null_excess"] for s in range(5)]
assert M[("action3d", "bone_motion", "fixed_frame0", 3)]["null_excess"] == max(_wa), "worked example A is described as the largest of its five seeds"

# ====================================================================== RQ4 (per class)
for ds in DS:
    sub = [r for r in small if r["dataset"] == ds and r["stream"] == "joint"]
    mac(f"pc{DSN[ds]}Mean", f3(np.nanmean([r["per_class_ece_mean"] for r in sub])))
    mac(f"pc{DSN[ds]}Max", f3(np.nanmean([r["per_class_ece_max"] for r in sub])))
    mac(f"cw{DSN[ds]}", f3(np.mean([r["classwise_ece"] for r in sub])))
pcp = read_csv("per_class_pooled.csv")
for ds in DS:
    v = [float(x["per_class_ece"]) for x in pcp if x["dataset"] == ds]
    mac(f"pcPool{DSN[ds]}Min", f2(min(v)))
    mac(f"pcPool{DSN[ds]}Max", f2(max(v)))

# ====================================================================== ladder
lad = read_csv("ladder_by_model.csv")
L = {(x["model"], int(x["n_train"])): x for x in lad}
NS = sorted({int(x["n_train"]) for x in lad})
top = NS[-1]
mac("nTop", thou(top))
mac("foldRange", f"{NS[-1] / NS[0]:.0f}")
cols = [("stgcn", "ST-GCN"), ("agcn", "2s-AGCN"), ("ctrgcn", "CTR-GCN"),
        ("agcn_frozen", "AGCN frozen"), ("stgcn_fixed", "ST-GCN 6k")]
body = []
for n in NS:
    cells = []
    for m, _ in cols:
        x = L.get((m, n))
        cells.append("-- & --" if x is None else f"{float(x['top1_mean']):.3f} & {float(x['excess_mean']):.3f}")
    fl = np.mean([float(L[(m, n)]["null_floor_mean"]) for m, _ in cols if (m, n) in L])
    body.append(thou(n) + f" & {fl:.4f} & " + " & ".join(cells) + r"\\")
head = " & ".join(f"\\multicolumn{{2}}{{c}}{{{nm}}}" for _, nm in cols)
cm = "".join(f"\\cmidrule(lr){{{3 + 2 * i}-{4 + 2 * i}}}" for i in range(len(cols)))
(O / "tab_ladder.tex").write_text(r"""\begin{table*}[t]
\caption{NTU-60 training-set ladder (fixed X-Sub test set of 16{,}487 clips),
mean over three seeds. Each entry gives top-1 and the excess adaptive ECE
over the matched null; ``floor'' is the mean null $\aece$ at that rung. All
models except the last column are trained to their validation-NLL optimum;
``ST-GCN 6k'' uses the fixed 6{,}000-step budget, and ``AGCN frozen'' stops
updating its graph parameters after 1{,}000 steps.}
\label{tab:ladder}
\centering\footnotesize\setlength{\tabcolsep}{4pt}
\begin{tabular}{@{}rc""" + "cc" * len(cols) + r"""@{}}
\toprule
 & & """ + head + r"""\\
""" + cm + r"""
$n_{\mathrm{tr}}$ & floor & """ + " & ".join(["top-1 & exc."] * len(cols)) + r"""\\
\midrule
""" + "\n".join(body) + r"""
\bottomrule
\end{tabular}
\end{table*}
""")


def L_(m, n, k):
    return float(L[(m, n)][k])


for m, nm in (("stgcn", "ST"), ("agcn", "AG"), ("ctrgcn", "CT"), ("agcn_frozen", "FR"), ("stgcn_fixed", "FX")):
    ns = sorted(n for (mm, n) in L if mm == m)
    mac(f"exc{nm}First", f3(L_(m, ns[0], "excess_mean")))
    mac(f"exc{nm}Top", f3(L_(m, top, "excess_mean")))
    mac(f"acc{nm}First", f3(L_(m, ns[0], "top1_mean")))
    mac(f"acc{nm}Top", f3(L_(m, top, "top1_mean")))
    mac(f"T{nm}First", f2(L_(m, ns[0], "temperature_mean")))
    mac(f"T{nm}Top", f2(L_(m, top, "temperature_mean")))
    mac(f"rel{nm}First", f3(L_(m, ns[0], "brier_rel_mean")))
    mac(f"rel{nm}Top", f3(L_(m, top, "brier_rel_mean")))
    mac(f"ts{nm}Top", f3(L_(m, top, "ece_adaptive_ts_mean")))
_tn = [r["temperature"] for r in ntu if r["model"] in ("stgcn", "agcn", "ctrgcn")]
mac("nNTUruns", len(_tn))
mac("nNTUunder", sum(t < 1 for t in _tn))
mac("excSTtwok", f3(L_("stgcn", 2000, "excess_mean")))
mac("excFXfivek", f3(L_("stgcn_fixed", 5000, "excess_mean")))
mac("TFXfivek", f2(L_("stgcn_fixed", 5000, "temperature_mean")))
mac("TSTfivek", f2(L_("stgcn", 5000, "temperature_mean")))
mac("accSTtwentyk", f3(L_("stgcn", 20000, "top1_mean")))
mac("excSTtwentyk", f3(L_("stgcn", 20000, "excess_mean")))
mac("accGapAGTop", f3(L_("stgcn", top, "top1_mean") - L_("agcn", top, "top1_mean")))
mac("accGapCTTop", f3(L_("stgcn", top, "top1_mean") - L_("ctrgcn", top, "top1_mean")))
for m, nm in (("agcn", "AG"), ("ctrgcn", "CT")):
    ex = [L_(m, n, "excess_mean") for n in NS]
    mac(f"exc{nm}Min", f3(min(ex)))
    mac(f"exc{nm}Max", f3(max(ex)))
ts = [L_(m, n, "ece_adaptive_ts_mean") for (m, n) in L if m in ("stgcn", "agcn", "ctrgcn")]
mac("tsLadderMin", f3(min(ts)))
mac("tsLadderMax", f3(max(ts)))
_ts_cells = [r["ece_adaptive_temp"] for r in ntu if r["model"] in ("stgcn", "agcn", "ctrgcn")]
mac("tsCellMin", f3(min(_ts_cells)))
mac("tsCellMax", f3(max(_ts_cells)))
fl = [float(x["null_floor_mean"]) for x in lad]
mac("floorLadderMin", f"{min(fl):.4f}")
mac("floorLadderMax", f"{max(fl):.4f}")
_rat = [L_(m, n, "excess_mean") / L_(m, n, "null_floor_mean") for m in ("stgcn", "agcn", "ctrgcn") for n in NS]
mac("ratioMin", f"{min(_rat):.0f}")
mac("ratioMax", f"{max(_rat):.0f}")

conv = {(x["model"], int(x["n_train"])): x for x in read_csv("ladder_convergence.csv")}
for m, nm in (("stgcn", "ST"), ("agcn", "AG"), ("ctrgcn", "CT")):
    mac(f"step{nm}Top", f"{float(conv[(m, top)]['best_step_mean']):,.0f}".replace(",", "{,}"))
mac("epochsFXFirst", f"{float(conv[('stgcn_fixed', NS[0])]['epochs_at_best_mean']):.0f}")
mac("epochsFXTop", f"{float(conv[('stgcn_fixed', top)]['epochs_at_best_mean']):.0f}")
mac("maxStepsRun", thou(max(r["steps_run"] for r in ntu if r["model"] != "stgcn_fixed")))
_es = [r for r in ntu if r["model"] in ("agcn", "ctrgcn") and r["n_train"] == NS[0]]
mac("firstEvalStops", sum(r["best_step"] == 250 for r in _es))
mac("nFirstEvalCells", len(_es))
_fr2k = [Lr[("agcn_frozen", 2000, s)]["best_step"] for s in (0, 1, 2)]
mac("frTwokStepMin", thou(min(_fr2k)))
mac("frTwokStepMax", thou(max(_fr2k)))

# scaling law: OLS on the ST-GCN rung means (interval includes lack of fit)
_ns = sorted({r["n_train"] for r in ntu if r["model"] == "stgcn"})
_m = [np.mean([r["null_excess"] for r in ntu if r["model"] == "stgcn" and r["n_train"] == n]) for n in _ns]
b, lo, hi, r2 = HS.ols_loglog(_ns, _m)
mac("bST", neg(f"{b:.2f}"))
mac("bSTlo", neg(f"{lo:.2f}"))
mac("bSThi", neg(f"{hi:.2f}"))
mac("rtwoST", f"{r2:.2f}")
mac("nRungsST", len(_ns))
mac("stTopNullP", pfmt(max(r["null_p_value"] for r in ntu if r["model"] == "stgcn" and r["n_train"] == top)))
mac("nSimsNTU", thou(ntu[0]["null_sims"]))

# run-to-run noise: two nominally identical runs (one pair: an illustration)
gap = HS.identical_run_gap(rows)
assert gap is not None, "identical-run check needs both runs restored at or before the freeze step"
mac("noiseECE", f3(gap["ece"]))
mac("noiseAcc", f3(gap["top1"]))
# run-to-run noise estimated from the seeds: spread of aECE across the three
# seeds of each model and large rung, and the standard error it implies for a
# difference of two three-seed means (the quantity in Table pairs)
big = [n for n in NS if n >= 10000]
low = [n for n in NS if n <= 5000]
ECE = {k: r["ece_adaptive"] for k, r in Lr.items()}
_seeds = sorted({s for (_, _, s) in Lr})
_sd, _ng = HS.pooled_seed_sd(ECE, ["stgcn", "agcn", "ctrgcn"], big, seeds=_seeds)
_se = _sd * np.sqrt(2 / len(_seeds))
mac("noiseSeedSD", f"{_sd:.4f}")
mac("nSeedSDGroups", _ng)
mac("seDiff", f"{_se:.4f}")

# paired differences on the fixed test set (bootstrap over clips, Table pairs)
pr = read_csv("ladder_paired_comparisons.csv")
P = {(x["model_a"], x["model_b"], int(x["n_train"])): x for x in pr}
nbt = thou(int(pr[0].get("n_boot") or 500))
fam = [("agcn", "stgcn", "2s-AGCN $-$ ST-GCN"), ("ctrgcn", "stgcn", "CTR-GCN $-$ ST-GCN"),
       ("agcn_frozen", "agcn", "AGCN frozen $-$ 2s-AGCN"), ("stgcn_fixed", "stgcn", "ST-GCN 6k $-$ ST-GCN")]
body = []
for a, b_, nm in fam:
    cells = []
    for n in NS:
        x = P.get((a, b_, n))
        if x is None:
            cells.append("--")
            continue
        d = [Lr[(a, n, s)]["ece_adaptive"] - Lr[(b_, n, s)]["ece_adaptive"] for s in (0, 1, 2)]
        agree = int(sum(np.sign(d) == np.sign(np.mean(d))))
        cells.append(sgn(float(x["delta_ece_adaptive"])) + ("$^{\\dagger}$" if agree < 3 else ""))
    body.append(nm + " & " + " & ".join(cells) + r"\\")
(O / "tab_pairs.tex").write_text(r"""\begin{table*}[t]
\caption{Paired differences in adaptive ECE on the fixed NTU-60 test set
(A $-$ B; negative: A better calibrated), averaged over three seeds; within a
seed, A and B see the same training subsample. Every entry differs from zero
under a bootstrap over test clips (""" + nbt + r""" draws, Holm-corrected within each
family: the two graph rows form one family of 14 tests, the freezing and
budget rows one family each), but that test holds the trained models fixed and
ignores training non-determinism: across seeds, $\aece$ varies with a standard
deviation of about """ + MAC["noiseSeedSD"] + r""" at the large rungs, so a
difference of two three-seed means has a standard error of about """ + MAC["seDiff"] + r""".
$^{\dagger}$: only two of the three seeds agree in sign; all other entries
agree in all three seeds.}
\label{tab:pairs}
\centering\footnotesize
\begin{tabular}{@{}l""" + "c" * len(NS) + r"""@{}}
\toprule
$n_{\mathrm{tr}}$ & """ + " & ".join(thou(n) for n in NS) + r"""\\
\midrule
""" + "\n".join(body) + r"""
\bottomrule
\end{tabular}
\end{table*}
""")
assert all(x["verdict"] != "n.s." for x in pr), "Table pairs caption says every entry is significant"
for a, b_, nm in (("agcn", "stgcn", "AG"), ("ctrgcn", "stgcn", "CT"), ("agcn_frozen", "agcn", "FR"),
                  ("stgcn_fixed", "stgcn", "FX")):
    first = min(n for (aa, bb, n) in P if (aa, bb) == (a, b_))
    mac(f"d{nm}Top", sgn(float(P[(a, b_, top)]["delta_ece_adaptive"])))
    mac(f"d{nm}First", sgn(float(P[(a, b_, first)]["delta_ece_adaptive"])))
    mac(f"d{nm}TopAbs", f3(abs(float(P[(a, b_, top)]["delta_ece_adaptive"]))))
    mac(f"d{nm}FirstAbs", f3(abs(float(P[(a, b_, first)]["delta_ece_adaptive"]))))
mac("dFXfivekAbs", f3(abs(float(P[("stgcn_fixed", "stgcn", 5000)]["delta_ece_adaptive"]))))


def pd(a, b_, n):
    return float(P[(a, b_, n)]["delta_ece_adaptive"])


mac("dAGBigMin", f3(min(pd("agcn", "stgcn", n) for n in big)))
mac("dAGBigMax", f3(max(pd("agcn", "stgcn", n) for n in big)))
mac("dCTBigMin", f3(min(pd("ctrgcn", "stgcn", n) for n in big)))
mac("dCTBigMax", f3(max(pd("ctrgcn", "stgcn", n) for n in big)))
mac("dAGSmallMin", f3(-max(pd("agcn", "stgcn", n) for n in low)))
mac("dAGSmallMax", f3(-min(pd("agcn", "stgcn", n) for n in low)))
mac("dFRBigMax", f3(max(abs(pd("agcn_frozen", "agcn", n)) for n in big)))
mac("dFRfivek", f3(pd("agcn_frozen", "agcn", 5000)))
mac("dFRfivekAcc", f3(float(P[("agcn_frozen", "agcn", 5000)]["delta_top1"])))
mac("dFXfivekAcc", f3(abs(float(P[("stgcn_fixed", "stgcn", 5000)]["delta_top1"]))))
mac("excFRvsSTBigMin", f3(min(L_("agcn_frozen", n, "excess_mean") - L_("stgcn", n, "excess_mean") for n in big)))
mac("excFRvsSTBigMax", f3(max(L_("agcn_frozen", n, "excess_mean") - L_("stgcn", n, "excess_mean") for n in big)))
_bigd = [abs(pd(a, "stgcn", n)) for a in ("agcn", "ctrgcn") for n in big]
mac("seRatioMin", f"{min(_bigd) / _se:.0f}")
mac("seRatioMax", f"{max(_bigd) / _se:.0f}")
mac("frSERatio", f"{max(abs(pd('agcn_frozen', 'agcn', n)) for n in big) / _se:.1f}")
_frst = [L_("agcn_frozen", n, "excess_mean") - L_("stgcn", n, "excess_mean") for n in big]
mac("frSTseMin", f"{min(_frst) / _se:.0f}")
mac("frSTseMax", f"{max(_frst) / _se:.0f}")

# seed-level tests: separately trained runs are the units (one Holm family)
sl = HS.seed_level_tests(ECE, [("agcn", "stgcn"), ("ctrgcn", "stgcn")], {"low": low, "high": big})
SL = {(r["model_a"], r["regime"]): r for r in sl}
for a, nm in (("agcn", "AG"), ("ctrgcn", "CT")):
    hi_, lo_ = SL[(a, "high")], SL[(a, "low")]
    mac(f"seedPos{nm}", f"{hi_['n_a_higher']}/{hi_['n_pairs']}")
    mac(f"seedNeg{nm}Low", f"{lo_['n_nonzero'] - lo_['n_a_higher']}/{lo_['n_pairs']}")
    mac(f"pSeed{nm}Low", pfmt(lo_["p_holm"]))
    mac(f"pSeed{nm}High", pfmt(hi_["p_holm"]))
mac("seedPHolm", pfmt(max(SL[(a, "high")]["p_holm"] for a in ("agcn", "ctrgcn"))))
mac("nSeedTests", len(sl))
mac("lowRungMax", thou(max(low)))
mac("highRungMin", thou(min(big)))

# ====================================================================== RQ3 (methods)
# Every method changes the confidences and with them the estimator floor, so a
# raw ECE difference between methods mixes calibration with the floor shift.
# The methods are TESTED on the excess over each method's own floor (computed on
# the method's probabilities by Stage 9c of the v6 notebook) and on the
# binning-free Brier score and NLL; the raw adaptive ECE is reported only.
mc = read_csv("methods_comparison.csv")
MN = {"raw": "uncalibrated", "temperature": "temperature", "vector": "vector scaling",
      "label_smooth": "label smoothing", "focal": "focal loss", "mc_dropout": "MC dropout",
      "deep_ensemble": "deep ensemble (5)"}
MK = {"raw": "Raw", "temperature": "TS", "vector": "VS", "label_smooth": "LS", "focal": "FL",
      "mc_dropout": "MC", "deep_ensemble": "DE"}
PN = {"temperature": "temperature scaling", "vector": "vector scaling", "label_smooth": "label smoothing",
      "focal": "focal loss", "mc_dropout": "MC dropout", "deep_ensemble": "the deep ensemble"}
ORDER = ["temperature", "vector", "label_smooth", "focal", "mc_dropout", "deep_ensemble"]
allm = {x["method"]: x for x in mc if x["scope"] == "all"}
pc_rows = read_csv("methods_per_cell.csv")
assert "floor" in pc_rows[0] and pc_rows[0]["floor"] not in ("", None), (
    "methods_per_cell.csv has no per-method floors: these results predate the v6 notebook (Stage 9c)")
for r in pc_rows:
    for c in ("ece", "brier", "nll", "floor", "excess"):
        r[c] = float(r[c])
TESTED = ["excess", "brier", "nll"]
SK = {"ece": "ECE", "excess": "Exc", "brier": "Brier", "nll": "NLL"}
RES = {key: HS.method_comparison(pc_rows, ORDER, key=key) for key in ["ece"] + TESTED}
res_ece, eci = RES["ece"]
NU = res_ece["vector"]["n"]
mac("nUnits", NU)
mac("nBootUnits", thou(HS.N_BOOT_UNITS))
mac("nMethodFamilies", len(TESTED))
for key, (res, _) in RES.items():
    for m in ORDER:
        r, k, s_ = res[m], MK[m], SK[key]
        mac(f"d{s_}{k}", sgn(r["mean"]))
        mac(f"d{s_}{k}Abs", f3(abs(r["mean"])))
        mac(f"imp{s_}{k}", f"{r['k']}/{r['n']}")
        mac(f"imp{s_}{k}PerDS", ", ".join(f"{r['per_dataset'][ds][0]}/{r['per_dataset'][ds][1]}" for ds in DS))
        if key != "ece":
            mac(f"p{s_}{k}Rel", prel(r["p_holm"]))
            mac(f"p{s_}{k}", pfmt(r["p_holm"]))
            mac(f"v{s_}{k}", r["verdict"])
# each method's floor: mean over its cells, and its mean reduction per test set
# (excess = ECE - floor, so the floor reduction is the ECE reduction minus the
# excess reduction, averaged over the same cells and test sets)
FD = {m: RES["ece"][0][m]["mean"] - RES["excess"][0][m]["mean"] for m in ORDER}
for m in ["raw"] + ORDER:
    mac(f"mfloor{MK[m]}", f3(float(allm[m]["floor"])))
for m in ORDER:
    mac(f"dFloor{MK[m]}Abs", f3(abs(FD[m])))

# The abstract, RQ3 and the conclusion state these results in words.
_V = {(key, m): RES[key][0][m]["verdict"] for key in TESTED for m in ORDER}
assert all(_V[(key, m)] == "n.s." for key in ("brier", "nll") for m in ("temperature", "vector", "mc_dropout")), _V
assert all(_V[(key, m)] == "increases" for key in ("brier", "nll") for m in ("label_smooth", "focal")), _V
assert all(_V[(key, "deep_ensemble")] == "reduces" for key in ("brier", "nll")), _V
for m in ("temperature", "vector"):      # "the lower ECE after either rescaling is accounted for by a lower floor"
    assert _V[("excess", m)] == "n.s." and RES["ece"][0][m]["mean"] > 0 and FD[m] > 0.5 * RES["ece"][0][m]["mean"], m
_ls = RES["excess"][0]["label_smooth"]   # "label smoothing raises the excess over its own floor in every test set"
assert _V[("excess", "label_smooth")] == "increases" and _ls["k"] == 0 and _ls["n"] == NU, _ls
assert abs(RES["excess"][0]["vector"]["mean"]) < 0.25 * RES["ece"][0]["vector"]["mean"]   # "its excess barely moves"
_de = RES["ece"][0]["deep_ensemble"]["mean"]   # "higher by X, of which only Y is a higher floor"
assert _de < 0 and _de < FD["deep_ensemble"] < 0 and abs(FD["deep_ensemble"]) < 0.5 * abs(_de), (_de, FD)


def _cell(r, ci=True):
    out = sgn(r["mean"]) + (f" [{sgn(r['lo'])}, {sgn(r['hi'])}]" if ci else "")
    return out


rows_t = []
for m in ORDER:
    cells = [MN[m], sgn(RES["ece"][0][m]["mean"]), f"{RES['ece'][0][m]['k']}/{RES['ece'][0][m]['n']}"]
    for key in TESTED:
        r = RES[key][0][m]
        cells += [_cell(r), f"{r['k']}/{r['n']}", pfmt(r["p_holm"])]
    rows_t.append(" & ".join(cells) + r"\\")
hdr1 = r" & \multicolumn{2}{c}{$\aece$ (descriptive)}"
hdr2 = r"Method & $\Delta$ & $k$"
cmid = r"\cmidrule(lr){2-3}"
for i, key in enumerate(TESTED):
    nm = {"excess": r"excess $\aece$", "brier": "Brier score", "nll": "NLL"}[key]
    hdr1 += r" & \multicolumn{3}{c}{" + nm + "}"
    hdr2 += r" & $\Delta$ [95\% CI] & $k$ & $p_{\mathrm{Holm}}$"
    cmid += r"\cmidrule(lr){" + f"{4 + 3 * i}-{6 + 3 * i}" + "}"
(O / "tab_methods.tex").write_text(r"""\begin{table*}[t]
\caption{Six calibration methods against the uncalibrated ST-GCN on the three
small datasets (four streams, four splits, five seeds; the deep ensemble has
one cell per dataset, stream and split and is compared with the seed-mean of
the uncalibrated cells). Every method changes the confidences and with them the
estimator floor, so a difference in raw $\aece$ mixes calibration with the
shift of the floor; it is shown for reference only. The methods are tested on
the excess adaptive ECE over each method's own matched floor, the Brier score
and the NLL. $\Delta$: mean reduction (positive is better); the
interval is a cluster bootstrap over the """ + MAC["nUnits"] + r""" test sets
(dataset $\times$ split); $k$: test sets in which the method is better; $p$:
exact sign test over test sets, Holm-corrected over the six methods for each
score. The four test sets of a dataset share subjects, so the $p$-values are
approximate.}
\label{tab:methods}
\centering\footnotesize\setlength{\tabcolsep}{3pt}
\resizebox{\textwidth}{!}{%
\begin{tabular}{@{}l""" + "cc" + "ccc" * len(TESTED) + r"""@{}}
\toprule
""" + hdr1 + r"""\\
""" + cmid + r"""
""" + hdr2 + r"""\\
\midrule
""" + "\n".join(rows_t) + r"""
\bottomrule
\end{tabular}}
\end{table*}
""")
b2 = []
for m in ["raw"] + ORDER:
    x, k = allm[m], MK[m]
    b2.append(f"{MN[m]} & {float(x['ece']):.3f} & {float(x['cw_ece']):.4f} & {float(x['brier']):.3f} & "
              f"{float(x['rel']):.3f} & {float(x['res']):.3f} & {float(x['nll']):.3f} & {float(x['top1']):.3f}\\\\")
    for key, nmk in (("ece", "ece"), ("brier", "brier"), ("nll", "nll"), ("top1", "acc"), ("rel", "rel"), ("res", "res")):
        mac(f"{nmk}{k}", f3(float(x[key])))
    mac(f"cw{k}", f"{float(x['cw_ece']):.4f}")
(O / "tab_methods_metrics.tex").write_text(r"""\begin{table}[t]
\caption{Mean scores per method over the cells of Table~\ref{tab:methods}:
adaptive and classwise ECE \eqref{eq:cwece}, Brier score with its reliability
(REL, lower is better) and resolution (RES, higher is better) terms
\eqref{eq:murphy}, NLL and top-1. The binned scores ($\aece$,
$\ece_{\mathrm{cw}}$, REL) move with each method's floor.}
\label{tab:methods_metrics}
\centering\footnotesize\setlength{\tabcolsep}{2.5pt}
\begin{tabular}{@{}lccccccc@{}}
\toprule
Method & $\aece$ & $\ece_{\mathrm{cw}}$ & Brier & REL & RES & NLL & top-1\\
\midrule
""" + "\n".join(b2) + r"""
\bottomrule
\end{tabular}
\end{table}
""")
grid = {m: {ds: float([y for y in mc if y["scope"] == ds and y["method"] == m][0]["ece"]) for ds in DS}
        for m in ["raw"] + ORDER}
best = {ds: min(grid[m][ds] for m in grid) for ds in DS}
lines = [MN[m] + " & " + " & ".join((f"\\textbf{{{grid[m][ds]:.3f}}}" if abs(grid[m][ds] - best[ds]) < 1e-12
                                     else f"{grid[m][ds]:.3f}") for ds in DS) + r"\\" for m in grid]
(O / "tab_methods_grid.tex").write_text(r"""\begin{table}[t]
\caption{Mean adaptive ECE per dataset and method (all streams); bold: lowest
per dataset. Descriptive only: each dataset has four test sets, and the values
include each method's shift of the floor.}
\label{tab:methods_grid}
\centering\footnotesize
\begin{tabular}{@{}lccc@{}}
\toprule
Method & MSR-Action3D & DailyAct3D & UTKinect\\
\midrule
""" + "\n".join(lines) + r"""
\bottomrule
\end{tabular}
\end{table}
""")
for ds in DS:
    x = [y for y in mc if y["scope"] == ds and y["method"] == "deep_ensemble"][0]
    mac(f"dDE{DSN[ds]}", f3(abs(float(x["delta"]))))
# Fig. 4 data: excess, Brier and NLL panels
_fig = {"n_units": NU, "panels": []}
for key in TESTED:
    res, _ = RES[key]
    _fig["panels"].append({"key": key, "tested": True, "methods": {
        m: dict(mean=res[m]["mean"], lo=res[m]["lo"], hi=res[m]["hi"], k=res[m]["k"], n=res[m]["n"],
                p_holm=res[m].get("p_holm"), verdict=res[m]["verdict"])
        for m in ORDER}})
(O / "methods_split_level.json").write_text(json.dumps(_fig, indent=1))


def _holm_txt(p):
    return r"$p_{\mathrm{Holm}}$" + prel(p)


def _join(items):
    return items[0] if len(items) == 1 else ", ".join(items[:-1]) + " and " + items[-1]


def _cap(t):
    return t[0].upper() + t[1:]


# The excess, in words: significant changes (sign test, Holm), then changes
# whose cluster-bootstrap interval excludes zero although the sign test does
# not, then the rest.
rs = RES["excess"][0]
_sig = [m for m in ORDER if rs[m]["p_holm"] < 0.05]
_ci = [m for m in ORDER if m not in _sig and (rs[m]["lo"] > 0 or rs[m]["hi"] < 0)]
_rest = [m for m in ORDER if m not in _sig and m not in _ci]
_parts = []
for direction, word in ((1, "lowered"), (-1, "raised")):
    hit = [f"{PN[m]} ({rs[m]['k']}/{rs[m]['n']} test sets better, {_holm_txt(rs[m]['p_holm'])})"
           for m in _sig if np.sign(rs[m]["mean"]) == direction]
    if hit:
        _parts.append(f"{word} by {_join(hit)}")
_txt = "Measured against each method's own floor, the excess adaptive ECE is " + " and ".join(_parts) + "." \
    if _parts else "Measured against each method's own floor, no method changes the excess adaptive ECE significantly."
for direction, word in ((1, "lower"), (-1, "raise")):
    hit = [m for m in _ci if np.sign(rs[m]["mean"]) == direction]
    if hit:
        moved = [str(rs[m]["n"] - rs[m]["k"] if direction < 0 else rs[m]["k"]) + "/" + str(rs[m]["n"]) for m in hit]
        _txt += (f" {_cap(_join([PN[m] for m in hit]))} {word}{'s' if len(hit) == 1 else ''} it on average as well "
                 f"(interval{'s' if len(hit) > 1 else ''} excluding zero), but in only {_join(moved)} test sets, "
                 f"which is not significant after correction.")
if _rest and (_parts or _ci):
    _txt += f" {_cap(_join([PN[m] for m in _rest]))} {'does' if len(_rest) == 1 else 'do'} not change it significantly."
mac("methodsExcessSentence", _txt)

# The floors, in words: how far each method moves its own floor
_fl = {m: float(allm[m]["floor"]) for m in ["raw"] + ORDER}
assert _fl["temperature"] < _fl["raw"] and _fl["vector"] < _fl["raw"], _fl          # the rescalings lower it
assert _fl["label_smooth"] > _fl["raw"] and _fl["focal"] > _fl["raw"], _fl          # the losses raise it
_mv = sorted(ORDER, key=lambda m: abs(_fl[m] - _fl["raw"]))
assert set(_mv[:2]) == {"mc_dropout", "deep_ensemble"}, _mv                         # these move it least
mac("methodsFloorSentence",
    f"The floors move with the confidences: the two rescalings lower the mean floor from {f3(_fl['raw'])} to "
    f"{f3(_fl['temperature'])} (temperature) and {f3(_fl['vector'])} (vector scaling), label smoothing and focal "
    f"loss raise it to {f3(_fl['label_smooth'])} and {f3(_fl['focal'])}, and MC dropout and the ensemble move it "
    f"least ({f3(_fl['mc_dropout'])} and {f3(_fl['deep_ensemble'])}).")

mcp = Counter(r["mc_p"] for r in small)
mac("nMCsmallest", mcp[min(mcp)])
mac("nMCother", sum(v for p_, v in mcp.items() if p_ != min(mcp)))
mac("mcPmin", f"{min(mcp):.2f}")
mac("mcPrestMin", f"{min(p_ for p_ in mcp if p_ != min(mcp)):.1f}")
mac("mcPrestMax", f"{max(mcp):.1f}")

# stream fusion: the single joint stream against the four-stream logit average,
# each against its own floor, before and after a temperature fitted to it
fus = read_csv("stream_fusion.csv")
assert "floor" in fus[0] and fus[0]["floor"] not in ("", None), (
    "stream_fusion.csv has no floors: these results predate the v6 notebook (Stage 9)")
FU = {}
for ds in DS:
    for fu, nm in (("joint", "J"), ("4-stream", "F")):
        s = [x for x in fus if x["dataset"] == ds and x["fusion"] == fu]
        g = {c: float(np.mean([float(x[c]) for x in s])) for c in
             ("top1", "ece_adaptive", "ece_adaptive_ts", "floor", "floor_ts", "brier", "brier_ts", "nll", "nll_ts")}
        g["exc"], g["excts"] = g["ece_adaptive"] - g["floor"], g["ece_adaptive_ts"] - g["floor_ts"]
        FU[(ds, nm)] = g
        mac(f"fus{DSN[ds]}{nm}acc", f3(g["top1"]))
        mac(f"fus{DSN[ds]}{nm}ece", f3(g["ece_adaptive"]))
        mac(f"fus{DSN[ds]}{nm}ts", f3(g["ece_adaptive_ts"]))
        mac(f"fus{DSN[ds]}{nm}exc", neg(f3(g["exc"])))
        mac(f"fus{DSN[ds]}{nm}excts", neg(f3(g["excts"])))
        mac(f"fus{DSN[ds]}{nm}floor", f3(g["floor"]))
        mac(f"fus{DSN[ds]}{nm}floorts", f3(g["floor_ts"]))
        for c in ("brier", "nll"):
            mac(f"fus{DSN[ds]}{nm}{c}", f3(g[c]))
            mac(f"fus{DSN[ds]}{nm}{c}ts", f3(g[c + "_ts"]))
_f4 = [x for x in fus if x["fusion"] == "4-stream"]
mac("nFusCells", len(_f4))
mac("nFusUnder", sum(float(x["temperature"]) < 1 for x in _f4))
# The fusion paragraph states these in words.
assert sum(float(x["temperature"]) < 1 for x in _f4) >= 0.9 * len(_f4)            # fusions are under-confident
for ds in DS:
    J, F = FU[(ds, "J")], FU[(ds, "F")]
    assert F["brier_ts"] < F["brier"] and F["nll_ts"] < F["nll"], ds              # its temperature helps both scores
for ds in ("action3d", "daily"):
    J, F = FU[(ds, "J")], FU[(ds, "F")]
    assert F["exc"] > J["exc"] and F["excts"] < 0.5 * F["exc"], ds                 # fusion raises it, TS removes most
J, F = FU[("utkinect", "J")], FU[("utkinect", "F")]
assert F["exc"] < J["exc"] and F["excts"] > F["exc"], "utkinect"                    # fusion lowers it; TS raises it
assert F["floor_ts"] < F["floor"] and F["ece_adaptive_ts"] < F["ece_adaptive"], "utkinect"

# RQ3: how much of temperature scaling's gain is the floor moving (Stage 9b)
_tf = {x["dataset"]: x for x in read_csv("temperature_floor.csv")}
_xr, _xt, _fd = [], [], []
for ds in DS:
    sub = [r for r in small if r["dataset"] == ds]
    fr, ft = float(_tf[ds]["floor_raw"]), float(_tf[ds]["floor_ts"])
    mac(f"tfRaw{DSN[ds]}", f3(fr))
    mac(f"tfTS{DSN[ds]}", f3(ft))
    _xr.append(np.mean([r["ece_adaptive"] for r in sub]) - fr)
    _xt.append(np.mean([r["ece_adaptive_temp"] for r in sub]) - ft)
    _fd.append(fr - ft)
    mac(f"excRaw{DSN[ds]}", f3(_xr[-1]))
    mac(f"excTS{DSN[ds]}", f3(_xt[-1]))
_nsf = {int(float(x["n_sim"])) for x in _tf.values()}
assert len(_nsf) == 1, _nsf
mac("nSimFloor", thou(_nsf.pop()))      # null draws for the floor of a recalibrated, retrained or fused model
mac("tfDropMean", f3(np.mean(_fd)))
mac("excRawMean", f3(np.mean(_xr)))
mac("excTSMean", f3(np.mean(_xt)))
mac("eceTSDropMean", f3(np.mean(_xr) + np.mean(_fd) - np.mean(_xt)))
assert np.mean(_xt) >= np.mean(_xr), ("the abstract, RQ3 and the conclusion say the lower ECE after temperature "
                                      "scaling is accounted for by the floor; on these results it is not")

# ====================================================================== preprocessing
pc = json.loads((R / "prep_config.json").read_text())
lab = {"A legacy": "A & legacy & joint 0, $t_0$ & --", "B topology only": "B & corrected & joint 0, $t_0$ & --",
       "C frame0": "C & corrected & hip, $t_0$ & off", "D frame0+align": "D & corrected & hip, $t_0$ & on",
       "E perframe+align": "E & corrected & hip, all $t$ & on"}
lines = []
for k, v in pc["summary"].items():
    lead, rest = lab[k].split(" & ", 1)
    lines.append(lead + ("$^{\\star}$" if k == pc["chosen"] else "") + " & " + rest +
                 f" & {v['val_top1']:.3f} & {v['top1']:.3f} & {v['ece']:.3f} & {v['T']:.2f}\\\\")
(O / "tab_prep.tex").write_text(r"""\begin{table}[t]
\caption{Preprocessing ablation on MSR-Action3D (joint stream, ST-GCN, three
splits $\times$ three seeds). Selection used validation top-1 among
corrected-topology settings whose normalization is applied before the streams
are derived (C--E); $^{\star}$: chosen. Test columns are reported only.
$t_0$: first frame.}
\label{tab:prep}
\centering\footnotesize\setlength{\tabcolsep}{3pt}
\begin{tabular}{@{}llllcccc@{}}
\toprule
 & tree & center & align & val & test & $\aece$ & $T$\\
\midrule
""" + "\n".join(lines) + r"""
\bottomrule
\end{tabular}
\end{table}
""")
S_ = pc["summary"]
mac("prepValLegacy", f3(S_["A legacy"]["val_top1"]))
mac("prepValChosen", f3(S_[pc["chosen"]]["val_top1"]))
mac("prepTestLegacy", f3(S_["A legacy"]["top1"]))
mac("prepTestChosen", f3(S_[pc["chosen"]]["top1"]))
mac("prepTmin", f2(min(v["T"] for v in S_.values())))
mac("prepTmax", f2(max(v["T"] for v in S_.values())))

# ====================================================================== environment
# The experiments ran in the first sessions; a later statistics-only session
# (possibly on a CPU runtime) must not change the stack the paper reports.
env = json.loads((R / "environment" / "environment.json").read_text())
hist_f = R / "environment" / "environment_history.json"
hist = json.loads(hist_f.read_text()) if hist_f.exists() else [dict(env, session="current")]
_by = {h.get("session"): h for h in hist}
first = _by.get("v1", hist[0])
second = _by.get("v2", hist[min(1, len(hist) - 1)])
mac("envPython", first["python"])
mac("envTorch", first["packages"]["torch"].split("+")[0])
mac("envNumpy", first["packages"]["numpy"])
mac("envGPU", next((h["cuda"]["device"] for h in hist if h.get("cuda") and h["cuda"].get("device")), "GPU"))
mac("envCudaFirst", (first.get("cuda") or {}).get("version") or "n/a")
mac("envCudaLast", (second.get("cuda") or {}).get("version") or "n/a")
mac("nSessions", len(hist))

# p-values with their relation, for prose (avoids "p = <0.001")
for _nm, _p in (("pTestSetsPos", pe["p"]), ("stTopNullP", max(r["null_p_value"] for r in ntu if r["model"] == "stgcn" and r["n_train"] == top)),
                ("WAp", M[("action3d", "bone_motion", "fixed_frame0", 3)]["null_p_value"]),
                ("WBp", M[("daily", "bone", "rand0_frame0", 1)]["null_p_value"]),
                ("seedPHolm", max(SL[(a, "high")]["p_holm"] for a in ("agcn", "ctrgcn"))),
                ("pSeedAGLow", SL[("agcn", "low")]["p_holm"]), ("pSeedCTLow", SL[("ctrgcn", "low")]["p_holm"]),
                ("pSeedAGHigh", SL[("agcn", "high")]["p_holm"]), ("pSeedCTHigh", SL[("ctrgcn", "high")]["p_holm"])):
    mac(_nm + "Rel", prel(_p))
(O / "numbers.tex").write_text("% generated by make_numbers.py -- do not edit\n" +
                               "\n".join(f"\\newcommand{{\\{k}}}{{{v}}}" for k, v in sorted(MAC.items())) + "\n")
print(f"{len(MAC)} macros; tables:", sorted(p.name for p in O.glob("tab_*.tex")))


In [ ]:
%%writefile /content/make_figures.py
"""
make_figures.py RESULTS_DIR GENERATED_DIR OUT_DIR

Draws all six figures of the manuscript from the result tables of a run
(run make_numbers.py first: it writes GENERATED_DIR/methods_split_level.json).
Nothing here needs the stored predictions: the two figures that pool raw
predictions read the per-bin and per-class tables that Stage 10 of the
notebook writes (tables/reliability_bins.csv, tables/per_class_pooled.csv).
Inference units follow the paper: test sets (dataset x split) on the small
datasets; error bars over seeds on the ladder.
"""
import csv
import json
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.ticker import FixedLocator, FormatStrFormatter, NullFormatter  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parent))  # har3d_plotstyle.py ships alongside
import har3d_plotstyle as PS  # noqa: E402

R, G, O = Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3])
O.mkdir(parents=True, exist_ok=True)
T = R / "tables"
PS.apply()
rows = json.loads((R / "metrics.json").read_text())
# replication and environment re-runs (revision) are reported from their own tables
rows = [r for r in rows if not r["model"].endswith(("_rep", "_env"))]
LABEL = {"daily": "DailyActivity3D", "action3d": "MSR-Action3D", "utkinect": "UTKinect", "ntu60": "NTU-60"}
SMALL = ["action3d", "daily", "utkinect"]
MCOL = {"stgcn": PS.PALETTE["blue"], "agcn": PS.PALETTE["vermillion"], "ctrgcn": PS.PALETTE["green"],
        "agcn_frozen": PS.PALETTE["orange"], "stgcn_fixed": PS.PALETTE["skyblue"]}
MNAME = {"stgcn": "ST-GCN (fixed graph)", "agcn": "2s-AGCN (adaptive)", "ctrgcn": "CTR-GCN (channel-wise)",
         "agcn_frozen": "2s-AGCN, graph frozen", "stgcn_fixed": "ST-GCN, fixed step budget"}
lad = [r for r in rows if r["dataset"] == "ntu60"
       and not (r["model"] == "agcn_frozen" and r.get("frozen_at") is None)]


def read_csv(name):
    return list(csv.DictReader(open(T / name)))


# ---------------------------------------------------------------- fig 1
# Equal-mass reliability diagrams, ST-GCN joint stream pooled over splits and
# seeds: bars span each bin's confidence range at the bin's accuracy; the dot
# marks the bin's (mean confidence, accuracy).
rb = read_csv("reliability_bins.csv")
fig, axes = plt.subplots(1, 3, figsize=(7.35, 2.7), sharey=True)
for ax, ds in zip(axes, SMALL):
    b = sorted((x for x in rb if x["dataset"] == ds), key=lambda x: int(x["bin"]))
    lo = np.array([float(x["conf_lo"]) for x in b]); hi = np.array([float(x["conf_hi"]) for x in b])
    acc = np.array([float(x["accuracy"]) for x in b]); mconf = np.array([float(x["mean_conf"]) for x in b])
    ax.plot([0, 1], [0, 1], "--", color="0.45", lw=0.8, zorder=1, label="perfect calibration")
    ax.bar(lo, acc, width=hi - lo, align="edge", color=PS.ROLE["bar"], edgecolor="white", lw=0.4,
           zorder=2, label="accuracy of bin")
    ax.plot(mconf, acc, "o", ms=2.5, color=PS.PALETTE["blue"], zorder=4, label="(mean confidence, accuracy)")
    sub = [r for r in rows if (r["model"], r["dataset"], r["stream"]) == ("stgcn", ds, "joint")]
    e = [r["ece_adaptive"] for r in sub]
    ax.set_title(f"{LABEL[ds]}\naECE {np.mean(e):.3f} ± {np.std(e, ddof=1):.3f}, "
                 f"floor {np.mean([r['null_null_mean'] for r in sub]):.3f}", fontsize=8)
    ax.set_xlabel("confidence (equal-mass bins)"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_aspect("equal", adjustable="box")
axes[0].set_ylabel("accuracy")
axes[0].legend(loc="upper left", fontsize=6, handlelength=1.6)
PS.save(fig, str(O / "fig1_reliability")); plt.close(fig)

# ---------------------------------------------------------------- fig 2
fig, (axA, axB, axC) = plt.subplots(1, 3, figsize=(7.4, 2.7))
pts = [r for r in rows if r["model"] == "stgcn" and r["stream"] == "joint"]
for ds in ["action3d", "daily", "utkinect", "ntu60"]:
    sel = [r for r in pts if r["dataset"] == ds]
    x = [np.sqrt(r["n_classes"] / r["n"]) for r in sel]
    axA.scatter(x, [r["ece_adaptive"] for r in sel], s=14, color=PS.ROLE[ds], label=LABEL[ds],
                edgecolor="white", lw=0.3, zorder=3)
    axA.scatter(x, [r["null_null_mean"] for r in sel], s=12, marker="x", color=PS.ROLE["null"], zorder=2,
                label="null floor" if ds == "action3d" else None)
axA.set_xscale("log"); axA.set_yscale("log")
axA.set_xlabel(r"$\sqrt{K/n_{\mathrm{test}}}$"); axA.set_ylabel("adaptive ECE")
axA.set_title("(a) observed vs null floor")
_C = [0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5]
_xv = [np.sqrt(r["n_classes"] / r["n"]) for r in pts]
_yv = [r["ece_adaptive"] for r in pts] + [r["null_null_mean"] for r in pts]
axA.set_xlim(min(_xv) / 1.3, max(_xv) * 1.3)
for ax_, vals in ((axA.xaxis, _xv), (axA.yaxis, _yv)):
    lo_, hi_ = min(vals), max(vals)
    ax_.set_major_locator(FixedLocator([c for c in _C if lo_ / 1.3 <= c <= hi_ * 1.3]))
    ax_.set_major_formatter(FormatStrFormatter("%g")); ax_.set_minor_formatter(NullFormatter())
axA.legend(fontsize=6)
for mdl in MCOL:
    by = defaultdict(list)
    for r in lad:
        if r["model"] == mdl:
            by[r["n_train"]].append(r)
    ns = np.array(sorted(by))
    ls = "--" if mdl in ("agcn_frozen", "stgcn_fixed") else "-"
    for ax, key in ((axB, "null_excess"), (axC, "top1")):
        mu = [np.mean([r[key] for r in by[n]]) for n in ns]
        sd = [np.std([r[key] for r in by[n]], ddof=1) for n in ns]
        ax.errorbar(ns, mu, yerr=sd, fmt="o" + ls, ms=3, capsize=2, color=MCOL[mdl], label=MNAME[mdl])
fl = [np.mean([r["null_null_mean"] for r in lad if (r["model"], r["n_train"]) == mn])
      for mn in sorted({(r["model"], r["n_train"]) for r in lad})]       # per variant and rung, as in the text
axB.axhline(0, color="0.7", lw=0.6)
axB.set_xscale("log"); axB.set_xlabel("NTU-60 training clips")
axB.set_ylabel("excess adaptive ECE (observed − floor)")
axB.set_title(f"(b) ladder, fixed test set\n(floor {min(fl):.4f}–{max(fl):.4f})")
axC.set_xscale("log"); axC.set_xlabel("NTU-60 training clips"); axC.set_ylabel("top-1")
axC.set_title("(c) accuracy"); axC.legend(fontsize=5.8, loc="lower right")
PS.save(fig, str(O / "fig2_scale")); plt.close(fig)

# ---------------------------------------------------------------- fig 3
fig, ax = plt.subplots(figsize=(3.6, 2.6))
xs, w = np.arange(len(SMALL)), 0.36
st = [r for r in rows if r["model"] == "stgcn" and r["dataset"] in SMALL]
for off, key, lab, col in ((-w / 2, "ece_adaptive", "raw", PS.ROLE["observed"]),
                           (w / 2, "ece_adaptive_temp", "temperature scaled", PS.ROLE["excess"])):
    mu, lo, hi = [], [], []
    for d in SMALL:
        sub = [r for r in st if r["dataset"] == d]
        per = defaultdict(list)
        for r in sub:
            per[r["split"]].append(r[key])
        tm = [np.mean(v) for v in per.values()]          # one mean per test set
        mu.append(np.mean([r[key] for r in sub])); lo.append(min(tm)); hi.append(max(tm))
    ax.bar(xs + off, mu, w, color=col, label=lab,
           yerr=[np.subtract(mu, lo), np.subtract(hi, mu)], capsize=2, error_kw={"lw": 0.8})
flo = [np.mean([r["null_null_mean"] for r in st if r["dataset"] == d]) for d in SMALL]
ax.scatter(xs - w / 2, flo, marker="_", s=160, lw=1.6, color="k", zorder=5, label="null floor, raw")
if (T / "temperature_floor.csv").exists():
    tf = {x["dataset"]: float(x["floor_ts"]) for x in read_csv("temperature_floor.csv")}
    ax.scatter(xs + w / 2, [tf[d] for d in SMALL], marker="_", s=160, lw=1.6, color="0.45", zorder=5,
               label="null floor, scaled")
ax.set_xticks(xs); ax.set_xticklabels([LABEL[d] for d in SMALL], fontsize=7)
ax.set_ylabel("adaptive ECE"); ax.legend(fontsize=6.5, loc="upper left")
PS.save(fig, str(O / "fig3_tempscaling")); plt.close(fig)

# ---------------------------------------------------------------- fig 4
# Reduction versus the uncalibrated model (positive is better) for each score,
# with cluster-bootstrap intervals over test sets. Tested scores are coloured by
# the Holm-corrected sign test; raw adaptive ECE (when each method's floor is
# not available) is shown in grey because it includes the shift of the floor.
sl = json.loads((G / "methods_split_level.json").read_text())
order = ["temperature", "vector", "label_smooth", "focal", "mc_dropout", "deep_ensemble"]
DISP = {"temperature": "temperature", "vector": "vector scaling", "label_smooth": "label smoothing",
        "focal": "focal loss", "mc_dropout": "MC dropout", "deep_ensemble": "deep ensemble (5)"}
TITLE = {"ece": "adaptive ECE (not tested)", "excess": "excess adaptive ECE", "brier": "Brier score", "nll": "NLL"}
fig, axes = plt.subplots(1, len(sl["panels"]), figsize=(7.2, 2.35), sharey=True)
yb = np.arange(len(order))
for ax, pan in zip(axes, sl["panels"]):
    M = pan["methods"]
    mu = np.array([M[m]["mean"] for m in order])
    lo = np.array([M[m]["lo"] for m in order]); hi = np.array([M[m]["hi"] for m in order])
    col = [PS.ROLE["null"] if not pan["tested"] else
           PS.PALETTE["green"] if M[m]["verdict"] == "reduces" else
           PS.PALETTE["vermillion"] if M[m]["verdict"] == "increases" else PS.PALETTE["skyblue"] for m in order]
    ax.barh(yb, mu, xerr=[mu - lo, hi - mu], color=col, capsize=2, error_kw={"lw": 0.8}, height=0.65)
    ax.axvline(0, color="0.3", lw=0.8)
    ax.set_title(TITLE[pan["key"]], fontsize=8)
    ax.set_xlabel("reduction vs. uncalibrated", fontsize=7)
    span = max(np.abs(np.concatenate([lo, hi])).max(), 1e-3) * 1.15
    ax.set_xlim(-span, span)
axes[0].set_yticks(yb); axes[0].set_yticklabels([DISP[m] for m in order]); axes[0].invert_yaxis()
PS.save(fig, str(O / "fig4_methods")); plt.close(fig)

# ---------------------------------------------------------------- fig 5
pcp = read_csv("per_class_pooled.csv")
fig, axes = plt.subplots(1, 3, figsize=(7.35, 2.4))
for ax, ds in zip(axes, SMALL):
    c = sorted((x for x in pcp if x["dataset"] == ds), key=lambda x: -float(x["per_class_ece"]))
    e = [float(x["per_class_ece"]) for x in c]; r1 = [float(x["one_minus_recall"]) for x in c]
    ax.bar(np.arange(len(c)), e, color=PS.ROLE[ds], width=0.8, label="per-class ECE")
    ax.plot(np.arange(len(c)), r1, "k.", ms=3, label="1 − class recall")
    ax.set_title(LABEL[ds], fontsize=8); ax.set_xlabel("class (sorted by ECE)")
    ax.set_xticks([]); ax.set_ylim(0, 1)
axes[0].set_ylabel("per-class ECE / 1 − recall"); axes[0].legend(fontsize=6, loc="upper right")
PS.save(fig, str(O / "fig5_perclass")); plt.close(fig)

# ---------------------------------------------------------------- fig 6
mrows = {r["method"]: r for r in read_csv("methods_comparison.csv") if r["scope"] == "all"}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.0, 2.6))
sub = order[1:]
xb = np.arange(len(sub))
a1.bar(xb - 0.2, [float(mrows[m]["rel"]) - float(mrows["raw"]["rel"]) for m in sub], 0.4,
       label="Δ reliability (REL; < 0 better)", color=PS.PALETTE["vermillion"])
a1.bar(xb + 0.2, [float(mrows[m]["res"]) - float(mrows["raw"]["res"]) for m in sub], 0.4,
       label="Δ resolution (RES; > 0 better)", color=PS.PALETTE["blue"])
a1.axhline(0, color="0.4", lw=0.8)
a1.set_xticks(xb); a1.set_xticklabels([DISP[m] for m in sub], rotation=35, ha="right", fontsize=6.5)
a1.set_ylabel("change vs. uncalibrated"); a1.set_title("(a) small datasets, by method")
a1.legend(fontsize=6, loc="best", framealpha=0.8)
for mdl in MCOL:
    by = defaultdict(list)
    for r in lad:
        if r["model"] == mdl:
            by[r["n_train"]].append(r["brier_rel"])
    ns = sorted(by)
    a2.errorbar(ns, [np.mean(by[n]) for n in ns], yerr=[np.std(by[n], ddof=1) for n in ns],
                fmt="o--" if mdl in ("agcn_frozen", "stgcn_fixed") else "o-", ms=3, capsize=2,
                color=MCOL[mdl], label=MNAME[mdl])
a2.set_xscale("log"); a2.set_xlabel("NTU-60 training clips")
a2.set_ylabel("Brier reliability (REL)"); a2.set_title("(b) NTU-60 ladder")
a2.legend(fontsize=5.8, loc="best", framealpha=0.8)
PS.save(fig, str(O / "fig6_brier")); plt.close(fig)
print("figures 1-6 written to", O)


### Stage 0b — environment capture

In [ ]:
# =====================================================================
# STAGE 0b -- environment capture (reproducibility)
# =====================================================================
# Exact versions, hardware and the full configuration, written to
# results/environment/ so every reported number is traceable to a stack.
import platform, subprocess

ENVDIR = RESULTS / 'environment'
ENVDIR.mkdir(parents=True, exist_ok=True)
pkgs = ['numpy', 'scipy', 'torch', 'matplotlib', 'pandas', 'sklearn', 'h5py']
versions = {}
for m in pkgs:
    try:
        versions[m] = __import__(m).__version__
    except Exception as e:
        versions[m] = f'(not importable: {e})'

CONFIG = {k: v for k, v in globals().items()
          if k.isupper() and isinstance(v, (int, float, str, list, tuple, bool))}
env = {'python': sys.version.split()[0], 'platform': platform.platform(),
       'packages': versions, 'config': CONFIG}
try:
    import torch
    env['cuda'] = {'available': torch.cuda.is_available(), 'version': torch.version.cuda,
                   'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
except Exception:
    env['cuda'] = None
(ENVDIR / 'environment.json').write_text(json.dumps(env, indent=2, default=str))
# environment.json describes only the current session; environment_history.json
# keeps one record per distinct software stack, so runs trained in different
# sessions stay traceable (the paper's reproducibility statement reads it).
HIST = ENVDIR / 'environment_history.json'
hist = json.loads(HIST.read_text()) if HIST.exists() else []
rec = {k: env[k] for k in ('python', 'platform', 'packages', 'cuda')}
if hist and all(hist[-1].get(k) == rec[k] for k in rec):
    hist[-1]['last_seen'] = time.ctime()
else:
    hist.append({'session': f'session {len(hist) + 1}', 'note': '', **rec,
                 'first_seen': time.ctime(), 'last_seen': time.ctime()})
HIST.write_text(json.dumps(hist, indent=2, default=str))
try:
    frozen = subprocess.run([sys.executable, '-m', 'pip', 'freeze'],
                            capture_output=True, text=True, timeout=120).stdout
    (ENVDIR / 'requirements.lock.txt').write_text(frozen)
except Exception as e:
    (ENVDIR / 'requirements.lock.txt').write_text(f'# pip freeze failed: {e}\n')
(ENVDIR / 'requirements.txt').write_text(
    ''.join(f'{m}=={v}\n' for m, v in versions.items() if not v.startswith('(')))
print('environment captured ->', ENVDIR)
for m, v in versions.items():
    print(f'  {m:12} {v}')
print('  cuda        ', env['cuda'])

## Stage 1 — test suite
117 assertions; the notebook stops here if any fail.

In [ ]:
# =====================================================================
# STAGE 1 -- run the test suite (stops the notebook on any failure)
# =====================================================================
# Every assertion guards something that would silently corrupt a calibration
# number rather than raise: metric identities, the Brier decomposition,
# normalisation invariances, subject-disjoint splits, the null model, the
# backbones, graph freezing and early stopping.
import subprocess
r = subprocess.run([sys.executable, 'test_har3d.py'], capture_output=True,
                   text=True, cwd='/content')
print(r.stdout[-6000:])
failed = [l for l in r.stdout.splitlines() if l.lstrip().startswith('FAIL')]
if failed:
    print('\nFAILED ASSERTIONS:'); print('\n'.join(failed))
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError('test suite FAILED -- do not run any later stage until fixed')

## Stage 0c — v2 top-up (runs once)
Removes only the UTKinect cells and the 13 cap-limited ladder cells so the stages below retrain them.

In [ ]:
# =====================================================================
# STAGE 0c -- v2 top-up: remove ONLY the cells the first run got wrong
# =====================================================================
# Run once, after Stage 1. Safe to re-run: a marker file stops it from
# deleting anything a second time.
#
#  1. UTKinect: the 10-frame minimum removed 21 genuine short clips (half of
#     'push'). MIN_FRAMES is now 5, so every UTKinect cell is retrained.
#     MSR-Action3D and DailyActivity3D are unaffected (no clip of theirs was
#     dropped for length), so their cells are kept.
#  2. NTU ladder: every cell that ran into the old 15,000-step ceiling (it was
#     still improving) is retrained with the 40,000-step ceiling. Cells that
#     early-stopped below the ceiling are kept -- a higher ceiling cannot
#     change them.
# Afterwards just run every cell below in order: each stage retrains only
# what is missing and skips everything else.
from store import Key, LogitStore

MARK = RESULTS / '.topup_v2_done'
if MARK.exists():
    print('  [skip] v2 top-up already applied on', MARK.read_text())
else:
    if MIN_FRAMES != 5 or LADDER_MAX_STEPS < 40000:
        raise RuntimeError('set MIN_FRAMES = 5 and LADDER_MAX_STEPS = 40000 in Stage 0 first')
    STORE = LogitStore(LOGITS)
    keys = STORE.list_keys()
    if not keys:
        raise RuntimeError(f'no logits found in {LOGITS} -- the store must not be deleted')

    # (1) UTKinect
    ut = [STORE.path(k) for k in keys if k.dataset == 'utkinect']
    ut += list(LOGITS.glob('mc__utkinect__*.npz'))
    # (2) capped ladder cells
    capped = []
    for k in keys:
        if k.dataset != 'ntu60' or k.subset != 'test':
            continue
        info = STORE.load(k)['provenance'].get('train_info', {}) or {}
        if info.get('early_stop') and info.get('steps_run', 0) >= 15000:
            capped.append(k)
    print(f'UTKinect files to retrain: {len(ut)}')
    print(f'capped ladder cells to retrain: {len(capped)}')
    for k in capped:
        print(f'    {k.model:<12} n={k.split[8:]:<6} seed {k.seed}')
    if len(capped) != 13:
        print(f'  note: expected 13 capped cells, found {len(capped)}')
    for f in ut:
        f.unlink()
    for k in capped:
        for sub in ('val', 'test'):
            p = STORE.path(Key(k.model, k.dataset, k.stream, k.split, k.seed, sub))
            if p.exists():
                p.unlink()
    # training logs: drop UTKinect rows so they are not double-counted
    for f in RESULTS.glob('train_log_*.json'):
        rows = [r for r in json.loads(f.read_text()) if r.get('dataset') != 'utkinect']
        f.write_text(json.dumps(rows, indent=1))
    reset('04_topology_gate', '06_msr_logits', '06b_traintime_calib', '08_ladder',
          '08b_freeze', '09_eval', '09b_statistics', '09c_methods', '10_figures')
    MARK.write_text(time.ctime())
    print('\ndone. Now run every cell below in order.')

## Stage 0d — v3 statistics reset (runs once)
Reopens Stages 9–10 so the null test is recomputed with 20,000 draws and the corrected p-value. CPU only.

In [ ]:
# =====================================================================
# STAGE 0d -- v3: recompute the statistics (CPU only, nothing retrained)
# =====================================================================
# The null-model p-value was (#null >= observed)/S, which is 0 whenever the
# observed ECE beats every draw. A Monte-Carlo p-value must be
# (1 + #null >= observed)/(S + 1) (Phipson & Smyth 2010); with S = 500 its
# smallest value is 0.002, and after Holm over 240 cells no cell could be
# called separable. v3 uses the +1 form, 20,000 null draws on the small
# datasets, and the same +1 form for the paired-bootstrap p-values.
# This cell reopens Stages 9-10 once; Stage 9 reuses every stored metric and
# recomputes only the null test. No logits are deleted, no GPU is needed.
MARK3 = RESULTS / '.v3_stats_done'
if MARK3.exists():
    print('  [skip] v3 statistics reset already applied on', MARK3.read_text())
else:
    reset('09_eval', '09b_statistics', '09c_methods', '10_figures')
    MARK3.write_text(time.ctime())
    print('done. Run every cell below in order (CPU runtime is fine).')

## Stage 0e — v4 ladder-comparison reset (runs once)
Reopens Stage 9b so the paired ladder comparisons are recomputed with 5,000 bootstrap draws. CPU only.

In [ ]:
# =====================================================================
# STAGE 0e -- v4: recompute the ladder comparisons (CPU only, ~25 min)
# =====================================================================
# The paired ladder comparisons used 500 bootstrap draws; their p-value cannot
# fall below 2/501 = 0.004, and the largest Holm family has 14 comparisons
# (14 x 0.004 > 0.05), so no comparison in it could be declared significant
# however large the effect. v4 uses 5,000 draws (Stage 9b). This cell reopens
# Stage 9b once; nothing else is recomputed and nothing is retrained.
MARK4 = RESULTS / '.v4_pairs_done'
if MARK4.exists():
    print('  [skip] v4 reset already applied on', MARK4.read_text())
else:
    reset('09b_statistics')
    MARK4.write_text(time.ctime())
    print('done. Run every cell below in order (CPU runtime is fine).')

## Stage 0f — v5 migration (runs once)
Records the software stacks of sessions v1–v4. Nothing is retrained; Stages 9b, 9c and 10 below recompute the statistics. CPU only.

In [ ]:
# =====================================================================
# STAGE 0f -- v5: statistics at the level of independent units (CPU only)
# =====================================================================
# Nothing is retrained and no stored prediction changes. v5 changes what is
# computed from them, so Stages 9b, 9c and 10 simply run again below:
#   * the 20 cells of a small-dataset split share its test clips, so the
#     method comparisons and the pooled RQ1 test now run over the 12 test sets
#     (dataset x split), with cluster-bootstrap intervals (Stage 9c, 9b);
#   * backbone differences on the ladder are tested over seed-rung pairs,
#     which carry the run-to-run variance a bootstrap over test clips cannot
#     see (Stage 9b);
#   * the scaling-law interval is an OLS t-interval on the rung means;
#   * Stage 10 writes the per-bin and per-class tables behind Figs. 1 and 5,
#     then runs make_numbers.py and make_figures.py, so RESULTS/paper holds
#     every number, table and figure of the manuscript.
# It also records the software stack of the four earlier sessions in
# environment_history.json (it was overwritten each session until v5). This
# applies only to the run that v1-v4 produced, which the v4 marker identifies.
MARK5 = RESULTS / '.v5_units_done'
EARLIER = [
    ('v1', 'initial run: preprocessing ablation, all small-dataset training, NTU-60 ladder and ablations, evaluation', '2.11.0+cu128', '12.8'),
    ('v2', 'top-up: retrained every UTKinect run (minimum clip length 10 -> 5 frames) and the 13 ladder runs that had '
           'reached the old 15,000-step cap (ST-GCN at 10,000, 20,000 and 32,202 clips, seeds 0-2; frozen 2s-AGCN at '
           '20,000 clips seed 0 and 32,202 clips seeds 0-2); re-evaluated everything', '2.11.0+cu130', '13.0'),
    ('v3', 'statistics only: 20,000 null draws and the +1 Monte-Carlo p-value; nothing retrained', '2.11.0+cu130', '13.0'),
    ('v4', 'statistics only: 5,000 bootstrap draws for the paired ladder comparisons; nothing retrained', '2.11.0+cu130', '13.0'),
]
if MARK5.exists():
    print('  [skip] v5 already applied on', MARK5.read_text())
elif not (RESULTS / '.v4_pairs_done').exists():
    MARK5.write_text(time.ctime())
    print('  fresh run: nothing to migrate')
else:
    HIST = RESULTS / 'environment' / 'environment_history.json'
    hist = json.loads(HIST.read_text()) if HIST.exists() else []
    if not any(h.get('session') == 'v1' for h in hist):
        base = {'python': '3.13.15', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39'}
        pk = {'numpy': '2.1.3', 'scipy': '1.16.3', 'matplotlib': '3.10.0', 'pandas': '2.2.3', 'sklearn': '1.6.1'}
        old = [dict(base, session=s, note=n, packages=dict(pk, torch=t, h5py='3.16.0' if s == 'v1' else '3.14.0'),
                    cuda={'available': True, 'version': c,
                          'device': 'NVIDIA RTX PRO 6000 Blackwell Server Edition'}) for s, n, t, c in EARLIER]
        for h in hist:
            if h.get('session', '').startswith('session'):
                h['session'] = 'v5'; h['note'] = 'statistics only (v5); nothing retrained'
        HIST.write_text(json.dumps(old + hist, indent=2, default=str))
        print('  recorded the software stacks of sessions v1-v4 in', HIST.name)
    MARK5.write_text(time.ctime())
    print('done. Run every cell below in order (CPU runtime is fine).')

## Stage 0g — v6 migration (runs once)
Labels this session in the environment history. Nothing is retrained; Stages 9, 9b, 9c and 10 below compute every method's own floor and the statistics. CPU only.

In [ ]:
# =====================================================================
# STAGE 0g -- v6: every calibration method against its own floor (CPU only)
# =====================================================================
# Nothing is retrained and no stored prediction changes. Every calibration
# method changes the confidences and with them the estimator floor that its
# ECE is measured against, so v6 computes that floor on each method's own
# probabilities (Stage 9c) and on the fused models before and after their
# temperature (Stage 9), and tests the methods on the excess over it and on
# the binning-free Brier score and NLL. Stages 9, 9b, 9c and 10 always run, so
# this cell only labels the current session in environment_history.json.
MARK6 = RESULTS / '.v6_floors_done'
if MARK6.exists():
    print('  [skip] v6 already applied on', MARK6.read_text())
else:
    HIST = RESULTS / 'environment' / 'environment_history.json'
    hist = json.loads(HIST.read_text()) if HIST.exists() else []
    if any(h.get('session') == 'v5' for h in hist):       # the run that v1-v5 produced
        for h in hist:
            if str(h.get('session', '')).startswith('session'):
                h['session'] = 'v6'
                h['note'] = 'statistics only (v6): per-method and fused-model floors; nothing retrained'
        HIST.write_text(json.dumps(hist, indent=2, default=str))
        print('  sessions recorded:', [h.get('session') for h in hist])
    else:
        print('  fresh run: nothing to migrate')
    MARK6.write_text(time.ctime())
    print('done. Run every cell below in order (CPU runtime is fine).')

## Stage 0h — label this session in the environment history (v7)
The revision adds training runs, so this session gets its own record even if the software stack is unchanged.

In [ ]:
# =====================================================================
# STAGE 0h -- environment record of the revision (v7)      [CPU, instant]
# =====================================================================
# Stage 0b appends a record only when the software stack differs from the last
# one. The revision trains new runs (Stages 5b and 8c), so its sessions are
# recorded under their own label either way; Stage 8c additionally pins the
# exact stack of the replication in results/replication_env.json.
V7_NOTE = ('revision (v7): confirmatory replication of the backbone comparison (new seeds, one '
           'pinned environment), same-seed environment re-runs, preprocessing sensitivity runs; '
           'dependence-aware, power and estimator-sensitivity statistics on the stored logits')
HIST = RESULTS / 'environment' / 'environment_history.json'
hist = json.loads(HIST.read_text())
cur = {k: hist[-1][k] for k in ('python', 'platform', 'packages', 'cuda')}   # Stage 0b just refreshed it
v7 = [h for h in hist if str(h.get('session', '')).startswith('v7')]
if str(hist[-1].get('session', '')).startswith('session '):
    # a new stack, appended by Stage 0b in this session
    hist[-1].update(session='v7' if not v7 else f'v7.{len(v7) + 1}', note=V7_NOTE)
elif not str(hist[-1].get('session', '')).startswith('v7'):
    # same stack as the last recorded session: still a session of its own
    hist.append({'session': 'v7', 'note': V7_NOTE, **cur,
                 'first_seen': time.ctime(), 'last_seen': time.ctime()})
HIST.write_text(json.dumps(hist, indent=2, default=str))
for h in hist:
    print(f"  {h.get('session', '?'):<8} torch {h['packages'].get('torch')}  "
          f"CUDA {(h.get('cuda') or {}).get('version')}  {(h.get('cuda') or {}).get('device')}")

## Stage 2 — MSR-Action3D

In [ ]:
# =====================================================================
# STAGE 2 -- ingest MSR-Action3D            [CPU, ~3 min; seconds if cached]
# =====================================================================
# Skipped automatically when cache/msr_action3d.npz already exists on Drive
# (parsing is protocol-independent). Expect fx ~ 285.633, cx ~ 160.002: the
# canonical Kinect v1 intrinsics at 320x240. This dataset DEFINES the joint
# order that DailyActivity3D and UTKinect are matched against.
import har3d_action3d as A3
importlib.reload(A3)

if not (CACHE / 'msr_action3d.npz').exists():
    res = A3.ingest(DRIVE, CACHE)
    for k, v in res.items():
        if k != 'intrinsics':
            print(f'  {k}: {v}')
    if 'error' in res:
        raise RuntimeError(f"Action3D ingest failed: {res['error']}")
else:
    print('  [skip] cache/msr_action3d.npz present -- parsing not repeated')
mark_done('02_action3d')

z = np.load(CACHE / 'msr_action3d.npz', allow_pickle=False)
print('\ncache keys:', z.files, '| xyz', z['xyz'].shape, '| sequences', len(z['meta']))
print('confidence retained for', int(z['has_conf'].sum()), 'of', len(z['meta']))
print('\nlowest-quality sequences (reported, not auto-applied; the filter is Stage 4):')
_ = A3.exclusion_candidates(CACHE / 'msr_action3d.npz', k=10)

## Stage 3 — MSR-DailyActivity3D

In [ ]:
# =====================================================================
# STAGE 3 -- ingest MSR-DailyActivity3D     [CPU, ~15 min; seconds if cached]
# =====================================================================
# A zip of nine STORED Deflate64 packs (7-Zip leads the extractor chain), and
# the 20 joints are stored in a DIFFERENT ORDER -- recovered by matching mean
# pairwise-distance matrices against Action3D, never assumed.
# Expect 320 sequences.
import har3d_daily as D
importlib.reload(D)

if not (CACHE / 'msr_daily.npz').exists():
    res = D.ingest(DRIVE, CACHE)
    for k, v in res.items():
        print(f'  {k}: {v}')
    if 'error' in res:
        raise RuntimeError(f"DailyActivity3D ingest failed: {res['error']}")
else:
    print('  [skip] cache/msr_daily.npz present -- parsing not repeated')
mark_done('03_daily')

zz = np.load(CACHE / 'msr_daily.npz', allow_pickle=False)
print('\ncache keys:', zz.files, '| xyz', zz['xyz'].shape, '| sequences', len(zz['meta']))
print('joint permutation applied:', zz['joint_perm'].tolist())

## Stage 3b — UTKinect-Action3D

In [ ]:
# =====================================================================
# STAGE 3b -- ingest UTKinect-Action3D      [CPU, seconds]
# =====================================================================
# 199 valid clips (s10_e02 'carry' is NaN NaN). UTKinect's Kinect-SDK joint
# order is DOCUMENTED, so it is the one dataset where the blind joint-order
# matcher is checked against ground truth -- which validates the permutation
# recovered for DailyActivity3D in Stage 3.
import har3d_utkinect as U
importlib.reload(U)

if not (CACHE / 'utkinect.npz').exists():
    res = U.ingest(DRIVE, CACHE, work=str(WORK / 'utkinect'), check_perm=True)
    for k, v in res.items():
        print(f'  {k}: {v}')
    if 'error' in res:
        raise RuntimeError(f"UTKinect ingest failed: {res['error']} "
                           '(do NOT bypass with check_perm=False)')
else:
    print('  [skip] cache/utkinect.npz present -- parsing not repeated')

zu = np.load(CACHE / 'utkinect.npz', allow_pickle=False)
perm_ok = bool(np.array_equal(zu['joint_perm'], U.EXPECTED_SDK_PERM))
print('\ncache keys:', zu.files, '| xyz', zu['xyz'].shape, '| clips', len(zu['meta']))
print('matcher reproduced the documented Kinect SDK order:', perm_ok)
if not perm_ok:
    raise RuntimeError('joint-order matcher disagrees with the documented enum')
print('clips per action:', np.bincount(zu['meta'][:, 0], minlength=11)[1:].tolist())
(RESULTS / 'utkinect_perm_check.json').write_text(json.dumps(
    {'matches_documented_sdk_order': perm_ok,
     'joint_perm': zu['joint_perm'].tolist()}, indent=2))
mark_done('03b_utkinect')

## Stage 4 — topology gate and quality filter

In [ ]:
# =====================================================================
# STAGE 4 -- topology gate + data-quality filter      [CPU, seconds]
# =====================================================================
# (a) Re-derives bone length and rigidity from the frames and raises, naming
#     every bad edge. Rigidity = low cv OR low absolute sd, since sensor jitter
#     is roughly constant in metres. The legacy tree is run as a control and
#     MUST be rejected.
# (b) The sequence-level quality filter used by every training stage: a
#     sequence is dropped when more than MAX_ZERO_FRAC of its joint-frames are
#     all-zero (tracker lost the skeleton) or it has < MIN_FRAMES frames. The
#     excluded ids and counts go to results/data_filtering.json -- this is
#     what the paper reports instead of an undocumented 557-of-567 subset.
import har3d_v2 as V
importlib.reload(V)

DATASETS = [('action3d', 'msr_action3d.npz'),
            ('daily', 'msr_daily.npz'),
            ('utkinect', 'utkinect.npz')]


def seqs_of(cache):
    o = cache['offsets']
    return [cache['xyz'][o[i]:o[i + 1]] for i in range(len(o) - 1)]


filt = {}
for ds, npz in DATASETS:
    cache = V.load_cache(CACHE / npz)
    print(f'\n{ds}: {len(cache["meta"])} sequences')
    for r in V.verify_topology(seqs_of(cache)):
        print(f"    {r['name']:<16} -> {r['parent_name']:<16} "
              f"{r['len_m']:.3f} m  cv {r['cv']:.3f}  sd {r['sd_m']*1000:4.0f} mm")
    keep = V.quality_mask(cache['flags'], MAX_ZERO_FRAC, MIN_FRAMES)
    dropped = np.where(~keep)[0]
    meta = cache['meta']
    filt[ds] = {'n_raw': int(len(keep)), 'n_kept': int(keep.sum()),
                'n_dropped': int(len(dropped)),
                'dropped': [{'index': int(i), 'action': int(meta[i, 0]),
                             'subject': int(meta[i, 1]),
                             'episode': int(meta[i, 2]) if meta.shape[1] > 2 else None,
                             'n_frames': int(cache['flags'][i, 0]),
                             'zero_joint_frac': round(float(cache['flags'][i, 2]), 3)}
                            for i in dropped],
                'rule': f'zero_joint_frac > {MAX_ZERO_FRAC} or n_frames < {MIN_FRAMES}'}
    print(f'  quality filter: kept {keep.sum()} / {len(keep)} '
          f'(dropped {len(dropped)})')
    for d in filt[ds]['dropped'][:12]:
        print(f"    a{d['action']:02d} s{d['subject']:02d} e{d['episode']}: "
              f"{d['n_frames']} frames, {d['zero_joint_frac']:.0%} zero joints")

print('\nlegacy tree, as a control:')
try:
    V.verify_topology(seqs_of(V.load_cache(CACHE / 'msr_action3d.npz')),
                      parents=V.LEGACY_PARENTS)
    raise RuntimeError('LEGACY TREE PASSED -- the topology check is broken')
except AssertionError as e:
    print('  correctly rejected,', str(e).count('->'), 'edges named')

(RESULTS / 'data_filtering.json').write_text(json.dumps(filt, indent=2))
print('\nwrote', RESULTS / 'data_filtering.json')
mark_done('04_topology_gate')

## Stage 5 — preprocessing ablation (GPU)
Selection on validation accuracy only.

In [ ]:
# =====================================================================
# STAGE 5 -- preprocessing ablation (MSR-Action3D)   [GPU, ~20 min]
# =====================================================================
#      topology    centring                       align
#   A  legacy      legacy (frame-0 on joint 0)     -
#   B  corrected   legacy                          -
#   C  corrected   frame-0 on the hip              off
#   D  corrected   frame-0 on the hip              on
#   E  corrected   per-frame on the hip            on
#
# SELECTION USES VALIDATION ACCURACY ONLY, among corrected-topology configs
# (C/D/E). Test metrics are computed and reported but never used to choose --
# the earlier notebook selected on test top-1, which is test-set leakage.
# The legacy tree is measured and reported but can never be adopted: 11 of its
# edges are not bones. Same quality filter, splits, seeds and early stopping as
# the main experiment.
import har3d_v2 as V, har3d_models_v2 as M2
importlib.reload(V); importlib.reload(M2)

ABLATION_SPLITS = 3

if not stage_done('05_ablation'):

    def legacy_build(cache, stream, parents, keep, T=T_FRAMES):
        off, meta = cache['offsets'], cache['meta']
        X = np.zeros((len(off) - 1, 3, T, len(parents)), np.float32)
        for i in range(len(off) - 1):
            x = V.make_stream(cache['xyz'][off[i]:off[i + 1]], stream, parents)
            x = V.resample(x, T)
            x = x - x[0:1, 0:1, :]
            s = np.linalg.norm(x, axis=-1).mean()
            X[i] = (x / s if s > 1e-6 else x).transpose(2, 0, 1)
        return X[keep]

    cache = V.load_cache(CACHE / 'msr_action3d.npz')
    S = 'joint'
    cfgs = {'C frame0': dict(centre='frame0', align=False),
            'D frame0+align': dict(centre='frame0', align=True),
            'E perframe+align': dict(centre='perframe', align=True)}
    built = {}
    for k, v in cfgs.items():
        built[k], y, subj, keep = V.masked_tensors(cache, S, T_FRAMES, v,
                                                   MAX_ZERO_FRAC, MIN_FRAMES)
    Xa = legacy_build(cache, S, V.LEGACY_PARENTS, keep)
    Xb = legacy_build(cache, S, V.MSR_PARENTS, keep)
    print(f'{len(y)} sequences after the quality filter')

    splits = V.subject_splits(subj, labels=y, n_splits=ABLATION_SPLITS, seed=0,
                              n_test_subjects=4, n_val_subjects=2)
    A_leg, A_new = V.adjacency(V.LEGACY_PARENTS), V.adjacency(V.MSR_PARENTS)
    runs = ([('A legacy', Xa, A_leg, None, False),
             ('B topology only', Xb, A_new, None, True)]
            + [(k, built[k], A_new, cfgs[k], True) for k in cfgs])

    print(f"\n{'config':<18}{'val top1':>10}{'test top1':>10}{'test aECE':>10}"
          f"{'T':>7}{'epochs':>8}")
    summary = {}
    for tag, X, Adj, cfg, corrected in runs:
        r = {m: [] for m in ('val_top1', 'val_nll', 'top1', 'ece', 'brier', 'T', 'best_epoch')}
        for sp in splits:
            for seed in ABLATION_SEEDS:
                out, _ = M2.train_and_logits(
                    X, y, sp, int(y.max()) + 1, Adj, max_epochs=MAX_EPOCHS_MSR,
                    patience=PATIENCE_MSR, min_epochs=MIN_EPOCHS_MSR,
                    seed=seed, verbose=False)
                Lv, yv = out['val']; Lt, yt = out['test']
                pv, pt = V.softmax(Lv), V.softmax(Lt)
                r['val_top1'].append(V.top1(pv, yv)); r['val_nll'].append(V.nll(pv, yv))
                r['top1'].append(V.top1(pt, yt)); r['ece'].append(V.adaptive_ece(pt, yt))
                r['brier'].append(V.brier(pt, yt)); r['T'].append(V.fit_temperature(Lv, yv))
                r['best_epoch'].append(out['info']['best_epoch'])
        summary[tag] = {m: float(np.mean(v)) for m, v in r.items()}
        summary[tag].update({f'{m}_sd': float(np.std(v, ddof=1)) for m, v in r.items()},
                            cfg=cfg, corrected=corrected)
        s_ = summary[tag]
        print(f"{tag:<18}{s_['val_top1']:>10.4f}{s_['top1']:>10.4f}{s_['ece']:>10.4f}"
              f"{s_['T']:>7.3f}{s_['best_epoch']:>8.1f}")

    ok = {k: v for k, v in summary.items() if v['corrected'] and v['cfg']}
    best = max(ok, key=lambda k: (ok[k]['val_top1'], -ok[k]['val_nll']))
    chosen = ok[best]['cfg']
    PREP_TAG = ('perframe' if chosen['centre'] == 'perframe' else 'frame0') + \
               ('_align' if chosen['align'] else '')
    print(f'\nchosen on VALIDATION accuracy (corrected topology only): {best} '
          f'-> {chosen}, tag {PREP_TAG!r}')
    leg = summary['A legacy']
    print(f"legacy vs chosen, test top-1: {leg['top1']:.4f} vs {ok[best]['top1']:.4f}; "
          f"test aECE {leg['ece']:.4f} vs {ok[best]['ece']:.4f}  (reported only)")
    Ts = {k: round(v['T'], 3) for k, v in summary.items()}
    if min(Ts.values()) < 1.0 < max(Ts.values()):
        print(f'!! fitted temperature crosses 1.0 across configs: {Ts}')

    (RESULTS / 'prep_config.json').write_text(json.dumps(
        {'chosen': best, 'cfg': chosen, 'prep_tag': PREP_TAG,
         'selection': 'mean validation top-1 over splits x seeds, corrected topology only',
         'splits': ABLATION_SPLITS, 'seeds': ABLATION_SEEDS,
         'summary': {k: {i: j for i, j in v.items() if i != 'cfg'}
                     for k, v in summary.items()}}, indent=2))
    print('wrote', RESULTS / 'prep_config.json')
    mark_done('05_ablation')
else:
    meta = json.loads((RESULTS / 'prep_config.json').read_text())
    print('  [skip] stage 5 already done --', meta['chosen'], meta['cfg'])

## Stage 6 — small-dataset training (GPU)

In [ ]:
# =====================================================================
# STAGE 6 -- MSR-Action3D, DailyActivity3D, UTKinect -> logits   [GPU, ~1 h]
# =====================================================================
# ST-GCN, 4 streams x (3 random subject-disjoint splits + fixed split) x
# 5 seeds. Early stopping on validation NLL (best weights restored). MC dropout
# (MC_PASSES passes) with the inference dropout rate chosen from MC_P_GRID on
# validation NLL. The 5 seeds form the 5-member deep ensemble in Stage 9c.
# Validation AND test logits are stored for every cell; resumable.
import har3d_v2 as V, har3d_models_v2 as M2, har3d_grid as G
importlib.reload(V); importlib.reload(M2); importlib.reload(G)
from store import LogitStore

meta = json.loads((RESULTS / 'prep_config.json').read_text())
PREP_CFG, PREP_TAG = meta['cfg'], meta['prep_tag']
print('preprocessing (chosen on validation in Stage 5):', PREP_CFG, PREP_TAG)

if not stage_done('06_msr_logits'):
    log = G.run_grid(CACHE, LOGITS, PREP_CFG, PREP_TAG, model_tag='stgcn',
                     streams=STREAMS, seeds=SEEDS, mc_passes=MC_PASSES,
                     mc_p_grid=MC_P_GRID, T=T_FRAMES, max_epochs=MAX_EPOCHS_MSR,
                     patience=PATIENCE_MSR, min_epochs=MIN_EPOCHS_MSR,
                     max_zero_frac=MAX_ZERO_FRAC, min_frames=MIN_FRAMES)
    prev = RESULTS / 'train_log_stgcn.json'
    old = json.loads(prev.read_text()) if prev.exists() else []
    prev.write_text(json.dumps(old + log['cells'], indent=1))
    print(f"\nproduced {log['produced']} files, skipped {log['skipped']} existing cells")
    mark_done('06_msr_logits')
else:
    print('  [skip] stage 6 already done')
LogitStore(LOGITS).inventory()

## Stage 6b — label smoothing and focal loss (GPU)

In [ ]:
# =====================================================================
# STAGE 6b -- training-time calibration: label smoothing, focal loss  [GPU, ~1.5 h]
# =====================================================================
# Identical grid, filter, splits, seeds and early stopping as Stage 6; only
# the loss changes. Label smoothing eps=0.1 (Szegedy et al. 2016), focal
# gamma=2 (Lin et al. 2017; Mukhoti et al. 2020). Model tags 'stgcn_ls' and
# 'stgcn_focal'. Resumable.
import har3d_grid as G
importlib.reload(G)
from store import LogitStore

meta = json.loads((RESULTS / 'prep_config.json').read_text())
PREP_CFG, PREP_TAG = meta['cfg'], meta['prep_tag']
METHODS = [('stgcn_ls', dict(loss='ls', smoothing=0.1)),
           ('stgcn_focal', dict(loss='focal', gamma=2.0))]

if not stage_done('06b_traintime_calib'):
    for tag, kw in METHODS:
        log = G.run_grid(CACHE, LOGITS, PREP_CFG, PREP_TAG, model_tag=tag,
                         loss_kw=kw, streams=STREAMS, seeds=SEEDS, mc_passes=0,
                         T=T_FRAMES, max_epochs=MAX_EPOCHS_MSR,
                         patience=PATIENCE_MSR, min_epochs=MIN_EPOCHS_MSR,
                         max_zero_frac=MAX_ZERO_FRAC, min_frames=MIN_FRAMES)
        (RESULTS / f'train_log_{tag}.json').write_text(json.dumps(log['cells'], indent=1))
        print(f"{tag}: produced {log['produced']}, skipped {log['skipped']}")
    mark_done('06b_traintime_calib')
else:
    print('  [skip] stage 6b already done')
LogitStore(LOGITS).inventory()

## Stage 5b — preprocessing sensitivity of the calibration conclusions (GPU, ~45 min, resumable)
Reviewer 2, point 14. The main experiment uses the preprocessing chosen on validation accuracy in Stage 5. Here the same ST-GCN grid (joint stream, the same four test sets, seeds 0–2, same early stopping) is retrained under the alternative preprocessings, and the logits are kept, so that Stage 9g can run the matched-null test on every variant. Logits go to `runs/<RUN_TAG>/logits_prep/`, apart from the main store.

In [ ]:
# =====================================================================
# STAGE 5b -- preprocessing sensitivity runs        [GPU, ~45 min; resumable]
# =====================================================================
#   frame0            centred on the hip of frame 0, no alignment   = MAIN experiment (already stored)
#   frame0_align      centred on the hip of frame 0, aligned        all three datasets
#   perframe_align    centred per frame, aligned                     all three datasets
#   legacy            legacy tree and legacy centring               MSR-Action3D only (11 edges are not bones)
#   legacy_centring   corrected tree, legacy centring               MSR-Action3D only
# Same quality filter, same subject-disjoint test sets (three random + the
# fixed split), seeds ABLATION_SEEDS, early stopping on validation NLL.
# Stage 5 did not keep its logits, so its runs could not be tested against the
# null; these can. Nothing here is used to select anything.
import har3d_v2 as V, har3d_models_v2 as M2, har3d_grid as G
importlib.reload(V); importlib.reload(M2); importlib.reload(G)
from store import Key, LogitStore

PREP_LOGITS = RUN / 'logits_prep'
PREP_LOGITS.mkdir(parents=True, exist_ok=True)
PREP_ALT = {'frame0_align': dict(centre='frame0', align=True),
            'perframe_align': dict(centre='perframe', align=True)}
PREP_LEGACY = ('legacy', 'legacy_centring')


def legacy_build(cache, stream, parents, keep, T=T_FRAMES):
    """The preprocessing of the first version of the pipeline (Stage 5, configs A and B)."""
    off = cache['offsets']
    X = np.zeros((len(off) - 1, 3, T, len(parents)), np.float32)
    for i in range(len(off) - 1):
        x = V.make_stream(cache['xyz'][off[i]:off[i + 1]], stream, parents)
        x = V.resample(x, T)
        x = x - x[0:1, 0:1, :]
        s = np.linalg.norm(x, axis=-1).mean()
        X[i] = (x / s if s > 1e-6 else x).transpose(2, 0, 1)
    return X[keep]


def _has_gpu():
    import torch
    return torch.cuda.is_available()


if not RUN_PREP_SENSITIVITY:
    print('  RUN_PREP_SENSITIVITY is False -- skipped')
elif stage_done('05b_prep_sensitivity'):
    print('  [skip] stage 5b already done')
elif not (_has_gpu() or ALLOW_CPU_TRAINING):
    print('  !! no GPU in this runtime -- Stage 5b skipped for now. Switch to a GPU runtime '
          '(Runtime > Change runtime type), re-run Stage 0 and this cell.')
else:
    t0 = time.time()
    for tag, cfg in PREP_ALT.items():
        log = G.run_grid(CACHE, PREP_LOGITS, cfg, tag, model_tag='stgcn', streams=('joint',),
                         seeds=ABLATION_SEEDS, mc_passes=0, T=T_FRAMES,
                         max_epochs=MAX_EPOCHS_MSR, patience=PATIENCE_MSR,
                         min_epochs=MIN_EPOCHS_MSR, max_zero_frac=MAX_ZERO_FRAC,
                         min_frames=MIN_FRAMES, verbose=False)
        print(f"  {tag:<16} produced {log['produced']} files, skipped {log['skipped']} cells "
              f"({time.time() - t0:.0f} s)")
    cache = V.load_cache(CACHE / 'msr_action3d.npz')
    _, y5, subj5, keep5 = V.masked_tensors(cache, 'joint', T_FRAMES, dict(centre='frame0', align=False),
                                           MAX_ZERO_FRAC, MIN_FRAMES)
    pstore, made = LogitStore(PREP_LOGITS), 0
    for tag, parents in (('legacy', V.LEGACY_PARENTS), ('legacy_centring', V.MSR_PARENTS)):
        X5, A5 = legacy_build(cache, 'joint', parents, keep5), V.adjacency(parents)
        for sp in G.splits_for(subj5, y5):
            name = f"{sp['name']}_{tag}"
            for seed in ABLATION_SEEDS:
                if pstore.has(Key('stgcn', 'action3d', 'joint', name, seed, 'test')):
                    continue
                out, _ = M2.train_and_logits(X5, y5, sp, int(y5.max()) + 1, A5,
                                             max_epochs=MAX_EPOCHS_MSR, patience=PATIENCE_MSR,
                                             min_epochs=MIN_EPOCHS_MSR, seed=seed, verbose=False)
                prov = dict(arch='stgcn', method='stgcn', stream='joint', T=T_FRAMES, prep=tag,
                            subjects=sp['subjects'], train_info=out['info'])
                for sub in ('val', 'test'):
                    pstore.save(Key('stgcn', 'action3d', 'joint', name, seed, sub), *out[sub],
                                provenance=prov)
                made += 1
        print(f'  {tag:<16} done ({time.time() - t0:.0f} s)')
    print(f'  legacy variants: {made} runs trained')
    mark_done('05b_prep_sensitivity')
print('  preprocessing-sensitivity logit files:', len(list(PREP_LOGITS.glob('*__test.npz'))))

## Stage 7 — NTU-60 tensors (CPU High-RAM, skipped if already on Drive)

In [ ]:
# =====================================================================
# STAGE 7 -- NTU-60 raw -> tensors      [CPU HIGH-RAM, 1-2 h; skipped if present]
# =====================================================================
# Skipped when preprocessed/ntu/NTU60_CS.npz AND performer.txt are already on
# Drive (they are protocol-independent). Otherwise: >>> switch to a CPU
# High-RAM runtime, re-run Stage 0, then this cell <<< -- it peaks at 12-18 GB.
# performer.txt (the subject id of every sample) is copied to Drive so the
# ladder's subject-disjoint validation can be rebuilt in a fresh runtime.
import har3d_ntu as N
importlib.reload(N)

NTU_DIR = PREP / 'ntu'
have_npz = (NTU_DIR / 'NTU60_CS.npz').exists()
have_perf = (NTU_DIR / 'performer.txt').exists()

if have_npz and have_perf:
    print('  [skip] NTU60_CS.npz and performer.txt already on Drive')
elif have_npz and not have_perf:
    # tensors exist but the subject list was never copied: fetch it from the
    # repository's statistics folder (cheap, no regeneration).
    repo = N.clone_repo()
    src = N.find_performer_file(drive_root=None, repo=str(repo))
    NTU_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, NTU_DIR / 'performer.txt')
    print('  copied performer.txt ->', NTU_DIR)
else:
    repo = N.clone_repo()
    N.patch_numpy_aliases(repo)
    N.place_raw(DRIVE, repo, ntu120=False)
    N.check_statistics(repo, ntu120=False)
    N.run_datagen(repo, ntu120=False)
    N.verify_outputs(repo, ntu120=False)
    N.push_to_drive(DRIVE, repo, ntu120=False)

perf = N.find_performer_file(drive_root=DRIVE)
print('performer file:', perf)
print('train performers:', len(N.train_performers(perf)),
      '| test performers:', len(N.test_performers(perf)))
mark_done('07_ntu_tensors')

## Stage 8 — NTU-60 training-set ladder (GPU, High-RAM)
Always run this cell before 8b in a new runtime (it defines the loader and skips instantly once done).

In [ ]:
# =====================================================================
# STAGE 8 -- NTU-60 TRAINING-SET LADDER, three backbones     [GPU, long; resumable]
# =====================================================================
# >>> GPU runtime (L4/A100, High-RAM). Re-run Stage 0 first. <<<
#
# Fixed X-Sub test set (16,487 clips) and a fixed subject-disjoint validation
# set for EVERY rung, seed and backbone; only the number of training samples
# moves (500 -> 40,091 minus validation). Each rung trains with EARLY STOPPING
# on validation NLL (best weights restored), so no rung is under- or over-
# trained by a shared step budget.
#
# Backbones (one shared trunk; only the graph layer differs):
#   stgcn   fixed anatomical graph           (Yan et al. 2018)
#   agcn    A + learned B + data-dependent C  (Shi et al. 2019)
#   ctrgcn  channel-wise topology refinement  (Chen et al. 2021)
# Optional ablation: 'stgcn_fixed' = ST-GCN with the old fixed 6,000-step
# cosine budget, to show what the early-stopping change does.
#
# Order of work: ST-GCN first, then 2s-AGCN, CTR-GCN, then the fixed-budget
# ablation. Every cell is stored as soon as it finishes; if Colab disconnects,
# re-run Stage 0 and this cell and it continues where it stopped.
import har3d_scale as S, har3d_ntu as N, har3d_backbones as HB
importlib.reload(S); importlib.reload(N); importlib.reload(HB)
from store import LogitStore

STORE = LogitStore(LOGITS)


def load_ntu_ladder():
    npz = PREP / 'ntu' / 'NTU60_CS.npz'
    if not npz.exists():
        raise FileNotFoundError('NTU60_CS.npz not on Drive -- run Stage 7')
    perf = N.find_performer_file(drive_root=DRIVE)      # raises if absent
    x_tr, y_tr, x_te, y_te = S.load_ntu_npz(npz)
    val_idx, pool_idx, _, proto = S.ladder_protocol(
        x_tr, y_tr, perf, NTU_VAL_SUBJECTS, seed=0,
        record=RESULTS / 'ladder_protocol.json')
    print(f"  validation subjects {proto['val_subjects']} -> {proto['n_val']} clips; "
          f"training pool {proto['n_pool']}; test {len(y_te)}")
    Xval = S.prepare_ntu_tensors(x_tr[val_idx], LADDER_STREAM, T=T_FRAMES, verbose=False)
    Xte = S.prepare_ntu_tensors(x_te, LADDER_STREAM, T=T_FRAMES, verbose=False)
    return dict(x_tr=x_tr, y_tr=y_tr, pool=pool_idx, Xval=Xval,
                yval=y_tr[val_idx], Xte=Xte, yte=y_te, proto=proto)


LADDER_COMMON = dict(stream=LADDER_STREAM, T=T_FRAMES, max_steps=LADDER_MAX_STEPS,
                     eval_every=LADDER_EVAL_EVERY, patience=LADDER_PATIENCE,
                     min_steps=LADDER_MIN_STEPS)

if not stage_done('08_ladder'):
    if 'NTU_DATA' not in globals():
        NTU_DATA = load_ntu_ladder()
    D8 = NTU_DATA
    jobs = [(arch, arch, True) for arch in LADDER_BACKBONES]
    if RUN_FIXED_BUDGET_ABLATION:
        jobs.append(('stgcn_fixed', 'stgcn', False))
    rows = []
    for tag, arch, es in jobs:
        rows += S.run_ladder(STORE, D8['x_tr'], D8['y_tr'], D8['pool'], D8['Xval'],
                             D8['yval'], D8['Xte'], D8['yte'], model_tag=tag,
                             arch=arch, rungs=LADDER_N, seeds=LADDER_SEEDS,
                             early_stop=es, fixed_steps=FIXED_BUDGET_STEPS,
                             protocol=D8['proto'], **LADDER_COMMON)
    prev = RESULTS / 'ladder_progress.json'
    old = json.loads(prev.read_text()) if prev.exists() else []
    prev.write_text(json.dumps(old + rows, indent=1))
    mark_done('08_ladder')
else:
    print('  [skip] stage 8 already done')
LogitStore(LOGITS).inventory()

## Stage 8b — adjacency-freezing ablation (GPU)

In [ ]:
# =====================================================================
# STAGE 8b -- adjacency-freezing ablation (2s-AGCN)      [GPU; resumable]
# =====================================================================
# Tests the causal claim directly: is the 2s-AGCN calibration difference
# produced by the LEARNED graph continuing to adapt? Identical to the 'agcn'
# ladder (same pool, validation, subsamples, early stopping) except that the
# graph-generating parameters (B, alpha, theta, phi) stop updating after
# FREEZE_AFTER_STEPS steps; the rest of the network keeps training.
# Stored as model 'agcn_frozen'. Run after Stage 8 (needs the same protocol
# file); can be skipped with RUN_FREEZE_ABLATION = False.
import har3d_scale as S
importlib.reload(S)
from store import LogitStore

STORE = LogitStore(LOGITS)
if not RUN_FREEZE_ABLATION:
    print('  RUN_FREEZE_ABLATION is False -- skipped')
elif not stage_done('08b_freeze'):
    if 'NTU_DATA' not in globals():
        NTU_DATA = load_ntu_ladder()        # defined in the Stage 8 cell
    D8 = NTU_DATA
    rows = S.run_ladder(STORE, D8['x_tr'], D8['y_tr'], D8['pool'], D8['Xval'],
                        D8['yval'], D8['Xte'], D8['yte'], model_tag='agcn_frozen',
                        arch='agcn_frozen', rungs=FREEZE_RUNGS, seeds=LADDER_SEEDS,
                        early_stop=True, freeze_graph_after=FREEZE_AFTER_STEPS,
                        protocol=D8['proto'], **LADDER_COMMON)
    prev = RESULTS / 'ladder_progress.json'
    old = json.loads(prev.read_text()) if prev.exists() else []
    prev.write_text(json.dumps(old + rows, indent=1))
    mark_done('08b_freeze')
else:
    print('  [skip] stage 8b already done')

## Stage 8c — confirmatory replication of the backbone comparison (GPU, High-RAM, long; resumable)
Reviewer 2, points 2, 3, 4 and 11. New seeds for the three backbones at the three large rungs, all trained in **one** software environment, with the hypothesis written to `results/replication_protocol.json` before the first run. The original runs (seeds 0–2) are not touched and not retrained. Also stored for every new run: the test logits at fixed step counts (matched training durations) and at the best validation accuracy (an alternative stopping rule). Then seed 0 at 10,000 clips is retrained for each backbone in today's environment, to measure how far a change of environment moves the numbers.

If Colab disconnects: reconnect to the **same GPU type**, run Stage 0 (and the module cells), the Stage 8 cell, then this cell. It continues with the first missing run.

In [ ]:
# =====================================================================
# STAGE 8c -- confirmatory replication, one pinned environment   [GPU, long; resumable]
# =====================================================================
# >>> GPU runtime, High-RAM. Run Stage 0, the module cells and the Stage 8
#     cell first (it defines the NTU loader and skips instantly). <<<
#
# WHAT IS TRAINED (model tags are new, so nothing stored is overwritten)
#   stgcn_rep / agcn_rep / ctrgcn_rep   seeds REPL_SEEDS at rungs REPL_RUNGS,
#        the protocol of Stage 8 exactly (same pool, validation set, test set,
#        subsamples per (n, seed), early stopping on validation NLL). On top of
#        the main output each run also stores the test logits
#          - at steps REPL_SNAPSHOT_STEPS (training continues to the last
#            snapshot step if early stopping fires before it; the stopping
#            decision and the restored weights are frozen at that moment, so
#            the main output is the Stage 8 protocol, unit-tested)
#          - of the weights with the best validation ACCURACY
#   stgcn_env / agcn_env / ctrgcn_env   seed ENV_RERUN_SEED at ENV_RERUN_RUNG
#        clips: the Stage 8 run repeated in today's environment, nothing else
#        changed, to compare with the stored original.
#   seeds REPL_EXTRA_SEEDS              last, only if RUN_REPL_EXTRA.
#
# WHAT IS FIXED BEFORE TRAINING (results/replication_protocol.json)
#   Hypotheses: at >= 10,000 training clips the excess adaptive ECE of 2s-AGCN
#   and of CTR-GCN exceeds that of ST-GCN. Statistic: per seed, the difference
#   in excess adaptive ECE averaged over the three rungs. Unit: the seed.
#   Tests: exact sign-flip test and t-interval over seeds, on the new seeds
#   alone and on all seeds. The first-version comparison (seeds 0-2), where the
#   10,000-clip threshold was chosen after seeing the curves, is exploratory.
#
# ONE ENVIRONMENT: the software stack and the GPU of the first replication run
# are written to results/replication_env.json; a later session with a
# different stack or GPU refuses to continue (ALLOW_ENV_MIX = True overrides,
# and every run records its own environment in its provenance either way).
import platform
import har3d_scale as S
importlib.reload(S)
from store import Key, LogitStore
import torch

STORE = LogitStore(LOGITS)
CKPT = RUN / 'logits_ckpt'
CKPT.mkdir(parents=True, exist_ok=True)

REPL_PLAN = RESULTS / 'replication_protocol.json'
if not REPL_PLAN.exists():
    REPL_PLAN.write_text(json.dumps({
        'registered': time.ctime(),
        'status': 'written before any replication run was trained',
        'hypotheses': ['H1: excess adaptive ECE of 2s-AGCN > ST-GCN at >= 10,000 training clips',
                       'H2: excess adaptive ECE of CTR-GCN > ST-GCN at >= 10,000 training clips'],
        'rungs': REPL_RUNGS, 'new_seeds': REPL_SEEDS, 'optional_seeds': REPL_EXTRA_SEEDS,
        'statistic': 'per seed: difference in excess adaptive ECE (15 equal-mass bins, matched '
                     'null floor) averaged over the three rungs',
        'unit': 'seed (independently initialised and subsampled training runs)',
        'tests': ['exact sign-flip test over seeds, two-sided',
                  't-interval of the mean per-seed difference',
                  'reported for the new seeds alone and for all seeds'],
        'protocol': 'identical to Stage 8: same pool, validation subjects, test set, '
                    'stratified subsample per (n, seed), early stopping on validation NLL',
        'secondary': ['same comparison at matched training steps ' + str(REPL_SNAPSHOT_STEPS),
                      'same comparison at the best-validation-accuracy checkpoint',
                      'same-seed re-runs in the replication environment vs the stored originals'],
        'exploratory': 'the comparison on seeds 0-2 in the first version (threshold chosen post hoc)',
    }, indent=2))
    print('  replication protocol registered ->', REPL_PLAN.name)

STACK = {'python': sys.version.split()[0], 'numpy': np.__version__, 'torch': torch.__version__,
         'cuda': torch.version.cuda,
         'cudnn': torch.backends.cudnn.version() if torch.cuda.is_available() else None,
         'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}


def _missing(tag, rungs, seeds):
    n_pool = json.loads((RESULTS / 'ladder_protocol.json').read_text())['n_pool']
    return [(n, s) for n in rungs for s in seeds
            if not STORE.has(Key(tag, 'ntu60', LADDER_STREAM, f'ladder_n{min(n, n_pool)}', s, 'test'))]


REPL_KW = dict(snapshot_steps=REPL_SNAPSHOT_STEPS, track_best_acc=True,
               run_until=max(REPL_SNAPSHOT_STEPS), extras_dir=CKPT)
jobs = [(f'{a}_rep', a, REPL_RUNGS, [s], REPL_KW, 'confirmatory replication')
        for s in REPL_SEEDS for a in LADDER_BACKBONES]
if RUN_ENV_RERUNS:
    jobs += [(f'{a}_env', a, [ENV_RERUN_RUNG], [ENV_RERUN_SEED], {}, 'environment re-run of a Stage 8 cell')
             for a in LADDER_BACKBONES]
if RUN_REPL_EXTRA:
    jobs += [(f'{a}_rep', a, REPL_RUNGS, [s], REPL_KW, 'confirmatory replication')
             for s in REPL_EXTRA_SEEDS for a in LADDER_BACKBONES]
todo = [(j, _missing(j[0], j[2], j[3])) for j in jobs]
todo = [(j, m) for j, m in todo if m]
n_runs = sum(len(m) for _, m in todo)

# time estimate from the stored original runs (seconds per step of each backbone)
def _estimate():
    rate, steps = {}, {}
    for a in LADDER_BACKBONES:
        sec = st = 0.0
        for k in STORE.list_keys():
            if k.model == a and k.dataset == 'ntu60' and k.subset == 'test':
                info = STORE.load(k)['provenance'].get('train_info', {}) or {}
                if info.get('seconds') and info.get('steps_run'):
                    sec += info['seconds']; st += info['steps_run']
                    steps.setdefault((a, k.split), []).append(info['steps_run'])
        rate[a] = sec / st if st else None
    tot = 0.0
    n_pool = json.loads((RESULTS / 'ladder_protocol.json').read_text())['n_pool']
    for (tag, a, rungs, seeds, kw, _), miss in todo:
        for n, s in miss:
            ref = steps.get((a, f'ladder_n{min(n, n_pool)}'))
            if rate[a] is None or not ref:
                return None
            tot += rate[a] * max(float(np.mean(ref)), kw.get('run_until', 0))
    return tot / 3600


if not todo:
    print('  [skip] every replication and environment run is already stored')
elif not (torch.cuda.is_available() or ALLOW_CPU_TRAINING):
    print(f'  !! no GPU in this runtime -- {n_runs} training runs are still missing. Switch to a GPU '
          f'runtime (High-RAM), re-run Stage 0, the module cells, the Stage 8 cell and this cell.')
else:
    ENVF = RESULTS / 'replication_env.json'
    if ENVF.exists():
        pinned = json.loads(ENVF.read_text())['stack']
        diff = {k: (pinned.get(k), STACK[k]) for k in STACK if pinned.get(k) != STACK[k]}
        if diff and not ALLOW_ENV_MIX:
            raise RuntimeError(
                'the replication was started in a different environment; (pinned, now) = '
                f'{diff}. Reconnect until Colab gives the same GPU type and library versions, or '
                'set ALLOW_ENV_MIX = True in Stage 0 to continue anyway (the mix is recorded '
                'and reported).')
        if diff:
            print('  !! continuing in a different environment (ALLOW_ENV_MIX):', diff)
    else:
        ENVF.write_text(json.dumps({'pinned': time.ctime(), 'stack': STACK,
                                    'platform': platform.platform()}, indent=2))
        print('  replication environment pinned:', STACK)
    est = _estimate()
    print(f'  {n_runs} training runs to do' +
          (f'; about {est:.1f} GPU hours at the speed of the original runs' if est else ''))
    if 'NTU_DATA' not in globals():
        NTU_DATA = load_ntu_ladder()        # defined in the Stage 8 cell
    D8 = NTU_DATA
    PROG = RESULTS / 'replication_progress.json'
    for (tag, arch, rungs, seeds, kw, what), _ in todo:
        rows = S.run_ladder(STORE, D8['x_tr'], D8['y_tr'], D8['pool'], D8['Xval'], D8['yval'],
                            D8['Xte'], D8['yte'], model_tag=tag, arch=arch, rungs=rungs,
                            seeds=seeds, early_stop=True, protocol=D8['proto'],
                            extra_provenance={'environment': STACK, 'revision_run': what},
                            **LADDER_COMMON, **kw)
        old = json.loads(PROG.read_text()) if PROG.exists() else []
        PROG.write_text(json.dumps(old + rows, indent=1))
    print('\n  all requested replication and environment runs are stored')

for tag in [f'{a}_{x}' for x in ('rep', 'env') for a in LADDER_BACKBONES]:
    have = sorted((k.split, k.seed) for k in STORE.list_keys()
                  if k.model == tag and k.subset == 'test')
    if have:
        print(f'  {tag:<11} {len(have)} test cells, seeds {sorted({s for _, s in have})}')
print('  matched-step / best-accuracy checkpoint files:', len(list(CKPT.glob('ck__*.npz'))))

## Stage 9 — evaluate every cell (CPU)
Cached by logit hash; the stream fusions, with their floors, Brier and NLL before and after their temperature, are recomputed every time.

In [ ]:
# =====================================================================
# STAGE 9 -- evaluate every stored cell        [CPU, ~10-20 min first time; ~3 min after]
# =====================================================================
# One row per test cell in results/metrics.json: accuracy, equal-width and
# adaptive ECE, classwise ECE, Brier and its Murphy decomposition
# (reliability, resolution, uncertainty, within-bin), NLL, AURC, validation-
# fitted temperature and post-TS metrics, bootstrap CIs, and the NULL MODEL
# (observed adaptive ECE vs the floor a perfectly calibrated model with the
# same confidence profile would report). Rows are cached by logit hash, so
# re-running after new cells arrive only evaluates the new ones.
import csv
import har3d_v2 as V, har3d_calib as C
importlib.reload(V); importlib.reload(C)
from store import Key, LogitStore

STORE = LogitStore(LOGITS)
TABDIR = RESULTS / 'tables'; TABDIR.mkdir(parents=True, exist_ok=True)
# Null draws: the small-dataset separability test is Holm-corrected over 240
# cells, and a Monte-Carlo p-value cannot fall below 1/(S+1), so S must exceed
# 240/0.05 by a wide margin. 20,000 draws -> smallest Holm-adjusted p ~0.012.
N_SIM_SMALL, N_SIM_NTU = 20000, 500
N_BOOT_SMALL, N_BOOT_NTU = 1000, 200
NULL_VERSION = 'v3-plus1'          # bump to force the null to be recomputed

mfile = RESULTS / 'metrics.json'
cached = {}
if mfile.exists():
    for r in json.loads(mfile.read_text()):
        cached[(r['file'], r['sha1'])] = r

rows, new = [], 0
for k in STORE.list_keys():
    if k.subset != 'test':
        continue
    d = STORE.load(k)
    sha = d['provenance'].get('logit_sha1')
    nsim = N_SIM_NTU if k.dataset == 'ntu60' else N_SIM_SMALL
    old = cached.get((k.fname(), sha))
    if old is not None and old.get('null_sims') == nsim and old.get('null_version') == NULL_VERSION:
        rows.append(old); continue
    if old is not None:            # metrics still valid; only the null changes
        sig = C.ece_significance(V.softmax(d['logits']), d['labels'], n_sim=nsim, seed=k.seed)
        old.update({f'null_{a}': b for a, b in sig.items()})
        old.update(null_sims=nsim, null_version=NULL_VERSION)
        rows.append(old); new += 1; continue
    vk = Key(k.model, k.dataset, k.stream, k.split, k.seed, 'val')
    lv = yv = None
    if STORE.has(vk):
        dv = STORE.load(vk); lv, yv = dv['logits'], dv['labels']
    nb = N_BOOT_NTU if k.dataset == 'ntu60' else N_BOOT_SMALL
    m = V.evaluate(d['logits'], d['labels'], lv, yv, n_boot=nb, seed=k.seed)
    sig = C.ece_significance(V.softmax(d['logits']), d['labels'], n_sim=nsim, seed=k.seed)
    m.update({f'null_{a}': b for a, b in sig.items()})
    m.update(null_sims=nsim, null_version=NULL_VERSION)
    info = d['provenance'].get('train_info', {}) or {}
    m.update(file=k.fname(), sha1=sha, model=k.model, dataset=k.dataset,
             stream=k.stream, split=k.split, seed=k.seed,
             n_train=d['provenance'].get('n_train'),
             best_step=info.get('best_step'), steps_run=info.get('steps_run'),
             best_epoch=info.get('best_epoch'), epochs_run=info.get('epochs_run'),
             frozen_at=info.get('frozen_at'), mc_p=info.get('mc_p'))
    rows.append(m); new += 1

mfile.write_text(json.dumps(rows, indent=1, default=float))
print(f'{len(rows)} test cells ({new} newly evaluated) -> {mfile}\n')

# ---- summaries: dataset x model x stream, split-then-seed aggregation ----
small = [r for r in rows if r['dataset'] != 'ntu60']
for ds in sorted({r['dataset'] for r in small}):
    for mdl in sorted({r['model'] for r in small if r['dataset'] == ds}):
        for st in STREAMS:
            sub = [r for r in small if (r['dataset'], r['model'], r['stream']) == (ds, mdl, st)]
            if not sub:
                continue
            a = {mt: V.aggregate_nested(sub, mt) for mt in
                 ('top1', 'ece_adaptive', 'brier', 'nll', 'temperature')}
            sep = sum(r['null_separable'] for r in sub)
            print(f"{ds:<9}{mdl:<12}{st:<13} top1 {a['top1']['mean']:.3f}"
                  f"±{a['top1']['std_across_splits']:.3f}  aECE {a['ece_adaptive']['mean']:.3f}"
                  f"  floor {np.mean([r['null_null_mean'] for r in sub]):.3f}"
                  f"  separable {sep}/{len(sub)}  T {a['temperature']['mean']:.2f}")

# ---- ladder convergence table (what early stopping chose, per rung) ----
lad = [r for r in rows if r['dataset'] == 'ntu60']
if lad:
    with open(TABDIR / 'ladder_convergence.csv', 'w', newline='') as fh:
        w = csv.writer(fh)
        w.writerow(['model', 'n_train', 'n_seeds', 'best_step_mean', 'steps_run_mean',
                    'epochs_at_best_mean', 'top1_mean', 'top1_sd', 'frozen_at'])
        for mdl in sorted({r['model'] for r in lad}):
            for n in sorted({r['n_train'] for r in lad if r['model'] == mdl}):
                g = [r for r in lad if r['model'] == mdl and r['n_train'] == n]
                bs = [r['best_step'] or 0 for r in g]
                w.writerow([mdl, n, len(g), f'{np.mean(bs):.0f}',
                            f"{np.mean([r['steps_run'] or 0 for r in g]):.0f}",
                            f'{np.mean(bs) * 64 / n:.1f}',
                            f"{np.mean([r['top1'] for r in g]):.4f}",
                            f"{np.std([r['top1'] for r in g], ddof=1) if len(g) > 1 else 0:.4f}",
                            g[0]['frozen_at']])
    print('\nwrote', TABDIR / 'ladder_convergence.csv')
    unfrozen = [r for r in lad if r['model'] == 'agcn_frozen' and r['frozen_at'] is None]
    if unfrozen:
        print(f'  note: {len(unfrozen)} agcn_frozen cells early-stopped before the '
              f'freeze step; they are excluded from the freezing comparison')

# ---- temperature stability: per-split vs pooled (ST-GCN, joint) ----
print('\ntemperature stability (ST-GCN, joint stream)')
for ds in sorted({r['dataset'] for r in small}):
    vl, vy, per = [], [], []
    for k in STORE.list_keys():
        if (k.subset, k.dataset, k.stream, k.model) != ('val', ds, 'joint', 'stgcn'):
            continue
        d = STORE.load(k); vl.append(d['logits']); vy.append(d['labels'])
        per.append(V.fit_temperature(d['logits'], d['labels']))
    if per:
        Tp, npool = V.fit_temperature_pooled(vl, vy)
        print(f'  {ds:<9} per-split T {np.mean(per):.3f} ± {np.std(per):.3f} '
              f'(range {min(per):.2f}-{max(per):.2f}); pooled T {Tp:.3f} over {npool}')

# ---- stream fusion, with a temperature fitted to the fusion itself ----
# v6: a fused model has its own confidences and hence its own floor, so the
# floor (N_SIM_FLOOR null draws: a mean, not a p-value) and the binning-free
# Brier score and NLL are recorded before and after the fusion's temperature.
N_SIM_FLOOR = 1000
fus = []
for k in STORE.list_keys():
    if k.model != 'stgcn' or k.stream != 'joint' or k.subset != 'test' or k.dataset == 'ntu60':
        continue
    for nm, st in (('joint', ('joint',)), ('joint+bone', ('joint', 'bone')),
                   ('4-stream', tuple(STREAMS))):
        try:
            z, y, used = STORE.ensemble_logits(k, st)
            zv, yv, _ = STORE.ensemble_logits(
                Key(k.model, k.dataset, 'joint', k.split, k.seed, 'val'), st)
        except (FileNotFoundError, ValueError):
            continue
        if len(used) != len(st):
            continue
        Tf = V.fit_temperature(zv, yv)
        p0, p1 = V.softmax(z), V.softmax(z, Tf)
        fus.append([k.dataset, nm, k.split, k.seed, V.top1(p0, y),
                    V.adaptive_ece(p0, y), V.adaptive_ece(p1, y),
                    C.ece_null(p0, n_sim=N_SIM_FLOOR, seed=k.seed)[1],
                    C.ece_null(p1, n_sim=N_SIM_FLOOR, seed=k.seed)[1],
                    V.brier(p0, y), V.brier(p1, y), V.nll(p0, y), V.nll(p1, y), Tf])
with open(TABDIR / 'stream_fusion.csv', 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['dataset', 'fusion', 'split', 'seed', 'top1', 'ece_adaptive', 'ece_adaptive_ts',
                'floor', 'floor_ts', 'brier', 'brier_ts', 'nll', 'nll_ts', 'temperature'])
    w.writerows([r[:4] + [f'{v:.4f}' for v in r[4:]] for r in fus])
print('wrote', TABDIR / 'stream_fusion.csv', f'({len(fus)} rows)')
for ds in sorted({r[0] for r in fus}):
    g = [r for r in fus if r[0] == ds and r[1] == '4-stream']
    if g:
        m = np.mean([r[4:13] for r in g], 0)
        print(f'  {ds:<9} 4-stream fusion: top-1 {m[0]:.3f}  aECE {m[1]:.3f} (floor {m[3]:.3f}) -> '
              f'{m[2]:.3f} (floor {m[4]:.3f}) after its temperature;  Brier {m[5]:.3f} -> {m[6]:.3f}, '
              f'NLL {m[7]:.3f} -> {m[8]:.3f}')
mark_done('09_eval')

## Stage 9b — statistics over test sets and seed–rung pairs (CPU)
Kept from v6 when already done (its inputs did not change); the seed-level re-analysis of the revision is Stage 9h.

In [ ]:
# =====================================================================
# STAGE 9b -- publication statistics          [CPU, ~25 min]
# =====================================================================
# Inference is made over coarse units, not cells (har3d_stats):
#   small datasets  test sets (dataset x split): the 20 cells of a split share
#                   its test clips, so they are not 20 observations. The four
#                   test sets of a dataset share one or two subjects pairwise,
#                   so p-values over them are approximate; the agreement within
#                   each dataset is reported with them
#   NTU ladder      seed-rung pairs: separately trained runs, which carry the
#                   run-to-run variance a bootstrap over clips cannot see
#  (1) Per-backbone ladder table; scaling law log(excess) = a + b log(n) by OLS
#      on the rung means, with a t-interval that includes lack of fit.
#  (2) Paired differences on the FIXED test set, rung by rung (bootstrap over
#      test clips, models held fixed; Holm within each family). Reported as
#      differences on this test set, not as evidence about training.
#  (3) Seed-level backbone tests: exact sign tests over seed-rung pairs, one
#      Holm family; the run-to-run spread across seeds at the large rungs; and
#      the gap between two nominally identical runs.
#  (4) RQ1: per-cell separability from the null (Holm over all cells), and the
#      pooled test: excess averaged within each test set, sign test over sets.
#  (5) RQ3 temperature scaling per dataset x stream -- DESCRIPTIVE ONLY: four
#      test sets per row are too few for a test.
#  (6) Null floor after temperature scaling (it changes the confidences, so it
#      changes the floor), ST-GCN small-dataset cells, for Fig. 3.
# Everything is written to results/tables/*.csv.
# v7: the inputs of this stage did not change in the revision, so a completed run is
# kept as it is (RERUN_OLD_STATS = True in Stage 0 recomputes it).
if stage_done('09b_statistics') and not RERUN_OLD_STATS:
    print('  [skip] stage 9b already done -- its tables are kept')
else:
    import csv
    from collections import defaultdict
    import har3d_v2 as V, har3d_calib as C, har3d_stats as HS
    importlib.reload(V); importlib.reload(C); importlib.reload(HS)
    from store import Key, LogitStore

    STORE = LogitStore(LOGITS)
    TABDIR = RESULTS / 'tables'; TABDIR.mkdir(parents=True, exist_ok=True)
    rows = json.loads((RESULTS / 'metrics.json').read_text())
    # the replication and environment re-runs of the revision (Stage 8c) are analysed in Stage 9h only
    rows = [r for r in rows if not r['model'].endswith(('_rep', '_env'))]


    def write_csv(path, header, body):
        with open(path, 'w', newline='') as fh:
            w = csv.writer(fh); w.writerow(header); w.writerows(body)
        print('  wrote', path.name, f'({len(body)} rows)')


    def msd(v):
        v = [x for x in v if x is not None and np.isfinite(x)]
        return (float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else 0.0) if v else (np.nan, np.nan)


    # ---------------------------------------------------------------- (1)
    print('=' * 70, '\n(1) NTU-60 ladder per backbone\n' + '=' * 70)
    lad = [r for r in rows if r['dataset'] == 'ntu60'
           and not (r['model'] == 'agcn_frozen' and r['frozen_at'] is None)]
    MODELS = [m for m in ('stgcn', 'agcn', 'ctrgcn', 'agcn_frozen', 'stgcn_fixed')
              if any(r['model'] == m for r in lad)]
    body, law = [], []
    for mdl in MODELS:
        by_n = defaultdict(list)
        for r in lad:
            if r['model'] == mdl:
                by_n[r['n_train']].append(r)
        ns = np.array(sorted(by_n))
        for n in ns:
            g = by_n[n]
            line = [mdl, int(n), len(g)]
            for mt in ('top1', 'ece_adaptive', 'null_null_mean', 'null_excess',
                       'classwise_ece', 'brier', 'brier_rel', 'brier_res', 'nll',
                       'ece_adaptive_temp', 'temperature'):
                mu, sd = msd([r.get(mt) for r in g])
                line += [f'{mu:.4f}', f'{sd:.4f}']
            body.append(line)
        exc = np.array([np.mean([r['null_excess'] for r in by_n[n]]) for n in ns])
        if len(ns) >= 3 and (exc > 0).all():
            b, lo, hi, r2 = HS.ols_loglog(ns, exc)
            law.append([mdl, f'{b:.4f}', f'{lo:.4f}', f'{hi:.4f}', f'{r2:.4f}', len(ns), int(ns[-1])])
            print(f'  {mdl:<12} b = {b:+.3f} [{lo:+.3f}, {hi:+.3f}]  R2 {r2:.3f}  (OLS on {len(ns)} rung means)')
    hdr = ['model', 'n_train', 'n_seeds']
    for mt in ('top1', 'ece_adaptive', 'null_floor', 'excess', 'classwise_ece', 'brier',
               'brier_rel', 'brier_res', 'nll', 'ece_adaptive_ts', 'temperature'):
        hdr += [mt + '_mean', mt + '_sd']
    write_csv(TABDIR / 'ladder_by_model.csv', hdr, body)
    write_csv(TABDIR / 'scaling_law_by_model.csv',
              ['model', 'exponent_b', 'ci_lo', 'ci_hi', 'r2', 'n_rungs_fit', 'max_n_fit'], law)

    # ---------------------------------------------------------------- (2)
    print('\n' + '=' * 70, '\n(2) paired differences on the fixed NTU test set (bootstrap over clips)\n' + '=' * 70)
    PAIRS = [('agcn', 'stgcn', 'graph'), ('ctrgcn', 'stgcn', 'graph'),
             ('agcn_frozen', 'agcn', 'freeze'), ('stgcn_fixed', 'stgcn', 'budget')]
    # The largest Holm family has 14 comparisons; a bootstrap p-value cannot fall
    # below 2/(B+1), so B must exceed 2*14/0.05 = 560 by a wide margin.
    N_BOOT_PAIR = 5000


    def ladder_probs(mdl, n, seed):
        d = STORE.load(Key(mdl, 'ntu60', LADDER_STREAM, f'ladder_n{n}', seed, 'test'))
        return V.softmax(d['logits']), d['labels']


    pair_rows = []
    for a, b, fam in PAIRS:
        na = {(r['n_train'], r['seed']) for r in lad if r['model'] == a}
        nb = {(r['n_train'], r['seed']) for r in lad if r['model'] == b}
        for n in sorted({n for n, _ in na & nb}):
            seeds = sorted(s for (m, s) in na & nb if m == n)
            PA, PB, y = [], [], None
            for s in seeds:
                pa, ya = ladder_probs(a, n, s); pb, yb = ladder_probs(b, n, s)
                assert np.array_equal(ya, yb), 'test labels differ -- store is inconsistent'
                PA.append(pa); PB.append(pb); y = ya
            d_obs = np.mean([V.adaptive_ece(pa, y) - V.adaptive_ece(pb, y) for pa, pb in zip(PA, PB)])
            CA = [(pa.max(-1), (pa.argmax(-1) == y).astype(float)) for pa in PA]
            CB = [(pb.max(-1), (pb.argmax(-1) == y).astype(float)) for pb in PB]
            acc_d = np.mean([V.top1(pa, y) - V.top1(pb, y) for pa, pb in zip(PA, PB)])
            rng = np.random.default_rng(n); bs = np.empty(N_BOOT_PAIR)
            for i in range(N_BOOT_PAIR):
                idx = rng.integers(0, len(y), len(y))
                bs[i] = np.mean([C.adaptive_ece_cc(ca[0][idx], ca[1][idx]) - C.adaptive_ece_cc(cb[0][idx], cb[1][idx])
                                 for ca, cb in zip(CA, CB)])
            p = float(min(1.0, 2 * (1 + min((bs >= 0).sum(), (bs <= 0).sum())) / (N_BOOT_PAIR + 1)))
            pair_rows.append([fam, a, b, n, len(seeds), f'{acc_d:+.4f}', f'{d_obs:+.4f}',
                              f'{np.quantile(bs, .025):+.4f}', f'{np.quantile(bs, .975):+.4f}', p, N_BOOT_PAIR])
    for fam in sorted({r[0] for r in pair_rows}):
        idx = [i for i, r in enumerate(pair_rows) if r[0] == fam]
        adj = HS.holm([pair_rows[i][9] for i in idx])
        for i, pa in zip(idx, adj):
            r = pair_rows[i]
            nb_ = r.pop()
            r[9] = f'{r[9]:.4f}'
            r += [f'{pa:.4f}', ('A lower ECE' if float(r[6]) < 0 else 'A higher ECE')
                  if pa < 0.05 else 'n.s.', nb_]
    for r in pair_rows:
        print(f'  {r[1]:>11} vs {r[2]:<6} n={r[3]:<6} dAcc {r[5]}  dECE {r[6]} '
              f'[{r[7]}, {r[8]}]  Holm p {r[10]}  {r[11]}')
    write_csv(TABDIR / 'ladder_paired_comparisons.csv',
              ['family', 'model_a', 'model_b', 'n_train', 'n_seeds', 'delta_top1',
               'delta_ece_adaptive', 'ci_lo', 'ci_hi', 'p_raw', 'p_holm', 'verdict', 'n_boot'], pair_rows)

    # ---------------------------------------------------------------- (3)
    print('\n' + '=' * 70, '\n(3) seed-level backbone tests (separately trained runs)\n' + '=' * 70)
    gap = HS.identical_run_gap(rows, n=2000, freeze_step=FREEZE_AFTER_STEPS)
    if gap:
        print(f"  run-to-run noise: frozen and unfrozen 2s-AGCN at 2,000 clips, seed 0, both restore step "
              f"{gap['step']} (before the freeze): |d aECE| {gap['ece']:.4f}, |d top-1| {gap['top1']:.4f}")
    ECE = {(r['model'], r['n_train'], r['seed']): r['ece_adaptive'] for r in lad}
    NS_ = sorted({r['n_train'] for r in lad if r['model'] == 'stgcn'})
    REG = {'low': [n for n in NS_ if n <= 5000], 'high': [n for n in NS_ if n >= 10000]}
    sl = HS.seed_level_tests(ECE, [(m, 'stgcn') for m in ('agcn', 'ctrgcn') if m in MODELS], REG,
                             seeds=tuple(LADDER_SEEDS))
    for r in sl:
        print(f"  {r['model_a']:>7} - stgcn  {r['regime']:<5} A higher in {r['n_a_higher']}/{r['n_pairs']} "
              f"seed-rung pairs, mean {r['mean_delta']:+.4f}, Holm p {r['p_holm']:.4f}")
    _sd, _ng = HS.pooled_seed_sd(ECE, [m for m in ('stgcn', 'agcn', 'ctrgcn') if m in MODELS], REG['high'],
                                 seeds=tuple(LADDER_SEEDS))
    print(f"  run-to-run spread: across-seed s.d. of aECE {_sd:.4f} (RMS over {_ng} backbone-rung groups "
          f">= 10,000 clips); s.e. of a difference of two {len(LADDER_SEEDS)}-seed means "
          f"{_sd * np.sqrt(2 / len(LADDER_SEEDS)):.4f}")
    write_csv(TABDIR / 'ladder_seed_level.csv',
              ['model_a', 'model_b', 'regime', 'n_pairs', 'n_a_higher', 'mean_delta', 'p', 'p_holm'],
              [[r['model_a'], r['model_b'], r['regime'], r['n_pairs'], r['n_a_higher'],
                f"{r['mean_delta']:+.4f}", f"{r['p']:.4f}", f"{r['p_holm']:.4f}"] for r in sl])

    # ---------------------------------------------------------------- (4)
    print('\n' + '=' * 70, '\n(4) RQ1 separability from the null (ST-GCN, small datasets)\n' + '=' * 70)
    rq1 = [r for r in rows if r['model'] == 'stgcn' and r['dataset'] != 'ntu60']
    padj = HS.holm([r['null_p_value'] for r in rq1])
    body = [[r['dataset'], r['stream'], r['split'], r['seed'], r['n'], r['n_classes'],
             f"{r['ece_adaptive']:.4f}", f"{r['null_null_mean']:.4f}",
             f"{r['null_excess']:.4f}", f"{r['null_p_value']:.4f}", f'{pa:.4f}',
             'yes' if pa < 0.05 else 'no'] for r, pa in zip(rq1, padj)]
    write_csv(TABDIR / 'rq1_separability.csv',
              ['dataset', 'stream', 'split', 'seed', 'n', 'K', 'ece_adaptive', 'null_mean',
               'excess', 'p_raw', 'p_holm', 'separable_holm'], body)
    surv = defaultdict(lambda: [0, 0, 0])
    for r in body:
        s = surv[(r[0], r[1])]; s[1] += 1; s[0] += r[-1] == 'yes'; s[2] += float(r[9]) < 0.05
    for key in sorted(surv):
        s = surv[key]
        print(f'  {key[0]:<9}{key[1]:<13} separable {s[0]}/{s[1]} after Holm ({s[2]} uncorrected)')
    pe = HS.pooled_excess(rq1)
    print(f"\n  pooled: excess > 0 in {pe['k_pos']}/{pe['n']} test sets, mean {pe['mean']:+.4f}, "
          f"exact sign test p = {pe['p']:.4g} (approximate: the test sets of a dataset share subjects)")
    print('  per dataset: ' + ', '.join(f'{ds} {k}/{n}' for ds, (k, n) in sorted(pe['per_dataset'].items())))
    write_csv(TABDIR / 'rq1_pooled_test_sets.csv', ['dataset', 'split', 'mean_excess'],
              [[u[0], u[1], f'{v:.5f}'] for u, v in zip(pe['units'], pe['unit_excess'])])

    # ---------------------------------------------------------------- (5)
    print('\n' + '=' * 70, '\n(5) RQ3 temperature scaling per dataset x stream (descriptive)\n' + '=' * 70)
    grp = defaultdict(list)
    for r in rq1:
        grp[(r['dataset'], r['stream'])].append(r)
    body = []
    for (ds, st), g in sorted(grp.items()):
        d = [r['ece_adaptive'] - r['ece_adaptive_temp'] for r in g]
        _, u = HS.unit_means(d, [r['split'] for r in g])
        body.append([ds, st, len(d), int(sum(x > 0 for x in d)), f'{np.mean(d):+.4f}',
                     len(u), int((u > 0).sum()), f'{u.min():+.4f}', f'{u.max():+.4f}'])
        print(f'  {ds:<9}{st:<13} reduced in {body[-1][3]}/{len(d)} cells and {body[-1][6]}/{len(u)} '
              f'test sets, mean reduction {np.mean(d):+.4f}')
    write_csv(TABDIR / 'rq3_temperature_scaling.csv',
              ['dataset', 'stream', 'n_cells', 'n_cells_reduced', 'mean_reduction', 'n_test_sets',
               'n_test_sets_reduced', 'test_set_min', 'test_set_max'], body)

    # ---------------------------------------------------------------- (6)
    print('\n' + '=' * 70, '\n(6) null floor after temperature scaling (ST-GCN, small datasets)\n' + '=' * 70)
    N_SIM_TS = 1000                    # a floor MEAN, not a p-value: 1,000 draws suffice
    fl = defaultdict(lambda: [[], []])
    for r in rq1:
        d = STORE.load(Key('stgcn', r['dataset'], r['stream'], r['split'], r['seed'], 'test'))
        _, mu_ts, _ = C.ece_null(V.softmax(d['logits'], r['temperature']), n_sim=N_SIM_TS, seed=r['seed'])
        fl[r['dataset']][0].append(r['null_null_mean']); fl[r['dataset']][1].append(mu_ts)
    body = [[ds, len(v[0]), f'{np.mean(v[0]):.5f}', f'{np.mean(v[1]):.5f}', N_SIM_TS] for ds, v in sorted(fl.items())]
    for b_ in body:
        print(f'  {b_[0]:<9} floor raw {b_[2]}  after temperature scaling {b_[3]}  ({b_[1]} cells)')
    write_csv(TABDIR / 'temperature_floor.csv', ['dataset', 'n_cells', 'floor_raw', 'floor_ts', 'n_sim'], body)
    mark_done('09b_statistics')

## Stage 9c — six methods, each against its own floor (CPU, ~10 min)
Tested on the excess over each method's own floor, the Brier score and the NLL, over the 12 test sets; raw ECE for reference.
Kept from v6 when already done; the subject-level version is Stage 9f.

In [ ]:
# =====================================================================
# STAGE 9c -- six calibration methods vs the uncalibrated baseline  [CPU, ~10 min]
# =====================================================================
#   raw            uncalibrated softmax
#   temperature    Guo et al. 2017, fitted on validation logits
#   vector         per-class scale + bias, L2 towards identity, validation
#   label_smooth   Szegedy et al. 2016 (eps 0.1), Stage 6b
#   focal          Lin et al. 2017 (gamma 2), Stage 6b
#   mc_dropout     Gal & Ghahramani 2016, MC_PASSES passes, rate tuned on val NLL
#   deep_ensemble  Lakshminarayanan et al. 2017, mean probability of 5 seeds
# Every method changes the confidences, and with them the estimator floor its
# ECE is measured against, so a raw ECE difference between two methods mixes
# calibration with the shift of the floor. v6 therefore computes each cell's
# floor on the METHOD'S OWN probabilities (N_SIM_FLOOR null draws: a mean, not
# a p-value, as for the temperature-scaled floor of Stage 9b) and tests the
# methods on
#   excess   adaptive ECE minus that floor
#   brier    binning-free (no floor)
#   nll      binning-free (no floor)
# The raw adaptive ECE is kept for reference only (not tested). Per method also
# the Murphy decomposition of the Brier score (REL, RES, UNC), classwise ECE
# and accuracy (descriptive, over cells). The paired comparison against raw is
# made over TEST SETS (dataset x split): all cells of a split share its test
# clips, so the 240 cells are not 240 observations. Sign test over the 12 test
# sets, Holm across the six methods for each score, cluster-bootstrap
# intervals over test sets (har3d_stats.method_comparison). The four test sets
# of a dataset share one or two subjects pairwise, so the p-values are
# approximate; the agreement within each dataset is printed and written too.
# ST-GCN, all four streams, all splits, three small datasets.
# v7: the inputs of this stage did not change in the revision, so a completed run is
# kept as it is (RERUN_OLD_STATS = True in Stage 0 recomputes it).
if stage_done('09c_methods') and not RERUN_OLD_STATS:
    print('  [skip] stage 9c already done -- its tables are kept')
else:
    import csv
    from collections import defaultdict
    import har3d_v2 as V, har3d_calib as C, har3d_stats as HS
    importlib.reload(V); importlib.reload(C); importlib.reload(HS)
    from store import Key, LogitStore

    STORE = LogitStore(LOGITS)
    TABDIR = RESULTS / 'tables'; TABDIR.mkdir(parents=True, exist_ok=True)
    SMALL = ('action3d', 'daily', 'utkinect')
    METHODS = ['raw', 'temperature', 'vector', 'label_smooth', 'focal', 'mc_dropout',
               'deep_ensemble']
    N_SIM_FLOOR = 1000                 # a floor MEAN, not a p-value: 1,000 draws suffice
    SCORES = [('excess', True), ('brier', True), ('nll', True), ('ece', False)]   # (score, tested)


    def metrics(p, y, seed):
        """Every score of one cell. The floor is the null mean of THIS method's
        probabilities; the same seed for every method of a cell gives the floors
        common random numbers, so their differences carry little Monte-Carlo noise."""
        bd = V.brier_decomposition(p, y)
        e = V.adaptive_ece(p, y)
        fl = C.ece_null(p, n_sim=N_SIM_FLOOR, seed=seed)[1]
        return {'ece': e, 'floor': fl, 'excess': e - fl, 'ece_ew': V.ece(p, y),
                'cw_ece': V.classwise_ece(p, y), 'brier': bd['brier'], 'rel': bd['rel'],
                'res': bd['res'], 'unc': bd['unc'], 'nll': V.nll(p, y), 'top1': V.top1(p, y)}


    per_cell, members = [], defaultdict(list)
    keys = [k for k in STORE.list_keys()
            if k.model == 'stgcn' and k.subset == 'test' and k.dataset in SMALL]
    t0 = time.time()
    for i, k in enumerate(keys):
        d = STORE.load(k); z, y = d['logits'], d['labels']
        cell = dict(dataset=k.dataset, stream=k.stream, split=k.split, seed=k.seed)
        probs = {'raw': V.softmax(z)}
        vk = Key('stgcn', k.dataset, k.stream, k.split, k.seed, 'val')
        if STORE.has(vk):
            dv = STORE.load(vk)
            probs['temperature'] = V.softmax(z, V.fit_temperature(dv['logits'], dv['labels']))
            w, b = C.fit_vector_scaling(dv['logits'], dv['labels'], l2=1e-2)
            probs['vector'] = C.apply_vector_scaling(z, w, b)
        for tag, nm in (('stgcn_ls', 'label_smooth'), ('stgcn_focal', 'focal')):
            mk = Key(tag, k.dataset, k.stream, k.split, k.seed, 'test')
            if STORE.has(mk):
                dm = STORE.load(mk)
                if np.array_equal(dm['labels'], y):
                    probs[nm] = V.softmax(dm['logits'])
        f = LOGITS / f'mc__{k.dataset}__stgcn__{k.stream}__{k.split}__s{k.seed}__test.npz'
        if f.exists():
            zz = np.load(f, allow_pickle=False)
            if np.array_equal(zz['labels'], y):
                probs['mc_dropout'] = C.mc_dropout_probs(zz['logits'])
        for nm, p in probs.items():
            per_cell.append({'method': nm, **cell, **metrics(p, y, k.seed)})
        members[(k.dataset, k.stream, k.split)].append((k.seed, probs['raw'], y))
        if (i + 1) % 40 == 0 or i + 1 == len(keys):
            print(f'  {i + 1}/{len(keys)} cells ({time.time() - t0:.0f} s)')

    N_ENS = len(SEEDS)
    for (ds, st, sp), mem in members.items():
        if len(mem) < N_ENS or not all(np.array_equal(m[2], mem[0][2]) for m in mem):
            continue
        mem = sorted(mem, key=lambda m: m[0])[:N_ENS]
        # seed -1 marks the ensemble (one cell per dataset, stream and split); its
        # floor uses null seed 0
        per_cell.append({'method': 'deep_ensemble', 'dataset': ds, 'stream': st, 'split': sp,
                         'seed': -1, **metrics(np.mean([m[1] for m in mem], 0), mem[0][2], 0)})

    cols = ['ece', 'floor', 'excess', 'ece_ew', 'cw_ece', 'brier', 'rel', 'res', 'unc', 'nll', 'top1']
    with open(TABDIR / 'methods_per_cell.csv', 'w', newline='') as fh:
        w = csv.writer(fh); w.writerow(['method', 'dataset', 'stream', 'split', 'seed'] + cols)
        for r in per_cell:
            w.writerow([r['method'], r['dataset'], r['stream'], r['split'], r['seed']]
                       + [f'{r[c]:.5f}' for c in cols])

    # descriptive means per scope (all cells, and per dataset); 'delta' is the raw
    # aECE reduction (reference only); the deep ensemble is paired with the
    # seed-mean of raw for its split
    raw = {(r['dataset'], r['stream'], r['split'], r['seed']): r for r in per_cell if r['method'] == 'raw'}
    raw_split = defaultdict(list)
    for r in raw.values():
        raw_split[(r['dataset'], r['stream'], r['split'])].append(r)
    summary = []
    for scope in ['all'] + list(SMALL):
        for m in METHODS:
            rs = [r for r in per_cell if r['method'] == m and scope in ('all', r['dataset'])]
            if not rs:
                continue
            dl = {}
            for sc in ('ece', 'excess', 'brier', 'nll'):
                if m == 'raw':
                    dl[sc] = 0.0
                elif m == 'deep_ensemble':
                    dl[sc] = float(np.mean([np.mean([q[sc] for q in raw_split[(r['dataset'], r['stream'], r['split'])]])
                                            - r[sc] for r in rs]))
                else:
                    dl[sc] = float(np.mean([raw[(r['dataset'], r['stream'], r['split'], r['seed'])][sc] - r[sc]
                                            for r in rs]))
            summary.append({'scope': scope, 'method': m, 'n_cells': len(rs), 'delta': dl['ece'],
                            'delta_excess': dl['excess'], 'delta_brier': dl['brier'], 'delta_nll': dl['nll'],
                            **{c: float(np.mean([r[c] for r in rs])) for c in cols}})
    hdr = ['scope', 'method', 'n_cells', 'delta', 'delta_excess', 'delta_brier', 'delta_nll'] + cols
    with open(TABDIR / 'methods_comparison.csv', 'w', newline='') as fh:
        w = csv.writer(fh); w.writerow(hdr)
        for r in summary:
            w.writerow([r[h] if not isinstance(r[h], float) else f'{r[h]:.5f}' for h in hdr])

    # inference over test sets, one Holm family per tested score
    out = []
    for sc, tested in SCORES:
        res, sci = HS.method_comparison(per_cell, METHODS[1:], key=sc)
        out.append(['', sc, 'yes' if tested else 'no (reference)', 'raw', '', '', '', '', '', '', '', '', 'baseline',
                    f"{sci['raw'][0]:.5f}", f"{sci['raw'][1]:.5f}"])
        print(f"\n{sc}{'' if tested else ' (reference only: includes each method shift of the floor)'}")
        print(f"  {'method':<14}{'reduction':>10}  {'95% CI over test sets':<20}{'better':>8}  "
              f"{'per dataset':<16}{'Holm p':>8}  verdict")
        for m in METHODS[1:]:
            r = res[m]
            pdtxt = ' '.join(f"{a}/{b}" for a, b in (r['per_dataset'][ds] for ds in SMALL if ds in r['per_dataset']))
            print(f"  {m:<14}{r['mean']:>+10.4f}  [{r['lo']:+.4f}, {r['hi']:+.4f}]  "
                  f"{str(r['k']) + '/' + str(r['n']):>8}  {pdtxt:<16}{r['p_holm']:>8.4f}  "
                  f"{r['verdict'] if tested else '(not tested)'}")
            out.append(['', sc, 'yes' if tested else 'no (reference)', m, r['n'], r['k'], pdtxt,
                        f"{r['mean']:.5f}", f"{r['lo']:.5f}", f"{r['hi']:.5f}", f"{r['p']:.5f}",
                        f"{r['p_holm']:.5f}", r['verdict'] if tested else 'not tested',
                        f"{sci[m][0]:.5f}", f"{sci[m][1]:.5f}"])
    with open(TABDIR / 'methods_split_level.csv', 'w', newline='') as fh:
        w = csv.writer(fh)
        w.writerow(['note', 'score', 'tested', 'method', 'n_test_sets', 'n_improved', 'improved_per_dataset',
                    'mean_reduction', 'ci_lo', 'ci_hi', 'p', 'p_holm', 'verdict', 'score_ci_lo', 'score_ci_hi'])
        w.writerows([['p approximate: the test sets of a dataset share subjects'] + r[1:] if i == 0 else r
                     for i, r in enumerate(out)])

    allm = {r['method']: r for r in summary if r['scope'] == 'all'}
    print(f"\n{'method':<14}{'aECE':>8}{'floor':>8}{'excess':>8}{'Brier':>8}{'NLL':>8}{'top-1':>8}  (means over cells)")
    for m in METHODS:
        a = allm[m]
        print(f"{m:<14}{a['ece']:>8.4f}{a['floor']:>8.4f}{a['excess']:>8.4f}{a['brier']:>8.4f}{a['nll']:>8.4f}{a['top1']:>8.4f}")
    print('\nwrote methods_per_cell.csv, methods_comparison.csv and methods_split_level.csv')
    mark_done('09c_methods')

## Stage 9d — dependence-aware inference on the small datasets (CPU, ~15 min)
Reviewer 1, points 1 and 5; Reviewer 2, points 1, 5, 7 and 16. Every test cell is loaded with the identity of its clips and subjects, and the first-version conclusions are re-examined: with other estimators and bin counts, with a null that keeps the dependence between cells that score the same clips, with a null that adds subject effects, per dataset, and with subjects as the units.

In [ ]:
# =====================================================================
# STAGE 9d -- dependence-aware inference, small datasets      [CPU, ~15 min]
# =====================================================================
# The 240 small-dataset cells are not 240 observations, and the 12 test sets
# are not 12 either: cells of a split score the same clips, and the four test
# sets of a dataset share subjects. This stage re-derives the conclusions of
# the first version without those independence assumptions.
#  (1) Estimator sensitivity. Every cell is tested against its matched null
#      with eight estimators: equal-mass ECE with 5, 10, 15, 20, 30 bins,
#      equal-width ECE, the debiased squared calibration error and the
#      binning-free KS calibration error. Top-label calibration error depends
#      on a prediction only through (confidence, correct), and under the null
#      the prediction is correct with probability equal to its confidence, so
#      the Bernoulli(confidence) draw used here IS the full-vector label draw
#      of the first version for every top-label statistic (unit-tested).
#  (2) Subject heterogeneity: is the calibration residual more variable between
#      subjects than independent labels allow? Over-dispersion factor and the
#      s.d. tau of a subject effect on the success probability, per dataset.
#  (3) Pooled excess under four nulls of increasing dependence:
#        independent   every cell drawn on its own (the implicit assumption of
#                      a test that treats cells or test sets as independent)
#        coupled       one uniform per CLIP decides the correctness of every
#                      model that predicts it (splits, streams, seeds): the
#                      strongest positive dependence the marginals allow
#        clustered     coupled + a subject effect N(0, tau^2) on the success
#                      probability of all clips of a subject, tau from (2)
#        tau grid      the same with fixed tau, to find where the conclusion
#                      would change
#      all datasets and per dataset, Monte-Carlo p-values.
#  (4) Per-cell separability under the clustered null (Holm over the cells).
#  (5) Subjects as units: mean over-confidence (confidence - correct) per
#      subject over every prediction made for that subject; sign test and
#      bootstrap over subjects. |mean gap| is a lower bound of the calibration
#      error and needs no bins and no null model.
import csv
from collections import defaultdict
import har3d_v2 as V, har3d_calib as C, har3d_grid as G, har3d_stats as HS, har3d_review as RV
importlib.reload(V); importlib.reload(C); importlib.reload(G); importlib.reload(HS); importlib.reload(RV)
from store import Key, LogitStore

STORE = LogitStore(LOGITS)
TABDIR = RESULTS / 'tables'; TABDIR.mkdir(parents=True, exist_ok=True)
SMALL = ('action3d', 'daily', 'utkinect')
_meta = json.loads((RESULTS / 'prep_config.json').read_text())
PREP_CFG, PREP_TAG = _meta['cfg'], _meta['prep_tag']
S_CELL, S_POOL, S_GRID = 20000, 5000, 2000      # null draws: per cell, pooled, tau grid
TAU_GRID = (0.02, 0.04, 0.06, 0.08, 0.10, 0.15)


def write_csv(path, header, body):
    with open(path, 'w', newline='') as fh:
        w = csv.writer(fh); w.writerow(header); w.writerows(body)
    print('  wrote', path.name, f'({len(body)} rows)')


def fmt(x, d=5):
    return x if isinstance(x, (str, int, np.integer)) else f'{float(x):.{d}f}'


def small_index(prep_tag=None):
    """Per dataset: labels and subjects of the quality-filtered clips and the
    test indices of every split, rebuilt exactly as the training grid built
    them (har3d_grid.run_grid). Clip i of a dataset is the same recording in
    every split, stream and seed."""
    prep_tag = prep_tag or PREP_TAG
    idx = {}
    for ds, npz in G.DATASETS:
        cache = V.load_cache(CACHE / npz)
        y = cache['meta'][:, 0].astype(np.int64) - 1
        subj = cache['meta'][:, 1].astype(np.int64)
        keep = (V.quality_mask(cache['flags'], MAX_ZERO_FRAC, MIN_FRAMES) if 'flags' in cache
                else np.ones(len(y), bool))
        y, subj = y[keep], subj[keep]
        idx[ds] = dict(y=y, subj=subj, n=len(y),
                       splits={f"{sp['name']}_{prep_tag}": sp for sp in G.splits_for(subj, y)})
    return idx


def load_cells(store, idx, model='stgcn', streams=None, seeds=None, logits=False):
    """Test cells of `model` with clip and subject ids. Raises if the stored
    labels are not the labels of the reconstructed test clips."""
    cells = []
    for k in store.list_keys():
        if k.model != model or k.subset != 'test' or k.dataset not in idx:
            continue
        if (streams and k.stream not in streams) or (seeds is not None and k.seed not in seeds):
            continue
        ix = idx[k.dataset]
        if k.split not in ix['splits']:
            continue
        d = store.load(k)
        te = ix['splits'][k.split]['test']
        if not np.array_equal(d['labels'], ix['y'][te]):
            raise RuntimeError(f'{k.fname()}: stored labels differ from the reconstructed test set')
        p = V.softmax(d['logits'])
        c = dict(key=k, dataset=k.dataset, stream=k.stream, split=k.split, seed=k.seed,
                 unit=(k.dataset, k.split), conf=p.max(-1), pred=p.argmax(-1),
                 correct=(p.argmax(-1) == d['labels']), labels=d['labels'], clip=te,
                 subj=ix['subj'][te], n=len(te))
        if logits:
            c['logits'] = d['logits']
        cells.append(c)
    return cells


def get_small_cells():
    """The 240 ST-GCN cells of the main experiment (cached for Stages 9e-9g)."""
    global SMALL_INDEX, SMALL_CELLS
    if 'SMALL_CELLS' not in globals():
        SMALL_INDEX = small_index()
        SMALL_CELLS = load_cells(STORE, SMALL_INDEX, logits=True)
        print(f'  {len(SMALL_CELLS)} cells loaded with clip and subject ids:',
              {ds: (SMALL_INDEX[ds]["n"], len(np.unique(SMALL_INDEX[ds]["subj"]))) for ds in SMALL},
              '(clips, subjects)')
    return SMALL_INDEX, SMALL_CELLS


def by_dataset(cells):
    out = defaultdict(list)
    for c in cells:
        out[c['dataset']].append(c)
    return out


def subj_index(ix):
    """Subject id per clip as 0..G-1 (for the subject effects of the null)."""
    return np.unique(ix['subj'], return_inverse=True)[1]


def pooled_under(cells, idx, S, names=('aece15',), tau=None, seed=0, independent=False):
    """Observed values and null draws of every cell, datasets stacked
    (independent across datasets). tau: None, or {dataset: tau}."""
    obs, nul, units, dsl = {n: [] for n in names}, {n: [] for n in names}, [], []
    for di, (ds, cs) in enumerate(by_dataset(cells).items()):
        if independent:
            parts = [RV.coupled_null([c], idx[ds]['n'], S=S, seed=seed + 7919 * (di + 1) + i, names=names)
                     for i, c in enumerate(cs)]
            r = {n: {'obs': np.concatenate([q[n]['obs'] for q in parts]),
                     'null': np.concatenate([q[n]['null'] for q in parts])} for n in names}
        else:
            t = (tau or {}).get(ds, 0.0)
            r = RV.coupled_null(cs, idx[ds]['n'], S=S, seed=seed + di, names=names, tau=t,
                                subj_of_clip=subj_index(idx[ds]) if t > 0 else None)
        for n in names:
            obs[n].append(r[n]['obs']); nul[n].append(r[n]['null'])
        units += [c['unit'] for c in cs]; dsl += [ds] * len(cs)
    return ({n: np.concatenate(obs[n]) for n in names}, {n: np.concatenate(nul[n]) for n in names},
            units, dsl)


if stage_done('09d_dependence') and not RERUN_REVIEW_STATS:
    print('  [skip] stage 9d already done (RERUN_REVIEW_STATS = True recomputes it)')
else:
    t0 = time.time()
    IDX, CELLS = get_small_cells()
    NAMES = RV.ESTIMATORS

    # ------------------------------------------------------------ (1)
    print('=' * 70, '\n(1) estimator and bin-count sensitivity, per cell\n' + '=' * 70)
    per_est = defaultdict(list)
    for c in CELLS:
        c['tests'] = RV.cell_tests(c['conf'], c['correct'], S=S_CELL, seed=c['seed'], names=NAMES)
        for nm in NAMES:
            per_est[nm].append(c['tests'][nm]['p'])
    adj = {nm: HS.holm(per_est[nm]) for nm in NAMES}
    body = []
    for i, c in enumerate(CELLS):
        for nm in NAMES:
            t = c['tests'][nm]
            body.append([c['dataset'], c['stream'], c['split'], c['seed'], c['n'], nm,
                         fmt(t['obs']), fmt(t['null_mean']), fmt(t['null_p95']), fmt(t['excess']),
                         fmt(t['p']), fmt(adj[nm][i])])
    write_csv(TABDIR / 'estimator_cells.csv',
              ['dataset', 'stream', 'split', 'seed', 'n', 'estimator', 'observed', 'floor',
               'null_p95', 'excess', 'p', 'p_holm'], body)

    # ------------------------------------------------------------ (2)
    print('=' * 70, '\n(2) subject heterogeneity of the calibration residual\n' + '=' * 70)
    for c in CELLS:
        c['het'] = RV.subject_heterogeneity(c['conf'], c['correct'], c['subj'], S=2000, seed=c['seed'])
    TAU, body = {}, []
    for ds, cs in by_dataset(CELLS).items():
        phi = np.array([c['het']['phi'] for c in cs]); t2 = np.array([c['het']['tau2'] for c in cs])
        ph = np.array([c['het']['p'] for c in cs])
        TAU[ds] = float(np.sqrt(max(t2.mean(), 0.0)))
        # per test set (its cells share the subjects): mean over-dispersion
        per_set = {u: float(np.mean([c['het']['phi'] for c in cs if c['unit'] == u]))
                   for u in dict.fromkeys(c['unit'] for c in cs)}
        body.append([ds, len(cs), len(np.unique(IDX[ds]['subj'])), fmt(np.median(phi), 3),
                     fmt(phi.mean(), 3), int((ph < 0.05).sum()), fmt(t2.mean(), 6), fmt(TAU[ds], 4),
                     ' '.join(f'{v:.2f}' for v in per_set.values())])
        print(f'  {ds:<9} over-dispersion phi median {np.median(phi):.2f} mean {phi.mean():.2f}; '
              f'{(ph < 0.05).sum()}/{len(cs)} cells p < 0.05; tau = {TAU[ds]:.4f}')
    write_csv(TABDIR / 'subject_heterogeneity.csv',
              ['dataset', 'n_cells', 'n_subjects', 'phi_median', 'phi_mean', 'cells_p_lt_0.05',
               'tau2_mean', 'tau', 'phi_mean_per_test_set'], body)

    # ------------------------------------------------------------ (3)
    print('=' * 70, '\n(3) pooled excess under nulls of increasing dependence\n' + '=' * 70)
    body, POOL = [], {}

    def add(null_name, tau_txt, nm, res):
        pd_ = res['per_dataset']
        body.append([null_name, tau_txt, nm, res['n_cells'], res['n_units'], fmt(res['mean_excess']),
                     fmt(res['null_sd']), fmt(res['z'], 2), fmt(res['p']), res['k_pos'], fmt(res['p_k']),
                     fmt(res['var_inflation'], 2)]
                    + [x for ds in SMALL for x in (fmt(pd_[ds]['mean_excess']), fmt(pd_[ds]['p']),
                                                   f"{pd_[ds]['k_pos']}/{pd_[ds]['n_units']}")])

    for null_name, kw, S_ in (('independent', dict(independent=True), S_POOL),
                              ('coupled', dict(), S_POOL),
                              ('clustered', dict(tau=TAU), S_POOL)):
        obs, nul, units, dsl = pooled_under(CELLS, IDX, S_, names=NAMES, **kw)
        for nm in NAMES:
            res = RV.pooled_from_null(obs[nm], nul[nm], units, dsl)
            POOL[(null_name, nm)] = res
            add(null_name, 'estimated' if null_name == 'clustered' else '0', nm, res)
        r = POOL[(null_name, 'aece15')]
        print(f"  {null_name:<12} mean excess {r['mean_excess']:.4f}  null s.d. {r['null_sd']:.4f}  "
              f"p {r['p']:.5f}  test sets > 0: {r['k_pos']}/{r['n_units']} (p {r['p_k']:.4f})  "
              f"variance x{r['var_inflation']:.1f}  ({time.time() - t0:.0f} s)")
    for tau in TAU_GRID:
        obs, nul, units, dsl = pooled_under(CELLS, IDX, S_GRID, tau={ds: tau for ds in SMALL})
        res = RV.pooled_from_null(obs['aece15'], nul['aece15'], units, dsl)
        POOL[('tau', tau)] = res
        add('clustered, fixed tau', f'{tau:g}', 'aece15', res)
        print(f"  tau = {tau:<5g} mean excess {res['mean_excess']:.4f}  p {res['p']:.4f}  "
              f"per dataset p " + ' '.join(f"{res['per_dataset'][ds]['p']:.3f}" for ds in SMALL))
    write_csv(TABDIR / 'pooled_dependence.csv',
              ['null', 'tau', 'estimator', 'n_cells', 'n_test_sets', 'mean_excess', 'null_sd', 'z',
               'p', 'test_sets_positive', 'p_test_sets_positive', 'variance_inflation']
              + [f'{ds}_{x}' for ds in SMALL for x in ('mean_excess', 'p', 'test_sets_positive')], body)

    # estimator summary: per-cell separability and the coupled pooled test
    body = []
    for nm in NAMES:
        ps = np.array(per_est[nm]); r = POOL[('coupled', nm)]; rc = POOL[('clustered', nm)]
        body.append([nm, RV.EST_LABEL[nm], len(ps), int((ps < 0.05).sum()), int((adj[nm] < 0.05).sum()),
                     fmt(np.mean([c['tests'][nm]['obs'] for c in CELLS])),
                     fmt(np.mean([c['tests'][nm]['null_mean'] for c in CELLS])),
                     fmt(r['mean_excess']), fmt(r['z'], 2), fmt(r['p']), f"{r['k_pos']}/{r['n_units']}",
                     fmt(rc['p'])]
                    + [fmt(r['per_dataset'][ds]['p']) for ds in SMALL])
        print(f"  {nm:<8} separable {int((adj[nm] < 0.05).sum()):>3}/{len(ps)} (Holm), "
              f"{int((ps < 0.05).sum()):>3} at p < 0.05; pooled excess {r['mean_excess']:+.4f}, "
              f"coupled p {r['p']:.5f}, clustered p {rc['p']:.5f}")
    write_csv(TABDIR / 'estimator_sensitivity.csv',
              ['estimator', 'label', 'n_cells', 'cells_p_lt_0.05', 'cells_separable_holm',
               'mean_observed', 'mean_floor', 'pooled_excess', 'z_coupled', 'p_coupled',
               'test_sets_positive', 'p_clustered'] + [f'p_coupled_{ds}' for ds in SMALL], body)

    # ------------------------------------------------------------ (4)
    print('=' * 70, '\n(4) per-cell separability under the clustered null\n' + '=' * 70)
    obs, nul, units, dsl = pooled_under(CELLS, IDX, S_CELL, tau=TAU, seed=11)
    p_cl = np.array([RV.mc_p(nul['aece15'][i], obs['aece15'][i]) for i in range(len(units))])
    a_cl = HS.holm(p_cl)
    order = [c for ds in by_dataset(CELLS).values() for c in ds]      # the order pooled_under stacks
    p_iid = np.array([c['tests']['aece15']['p'] for c in order]); a_iid = HS.holm(p_iid)
    body = [[c['dataset'], c['stream'], c['split'], c['seed'], fmt(obs['aece15'][i]),
             fmt(c['tests']['aece15']['null_mean']), fmt(nul['aece15'][i].mean()),
             fmt(p_iid[i]), fmt(a_iid[i]), fmt(p_cl[i]), fmt(a_cl[i])] for i, c in enumerate(order)]
    write_csv(TABDIR / 'separability_clustered.csv',
              ['dataset', 'stream', 'split', 'seed', 'observed', 'floor_iid', 'floor_clustered',
               'p_iid', 'p_holm_iid', 'p_clustered', 'p_holm_clustered'], body)
    print(f'  separable after Holm: {(a_iid < 0.05).sum()}/{len(order)} under the i.i.d. null, '
          f'{(a_cl < 0.05).sum()}/{len(order)} under the clustered null '
          f'(p < 0.05 uncorrected: {(p_iid < 0.05).sum()} and {(p_cl < 0.05).sum()})')

    # ------------------------------------------------------------ (5)
    print('=' * 70, '\n(5) subjects as units: mean over-confidence per subject\n' + '=' * 70)
    acc = RV.SubjectAccumulator()
    for c in CELLS:
        acc.add(c['dataset'], c['subj'], c['conf'] - c['correct'])
    keys, gaps = acc.means()
    st = RV.subject_test(keys, gaps, n_boot=20000, seed=0)
    write_csv(TABDIR / 'subject_gap.csv', ['dataset', 'subject', 'n_predictions', 'mean_conf_minus_correct'],
              [[d, g, acc.n[(d, g)], fmt(v)] for (d, g), v in zip(keys, gaps)])
    print(f"  {st['n']} subjects; mean gap {st['mean']:+.4f} [{st['lo']:+.4f}, {st['hi']:+.4f}]; "
          f"over-confident for {st['k']}/{st['n']} (sign test p {st['p']:.4f}); per dataset {st['per_dataset']}")

    summary = {
        'n_cells': len(CELLS), 'null_draws': {'cell': S_CELL, 'pooled': S_POOL, 'tau_grid': S_GRID},
        'tau': TAU,
        'pooled': {nn: {k: v for k, v in POOL[(nn, 'aece15')].items() if k not in ('unit_keys', 'unit_excess')}
                   for nn in ('independent', 'coupled', 'clustered')},
        'tau_grid': {f'{t:g}': {'p': POOL[('tau', t)]['p'],
                                'per_dataset_p': {ds: POOL[('tau', t)]['per_dataset'][ds]['p'] for ds in SMALL}}
                     for t in TAU_GRID},
        'separable_holm': {'iid': int((a_iid < 0.05).sum()), 'clustered': int((a_cl < 0.05).sum())},
        'separable_p05': {'iid': int((p_iid < 0.05).sum()), 'clustered': int((p_cl < 0.05).sum())},
        'subject_gap': {k: v for k, v in st.items()},
    }
    (RESULTS / 'review_dependence.json').write_text(json.dumps(summary, indent=1, default=float))
    print('  wrote review_dependence.json')
    mark_done('09d_dependence')

## Stage 9e — power and size of the null test (CPU, ~15 min)
Reviewer 1, points 1 and 6; Reviewer 2, points 5 and 6. How large must a miscalibration be before the matched-null test detects it at these sample sizes, for one cell and for the pooled test, and how does the test behave when labels are not independent within a subject?

In [ ]:
# =====================================================================
# STAGE 9e -- power and size of the matched-null test        [CPU, ~15 min]
# =====================================================================
# Controlled miscalibration: the true success probability of a prediction with
# confidence c is q = sigmoid(logit(c) - b); b > 0 is over-confidence, b < 0
# under-confidence. The true top-label calibration error of that model is
# eps = mean |c - q|. Power = probability that the test rejects at level alpha.
#  (1) Every small-dataset cell with its own confidence profile: power curve,
#      minimum detectable calibration error (MDE: the eps at which power reaches
#      80%) at alpha = 0.05 and at the Holm level 0.05/240, both directions.
#  (2) The pooled test of Stage 9d (mean excess over the 12 test sets, coupled
#      null): power against a miscalibration common to every cell.
#  (3) NTU-60 test set (16,487 clips): MDE for the full-rung runs.
#  (4) Synthetic profiles: MDE as a function of the test-set size n, the number
#      of classes K and the mean confidence. For (1) and (4) the MDE of the
#      binning-free KS statistic is given next to that of the adaptive ECE.
#  (5) Size under within-subject dependence: rejection rate of a model that is
#      calibrated on average but has a subject effect N(0, tau^2) on its success
#      probability, under the i.i.d. null and under the cluster-aware null.
import csv
from collections import defaultdict
import har3d_v2 as V, har3d_review as RV
importlib.reload(RV)
from store import Key, LogitStore

STORE = LogitStore(LOGITS)
TABDIR = RESULTS / 'tables'
ALPHAS = (0.05, 0.05 / 240)
EPS_REF = (0.02, 0.05, 0.10)          # power is tabulated at these true errors
R_POWER, S_POWER = 300, 20000


def power_at(rows, eps, key, side=1):
    pts = sorted((r['eps'], r[key]) for r in rows if r['b'] == 0 or np.sign(r['b']) == side)
    return float(np.interp(eps, [p[0] for p in pts], [p[1] for p in pts]))


def mde_block(rows):
    out = {}
    for a, tag in zip(ALPHAS, ('05', 'holm')):
        for side, sn in ((1, 'over'), (-1, 'under')):
            out[f'mde_{sn}_{tag}'] = RV.mde(rows, alpha=a, side=side)
        for e in EPS_REF:
            out[f'power_{tag}_eps{e:g}'] = power_at(rows, e, f'power_{a:g}')
    return out


if stage_done('09e_power') and not RERUN_REVIEW_STATS:
    print('  [skip] stage 9e already done (RERUN_REVIEW_STATS = True recomputes it)')
else:
    t0 = time.time()
    IDX, CELLS = get_small_cells()          # defined in the Stage 9d cell
    dep = json.loads((RESULTS / 'review_dependence.json').read_text())
    TAU = dep['tau']
    cols = [f'mde_{sn}_{tag}' for tag in ('05', 'holm') for sn in ('over', 'under')] + \
           [f'power_{tag}_eps{e:g}' for tag in ('05', 'holm') for e in EPS_REF] + \
           ['mde_over_05_ks', 'mde_under_05_ks']

    # ------------------------------------------------------------ (1)
    print('=' * 70, '\n(1) power of the per-cell test, real confidence profiles\n' + '=' * 70)
    body, curves, PC = [], defaultdict(list), []
    for i, c in enumerate(CELLS):
        rows, fl, sd = RV.power_curve(c['conf'], S_null=S_POWER, R=R_POWER, alphas=ALPHAS, seed=i)
        blk = mde_block(rows)
        rows_ks, _, _ = RV.power_curve(c['conf'], S_null=S_POWER, R=R_POWER, alphas=(0.05,), seed=i, name='ks')
        blk['mde_over_05_ks'], blk['mde_under_05_ks'] = RV.mde(rows_ks, side=1), RV.mde(rows_ks, side=-1)
        PC.append(dict(dataset=c['dataset'], size=rows[0]['power_0.05'], **blk))
        body.append([c['dataset'], c['stream'], c['split'], c['seed'], c['n'],
                     fmt(c['conf'].mean(), 4), fmt(c['correct'].mean(), 4), fmt(fl), fmt(sd),
                     fmt(rows[0]['power_0.05'], 4)] + [fmt(blk[k], 4) for k in cols])
        for r in rows:
            curves[(c['dataset'], r['b'])].append((r['eps'], r['power_0.05'], r[f'power_{ALPHAS[1]:g}']))
        if (i + 1) % 60 == 0:
            print(f'  {i + 1}/{len(CELLS)} cells ({time.time() - t0:.0f} s)')
    write_csv(TABDIR / 'power_cells.csv',
              ['dataset', 'stream', 'split', 'seed', 'n', 'mean_conf', 'top1', 'floor', 'null_sd',
               'size_0.05'] + cols, body)
    write_csv(TABDIR / 'power_curves.csv', ['dataset', 'b', 'eps', 'power_0.05', 'power_holm240', 'n_cells'],
              [[ds, fmt(b, 3)] + [fmt(x, 4) for x in np.mean(v, 0)] + [len(v)]
               for (ds, b), v in sorted(curves.items())])
    body = []
    for ds in list(SMALL) + ['all']:
        g = [p for p in PC if ds in ('all', p['dataset'])]
        med = {k: float(np.median([p[k] for p in g])) for k in cols}
        body.append([ds, len(g), fmt(np.mean([p['size'] for p in g]), 4)] + [fmt(med[k], 4) for k in cols])
        print(f"  {ds:<9} MDE over-confident: {med['mde_over_05']:.3f} at alpha 0.05, "
              f"{med['mde_over_holm']:.3f} at the Holm level; under-confident {med['mde_under_05']:.3f} / "
              f"{med['mde_under_holm']:.3f}; power at eps 0.02 / 0.05 / 0.10 = "
              + ' / '.join(f"{med[f'power_05_eps{e:g}']:.2f}" for e in EPS_REF)
              + f"; KS statistic: MDE {med['mde_over_05_ks']:.3f} / {med['mde_under_05_ks']:.3f}")
    write_csv(TABDIR / 'power_summary.csv', ['dataset', 'n_cells', 'mean_size_0.05'] + [f'median_{k}' for k in cols], body)

    # ------------------------------------------------------------ (2)
    print('=' * 70, '\n(2) power of the pooled test (12 test sets, coupled null)\n' + '=' * 70)
    SHIFTS = (0.025, 0.05, 0.075, 0.1, 0.15, 0.2, 0.3, 0.4, 0.6, 0.8,
              -0.05, -0.1, -0.15, -0.2, -0.3, -0.4, -0.6, -0.8)
    body = []
    for scope in ['all'] + list(SMALL):
        groups = [(cs, IDX[ds]['n'], [c['unit'] for c in cs])
                  for ds, cs in by_dataset(CELLS).items() if scope in ('all', ds)]
        rows = RV.pooled_power(groups, SHIFTS, S=2000, seed=5)
        m_over = RV.mde([dict(r, **{'power_0.05': r['power']}) for r in rows], side=1)
        m_under = RV.mde([dict(r, **{'power_0.05': r['power']}) for r in rows], side=-1)
        for r in rows:
            body.append([scope, fmt(r['b'], 3), fmt(r['eps']), fmt(r['power'], 4), fmt(m_over, 4), fmt(m_under, 4)])
        print(f'  {scope:<9} size {rows[0]["power"]:.3f}; MDE of the pooled test: over-confident '
              f'{m_over:.4f}, under-confident {m_under:.4f}  ({time.time() - t0:.0f} s)')
    write_csv(TABDIR / 'power_pooled.csv', ['scope', 'b', 'eps', 'power', 'mde_over', 'mde_under'], body)

    # ------------------------------------------------------------ (3)
    print('=' * 70, '\n(3) NTU-60 test set\n' + '=' * 70)
    body = []
    n_pool = json.loads((RESULTS / 'ladder_protocol.json').read_text())['n_pool']
    for mdl in LADDER_BACKBONES:
        for seed in LADDER_SEEDS:
            k = Key(mdl, 'ntu60', LADDER_STREAM, f'ladder_n{n_pool}', seed, 'test')
            if not STORE.has(k):
                continue
            d = STORE.load(k)
            conf = V.softmax(d['logits']).max(-1)
            rows, fl, sd = RV.power_curve(conf, bs=(0.02, 0.04, 0.06, 0.08, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0, 2.0),
                                          S_null=2000, R=200, alphas=(0.05,), seed=seed)
            body.append([mdl, seed, len(conf), fmt(conf.mean(), 4), fmt(fl), fmt(sd),
                         fmt(rows[0]['power_0.05'], 3), fmt(RV.mde(rows, side=1), 4), fmt(RV.mde(rows, side=-1), 4)])
            print(f'  {mdl:<7} seed {seed}: floor {fl:.4f}, MDE {RV.mde(rows, side=1):.4f} (over), '
                  f'{RV.mde(rows, side=-1):.4f} (under)')
    write_csv(TABDIR / 'power_ntu.csv', ['model', 'seed', 'n', 'mean_conf', 'floor', 'null_sd', 'size_0.05',
                                         'mde_over_05', 'mde_under_05'], body)

    # ------------------------------------------------------------ (4)
    print('=' * 70, '\n(4) synthetic profiles: n x K x mean confidence\n' + '=' * 70)
    body = []
    for K_ in (10, 16, 20, 60):
        for mc_ in (0.5, 0.7, 0.9):
            for n_ in (50, 100, 250, 500, 1000, 2000, 5000, 16487):
                conf = RV.synthetic_conf(n_, K_, mc_, seed=0)
                S_ = int(min(S_POWER, max(2000, 4e7 // n_)))
                al = ALPHAS if S_ >= 20000 else (0.05,)
                rows, fl, sd = RV.power_curve(conf, S_null=S_, R=R_POWER, alphas=al, seed=n_ + K_)
                rks, _, _ = RV.power_curve(conf, S_null=S_, R=R_POWER, alphas=(0.05,), seed=n_ + K_, name='ks')
                body.append([n_, K_, mc_, fmt(fl), fmt(sd), fmt(rows[0]['power_0.05'], 3),
                             fmt(RV.mde(rows, 0.05, 1), 4), fmt(RV.mde(rows, 0.05, -1), 4),
                             fmt(RV.mde(rows, ALPHAS[1], 1), 4) if len(al) > 1 else '',
                             fmt(RV.mde(rows, ALPHAS[1], -1), 4) if len(al) > 1 else '',
                             fmt(RV.mde(rks, 0.05, 1), 4), fmt(RV.mde(rks, 0.05, -1), 4)])
        print(f'  K = {K_} done ({time.time() - t0:.0f} s)')
    write_csv(TABDIR / 'power_synthetic.csv',
              ['n', 'K', 'mean_conf', 'floor', 'null_sd', 'size_0.05', 'mde_over_05', 'mde_under_05',
               'mde_over_holm', 'mde_under_holm', 'mde_over_05_ks', 'mde_under_05_ks'], body)

    # ------------------------------------------------------------ (5)
    print('=' * 70, '\n(5) size of the test under within-subject dependence\n' + '=' * 70)
    body = []
    for ds, cs in by_dataset(CELLS).items():
        taus = tuple(sorted({0.0, 0.03, 0.06, 0.10, 0.15, round(TAU[ds], 4)}))
        pick = [c for c in cs if c['stream'] == 'joint' and c['seed'] == 0]       # one cell per test set
        res = [RV.clustered_size(c['conf'], c['subj'], taus, S_null=2000, R=1000, seed=i)
               for i, c in enumerate(pick)]
        for j, tau in enumerate(taus):
            body.append([ds, fmt(tau, 4), 'estimated' if abs(tau - round(TAU[ds], 4)) < 1e-9 and tau > 0 else '',
                         len(pick), fmt(np.mean([r[j]['size_iid'] for r in res]), 4),
                         fmt(np.mean([r[j]['size_cluster'] for r in res]), 4),
                         fmt(np.mean([r[j]['floor_iid'] for r in res])),
                         fmt(np.mean([r[j]['floor_cluster'] for r in res]))])
        print(f'  {ds:<9} ' + '  '.join(f"tau {b[1]}: {b[4]}" for b in body[-len(taus):]) + '  (i.i.d. null)')
    write_csv(TABDIR / 'size_clustered.csv',
              ['dataset', 'tau', 'note', 'n_cells', 'size_iid_null', 'size_cluster_null', 'floor_iid',
               'floor_cluster'], body)
    mark_done('09e_power')

## Stage 9f — calibration methods with subjects as units; cross-fitted post-hoc calibration; per-class null (CPU, ~10 min)
Reviewer 2, points 1, 12, 13 and 15. The method comparison is repeated with human subjects as the units; temperature and vector scaling are refitted on a larger, subject-disjoint calibration set (cross-fitting over test subjects) and on the test set itself (an upper bound), to separate "too little calibration data" from "nothing for this method to fix"; the per-class ECE gets class counts and a matched null.

In [ ]:
# =====================================================================
# STAGE 9f -- methods at the subject level, calibration-set size, per-class null  [CPU, ~10 min]
# =====================================================================
#  (1) Six methods vs raw with SUBJECTS as units. Brier score and NLL are means
#      of per-clip losses, so the paired difference raw - method can be averaged
#      over every prediction made for a subject (all splits, streams, seeds in
#      which that subject is a test subject). One value per subject; exact sign
#      test and bootstrap over subjects (resampled within dataset); Holm over
#      the six methods. Subjects are disjoint units; no test set is counted
#      twice.
#  (2) Is post-hoc calibration data-starved or intrinsically ineffective here?
#        *_val      fitted on the validation subjects (1-2 subjects): the
#                   protocol of the paper
#        *_xfit     cross-fitted: for each test subject, fitted on the
#                   validation clips plus the test clips of the OTHER test
#                   subjects, applied to the held-out subject (4-6 subjects,
#                   still subject-disjoint)
#        *_oracle   fitted on the test set itself: an optimistic upper bound on
#                   what the method family can do, NOT a usable method
#      for temperature and vector scaling. Excess over each variant's own
#      floor (tested over test sets, as in Stage 9c), Brier and NLL (tested
#      over subjects).
#  (3) Pooled excess of every method under the coupled null: is miscalibration
#      still detectable after the method has been applied?
#  (4) Per-class ECE of Fig. 5 with the number of predictions and of distinct
#      clips per class and a matched (clip-coupled) null, Holm over classes.
import csv
from collections import defaultdict
import har3d_v2 as V, har3d_calib as C, har3d_stats as HS, har3d_review as RV
importlib.reload(RV)
from store import Key, LogitStore

STORE = LogitStore(LOGITS)
TABDIR = RESULTS / 'tables'
METHODS6 = ['temperature', 'vector', 'label_smooth', 'focal', 'mc_dropout', 'deep_ensemble']
VARIANTS = ['temperature_xfit', 'temperature_oracle', 'vector_xfit', 'vector_oracle']
L2_VECTOR = 1e-2                    # as in Stage 9c
N_SIM_FLOOR_REV = 1000


def clip_losses(p, y):
    """Per-clip Brier score and negative log-likelihood."""
    oh = np.zeros_like(p); oh[np.arange(len(y)), y] = 1.0
    return ((p - oh) ** 2).sum(1), -np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1.0))


def cell_scores(p, y, seed):
    conf, corr = p.max(-1), (p.argmax(-1) == y)
    t = RV.cell_tests(conf, corr, S=N_SIM_FLOOR_REV, seed=seed, names=('aece15',))['aece15']
    b, l = clip_losses(p, y)
    return dict(ece=t['obs'], floor=t['null_mean'], excess=t['excess'], brier=float(b.mean()),
                nll=float(l.mean()), top1=float(corr.mean()), conf=conf, correct=corr, b=b, l=l)


if stage_done('09f_methods_subjects') and not RERUN_REVIEW_STATS:
    print('  [skip] stage 9f already done (RERUN_REVIEW_STATS = True recomputes it)')
else:
    t0 = time.time()
    IDX, CELLS = get_small_cells()          # defined in the Stage 9d cell
    ALLM = ['raw'] + METHODS6 + VARIANTS
    per_cell, mcells = [], defaultdict(list)       # rows for the test-set tests; cells per method
    accB = {m: RV.SubjectAccumulator() for m in ALLM[1:]}
    accN = {m: RV.SubjectAccumulator() for m in ALLM[1:]}
    members, fits = defaultdict(list), []

    def record(m, c, sc, seed):
        per_cell.append(dict(method=m, dataset=c['dataset'], stream=c['stream'], split=c['split'],
                             seed=seed, **{k: sc[k] for k in ('ece', 'floor', 'excess', 'brier', 'nll', 'top1')}))
        mcells[m].append(dict(conf=sc['conf'], correct=sc['correct'], clip=c['clip'],
                              unit=c['unit'], dataset=c['dataset']))

    for i, c in enumerate(CELLS):
        k, z, y, sub = c['key'], c['logits'], c['labels'], c['subj']
        raw = cell_scores(V.softmax(z), y, k.seed)
        record('raw', c, raw, k.seed)
        probs = {}
        vk = Key('stgcn', k.dataset, k.stream, k.split, k.seed, 'val')
        if STORE.has(vk):
            dv = STORE.load(vk); zv, yv = dv['logits'], dv['labels']
            T_val = V.fit_temperature(zv, yv)
            probs['temperature'] = V.softmax(z, T_val)
            probs['vector'] = C.apply_vector_scaling(z, *C.fit_vector_scaling(zv, yv, l2=L2_VECTOR))
            # cross-fitting over the test subjects
            pt, pv, Ts, ncal = np.zeros_like(probs['temperature']), np.zeros_like(probs['temperature']), [], []
            for g in np.unique(sub):
                te, tr = sub == g, sub != g
                zc, yc = np.concatenate([zv, z[tr]]), np.concatenate([yv, y[tr]])
                Ts.append(V.fit_temperature(zc, yc)); ncal.append(len(yc))
                pt[te] = V.softmax(z[te], Ts[-1])
                pv[te] = C.apply_vector_scaling(z[te], *C.fit_vector_scaling(zc, yc, l2=L2_VECTOR))
            probs['temperature_xfit'], probs['vector_xfit'] = pt, pv
            T_or = V.fit_temperature(z, y)
            probs['temperature_oracle'] = V.softmax(z, T_or)
            probs['vector_oracle'] = C.apply_vector_scaling(z, *C.fit_vector_scaling(z, y, l2=L2_VECTOR))
            fits.append([k.dataset, k.stream, k.split, k.seed, len(yv), int(np.mean(ncal)), len(y),
                         fmt(T_val, 4), fmt(float(np.mean(Ts)), 4), fmt(T_or, 4)])
        for tag, nm in (('stgcn_ls', 'label_smooth'), ('stgcn_focal', 'focal')):
            mk = Key(tag, k.dataset, k.stream, k.split, k.seed, 'test')
            if STORE.has(mk):
                dm = STORE.load(mk)
                if np.array_equal(dm['labels'], y):
                    probs[nm] = V.softmax(dm['logits'])
        f = LOGITS / f'mc__{k.dataset}__stgcn__{k.stream}__{k.split}__s{k.seed}__test.npz'
        if f.exists():
            zz = np.load(f, allow_pickle=False)
            if np.array_equal(zz['labels'], y):
                probs['mc_dropout'] = C.mc_dropout_probs(zz['logits'])
        for nm, p in probs.items():
            sc = cell_scores(p, y, k.seed)
            record(nm, c, sc, k.seed)
            accB[nm].add(k.dataset, sub, raw['b'] - sc['b'])
            accN[nm].add(k.dataset, sub, raw['l'] - sc['l'])
        members[(k.dataset, k.stream, k.split)].append((k.seed, V.softmax(z), raw['b'], raw['l'], c))
        if (i + 1) % 60 == 0 or i + 1 == len(CELLS):
            print(f'  {i + 1}/{len(CELLS)} cells ({time.time() - t0:.0f} s)')
    for key3, mem in members.items():
        if len(mem) < len(SEEDS):
            continue
        mem = sorted(mem, key=lambda m: m[0])[:len(SEEDS)]
        c = mem[0][4]
        sc = cell_scores(np.mean([m[1] for m in mem], 0), c['labels'], 0)
        record('deep_ensemble', c, sc, -1)
        accB['deep_ensemble'].add(c['dataset'], c['subj'], np.mean([m[2] for m in mem], 0) - sc['b'])
        accN['deep_ensemble'].add(c['dataset'], c['subj'], np.mean([m[3] for m in mem], 0) - sc['l'])
    have = [m for m in ALLM if mcells[m]]
    write_csv(TABDIR / 'calibration_fits.csv',
              ['dataset', 'stream', 'split', 'seed', 'n_val', 'n_xfit', 'n_test', 'T_val', 'T_xfit_mean', 'T_oracle'], fits)

    # ------------------------------------------------------------ (1)+(2)
    print('=' * 70, '\n(1)-(2) methods and calibration-set variants: subjects (Brier, NLL), test sets (excess)\n' + '=' * 70)
    body = []
    for fam_name, fam in (('six methods', [m for m in METHODS6 if m in have]),
                          ('calibration-set variants', [m for m in VARIANTS if m in have])):
        if not fam:
            continue
        rb = RV.subject_method_tests(accB, fam, n_boot=20000)
        rn = RV.subject_method_tests(accN, fam, n_boot=20000)
        rx, _ = HS.method_comparison(per_cell, fam, key='excess')
        for m in fam:
            mean = {sc: float(np.mean([r[sc] for r in per_cell if r['method'] == m]))
                    for sc in ('excess', 'brier', 'nll', 'top1', 'ece', 'floor')}
            for sc, r, unit in (('brier', rb[m], 'subjects'), ('nll', rn[m], 'subjects'),
                                ('excess', rx[m], 'test sets')):
                body.append([fam_name, m, sc, unit, r['n'], r['k'],
                             ' '.join(f'{a}/{b}' for a, b in r['per_dataset'].values()),
                             fmt(mean[sc]), fmt(r['mean']), fmt(r['lo']), fmt(r['hi']), fmt(r['p']),
                             fmt(r['p_holm']), r['verdict']])
            print(f"  {m:<19} Brier {rb[m]['mean']:+.4f} ({rb[m]['k']}/{rb[m]['n']} subjects, Holm p "
                  f"{rb[m]['p_holm']:.4f}, {rb[m]['verdict']});  NLL {rn[m]['mean']:+.4f} "
                  f"({rn[m]['k']}/{rn[m]['n']}, {rn[m]['verdict']});  excess {rx[m]['mean']:+.4f} "
                  f"({rx[m]['k']}/{rx[m]['n']} test sets, {rx[m]['verdict']})")
    rawm = {sc: float(np.mean([r[sc] for r in per_cell if r['method'] == 'raw']))
            for sc in ('excess', 'brier', 'nll', 'top1', 'ece', 'floor')}
    for sc in ('brier', 'nll', 'excess'):
        body.insert(0, ['baseline', 'raw', sc, '', '', '', '', fmt(rawm[sc]), '', '', '', '', '', 'baseline'])
    write_csv(TABDIR / 'methods_subject_level.csv',
              ['family', 'method', 'score', 'unit', 'n_units', 'n_improved', 'improved_per_dataset',
               'mean_score', 'mean_reduction', 'ci_lo', 'ci_hi', 'p', 'p_holm', 'verdict'], body)

    # ------------------------------------------------------------ (3)
    print('=' * 70, '\n(3) pooled excess of every method under the coupled null\n' + '=' * 70)
    body = []
    for m in have:
        obs, nul, units, dsl = pooled_under(mcells[m], IDX, 2000, seed=3)
        r = RV.pooled_from_null(obs['aece15'], nul['aece15'], units, dsl)
        body.append([m, r['n_cells'], fmt(r['mean_excess']), fmt(r['null_sd']), fmt(r['z'], 2), fmt(r['p']),
                     f"{r['k_pos']}/{r['n_units']}"]
                    + [x for ds in SMALL for x in (fmt(r['per_dataset'][ds]['mean_excess']),
                                                   fmt(r['per_dataset'][ds]['p']))])
        print(f"  {m:<19} pooled excess {r['mean_excess']:+.4f}  coupled p {r['p']:.4f}  "
              f"test sets > 0: {r['k_pos']}/{r['n_units']}")
    write_csv(TABDIR / 'methods_pooled_null.csv',
              ['method', 'n_cells', 'pooled_excess', 'null_sd', 'z', 'p_coupled', 'test_sets_positive']
              + [f'{ds}_{x}' for ds in SMALL for x in ('excess', 'p')], body)

    # ------------------------------------------------------------ (4)
    print('=' * 70, '\n(4) per-class ECE with class counts and a matched null\n' + '=' * 70)
    body = []
    for ds, cs in by_dataset(CELLS).items():
        cs = [c for c in cs if c['stream'] == 'joint']
        conf = np.concatenate([c['conf'] for c in cs]); pred = np.concatenate([c['pred'] for c in cs])
        corr = np.concatenate([c['correct'] for c in cs]); clip = np.concatenate([c['clip'] for c in cs])
        yy = np.concatenate([c['labels'] for c in cs])
        K_ = int(IDX[ds]['y'].max()) + 1
        rows = RV.per_class_null(conf, pred, corr, clip, IDX[ds]['n'], K_, S=5000, seed=0)
        for r in rows:
            cl = r['class']
            body.append([ds, cl, r['n_pred'], r['n_clips'], int((IDX[ds]['y'] == cl).sum()),
                         fmt(1 - float((pred[yy == cl] == cl).mean()) if (yy == cl).any() else float('nan'), 4),
                         fmt(r['obs']), fmt(r['null_mean']), fmt(r['null_p95']), fmt(r['obs'] - r['null_mean']),
                         fmt(r['p']), fmt(r['p_holm'])])
        sep = sum(r['p_holm'] < 0.05 for r in rows)
        print(f"  {ds:<9} {len(rows)} classes; per-class ECE {min(r['obs'] for r in rows):.3f}-"
              f"{max(r['obs'] for r in rows):.3f}, floors {min(r['null_mean'] for r in rows):.3f}-"
              f"{max(r['null_mean'] for r in rows):.3f}; above their null after Holm: {sep}; "
              f"distinct clips per class {min(r['n_clips'] for r in rows)}-{max(r['n_clips'] for r in rows)}")
    write_csv(TABDIR / 'per_class_null.csv',
              ['dataset', 'class', 'n_predictions', 'n_distinct_clips', 'n_clips_of_class', 'one_minus_recall',
               'per_class_ece', 'floor', 'null_p95', 'excess', 'p', 'p_holm'], body)
    mark_done('09f_methods_subjects')

## Stage 9g — preprocessing sensitivity and training-only checks (CPU, ~5 min)
Reviewer 2, point 14. The matched-null analysis under every alternative preprocessing of Stage 5b, and a check that the skeleton tree and the joint order are confirmed by the training subjects of each split alone.

In [ ]:
# =====================================================================
# STAGE 9g -- preprocessing sensitivity; training-only checks   [CPU, ~5 min]
# =====================================================================
#  (1) For every preprocessing variant (Stage 5b) and dataset, on the joint
#      stream and seeds ABLATION_SEEDS: accuracy, adaptive ECE, floor, excess,
#      cells separable from the null, the pooled excess with its coupled
#      p-value, temperature, Brier, NLL. 'frame0' is the main experiment
#      restricted to the same stream and seeds.
#  (2) Was anything about the skeleton derived from test subjects?
#      The tree is a fixed anatomical prior (not learned); the gate of Stage 4
#      only VERIFIES it. Here the verification is repeated on the training
#      subjects of every split alone. The joint order of MSR-DailyActivity3D
#      was recovered by matching inter-joint distance signatures (no labels):
#      here it is re-derived from the training subjects of every split alone,
#      against MSR-Action3D training subjects; since the cache is stored in
#      canonical order, the answer must be the identity.
import contextlib, csv, io
from collections import defaultdict
import har3d_v2 as V, har3d_grid as G, har3d_stats as HS, har3d_review as RV
importlib.reload(RV)
from store import Key, LogitStore

TABDIR = RESULTS / 'tables'
PREP_LOGITS = RUN / 'logits_prep'
PREP_VARIANTS = [('frame0 (main)', None), ('frame0_align', 'frame0_align'),
                 ('perframe_align', 'perframe_align'), ('legacy_centring', 'legacy_centring'),
                 ('legacy', 'legacy')]

if stage_done('09g_prep_checks') and not RERUN_REVIEW_STATS:
    print('  [skip] stage 9g already done (RERUN_REVIEW_STATS = True recomputes it)')
else:
    t0 = time.time()
    IDX, CELLS = get_small_cells()          # defined in the Stage 9d cell

    # ------------------------------------------------------------ (1)
    print('=' * 70, '\n(1) calibration conclusions under alternative preprocessing\n' + '=' * 70)
    body, n_variants = [], 0
    pstore = LogitStore(PREP_LOGITS) if PREP_LOGITS.exists() else None
    for name, tag in PREP_VARIANTS:
        if tag is None:
            store, cells = LogitStore(LOGITS), [c for c in CELLS if c['stream'] == 'joint' and c['seed'] in ABLATION_SEEDS]
        elif pstore is None:
            continue
        else:
            store = pstore
            cells = load_cells(pstore, small_index(tag), streams=('joint',), seeds=ABLATION_SEEDS, logits=True)
        if not cells:
            continue
        n_variants += 1
        for c in cells:
            c['t'] = RV.cell_tests(c['conf'], c['correct'], S=20000, seed=c['seed'], names=('aece15',))['aece15']
            p = V.softmax(c['logits'])
            c['brier'], c['nll'] = V.brier(p, c['labels']), V.nll(p, c['labels'])
            vk = Key('stgcn', c['dataset'], c['stream'], c['split'], c['seed'], 'val')
            c['T'] = float('nan')
            if store.has(vk):
                dv = store.load(vk)
                c['T'] = V.fit_temperature(dv['logits'], dv['labels'])
        for ds, cs in by_dataset(cells).items():
            obs, nul, units, dsl = pooled_under(cs, IDX, 5000, seed=2)
            r = RV.pooled_from_null(obs['aece15'], nul['aece15'], units)
            adj = HS.holm([c['t']['p'] for c in cs])
            body.append([name, ds, len(cs), r['n_units'],
                         fmt(np.mean([c['correct'].mean() for c in cs]), 4),
                         fmt(np.mean([c['t']['obs'] for c in cs])), fmt(np.mean([c['t']['null_mean'] for c in cs])),
                         fmt(r['mean_excess']), fmt(r['p']), f"{r['k_pos']}/{r['n_units']}",
                         int(sum(c['t']['p'] < 0.05 for c in cs)), int((adj < 0.05).sum()),
                         fmt(np.mean([c['T'] for c in cs]), 3), fmt(np.mean([c['brier'] for c in cs])),
                         fmt(np.mean([c['nll'] for c in cs]))])
            print(f"  {name:<16} {ds:<9} {len(cs):>3} cells  top-1 {body[-1][4]}  aECE {body[-1][5]}  "
                  f"floor {body[-1][6]}  excess {r['mean_excess']:+.4f} (coupled p {r['p']:.4f})  "
                  f"separable {body[-1][11]}/{len(cs)}  T {body[-1][12]}")
    write_csv(TABDIR / 'prep_sensitivity.csv',
              ['preprocessing', 'dataset', 'n_cells', 'n_test_sets', 'top1', 'ece_adaptive', 'floor',
               'pooled_excess', 'p_coupled', 'test_sets_positive', 'cells_p_lt_0.05', 'cells_separable_holm',
               'temperature', 'brier', 'nll'], body)
    if n_variants <= 1:
        print('  !! only the main preprocessing is available -- run Stage 5b on a GPU runtime first')

    # ------------------------------------------------------------ (2)
    print('=' * 70, '\n(2) skeleton tree and joint order from training subjects only\n' + '=' * 70)

    def seqs_of(cache, rows):
        o = cache['offsets']
        return [cache['xyz'][o[i]:o[i + 1]] for i in rows]

    caches = {ds: V.load_cache(CACHE / npz) for ds, npz in G.DATASETS}
    keeps = {ds: np.flatnonzero(V.quality_mask(c['flags'], MAX_ZERO_FRAC, MIN_FRAMES) if 'flags' in c
                                else np.ones(len(c['meta']), bool)) for ds, c in caches.items()}
    body, all_ok = [], True
    for ds in SMALL:
        for name, sp in IDX[ds]['splits'].items():
            tr_rows = keeps[ds][sp['train']]
            seqs = seqs_of(caches[ds], tr_rows)
            with contextlib.redirect_stdout(io.StringIO()):
                rep = V.verify_topology(seqs, raise_on_fail=False)
                F = np.concatenate([np.asarray(s, np.float64) for s in seqs], 0)
                opt = V.tree_optimality(F[~(np.abs(F) < 1e-9).all(-1).any(-1)])
            # the rule of the Stage 4 gate: every edge rigid, or (some edges jittery but) the
            # tree is still the near-optimal explanation of the geometry and no edge too long
            ok = all(r['ok'] for r in rep) or ((1 / 1.10 <= opt['ratio'] <= 1.10) and opt['shared'] >= 8
                                               and all(r['len_m'] <= V.MAX_BONE_M for r in rep))
            perm_txt = ''
            if ds != 'action3d':
                ref = seqs_of(caches['action3d'], keeps['action3d'][IDX['action3d']['splits'][name]['train']][:120])
                with contextlib.redirect_stdout(io.StringIO()):
                    perm, info = V.derive_canonical_permutation(np.concatenate(seqs[:60], 0),
                                                                np.concatenate(ref, 0), verbose=False)
                ident = bool(np.array_equal(perm, np.arange(len(perm))))
                perm_txt = 'identity' if ident else 'DIFFERENT: ' + ' '.join(map(str, perm.tolist()))
                ok = ok and ident
            all_ok = all_ok and ok
            body.append([ds, name, ' '.join(map(str, sp['subjects']['train'])), len(tr_rows),
                         sum(r['ok'] for r in rep), len(rep), fmt(opt['ratio'], 3), opt['shared'],
                         perm_txt, 'pass' if ok else 'FAIL'])
            print(f"  {ds:<9} {name:<16} train subjects {sp['subjects']['train']}: "
                  f"{sum(r['ok'] for r in rep)}/{len(rep)} edges rigid, tree {opt['ratio']:.3f}x the "
                  f"data-derived optimum" + (f', joint order: {perm_txt}' if perm_txt else '')
                  + ('' if ok else '   <-- FAIL'))
    write_csv(TABDIR / 'train_only_checks.csv',
              ['dataset', 'split', 'train_subjects', 'n_train_sequences', 'edges_rigid', 'edges',
               'tree_length_ratio', 'edges_shared_with_optimum', 'joint_order_from_train_only', 'verdict'], body)
    print('  every split passes on its training subjects alone' if all_ok
          else '  !! at least one split does not pass on its training subjects alone -- see the table')
    print(f'  ({time.time() - t0:.0f} s)')
    if n_variants > 1:
        mark_done('09g_prep_checks')
    else:
        print('  [stage 9g NOT marked complete: the Stage 5b logits are missing]')

## Stage 9h — replication, matched training durations, environment, seed-level ladder analysis (CPU, ~10 min)
Reviewer 2, points 2, 3, 4, 7, 11 (and 1, 5 for the NTU test set). Always recomputed, so it can be run while the replication of Stage 8c is still incomplete: it uses whatever runs are stored and says what is missing. Run Stage 9 first (it evaluates the new cells).

In [ ]:
# =====================================================================
# STAGE 9h -- replication and seed-level ladder analysis      [CPU, ~10 min]
# =====================================================================
# Units are SEEDS: a seed fixes the initialisation and the training subsample
# of every rung, and the rungs of a seed are nested and share the test set, so
# the nine seed-rung pairs of the first version are not nine observations.
#  (1) Every large-rung cell: original (seeds 0-2), replication (new seeds, one
#      pinned environment), environment re-runs.
#  (2) Pre-specified comparison (results/replication_protocol.json): per seed,
#      the difference between two backbones averaged over the large rungs;
#      mean, t-interval and exact sign-flip test over seeds, for the new seeds
#      alone, the original seeds alone (exploratory) and all seeds. Also the
#      original low-rung regime, seed-level.
#  (3) The same comparison at MATCHED TRAINING STEPS (snapshots) and at the
#      best-validation-ACCURACY checkpoint, new seeds: does the difference
#      between backbones depend on where early stopping on validation NLL
#      happened to stop each of them?
#  (4) Same-seed environment re-runs against the stored originals, next to the
#      spread across seeds.
#  (5) Estimator sensitivity on the ladder: the comparison of (2) with every
#      estimator of Stage 9d.
#  (6) NTU test subjects (20 performers): subject heterogeneity, the clustered
#      null and the per-subject over-confidence, for the large-rung cells.
import csv
from collections import defaultdict
import har3d_v2 as V, har3d_stats as HS, har3d_review as RV, har3d_ntu as N
importlib.reload(RV)
from store import Key, LogitStore

STORE = LogitStore(LOGITS)
TABDIR = RESULTS / 'tables'; TABDIR.mkdir(parents=True, exist_ok=True)
CKPT = RUN / 'logits_ckpt'
NAME = {'stgcn': 'ST-GCN', 'agcn': '2s-AGCN', 'ctrgcn': 'CTR-GCN'}
COMPARISONS = [('agcn', 'stgcn'), ('ctrgcn', 'stgcn'), ('ctrgcn', 'agcn')]
n_pool = json.loads((RESULTS / 'ladder_protocol.json').read_text())['n_pool']
BIG = sorted({min(n, n_pool) for n in REPL_RUNGS})
LOW = sorted({min(n, n_pool) for n in LADDER_N} - set(BIG))
METRICS = [('null_excess', 'excess adaptive ECE'), ('ece_adaptive', 'adaptive ECE'),
           ('nll', 'NLL'), ('brier', 'Brier score'), ('top1', 'top-1 accuracy')]

if 'write_csv' not in globals():
    def write_csv(path, header, body):
        with open(path, 'w', newline='') as fh:
            w = csv.writer(fh); w.writerow(header); w.writerows(body)
        print('  wrote', path.name, f'({len(body)} rows)')

    def fmt(x, d=5):
        return x if isinstance(x, (str, int, np.integer)) else f'{float(x):.{d}f}'

t0 = time.time()
rows = [r for r in json.loads((RESULTS / 'metrics.json').read_text()) if r['dataset'] == 'ntu60']
# (backbone, n_train, seed) -> metrics row; origin of the run
CELL, ORIGIN, ENVR = {}, {}, {}
for r in rows:
    m = r['model']
    if m in NAME:
        CELL[(m, r['n_train'], r['seed'])] = r; ORIGIN[(m, r['n_train'], r['seed'])] = 'original'
    elif m.endswith('_rep') and m[:-4] in NAME:
        CELL[(m[:-4], r['n_train'], r['seed'])] = r; ORIGIN[(m[:-4], r['n_train'], r['seed'])] = 'replication'
    elif m.endswith('_env') and m[:-4] in NAME:
        ENVR[(m[:-4], r['n_train'], r['seed'])] = r
missing = [k.fname() for k in STORE.list_keys()
           if k.subset == 'test' and k.model.endswith(('_rep', '_env'))
           and k.fname() not in {r['file'] for r in rows}]
if missing:
    raise RuntimeError(f'{len(missing)} replication cells are stored but not in metrics.json -- run Stage 9 first')
orig_seeds = sorted({s for (m, n, s), o in ORIGIN.items() if o == 'original'})
new_seeds = sorted({s for (m, n, s), o in ORIGIN.items() if o == 'replication'})
print(f'  original seeds {orig_seeds}; replication seeds {new_seeds or "none yet"}; '
      f'environment re-runs {len(ENVR)}')

# ---------------------------------------------------------------- (1)
body = []
for (m, n, s), r in sorted(CELL.items()):
    if n in BIG:
        body.append([m, n, s, ORIGIN[(m, n, s)], fmt(r['top1'], 4), fmt(r['ece_adaptive']),
                     fmt(r['null_null_mean']), fmt(r['null_excess']), fmt(r['null_p_value']),
                     fmt(r['nll'], 4), fmt(r['brier'], 4), fmt(r['temperature'], 3),
                     r['best_step'], r['steps_run']])
write_csv(TABDIR / 'replication_cells.csv',
          ['backbone', 'n_train', 'seed', 'origin', 'top1', 'ece_adaptive', 'floor', 'excess', 'null_p',
           'nll', 'brier', 'temperature', 'best_step', 'steps_run'], body)


# ---------------------------------------------------------------- (2)
def seed_diffs(a, b, rungs, seeds, value):
    """Per seed: mean over `rungs` of value(a) - value(b); seeds with every rung present."""
    out = {}
    for s in seeds:
        if all((m, n, s) in CELL for m in (a, b) for n in rungs):
            out[s] = float(np.mean([value(a, n, s) - value(b, n, s) for n in rungs]))
    return out


def summarise(tag, a, b, regime, rungs, metric, label, value, body, verbose=False):
    sets = [('original seeds (exploratory)', orig_seeds), ('new seeds (confirmatory)', new_seeds),
            ('all seeds', orig_seeds + new_seeds)]
    out = {}
    for set_name, seeds in sets:
        d = seed_diffs(a, b, rungs, seeds, value)
        if not d:
            continue
        ss = RV.seed_summary(list(d.values()))
        pairs = [value(a, n, s) - value(b, n, s) for s in d for n in rungs]
        out[set_name] = ss
        body.append([tag, a, b, regime, metric, set_name, ss['n'], ' '.join(map(str, d)),
                     ' '.join(f'{v:+.4f}' for v in d.values()), fmt(ss['mean']), fmt(ss['sd']),
                     fmt(ss['lo']), fmt(ss['hi']), ss['n_pos'], fmt(ss['p_t']), fmt(ss['p_flip']),
                     fmt(2.0 / 2 ** ss['n']), f"{int(np.sum(np.array(pairs) > 0))}/{len(pairs)}"])
        if verbose:
            print(f"    {set_name:<30} n = {ss['n']}: mean {ss['mean']:+.4f} "
                  f"[{ss['lo']:+.4f}, {ss['hi']:+.4f}]  positive in {ss['n_pos']}/{ss['n']} seeds  "
                  f"sign-flip p {ss['p_flip']:.4f} (smallest possible {2.0 / 2 ** ss['n']:.4f})  t p {ss['p_t']:.4f}")
    return out


HDR = ['analysis', 'model_a', 'model_b', 'regime', 'metric', 'seed_set', 'n_seeds', 'seeds',
       'per_seed_difference', 'mean', 'sd', 'ci_lo', 'ci_hi', 'n_positive', 'p_t', 'p_sign_flip',
       'p_sign_flip_minimum', 'seed_rung_pairs_positive']
print('=' * 70, '\n(2) seed-level backbone comparison (pre-specified at the large rungs)\n' + '=' * 70)
body = []
for a, b in COMPARISONS:
    for regime, rungs in (('>= 10,000 clips', BIG), ('<= 5,000 clips', LOW)):
        for metric, label in METRICS:
            verbose = metric == 'null_excess' and regime.startswith('>')
            if verbose:
                print(f'  {NAME[a]} minus {NAME[b]}, {label}, {regime}')
            summarise('main checkpoint (validation NLL)', a, b, regime, rungs, metric, label,
                      lambda m, n, s, k=metric: CELL[(m, n, s)][k], body, verbose)
write_csv(TABDIR / 'replication_seed_level.csv', HDR, body)


# ---------------------------------------------------------------- (3)
def cc_of(logits, labels):
    p = V.softmax(logits)
    return p.max(-1), (p.argmax(-1) == labels), p


def ck_metrics(path, seed):
    z = np.load(path, allow_pickle=False)
    conf, corr, p = cc_of(z['logits'], z['labels'])
    t = RV.cell_tests(conf, corr, S=500, seed=seed, names=('aece15',))['aece15']
    return dict(step=int(z['step']), top1=float(corr.mean()), ece=t['obs'], floor=t['null_mean'],
                excess=t['excess'], nll=V.nll(p, z['labels']), val_nll=float(z['val_nll']),
                val_top1=float(z['val_top1']))


print('=' * 70, '\n(3) matched training steps and the best-validation-accuracy checkpoint (new seeds)\n' + '=' * 70)
CK = {}
for f in sorted(CKPT.glob('ck__ntu60__*.npz')) if CKPT.exists() else []:
    _, _, tag, stream, split, sd, ck = f.stem.split('__')
    if not tag.endswith('_rep') or tag[:-4] not in NAME:
        continue
    CK[(tag[:-4], int(split.replace('ladder_n', '')), int(sd[1:]), ck)] = ck_metrics(f, int(sd[1:]))
ck_tags = sorted({k[3] for k in CK}, key=lambda t: (t == 'acc', int(t[1:]) if t != 'acc' else 0))
body = [[m, n, s, ck, c['step'], fmt(c['top1'], 4), fmt(c['ece']), fmt(c['floor']), fmt(c['excess']),
         fmt(c['nll'], 4), fmt(c['val_nll'], 4), fmt(c['val_top1'], 4)] for (m, n, s, ck), c in sorted(CK.items())]
write_csv(TABDIR / 'replication_checkpoints.csv',
          ['backbone', 'n_train', 'seed', 'checkpoint', 'step', 'top1', 'ece_adaptive', 'floor', 'excess',
           'nll', 'val_nll', 'val_top1'], body)
body, body2 = [], []
for ck in ck_tags:
    what = 'best validation accuracy' if ck == 'acc' else f'step {int(ck[1:])}'
    for m in NAME:                                   # descriptive: mean per backbone and rung
        for n in BIG:
            g = [c for (mm, nn, s, cc), c in CK.items() if (mm, nn, cc) == (m, n, ck)]
            if g:
                body2.append([what, m, n, len(g), fmt(np.mean([c['step'] for c in g]), 0),
                              fmt(np.mean([c['top1'] for c in g]), 4), fmt(np.mean([c['excess'] for c in g])),
                              fmt(np.mean([c['nll'] for c in g]), 4)])
    for a, b in COMPARISONS:
        for metric in ('excess', 'nll', 'top1'):
            d = {}
            for s in new_seeds:
                if all((m, n, s, ck) in CK for m in (a, b) for n in BIG):
                    d[s] = float(np.mean([CK[(a, n, s, ck)][metric] - CK[(b, n, s, ck)][metric] for n in BIG]))
            if not d:
                continue
            ss = RV.seed_summary(list(d.values()))
            pairs = [CK[(a, n, s, ck)][metric] - CK[(b, n, s, ck)][metric] for s in d for n in BIG]
            body.append([what, a, b, '>= 10,000 clips', metric, 'new seeds', ss['n'], ' '.join(map(str, d)),
                         ' '.join(f'{v:+.4f}' for v in d.values()), fmt(ss['mean']), fmt(ss['sd']),
                         fmt(ss['lo']), fmt(ss['hi']), ss['n_pos'], fmt(ss['p_t']), fmt(ss['p_flip']),
                         fmt(2.0 / 2 ** ss['n']), f"{int(np.sum(np.array(pairs) > 0))}/{len(pairs)}"])
            if metric == 'excess':
                print(f"  {what:<26} {NAME[a]:>8} - {NAME[b]:<7} excess {ss['mean']:+.4f}  "
                      f"per seed {' '.join(f'{v:+.4f}' for v in d.values())}  pairs > 0: {body[-1][-1]}")
write_csv(TABDIR / 'replication_matched_steps.csv', HDR, body)
write_csv(TABDIR / 'replication_checkpoint_means.csv',
          ['checkpoint', 'backbone', 'n_train', 'n_seeds', 'mean_step', 'top1', 'excess', 'nll'], body2)
if not CK:
    print('  no checkpoint files yet (Stage 8c has not produced any)')

# ---------------------------------------------------------------- (4)
print('=' * 70, '\n(4) same-seed re-runs in the replication environment\n' + '=' * 70)
body = []
for (m, n, s), r in sorted(ENVR.items()):
    o = CELL.get((m, n, s))
    if o is None or ORIGIN[(m, n, s)] != 'original':
        continue
    do = STORE.load(Key(m, 'ntu60', LADDER_STREAM, f'ladder_n{n}', s, 'test'))
    dn = STORE.load(Key(f'{m}_env', 'ntu60', LADDER_STREAM, f'ladder_n{n}', s, 'test'))
    agree = float((do['logits'].argmax(-1) == dn['logits'].argmax(-1)).mean())
    ident = bool(np.array_equal(do['logits'], dn['logits']))
    sd = {k: float(np.std([CELL[(m, n, q)][k] for q in orig_seeds if (m, n, q) in CELL], ddof=1))
          for k in ('top1', 'null_excess', 'nll')}
    body.append([m, n, s, do['provenance'].get('saved', ''), dn['provenance'].get('saved', ''),
                 json.dumps(dn['provenance'].get('environment', {})), ident, fmt(agree, 4),
                 o['best_step'], r['best_step'], fmt(o['top1'], 4), fmt(r['top1'], 4),
                 fmt(o['null_excess']), fmt(r['null_excess']), fmt(o['nll'], 4), fmt(r['nll'], 4),
                 fmt(r['top1'] - o['top1'], 4), fmt(r['null_excess'] - o['null_excess']),
                 fmt(r['nll'] - o['nll'], 4), fmt(sd['top1'], 4), fmt(sd['null_excess']), fmt(sd['nll'], 4)])
    print(f"  {NAME[m]:<8} n = {n}, seed {s}: bit-identical logits: {ident}; same prediction on "
          f"{100 * agree:.1f}% of clips; top-1 {o['top1']:.4f} -> {r['top1']:.4f}; excess "
          f"{o['null_excess']:.4f} -> {r['null_excess']:.4f} (s.d. across the original seeds "
          f"{sd['null_excess']:.4f}); best step {o['best_step']} -> {r['best_step']}")
write_csv(TABDIR / 'environment_reruns.csv',
          ['backbone', 'n_train', 'seed', 'original_saved', 'rerun_saved', 'rerun_environment',
           'bit_identical', 'prediction_agreement', 'best_step_original', 'best_step_rerun',
           'top1_original', 'top1_rerun', 'excess_original', 'excess_rerun', 'nll_original', 'nll_rerun',
           'delta_top1', 'delta_excess', 'delta_nll', 'seed_sd_top1', 'seed_sd_excess', 'seed_sd_nll'], body)
if not body:
    print('  no environment re-runs yet')

# ---------------------------------------------------------------- (5) and (6)
print('=' * 70, '\n(5) estimator sensitivity on the ladder; (6) NTU test subjects\n' + '=' * 70)
try:
    PERF = N.test_performers(N.find_performer_file(drive_root=DRIVE))
except FileNotFoundError as e:
    PERF = None
    print('  performer file not found -- the subject analysis of the NTU test set is skipped:', e)
EST, body6, gap_rows = {}, [], []
keys = [k for k in STORE.list_keys() if k.dataset == 'ntu60' and k.subset == 'test'
        and (k.model in NAME or (k.model.endswith('_rep') and k.model[:-4] in NAME))]
for i, k in enumerate(keys):
    d = STORE.load(k)
    conf, corr, _ = cc_of(d['logits'], d['labels'])
    m, n = k.model.replace('_rep', ''), int(k.split.replace('ladder_n', ''))
    EST[(m, n, k.seed)] = RV.cell_tests(conf, corr, S=1000, seed=k.seed)
    if PERF is not None and n in BIG:
        if len(PERF) != len(conf):
            raise RuntimeError(f'{len(PERF)} test performers vs {len(conf)} test clips')
        het = RV.subject_heterogeneity(conf, corr, PERF, S=200, seed=k.seed)
        tau = float(np.sqrt(max(het['tau2'], 0.0)))
        sidx = np.unique(PERF, return_inverse=True)[1]
        cn = RV.coupled_null([dict(conf=conf, correct=corr, clip=np.arange(len(conf)))], len(conf), S=500,
                             seed=k.seed, tau=tau, subj_of_clip=sidx if tau > 0 else None)['aece15']
        g = np.array([(conf[sidx == j] - corr[sidx == j]).mean() for j in range(sidx.max() + 1)])
        body6.append([m, n, k.seed, ORIGIN.get((m, n, k.seed), ''), het['G'], fmt(het['phi'], 2), fmt(het['p']),
                      fmt(tau, 4), fmt(cn['obs'][0]), fmt(EST[(m, n, k.seed)]['aece15']['null_mean']),
                      fmt(cn['null'][0].mean()), fmt(RV.mc_p(cn['null'][0], cn['obs'][0])),
                      fmt(g.mean(), 4), int((g > 0).sum()), len(g)])
    if (i + 1) % 20 == 0:
        print(f'  {i + 1}/{len(keys)} ladder cells ({time.time() - t0:.0f} s)')
body = [[m, n, s, ORIGIN.get((m, n, s), ''), nm, fmt(t[nm]['obs']), fmt(t[nm]['null_mean']), fmt(t[nm]['excess']),
         fmt(t[nm]['p'])] for (m, n, s), t in sorted(EST.items()) for nm in RV.ESTIMATORS]
write_csv(TABDIR / 'estimator_ladder_cells.csv',
          ['backbone', 'n_train', 'seed', 'origin', 'estimator', 'observed', 'floor', 'excess', 'p'], body)
body = []
for nm in RV.ESTIMATORS:
    for a, b in COMPARISONS:
        summarise(f'estimator: {RV.EST_LABEL[nm]}', a, b, '>= 10,000 clips', BIG, nm, nm,
                  lambda m, n, s, k=nm: EST[(m, n, s)][k]['excess'], body)
    r = [x for x in body if x[0].endswith(RV.EST_LABEL[nm]) and x[5] == 'all seeds']
    print(f'  {nm:<8} ' + ';  '.join(f"{NAME[x[1]]} - {NAME[x[2]]}: {float(x[9]):+.4f} ({x[13]}/{x[6]} seeds)" for x in r))
write_csv(TABDIR / 'estimator_ladder_seed_level.csv', HDR, body)
if body6:
    write_csv(TABDIR / 'ntu_subject_dependence.csv',
              ['backbone', 'n_train', 'seed', 'origin', 'n_subjects', 'phi', 'p_overdispersion', 'tau',
               'ece_adaptive', 'floor_iid', 'floor_clustered', 'p_clustered', 'mean_subject_gap',
               'subjects_overconfident', 'subjects'], body6)
    sep = sum(float(r[11]) < 0.05 for r in body6)
    print(f'  NTU large rungs: {sep}/{len(body6)} cells separable under the clustered null (p < 0.05); '
          f'median tau {np.median([float(r[7]) for r in body6]):.4f}')

(RESULTS / 'review_replication.json').write_text(json.dumps({
    'original_seeds': orig_seeds, 'new_seeds': new_seeds, 'big_rungs': BIG,
    'n_checkpoints': len(CK), 'n_env_reruns': len(ENVR),
    'replication_env': (json.loads((RESULTS / 'replication_env.json').read_text())
                        if (RESULTS / 'replication_env.json').exists() else None),
    'complete': bool(new_seeds) and all((m, n, s) in CELL for m in NAME for n in BIG for s in new_seeds)},
    indent=1))
print(f'  wrote review_replication.json  ({time.time() - t0:.0f} s)')

## Stage 10 — figures and every number of the manuscript (CPU)
Writes `results/paper/` (generated/ + fig1–fig6); copy it next to the .tex in Overleaf.

In [ ]:
# =====================================================================
# STAGE 10 -- figures and every number of the manuscript     [CPU, ~3 min]
# =====================================================================
# Two figures pool the raw predictions; this stage writes the tables behind
# them, so that every figure can be redrawn from the result tables alone:
#   tables/reliability_bins.csv   Fig. 1: equal-MASS bins (the binning of the
#                                 adaptive ECE), ST-GCN joint stream pooled
#                                 over splits and seeds: bin range, mean
#                                 confidence, accuracy, count
#   tables/per_class_pooled.csv   Fig. 5: per-class ECE (one-vs-rest over the
#                                 clips predicted as each class, equal-mass
#                                 bins) and 1 - class recall, same pooling
# Then make_numbers.py writes RESULTS/paper/generated (numbers.tex, tab_*.tex)
# and make_figures.py draws all six figures into RESULTS/figures and
# RESULTS/paper. Copy RESULTS/paper/* next to the .tex in Overleaf.
import csv, shutil, subprocess
import har3d_v2 as V
importlib.reload(V)
from store import LogitStore

STORE = LogitStore(LOGITS)
FIGDIR = RESULTS / 'figures'; FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = RESULTS / 'tables'
PAPER = RESULTS / 'paper'; (PAPER / 'generated').mkdir(parents=True, exist_ok=True)
SMALL = ('action3d', 'daily', 'utkinect')


def pooled(ds, model='stgcn', stream='joint'):
    P, Y = [], []
    for k in STORE.list_keys():
        if (k.model, k.dataset, k.stream, k.subset) == (model, ds, stream, 'test'):
            d = STORE.load(k); P.append(V.softmax(d['logits'])); Y.append(d['labels'])
    return np.concatenate(P), np.concatenate(Y)


rb, pcp = [], []
for ds in SMALL:
    p, y = pooled(ds)
    conf, corr = p.max(-1), (p.argmax(-1) == y).astype(float)
    edges = np.unique(np.quantile(conf, np.linspace(0, 1, 16)))
    idx = np.clip(np.searchsorted(edges, conf, side='right') - 1, 0, len(edges) - 2)
    for b in range(len(edges) - 1):
        m = idx == b
        if m.any():
            rb.append([ds, b, f'{edges[b]:.6f}', f'{edges[b + 1]:.6f}', f'{conf[m].mean():.6f}',
                       f'{corr[m].mean():.6f}', int(m.sum())])
    pc = V.per_class_ece(p, y)
    rec = np.array([(p.argmax(-1)[y == c] == c).mean() if (y == c).any() else np.nan for c in range(p.shape[1])])
    for rank, c in enumerate(np.argsort(np.nan_to_num(pc, nan=-1))[::-1]):
        if np.isfinite(pc[c]):
            pcp.append([ds, rank, int(c), f'{pc[c]:.6f}', f'{1 - rec[c]:.6f}'])
    print(f'{ds:<9} {len(y)} pooled predictions; per-class ECE {np.nanmin(pc):.3f}-{np.nanmax(pc):.3f}')
for name, hdr, body in (('reliability_bins.csv', ['dataset', 'bin', 'conf_lo', 'conf_hi', 'mean_conf', 'accuracy', 'count'], rb),
                        ('per_class_pooled.csv', ['dataset', 'rank', 'class', 'per_class_ece', 'one_minus_recall'], pcp)):
    with open(TABDIR / name, 'w', newline='') as fh:
        w = csv.writer(fh); w.writerow(hdr); w.writerows(body)
    print('  wrote', name)

for cmd in ([sys.executable, '/content/make_numbers.py', str(RESULTS), str(PAPER / 'generated')],
            [sys.executable, '/content/make_figures.py', str(RESULTS), str(PAPER / 'generated'), str(FIGDIR)]):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        raise RuntimeError(f'{Path(cmd[1]).name} failed -- see the message above')
for f in FIGDIR.glob('fig*.pdf'):
    shutil.copy2(f, PAPER / f.name)
print('manuscript files in', PAPER, ':', sorted(p.name for p in PAPER.glob('*.pdf')),
      '+ generated/', sorted(p.name for p in (PAPER / 'generated').iterdir()))
mark_done('10_figures')

## Stage 11 — status

In [ ]:
# =====================================================================
# STAGE 11 -- run status and what to download
# =====================================================================
done = sorted(p.name[6:] for p in RESULTS.glob('.done_*'))
print('run tag       :', RUN_TAG)
print('stages done   :', done)
print('logit files   :', len(list(LOGITS.glob('*.npz'))))
for mdl in ('stgcn', 'agcn', 'ctrgcn', 'agcn_frozen', 'stgcn_fixed'):
    n = len(list(LOGITS.glob(f'ntu60__{mdl}__*__test.npz')))
    if n:
        print(f'  NTU ladder {mdl:<12}: {n} test cells')
print('\nfor the manuscript: copy the contents of this folder next to the .tex in Overleaf')
print('  ', RESULTS / 'paper', '(generated/ numbers and tables, fig1-fig6 PDFs)')
print('\nfor the code release, download these as well:')
for p in ('tables', 'figures', 'environment'):
    print('  ', RESULTS / p)
for f in ('metrics.json', 'prep_config.json', 'data_filtering.json', 'ladder_protocol.json',
          'utkinect_perm_check.json'):
    print('  ', RESULTS / f, '' if (RESULTS / f).exists() else '(missing)')

# ---------------------------------------------------------------- v7 (revision)
print('\n' + '=' * 70 + '\nREVISION (v7) -- what the reviewers asked for\n' + '=' * 70)
_n_pool = json.loads((RESULTS / 'ladder_protocol.json').read_text())['n_pool'] \
    if (RESULTS / 'ladder_protocol.json').exists() else None
_want = REPL_SEEDS + (REPL_EXTRA_SEEDS if RUN_REPL_EXTRA else [])
_ok = True
for mdl in LADDER_BACKBONES:
    have = {(int(p.name.split('__')[3].replace('ladder_n', '')), int(p.name.split('__')[4][1:]))
            for p in LOGITS.glob(f'ntu60__{mdl}_rep__*__test.npz')}
    need = {(min(n, _n_pool or n), s) for n in REPL_RUNGS for s in _want}
    _ok &= need <= have
    print(f'  replication {mdl:<7}: {len(have & need)}/{len(need)} runs stored'
          + ('' if need <= have else f'   missing (n, seed): {sorted(need - have)}'))
if RUN_ENV_RERUNS:
    n_env = len(list(LOGITS.glob('ntu60__*_env__*__test.npz')))
    _ok &= n_env >= len(LADDER_BACKBONES)
    print(f'  environment re-runs : {n_env}/{len(LADDER_BACKBONES)} stored')
_stages = ['09d_dependence', '09e_power', '09f_methods_subjects', '09g_prep_checks']
if RUN_PREP_SENSITIVITY:
    _stages.insert(0, '05b_prep_sensitivity')
for t in _stages:
    _ok &= stage_done(t)
    print(f'  stage {t:<22}: {"done" if stage_done(t) else "NOT DONE"}')
_rep = RESULTS / 'review_replication.json'
_rep_ok = _rep.exists() and json.loads(_rep.read_text()).get('complete')
_ok &= bool(_rep_ok)
print(f'  stage 09h (replication analysis): {"complete" if _rep_ok else "incomplete -- run Stage 9 and Stage 9h after the training has finished"}')
_new = ['estimator_cells.csv', 'estimator_sensitivity.csv', 'subject_heterogeneity.csv',
        'pooled_dependence.csv', 'separability_clustered.csv', 'subject_gap.csv', 'power_cells.csv',
        'power_curves.csv', 'power_summary.csv', 'power_pooled.csv', 'power_ntu.csv',
        'power_synthetic.csv', 'size_clustered.csv', 'calibration_fits.csv', 'methods_subject_level.csv',
        'methods_pooled_null.csv', 'per_class_null.csv', 'prep_sensitivity.csv', 'train_only_checks.csv',
        'replication_cells.csv', 'replication_seed_level.csv', 'replication_checkpoints.csv',
        'replication_matched_steps.csv', 'replication_checkpoint_means.csv', 'environment_reruns.csv',
        'estimator_ladder_cells.csv', 'estimator_ladder_seed_level.csv', 'ntu_subject_dependence.csv']
_miss = [f for f in _new if not (RESULTS / 'tables' / f).exists()]
print(f'  new result tables   : {len(_new) - len(_miss)}/{len(_new)}' + (f'   missing: {_miss}' if _miss else ''))
print()
if _ok and not _miss:
    print('  >>> EVERYTHING FOR THE REVISION IS READY. <<<')
    print('  Download the whole folder', RESULTS)
    print('  from Google Drive (right-click > Download; Drive zips it) and upload that zip to the chat.')
else:
    print('  Not finished yet. If Colab disconnected during Stage 8c: reconnect to the same GPU type,')
    print('  then Runtime > Run all. Finished runs are skipped; it continues with the first missing one.')